# Metabolic flux phenotyping

**An auditable representation of sampled metabolic states**: the whole Python
analysis of the project, in one notebook. Running it from top to bottom
reproduces every table, every figure, every supplementary table and every number
the manuscript quotes.

## What it does

A sampled flux vector is a complete metabolic state, but a list of thousands of
reaction rates is not a description of it. The notebook builds a map

$$v \longrightarrow \Phi(v)$$

from a flux vector to a vector of declared, interpretable descriptors (flux
organisation, pathway activity, metabolite turnover, cofactor allocation,
capacity use, exchange, and the coupling between inside and boundary), each
declared before it is computed, with its units, its requirements and what it
returns when a requirement is absent (a missing value, never a zero).

It then asks four questions of that representation:

1. **Does it keep what matters?** The same states, as raw fluxes and as
   $\Phi(v)$, under the same chain-held-out folds and the same models.
2. **How much of it is needed?** Panels selected inside every training fold,
   with the stability of the choice reported beside its accuracy.
3. **Does it respond to what was imposed, and only to that?** A graded
   restriction of G6PDH2 beside an inert peroxide axis, analysed at the level
   of the chain.
4. **Does it mean the same thing in another network?** The original RBC-GEM
   against its thermodynamically consistent subset, at four levels: support,
   response, panel and prediction.

## The repository and the project folder

The code lives in the `flux2features` repository: this notebook (`notebook/`),
`matlab/generateModelsAndSamples_flux2features.m` (models and sampling) and the
standalone `flux2feature.m` with its test. The data and every result live in a
separate project folder, found from the environment variable
`FLUX2FEATURES_PROJECT` or from a `project_dir.txt` next to the notebook that
holds its path:

| folder | what is in it |
|---|---|
| `data/` | `RBC-GEM.mat` and `metadata/` (registry, manifests, model comparison) |
| `results/raw/` | what the MATLAB script writes: `models/`, `fluxes/`, `samples/` |
| `results/processed/` | the chains, thinned to float32 (stage 0), and the list of excluded chains |
| `results/tables/` | every table, per model (`original/`, `thermo/`) and across them (`transfer/`) |
| `results/figures/` | every figure; `paper/` holds the seven of the manuscript |
| `results/supplementary/` | the supplementary tables and their workbook |
| `reports/paper/` | the manuscript source, its build script, `numbers.json` and the .docx |

## How it is organised

**The machinery.** Reading MATLAB files, the model, the samples, what a
descriptor is, the descriptor families, sampling diagnostics, the variance
stabilisation, the quality-control cascade, the distance to the healthy state,
the visual system. Definitions only; nothing runs.

**The stages.** One function each, named `run_NN_...`, in the order they depend
on each other: the integrity check of every chain, preparation, sampling
diagnostics, the descriptors and their quality control, the analyses of one
model, and the analyses that compare the two.

**Running everything.** The last cell calls `run_all()`: the export of the
MATLAB archives, every per-model stage for both reconstructions, and the
stages that compare them. Set `RUN_EVERYTHING = False` there to load the
definitions without running anything, then call any stage on its own after
`use_variant("original")` or `use_variant("thermo")`.

## Before reading any number

Every perturbed chain is checked against the bound it is named after
(`run_00b_constraint_integrity`), and the ones that fail are excluded with
their reason in `results/processed/excluded_chains.json`. The sampling
diagnostics (`run_02_sampling_qc`) state which mixing criteria each model meets.
Both are reported in the manuscript; neither is optional.

## 1. Imports and paths

Everything the notebook needs, imported once. The path block resolves the project
root whether the notebook is run from `code/` or from the project root, so
`data/` and `resultados/` are always found.

In [ ]:
import gzip
import io
import itertools
import json
import re
import struct
import zlib
from dataclasses import dataclass, field
from pathlib import Path

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.gridspec import GridSpec
from matplotlib.patches import FancyArrowPatch, FancyBboxPatch, Rectangle
from scipy import stats
from scipy.cluster.hierarchy import fcluster, linkage
from scipy.optimize import linprog
from scipy.spatial.distance import pdist, squareform
from scipy.stats import kendalltau, spearmanr
from sklearn.cluster import AgglomerativeClustering, KMeans
from sklearn.covariance import LedoitWolf
from sklearn.cross_decomposition import CCA
from sklearn.decomposition import PCA
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.feature_selection import mutual_info_regression
from sklearn.linear_model import ElasticNet, Ridge
from sklearn.metrics import adjusted_rand_score, silhouette_score
from sklearn.model_selection import GroupKFold
from sklearn.tree import DecisionTreeRegressor, export_text

# -- where things live -----------------------------------------------------
# The code (this notebook and the MATLAB scripts) lives in the flux2features
# repository; the data and every result live in a separate project folder,
# laid out the same on every machine:
#
#   data/                       RBC-GEM.mat and the curated metadata
#   results/raw/                what generateModelsAndSamples_flux2features.m writes:
#       models/ fluxes/ samples/
#   results/processed/samples/  the same chains, thinned, as float32 (stage 00)
#   results/tables/<variant>/   every table, per reconstruction
#   results/figures/            every figure
#   results/reports/            the written diagnostics
#   results/supplementary/      the supplementary tables
#   reports/paper/              the manuscript, its figures and its build script
#
# The project folder is found, in this order: the environment variable
# FLUX2FEATURES_PROJECT; a file project_dir.txt holding its path, next to the
# notebook or in any folder above it (the repository ignores that file); or,
# for a notebook kept inside the project folder, the folder above code/.
import os


def _project_root() -> Path:
    if os.environ.get("FLUX2FEATURES_PROJECT"):
        return Path(os.environ["FLUX2FEATURES_PROJECT"]).expanduser()
    here = Path.cwd()
    for folder in (here, *here.parents):
        pointer = folder / "project_dir.txt"
        if pointer.exists():
            return Path(pointer.read_text().strip()).expanduser()
    root = here
    while root.name in {"python", "code", "notebook", "notebooks"}:
        root = root.parent
    return root


ROOT = _project_root()
assert (ROOT / "results").exists(), (
    f"no results/ folder under {ROOT}: set FLUX2FEATURES_PROJECT to the project folder "
    f"(the one holding data/ and results/), or write its path in project_dir.txt next to "
    f"the notebook")

INPUT_DIR = ROOT / "data"                                 # the GEM as given
METADATA_DIR = ROOT / "data" / "metadata"                 # registry and manifests
RAW_DIR_MATLAB = ROOT / "results" / "raw"
MODEL_DIR = RAW_DIR_MATLAB / "models"                     # constrained models
FLUX_DIR = RAW_DIR_MATLAB / "fluxes"                      # FBA solutions
RAW_SAMPLE_DIR = RAW_DIR_MATLAB / "samples"               # the MATLAB archives
SAMPLE_DIR = ROOT / "results" / "processed" / "samples"   # what Python reads
TABLE_ROOT = ROOT / "results" / "tables"
FIGURE_ROOT = ROOT / "results" / "figures"
REPORT_DIR = ROOT / "results" / "reports"
SUPPLEMENTARY_DIR = ROOT / "results" / "supplementary"
PAPER_DIR = ROOT / "reports" / "paper"
PAPER_FIGURE_DIR = PAPER_DIR / "figures"


def ensure_dirs() -> None:
    for d in (METADATA_DIR, SAMPLE_DIR, TABLE_ROOT, FIGURE_ROOT, REPORT_DIR,
              SUPPLEMENTARY_DIR, PAPER_FIGURE_DIR):
        Path(d).mkdir(parents=True, exist_ok=True)


ensure_dirs()
print(f"project root: {ROOT}")

## 2. Analysis choices

Every decision that is a *choice* rather than a *result* is collected here: the perturbation levels, the pathway groups, the numerical tolerances and the random seed. Nothing below invents a constant of its own, so the provenance of any number in the paper ends in this cell.

> Central configuration for the metabolic flux phenotyping framework.

In [ ]:
from dataclasses import dataclass, field
from pathlib import Path



# --------------------------------------------------------------------------
# Experimental design
# --------------------------------------------------------------------------

#: Fractional upper bound applied to G6PDH2 (glucose-6-phosphate dehydrogenase).
G6PD_LEVELS = (0.10, 0.28, 0.50)

#: Fractional upper bound applied to EX_h2o2_e (hydrogen peroxide exchange).
H2O2_LEVELS = (0.50, 0.75, 1.00)

#: WHO-inspired ordinal severity labels for the G6PD axis.
G6PD_SEVERITY = {0.10: "severe", 0.28: "moderate", 0.50: "mild"}

#: Ordinal oxidative-load labels for the H2O2 axis. A *larger* permitted
#: exchange bound means the cell can move more peroxide across its boundary.
H2O2_LOAD = {0.50: "low", 0.75: "intermediate", 1.00: "high"}

N_CHAINS = 3
N_SAMPLES_PER_CHAIN_RAW = 10_000
THINNING = 10  # stride for a 10,000-point chain; see KEEP_PER_CHAIN

#: States kept per chain, by reconstruction. Every chain keeps the same
#: number, so no condition outweighs another; the stride is derived from each
#: chain's own length and written to its JSON, so a chain sampled with a
#: different number of points is thinned to the same count rather than
#: silently weighted differently.
KEEP_PER_CHAIN = {"original": 1000, "thermo": 1000}

#: Reactions per reconstruction; used to orient the sampling matrices.
N_REACTIONS = {"original": 3275, "thermo": 2580}


# --------------------------------------------------------------------------
# Key reactions and metabolites of the RBC-GEM
# --------------------------------------------------------------------------

G6PD_REACTION = "G6PDH2"
H2O2_EXCHANGE = "EX_h2o2_e"

#: Metabolites whose turnover defines the biochemical phenotype.
TURNOVER_METABOLITES = (
    "atp_c", "adp_c", "amp_c", "pi_c",
    "nadh_c", "nad_c", "nadph_c", "nadp_c",
    "gthrd_c", "gthox_c",
    "glc__D_c", "g6p_c", "f6p_c", "pyr_c", "lac__L_c",
    "h2o2_c", "o2_c", "o2s_c",
    "13dpg_c", "23dpg_c", "r5p_c", "ru5p__D_c", "6pgc_c",
)

#: Currency metabolites excluded from flux-weighted network statistics, where
#: their promiscuous participation would otherwise dominate every centrality.
CURRENCY_METABOLITES = (
    "h_c", "h_e", "h2o_c", "h2o_e", "pi_c", "pi_e", "ppi_c",
    "co2_c", "co2_e", "na1_c", "na1_e", "k_c", "k_e", "cl_c", "cl_e",
)

#: Curated pathway groupings. Keys are the labels used throughout the paper;
#: values are the RBC-GEM ``subSystems`` strings that map onto them. The
#: strings are verbatim from the model, so an unmatched entry is a bug rather
#: than a silent omission (:func:`mfp.annotation.reactions.check_pathway_map`).
PATHWAY_GROUPS: dict[str, tuple[str, ...]] = {
    "glycolysis": ("Glycolysis / Gluconeogenesis",),
    "pentose_phosphate": ("Pentose phosphate pathway",),
    "glutathione": ("Glutathione metabolism",),
    "redox_detox": ("Reactive species formation and detoxification",),
    "purine": ("Purine metabolism",),
    "pyrimidine": ("Pyrimidine metabolism",),
    "nucleotide": (
        "Nucleotide metabolism",
        "Amino sugar and nucleotide sugar metabolism",
    ),
    "pyruvate_tca": (
        "Pyruvate metabolism",
        "Tricarboxylic acid cycle and glyoxylate/dicarboxylate metabolism",
    ),
    "amino_acid": (
        "Alanine, aspartate and glutamate metabolism",
        "Glycine, serine and threonine metabolism",
        "Cysteine and methionine metabolism",
        "Arginine and proline metabolism",
        "Phenylalanine, tyrosine and tryptophan metabolism",
        "Valine, leucine, and isoleucine metabolism",
        "Histidine metabolism",
        "Lysine metabolism",
        "Selenocompound metabolism",
        "Sulfur metabolism",
        "Aminoacyl-tRNA biosynthesis",
        "Peptide metabolism",
    ),
    "lipid": (
        "Fatty acid oxidation",
        "Fatty acid activation (cytosolic)",
        "Fatty acid biosynthesis",
        "Fatty acid biosynthesis (even-chain)",
        "Fatty acid biosynthesis (unsaturated)",
        "Fatty acid elongation (even-chain)",
        "Fatty acid elongation (odd-chain)",
        "Fatty acid desaturation (even-chain)",
        "Acyl-CoA hydrolysis",
        "Carnitine shuttle (cytosolic)",
        "Glycerophospholipid metabolism",
        "Glycerolipid metabolism",
        "Sphingolipid metabolism",
        "Ether lipid metabolism",
        "Cholesterol metabolism",
        "Inositol phosphate metabolism",
        "Docosahexaenoic acid metabolism",
        "Eicosapentaenoic acid metabolism",
        "Arachidonic acid metabolism",
        "Linoleate metabolism",
        "Omega-3 fatty acid metabolism",
        "Omega-6 fatty acid metabolism",
        "Prostaglandin biosynthesis",
        "Leukotriene metabolism",
    ),
    "glycan": (
        "Glycosphingolipid biosynthesis - ganglio series",
        "Glycosphingolipid biosynthesis - lacto and neolacto series",
        "Glycosphingolipid biosynthesis - globo and isoglobo series",
        "N-glycan metabolism",
        "Starch and sucrose metabolism",
        "Galactose metabolism",
        "Fructose and mannose metabolism",
        "Pentose and glucuronate interconversions",
    ),
    "cofactor_vitamin": (
        "Nicotinate and nicotinamide metabolism",
        "Folate metabolism",
        "Riboflavin metabolism",
        "Thiamine metabolism",
        "Vitamin B6 metabolism",
        "Vitamin E metabolism",
        "Retinol metabolism",
        "Biopterin metabolism",
        "Pantothenate and CoA metabolism",
        "Ascorbate and aldarate metabolism",
    ),
    "porphyrin_hemoglobin": (
        "Porphyrin metabolism",
        "Hemoglobin binding and degradation",
    ),
    "xenobiotic_hormone": (
        "Estrogen metabolism",
        "Androgen metabolism",
        "Steroid metabolism",
        "Azathioprine and 6-mercaptopurine metabolism",
        "Ribavirin metabolism",
        "5-fluorouracil metabolism",
        "Renin-angiotensin system",
    ),
    "protein_turnover": (
        "Protein modification",
        "Ubiquitin mediated proteolysis",
    ),
    "transport": ("Transport, extracellular",),
    "pseudoreactions": ("Pseudoreactions", "Miscellaneous"),
}

#: The Rapoport-Luebering shunt is not annotated as a subsystem in RBC-GEM but
#: is central to erythrocyte physiology, so it is defined by reaction identity.
RAPOPORT_LUEBERING_RXNS = ("DPGM", "DPGase")

#: Reactions that define the oxidative branch of the pentose phosphate pathway.
OXIDATIVE_PPP_RXNS = ("G6PDH2", "PGL", "GND")


# --------------------------------------------------------------------------
# Numerical tolerances
# --------------------------------------------------------------------------

@dataclass(frozen=True)
class Tolerances:
    #: below this magnitude a flux is treated as numerically zero
    flux_zero: float = 1e-9
    #: below this range a reaction is treated as invariant across samples
    invariant_range: float = 1e-8
    #: relative tolerance used when checking mass balance of a sample
    mass_balance: float = 1e-6
    #: features with |Spearman rho| above this are considered redundant
    redundancy_rho: float = 0.95
    #: features with coefficient of variation below this are considered constant
    min_cv: float = 1e-6


TOL = Tolerances()


# --------------------------------------------------------------------------
# Analysis choices
# --------------------------------------------------------------------------

RANDOM_SEED = 20260812

#: Quantile cut-offs used to define "healthy-like" sampled states.
HEALTHY_LIKE_QUANTILES = (0.01, 0.05, 0.10)

#: Sample sizes probed in the robustness analysis.
ROBUSTNESS_SAMPLE_SIZES = (250, 500, 1000, 2000, 3000)


@dataclass
class Config:
    """Runtime configuration, overridable per analysis."""

    seed: int = RANDOM_SEED
    n_jobs: int = 1
    thinning: int = THINNING
    burn_in_fraction: float = 0.0
    tolerances: Tolerances = field(default_factory=lambda: TOL)

    def rng(self):
        import numpy as np

        return np.random.default_rng(self.seed)


CONFIG = Config()

The stages below refer to these settings as `config.X`. That name is bound here so the code reads the same as it does in a script.

In [ ]:
import types

config = types.SimpleNamespace(
    **{k: v for k, v in globals().items()
       if k.isupper() or k == "ensure_dirs"})
print(f"{len(config.__dict__)} settings bound")

## 3. The two models

Everything below runs twice, once per reconstruction. `use_variant` points
the inputs and the outputs at one of them; the stage functions never
mention a variant, so neither model can be analysed with the other's
settings by accident.

The sampler is named here too. `generateModelsAndSamples.m` calls
`sampleCbModel(..., 'CHRR', ...)` — coordinate hit-and-run with rounding —
so that, and not RHMC, is what produced these archives.

In [ ]:
"""The two model variants, and the paths that follow whichever one is active.

Everything downstream reads ``config.RAW_DIR``, ``config.RESULTS_DIR`` and
``config.FIGURE_DIR``. Those are *lazy* paths: they resolve against the active
variant at the moment they are used, so the same stage functions produce the
original-model tables or the thermodynamic-model tables without being edited.
"""
import os
from pathlib import Path


#: The two reconstructions compared in this study.
#:
#: ``original``  RBC-GEM as published, the structural and functional reference.
#: ``thermo``    the stoichiometrically and thermodynamically consistent subset
#:               produced by findStoichConsistentSubset + findThermoConsistent-
#:               FluxSubset. It is a *different* model, not a better one: it has
#:               fewer reactions, and the structural report in section 3 says
#:               exactly which.
MODEL_VARIANTS = {
    "original": {
        "label": "RBC-GEM (original)",
        "model_file": INPUT_DIR / "RBC-GEM.mat",
        "model_name": "RBC-GEM",
        "fba_file": FLUX_DIR / "v_or.mat",
        "role": "structural and functional reference",
    },
    "thermo": {
        "label": "RBC-GEM (thermodynamically consistent subset)",
        "model_file": MODEL_DIR / "thermoConsistModel.mat",
        "model_name": "thermoConsistModel",
        "fba_file": FLUX_DIR / "v_thermoOr.mat",
        "role": "robustness test of the representation",
    },
}

#: The sampler that produced the archives. Named here because the manuscript
#: must not describe a sampler the code did not run: generateModelsAndSamples.m
#: calls sampleCbModel(..., 'CHRR', ...), coordinate hit-and-run with rounding.
SAMPLER = "CHRR"

#: Every stage writes into one of these, so they are created with the variant
#: rather than by each stage separately.
TABLE_SUBDIRECTORIES = (
    "sampling", "phenotype", "features", "distance", "landscape", "ml",
    "interventions", "robustness", "benchmark",
)

ACTIVE_VARIANT = "original"


class LazyPath(os.PathLike):
    """A path that resolves when it is used, not when it is built.

    ``OUT = config.RESULTS_DIR / "features"`` at the top of a stage cell would
    otherwise freeze the variant that was active when the cell was executed.
    """

    __slots__ = ("_resolver", "_parts")

    def __init__(self, resolver, *parts):
        self._resolver = resolver
        self._parts = parts

    def _resolve(self) -> Path:
        base = Path(self._resolver())
        for part in self._parts:
            base = base / part
        return base

    def __truediv__(self, other) -> "LazyPath":
        return LazyPath(self._resolver, *self._parts, other)

    def __fspath__(self) -> str:
        return str(self._resolve())

    def __str__(self) -> str:
        return str(self._resolve())

    def __repr__(self) -> str:
        return f"LazyPath({self._resolve()})"

    def __eq__(self, other) -> bool:
        return self._resolve() == (other._resolve() if isinstance(other, LazyPath) else other)

    def __hash__(self) -> int:
        return hash(self._resolve())

    def __getattr__(self, name):
        return getattr(self._resolve(), name)


def variant_tables() -> Path:
    return TABLE_ROOT / ACTIVE_VARIANT


def variant_figures() -> Path:
    return FIGURE_ROOT / ACTIVE_VARIANT


def use_variant(name: str) -> None:
    """Point every output and every input at one of the two models."""
    global ACTIVE_VARIANT
    if name not in MODEL_VARIANTS:
        raise KeyError(f"unknown model variant {name!r}; choose from {list(MODEL_VARIANTS)}")
    ACTIVE_VARIANT = name
    spec = MODEL_VARIANTS[name]
    config.ACTIVE_VARIANT = name
    config.MODEL_FILE = spec["model_file"]
    config.MODEL_NAME = spec["model_name"]
    config.FBA_FILE = spec["fba_file"]
    config.MODEL_LABEL = spec["label"]
    for d in (variant_tables(), variant_figures()):
        d.mkdir(parents=True, exist_ok=True)
    for sub in TABLE_SUBDIRECTORIES:
        (variant_tables() / sub).mkdir(parents=True, exist_ok=True)
    print(f"active variant: {name}  ({spec['label']})")


# Bound onto the configuration namespace so the stages see them.
config.RAW_DIR = SAMPLE_DIR
config.SAMPLE_DIR = SAMPLE_DIR
config.MODEL_DIR = MODEL_DIR
config.FLUX_DIR = FLUX_DIR
config.INPUT_DIR = INPUT_DIR
config.METADATA_DIR = METADATA_DIR
config.REPORT_DIR = REPORT_DIR
config.RESULTS_DIR = LazyPath(variant_tables)
config.PROCESSED_DIR = LazyPath(variant_tables)
config.FIGURE_DIR = LazyPath(variant_figures)
config.MODEL_VARIANTS = MODEL_VARIANTS
config.SAMPLER = SAMPLER
config.ensure_dirs = ensure_dirs

use_variant("original")

## 4. Reading MATLAB files without SciPy's reader

The sampling was done in MATLAB, so the model arrives as a v5 `.mat` file. This is a small dependency-free reader for that format — it exists because the machine where the samples were extracted had no SciPy, and it is kept because it makes the notebook independent of one more library.

> A dependency-free reader for MATLAB level-5 ``.mat`` files.

In [ ]:
import struct
import zlib
from typing import Any

__all__ = ["Elem", "load", "to_list"]

# MATLAB data types -> struct format characters
_MI_TYPES = {
    1: "i1", 2: "u1", 3: "i2", 4: "u2", 5: "i4", 6: "u4",
    7: "f4", 9: "f8", 12: "i8", 13: "u8", 16: "u1", 17: "u2", 18: "u4",
}
_FMT = {"i1": "b", "u1": "B", "i2": "h", "u2": "H", "i4": "i",
        "u4": "I", "f4": "f", "f8": "d", "i8": "q", "u8": "Q"}
_SIZE = {"i1": 1, "u1": 1, "i2": 2, "u2": 2, "i4": 4,
         "u4": 4, "f4": 4, "f8": 8, "i8": 8, "u8": 8}

# MATLAB array classes
CLASS_CELL = 1
CLASS_STRUCT = 2
CLASS_OBJECT = 3
CLASS_CHAR = 4
CLASS_SPARSE = 5


class Elem:
    """A parsed MATLAB array."""

    __slots__ = ("cls", "dims", "name", "data", "fields")

    def __init__(self, cls: int, dims: tuple, name: str,
                 data: Any = None, fields: dict | None = None):
        self.cls = cls
        self.dims = dims
        self.name = name
        self.data = data
        self.fields = fields

    def __repr__(self) -> str:  # pragma: no cover - debugging aid
        fields = list(self.fields) if self.fields else None
        return f"<Elem class={self.cls} dims={self.dims} name={self.name!r} fields={fields}>"


def _read_tag(buf: bytes, off: int):
    """Return ``(dtype, nbytes, data_start, next_off)`` for the tag at ``off``."""
    (word,) = struct.unpack_from("<I", buf, off)
    if (word >> 16) != 0:  # small data element format
        return word & 0xFFFF, word >> 16, off + 4, off + 8
    dtype, nbytes = struct.unpack_from("<II", buf, off)
    padded = nbytes + ((8 - nbytes % 8) % 8)
    return dtype, nbytes, off + 8, off + 8 + padded


def _read_raw(buf: bytes, off: int):
    dtype, nbytes, start, nxt = _read_tag(buf, off)
    return dtype, buf[start:start + nbytes], nxt


def _decode_numbers(dtype: int, payload: bytes) -> tuple:
    kind = _MI_TYPES[dtype]
    n = len(payload) // _SIZE[kind]
    return struct.unpack(f"<{n}{_FMT[kind]}", payload[: n * _SIZE[kind]])


def _parse_matrix(buf: bytes, off: int):
    dtype, nbytes, start, nxt = _read_tag(buf, off)
    if dtype != 14:
        raise ValueError(f"expected miMATRIX (14), found {dtype}")
    end = start + nbytes
    pos = start

    _, flags_raw, pos = _read_raw(buf, pos)
    flags = struct.unpack_from("<II", flags_raw, 0)
    cls = flags[0] & 0xFF

    dt, dims_raw, pos = _read_raw(buf, pos)
    dims = _decode_numbers(dt, dims_raw)

    _, name_raw, pos = _read_raw(buf, pos)
    name = name_raw.decode("latin1")

    if cls in (CLASS_STRUCT, CLASS_OBJECT):
        if cls == CLASS_OBJECT:
            _, _class_name, pos = _read_raw(buf, pos)
        dt, len_raw, pos = _read_raw(buf, pos)
        name_len = _decode_numbers(dt, len_raw)[0]
        _, names_raw, pos = _read_raw(buf, pos)
        n_fields = len(names_raw) // name_len
        field_names = [
            names_raw[i * name_len:(i + 1) * name_len].split(b"\x00")[0].decode("latin1")
            for i in range(n_fields)
        ]
        n_elems = 1
        for d in dims:
            n_elems *= d
        fields: dict[str, Any] = {}
        for element in range(n_elems):
            for fname in field_names:
                sub, pos = _parse_matrix(buf, pos)
                if element == 0:
                    fields[fname] = sub
        return Elem(cls, dims, name, None, fields), nxt

    if cls == CLASS_CELL:
        n_elems = 1
        for d in dims:
            n_elems *= d
        items = []
        for _ in range(n_elems):
            sub, pos = _parse_matrix(buf, pos)
            items.append(sub)
        return Elem(cls, dims, name, items), nxt

    if cls == CLASS_SPARSE:
        dt, ir_raw, pos = _read_raw(buf, pos)
        ir = _decode_numbers(dt, ir_raw)
        dt, jc_raw, pos = _read_raw(buf, pos)
        jc = _decode_numbers(dt, jc_raw)
        dt, pr_raw, pos = _read_raw(buf, pos)
        pr = _decode_numbers(dt, pr_raw)
        return Elem(cls, dims, name, ("sparse", ir, jc, pr)), nxt

    if cls == CLASS_CHAR:
        dt, chars, pos = _read_raw(buf, pos)
        encoding = "utf-16-le" if dt in (4, 17) else "latin1"
        return Elem(cls, dims, name, chars.decode(encoding, "replace")), nxt

    if pos < end:
        dt, values, pos = _read_raw(buf, pos)
        return Elem(cls, dims, name, (dt, values)), nxt
    return Elem(cls, dims, name, None), nxt


def load(path: str) -> dict[str, Elem]:
    """Parse a level-5 ``.mat`` file into a mapping of variable name to element."""
    with open(path, "rb") as handle:
        raw = handle.read()
    if raw[:6] != b"MATLAB":
        raise ValueError(f"{path} is not a MATLAB level-5 file")
    out: dict[str, Elem] = {}
    off, n = 128, len(raw)
    while off < n - 8:
        dtype, nbytes, start, nxt = _read_tag(raw, off)
        if dtype == 15:  # miCOMPRESSED
            payload = zlib.decompress(raw[start:start + nbytes])
            elem, _ = _parse_matrix(payload, 0)
            out[elem.name] = elem
        elif dtype == 14:
            elem, _ = _parse_matrix(raw, off)
            out[elem.name] = elem
        off = nxt
    return out


def to_list(elem: Elem) -> list:
    """Return the numeric payload of a dense numeric ``Elem`` as a flat list."""
    if elem.data is None or elem.cls in (CLASS_STRUCT, CLASS_CELL, CLASS_CHAR):
        raise TypeError(f"element {elem.name!r} is not a dense numeric array")
    dtype, payload = elem.data
    return list(_decode_numbers(dtype, payload))

## 5. The model

Loading RBC-GEM into a plain dataclass: stoichiometric matrix, bounds, reaction and metabolite identifiers, subsystems and gene rules. 2,157 metabolites and 3,275 reactions.

> Loading and representing the RBC-GEM constraint-based model.

In [ ]:
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import scipy.io as sio
import scipy.sparse as sp

__all__ = ["GEM", "load_gem", "load_flux_vector", "load_fba_solution"]


def _as_str_array(value) -> np.ndarray:
    """Normalise the many shapes MATLAB cell arrays take after ``loadmat``."""
    out = []
    for item in np.atleast_1d(value):
        if isinstance(item, np.ndarray):
            flat = item.ravel()
            out.append(str(flat[0]) if flat.size else "")
        else:
            out.append(str(item))
    return np.array(out, dtype=object)


@dataclass
class GEM:
    """A minimal, analysis-oriented view of a COBRA model."""

    name: str
    S: sp.csr_matrix
    lb: np.ndarray
    ub: np.ndarray
    b: np.ndarray
    c: np.ndarray
    rxns: np.ndarray
    rxn_names: np.ndarray
    subsystems: np.ndarray
    mets: np.ndarray
    met_names: np.ndarray
    met_formulas: np.ndarray
    met_charges: np.ndarray
    genes: np.ndarray
    gene_names: np.ndarray
    rxn_gene_mat: sp.csr_matrix | None
    gr_rules: np.ndarray

    # -- basic properties -------------------------------------------------
    @property
    def n_reactions(self) -> int:
        return self.S.shape[1]

    @property
    def n_metabolites(self) -> int:
        return self.S.shape[0]

    def rxn_index(self, rxn_id: str) -> int:
        hits = np.flatnonzero(self.rxns == rxn_id)
        if hits.size == 0:
            raise KeyError(f"reaction {rxn_id!r} is not in the model")
        return int(hits[0])

    def met_index(self, met_id: str) -> int:
        hits = np.flatnonzero(self.mets == met_id)
        if hits.size == 0:
            raise KeyError(f"metabolite {met_id!r} is not in the model")
        return int(hits[0])

    def has_met(self, met_id: str) -> bool:
        return bool(np.any(self.mets == met_id))

    def has_rxn(self, rxn_id: str) -> bool:
        return bool(np.any(self.rxns == rxn_id))

    def with_bounds(self, changes: dict[str, tuple[float | None, float | None]]) -> "GEM":
        """Return a copy of the model with selected bounds replaced."""
        lb, ub = self.lb.copy(), self.ub.copy()
        for rxn_id, (low, high) in changes.items():
            idx = self.rxn_index(rxn_id)
            if low is not None:
                lb[idx] = low
            if high is not None:
                ub[idx] = high
        clone = GEM(**{**self.__dict__, "lb": lb, "ub": ub})
        return clone

    def carbon_count(self) -> np.ndarray:
        """Number of carbon atoms per metabolite, parsed from ``metFormulas``."""
        import re

        counts = np.zeros(self.n_metabolites)
        pattern = re.compile(r"C(\d*)(?![a-z])")
        for i, formula in enumerate(self.met_formulas):
            if not isinstance(formula, str) or not formula:
                continue
            match = pattern.search(formula)
            if match:
                counts[i] = int(match.group(1)) if match.group(1) else 1
        return counts


def load_gem(path: str | Path, name: str | None = None) -> GEM:
    """Read a COBRA model stored as a MATLAB v7 ``.mat`` file."""
    path = Path(path)
    raw = sio.loadmat(str(path), struct_as_record=False, squeeze_me=True)
    keys = [k for k in raw if not k.startswith("__")]
    if len(keys) != 1:
        raise ValueError(f"expected exactly one variable in {path.name}, found {keys}")
    model = raw[keys[0]]

    S = sp.csr_matrix(model.S)
    n_rxn = S.shape[1]

    def field(attr, default=None):
        return getattr(model, attr, default)

    rgm = field("rxnGeneMat")
    return GEM(
        name=name or keys[0],
        S=S,
        lb=np.asarray(model.lb, dtype=float).ravel(),
        ub=np.asarray(model.ub, dtype=float).ravel(),
        b=np.asarray(field("b", np.zeros(S.shape[0])), dtype=float).ravel(),
        c=np.asarray(field("c", np.zeros(n_rxn)), dtype=float).ravel(),
        rxns=_as_str_array(model.rxns),
        rxn_names=_as_str_array(field("rxnNames", model.rxns)),
        subsystems=_as_str_array(field("subSystems", np.array([""] * n_rxn))),
        mets=_as_str_array(model.mets),
        met_names=_as_str_array(field("metNames", model.mets)),
        met_formulas=_as_str_array(field("metFormulas", np.array([""] * S.shape[0]))),
        met_charges=np.asarray(field("metCharges", np.zeros(S.shape[0])), dtype=float).ravel(),
        genes=_as_str_array(field("genes", np.array([]))),
        gene_names=_as_str_array(field("geneNames", np.array([]))),
        rxn_gene_mat=sp.csr_matrix(rgm) if rgm is not None else None,
        gr_rules=_as_str_array(field("grRules", np.array([""] * n_rxn))),
    )


def load_flux_vector(path: str | Path) -> np.ndarray:
    """Read a plain numeric vector stored in a ``.mat`` file (e.g. ``v_or.mat``)."""
    raw = sio.loadmat(str(path), struct_as_record=False, squeeze_me=True)
    keys = [k for k in raw if not k.startswith("__")]
    if len(keys) != 1:
        raise ValueError(f"expected exactly one variable in {path}, found {keys}")
    return np.asarray(raw[keys[0]], dtype=float).ravel()


def load_fba_solution(path: str | Path) -> dict:
    """Read a COBRA ``optimizeCbModel`` solution structure."""
    raw = sio.loadmat(str(path), struct_as_record=False, squeeze_me=True)
    keys = [k for k in raw if not k.startswith("__")]
    sol = raw[keys[0]]
    out = {"objective": float(np.atleast_1d(sol.f)[0])}
    for attr in ("v", "x"):
        if hasattr(sol, attr):
            out["v"] = np.asarray(getattr(sol, attr), dtype=float).ravel()
            break
    out["status"] = int(np.atleast_1d(getattr(sol, "stat", 1))[0])
    return out

## 6. The samples (`run_00_export_samples`)

MATLAB produced one matrix per (condition, chain). This turns that collection into a single addressable dataset in which every sampled flux vector carries the biological context needed to interpret it: which condition, which chain, which severity level.

> The uniform data model: sampled flux matrices plus their metadata.

## 7. From MATLAB archives to arrays Python can hold (`run_00_export_samples`)

The only place a `.mat` file is opened. Each sampling archive — a dense
matrix of reactions by samples, about 174 MB of double precision — is read
with the parser above and written back as a thinned float32 matrix, a
full-resolution trace of the monitored reactions, and a small JSON record.
That is what makes sixty chains fit in memory at once, and it is why the
project needs no conversion script outside this notebook.

In [ ]:
"""Stage 0 — turn the MATLAB archives into something Python can read quickly.

This is the only place where a ``.mat`` file is opened, and it exists so that
the whole project is one notebook: no separate conversion script, nothing to
run outside this file.

A sampling archive is a dense matrix of reactions × samples saved by MATLAB as
a v7 file, which is a level-5 MAT container whose elements are zlib
compressed. Each one is read here and written back as three small files:

    <stem>.thin.f32.gz     (n_kept, n_reactions)  float32, C order
    <stem>.trace.f32.gz    (n_trace, n_samples)   float32, C order
    <stem>.json            shapes, thinning, trace rows, provenance

The thinned matrix is what every feature, distance and model is computed on.
The traces keep full resolution for the monitored reactions, because mixing
diagnostics must not be computed on thinned output. Going from ~180 MB of
double precision per chain to ~9 MB of compressed float32 is what makes it
possible to hold sixty chains in memory at once.

**How many states each chain keeps.** Every chain of a reconstruction keeps
the same number of states, `config.KEEP_PER_CHAIN[variant]`, by thinning it
uniformly. Keeping a fixed number per chain, rather than a fixed stride, keeps
the design balanced even if chains are sampled with different numbers of
points: an earlier thermo run returned 2,000 points for its perturbed chains
and 10,000 for its healthy ones, and a fixed stride would have given the
healthy condition five times the weight of every other. The stride actually
applied is written to each chain's JSON.

**Which axis is reactions.** The orientation is taken from the model, not
guessed from which axis is longer: a chain with fewer points than the model
has reactions is exactly where that guess fails.

The reader below is deliberately narrow: it finds the largest two-dimensional
numeric array in the file and ignores everything else. That is all a sampling
archive contains, and a narrow reader fails loudly on a file that holds
something else.
"""
import gzip
import json
import re
import struct
import zlib
from pathlib import Path

import numpy as np


__all__ = ["read_sample_matrix", "canonical_stem", "export_chain", "run_00_export_samples"]

MI_COMPRESSED, MI_MATRIX = 15, 14
MI_INT8, MI_INT32, MI_UINT32, MI_SINGLE, MI_DOUBLE = 1, 5, 6, 7, 9
NUMPY_OF = {MI_DOUBLE: "<f8", MI_SINGLE: "<f4", MI_INT32: "<i4",
            MI_UINT32: "<u4", MI_INT8: "|i1"}


def _tag(buffer: bytes, offset: int):
    """``(dtype, n_bytes, data_start, next_offset)`` of the element at ``offset``."""
    (word,) = struct.unpack_from("<I", buffer, offset)
    if word >> 16:                                   # small-data element format
        return word & 0xFFFF, word >> 16, offset + 4, offset + 8
    dtype, n_bytes = struct.unpack_from("<II", buffer, offset)
    padded = n_bytes + ((8 - n_bytes % 8) % 8)
    return dtype, n_bytes, offset + 8, offset + 8 + padded


def _matrix(buffer: bytes, offset: int):
    """Parse one miMATRIX into ``(name, array)``; numeric, non-complex only."""
    dtype, n_bytes, start, next_offset = _tag(buffer, offset)
    if dtype != MI_MATRIX:
        return None, None, next_offset
    position = start
    _, _, _, position = _tag(buffer, position)                      # array flags
    dtype_dims, n_dims, dims_start, position = _tag(buffer, position)
    dims = np.frombuffer(buffer, NUMPY_OF[dtype_dims], n_dims // 4, dims_start)
    _, n_name, name_start, position = _tag(buffer, position)
    name = buffer[name_start:name_start + n_name].decode("latin1")
    dtype_values, n_values, values_start, position = _tag(buffer, position)
    itemsize = np.dtype(NUMPY_OF[dtype_values]).itemsize
    values = np.frombuffer(buffer, NUMPY_OF[dtype_values], n_values // itemsize, values_start)
    return name, values.reshape(tuple(int(d) for d in dims), order="F"), next_offset


def read_sample_matrix(path: str | Path) -> tuple[str, np.ndarray]:
    """The largest 2-D numeric array in a MATLAB v7 file, with its variable name."""
    raw = Path(path).read_bytes()
    best_name, best = None, None
    offset = 128                                     # skip the text header
    while offset < len(raw) - 8:
        dtype, n_bytes, start, next_offset = _tag(raw, offset)
        if dtype == MI_COMPRESSED:
            name, array, _ = _matrix(zlib.decompress(raw[start:start + n_bytes]), 0)
        elif dtype == MI_MATRIX:
            name, array, _ = _matrix(raw, offset)
        else:
            name, array = None, None
        if array is not None and array.ndim == 2 and (best is None or array.size > best.size):
            best_name, best = name, array
        offset = next_offset
    if best is None:
        raise ValueError(f"{Path(path).name}: no two-dimensional numeric array found")
    return best_name, best


# --------------------------------------------------------------- naming
_LEVEL = {"0_1": "0_10", "0_28": "0_28", "0_5": "0_50", "1": "1_00", "0_75": "0_75"}
_RAW_CONDITION = re.compile(
    r"^(?P<variant>thermo_)?G6PD_(?P<g6pd>[0-9_]+)_h2o2_(?P<h2o2>[0-9_]+)"
    r"(?:_samples|_chain)(?P<chain>\d+)$")
_RAW_HEALTHY = re.compile(r"^(?P<variant>thermo)?[Mm]odelOr(?:_samples|_chain)(?P<chain>\d+)$")
_ALREADY = re.compile(r"^(original|thermo)_.*_chain\d+$")


def canonical_stem(stem: str) -> tuple[str, str, float, float]:
    """MATLAB output name -> ``(canonical stem, variant, g6pd level, h2o2 level)``."""
    if _ALREADY.match(stem):
        info = parse_condition(stem)
        return stem, info["model_variant"], info["g6pd_level"], info["h2o2_level"]
    healthy = _RAW_HEALTHY.match(stem)
    if healthy:
        variant = "thermo" if healthy.group("variant") else "original"
        return f"{variant}_healthy_chain{int(healthy.group('chain')):02d}", variant, 1.0, 1.0
    match = _RAW_CONDITION.match(stem)
    if not match:
        raise ValueError(f"unrecognised archive name: {stem!r}")
    variant = "thermo" if match.group("variant") else "original"
    g6pd, h2o2 = _LEVEL[match.group("g6pd")], _LEVEL[match.group("h2o2")]
    name = f"{variant}_G6PD_{g6pd}_H2O2_{h2o2}_chain{int(match.group('chain')):02d}"
    return name, variant, float(g6pd.replace("_", ".")), float(h2o2.replace("_", "."))


def export_chain(path: Path, out_dir: Path, *, n_trace: int = 200) -> str:
    """Convert one archive. Returns a one-line report."""
    stem = Path(path).stem
    name, variant, g6pd, h2o2 = canonical_stem(stem)
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    if (out_dir / f"{name}.json").exists():
        return f"{name}: already exported"

    variable, samples = read_sample_matrix(path)
    expected = config.N_REACTIONS[variant]
    if samples.shape[0] != expected:
        if samples.shape[1] != expected:
            raise ValueError(f"{stem}: neither axis of {samples.shape} is the "
                             f"{expected} reactions of the {variant} reconstruction")
        samples = samples.T
    n_reactions, n_samples = samples.shape

    keep = config.KEEP_PER_CHAIN[variant]
    if n_samples < keep:
        raise ValueError(f"{stem}: {n_samples} points, fewer than the {keep} to keep")
    thin = n_samples // keep
    kept = np.ascontiguousarray(samples[:, ::thin][:, :keep].T, dtype=np.float32)
    rows = np.unique(np.linspace(0, n_reactions - 1, min(n_trace, n_reactions)).astype(int))
    trace = np.ascontiguousarray(samples[rows, :], dtype=np.float32)

    for kind, array in (("thin", kept), ("trace", trace)):
        with gzip.open(out_dir / f"{name}.{kind}.f32.gz", "wb", compresslevel=6) as handle:
            handle.write(array.tobytes(order="C"))

    (out_dir / f"{name}.json").write_text(json.dumps({
        "file": Path(path).name, "variable": variable, "model_variant": variant,
        "condition": "healthy" if g6pd == 1.0 else f"G6PD{g6pd:g}_H2O2{h2o2:g}",
        "g6pd_level": g6pd, "h2o2_level": h2o2, "sampler": config.SAMPLER, "keep_per_chain": int(keep),
        "n_reactions": int(n_reactions), "n_samples_total": int(n_samples),
        "thin": int(thin), "n_samples_kept": int(kept.shape[0]), "kept_columns_start": 0,
        "trace_rows": [int(r) for r in rows], "trace_n_samples": int(n_samples),
        "thin_shape": [int(kept.shape[0]), int(kept.shape[1])],
        "trace_shape": [int(trace.shape[0]), int(trace.shape[1])],
        "dtype": "float32", "order": "C",
    }))
    return (f"{name}: {n_reactions} × {n_samples} → thin {kept.shape}, "
            f"trace {trace.shape}")


def run_00_export_samples(pattern: str = "*.mat") -> None:
    """Convert every archive under results/raw/samples that is not yet exported."""
    raw_dir = Path(RAW_SAMPLE_DIR)
    out_dir = Path(SAMPLE_DIR)
    if not raw_dir.exists():
        print(f"no raw archives at {raw_dir}; nothing to convert")
        return
    archives = sorted(p for p in raw_dir.glob(pattern) if "_samples" in p.name or "_chain" in p.name)
    print(f"{len(archives)} archives found")
    for path in archives:
        try:
            print(export_chain(path, out_dir), flush=True)
        except Exception as error:                   # noqa: BLE001 - one bad file must not stop the rest
            print(f"{path.name}: ERROR {error}", flush=True)

    exported = sorted(out_dir.glob("*.json"))
    print(f"\n{len(exported)} chains available in {out_dir}")

In [ ]:
import gzip
import json
import re
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd


__all__ = ["SampleChain", "SampleStore", "parse_condition", "build_metadata"]

#: Canonical export stem: model_variant / condition / chain. Written by
#: stage 00 of this notebook (run_00_export_samples), e.g.
#:     original_G6PD_0_28_H2O2_0_75_chain02
#:     thermo_healthy_chain01
_STEM_RE = re.compile(
    r"^(?P<variant>original|thermo)_"
    r"(?:healthy|G6PD_(?P<g6pd>\d+_\d+)_H2O2_(?P<h2o2>\d+_\d+))"
    r"_chain(?P<chain>\d+)$"
)
#: The pre-reorganisation names, still accepted so that an old archive loads.
_LEGACY_RE = re.compile(r"^G6PD_(?P<g6pd>[0-9_]+)_h2o2_(?P<h2o2>[0-9_]+)_samples(?P<chain>\d+)$")
_LEGACY_HEALTHY_RE = re.compile(r"^modelOr_samples(?P<chain>\d+)$")


def _decode_level(token: str) -> float:
    """``"0_28"`` -> 0.28, ``"1_00"`` -> 1.0."""
    return float(token.replace("_", ".", 1)) if "_" in token else float(token)


def parse_condition(stem: str) -> dict:
    """Decode an export stem into a structured condition record."""
    match = _STEM_RE.match(stem)
    if match:
        healthy = match.group("g6pd") is None
        g6pd = 1.0 if healthy else _decode_level(match.group("g6pd"))
        h2o2 = 1.0 if healthy else _decode_level(match.group("h2o2"))
        return {
            "model_variant": match.group("variant"),
            "condition": "healthy" if healthy else f"G6PD{g6pd:g}_H2O2{h2o2:g}",
            "state": "healthy" if healthy else "disease",
            "g6pd_level": g6pd,
            "h2o2_level": h2o2,
            "g6pd_severity": "none" if healthy else G6PD_SEVERITY.get(g6pd, "unknown"),
            "h2o2_load": "reference" if healthy else H2O2_LOAD.get(h2o2, "unknown"),
            "chain": int(match.group("chain")),
        }
    legacy_healthy = _LEGACY_HEALTHY_RE.match(stem)
    if legacy_healthy:
        return {
            "model_variant": "original", "condition": "healthy", "state": "healthy",
            "g6pd_level": 1.0, "h2o2_level": 1.0, "g6pd_severity": "none",
            "h2o2_load": "reference", "chain": int(legacy_healthy.group("chain")),
        }
    legacy = _LEGACY_RE.match(stem)
    if not legacy:
        raise ValueError(f"unrecognised sampling file stem: {stem!r}")
    g6pd, h2o2 = _decode_level(legacy.group("g6pd")), _decode_level(legacy.group("h2o2"))
    return {
        "model_variant": "original", "condition": f"G6PD{g6pd:g}_H2O2{h2o2:g}",
        "state": "disease", "g6pd_level": g6pd, "h2o2_level": h2o2,
        "g6pd_severity": G6PD_SEVERITY.get(g6pd, "unknown"),
        "h2o2_load": H2O2_LOAD.get(h2o2, "unknown"), "chain": int(legacy.group("chain")),
    }


@dataclass
class SampleChain:
    """One MCMC chain: a thinned flux matrix plus full-resolution traces."""

    stem: str
    meta: dict
    thin: np.ndarray            # (n_kept, n_reactions), float64
    trace: np.ndarray | None    # (n_trace_reactions, n_raw_samples)
    trace_rows: np.ndarray      # reaction indices of the trace rows

    @property
    def n_samples(self) -> int:
        return self.thin.shape[0]


class SampleStore:
    """Lazily loads the exported sampling matrices of one model variant.

    The export directory holds both models. A store is always scoped to one of
    them, because their flux vectors have different lengths and must never be
    concatenated by accident.
    """

    def __init__(self, directory: str | Path, variant: str | None = None, *,
                 include_excluded: bool = False):
        self.directory = Path(directory)
        self.variant = variant or ACTIVE_VARIANT
        stems = sorted(
            p.name[: -len(".json")] for p in self.directory.glob("*.json")
            if _STEM_RE.match(p.name[: -len(".json")]) or _LEGACY_RE.match(p.name[: -len(".json")])
            or _LEGACY_HEALTHY_RE.match(p.name[: -len(".json")])
        )
        stems = [s for s in stems if parse_condition(s)["model_variant"] == self.variant]
        # Chains that failed the constraint-integrity check (stage 00b) are
        # skipped by every stage; the check itself reads them all.
        self.excluded = self._excluded() if not include_excluded else set()
        self.stems = [s for s in stems if s not in self.excluded]
        if not self.stems:
            raise FileNotFoundError(
                f"no exported sampling files for variant {self.variant!r} in {self.directory}"
            )

    def _excluded(self) -> set[str]:
        path = self.directory.parent / "excluded_chains.json"
        if not path.exists():
            return set()
        return {entry["stem"] for entry in json.loads(path.read_text())["excluded"]}

    # -- loading --------------------------------------------------------
    def _read_array(self, stem: str, kind: str, shape: list[int]) -> np.ndarray:
        path = self.directory / f"{stem}.{kind}.f32.gz"
        with gzip.open(path, "rb") as handle:
            raw = handle.read()
        expected = int(np.prod(shape))
        arr = np.frombuffer(raw, dtype="<f4")
        if arr.size != expected:
            raise ValueError(
                f"{path.name}: expected {expected} values for shape {shape}, found {arr.size}"
            )
        return arr.reshape(shape).astype(np.float64)

    def load_chain(self, stem: str, *, with_trace: bool = False) -> SampleChain:
        meta = json.loads((self.directory / f"{stem}.json").read_text())
        thin = self._read_array(stem, "thin", meta["thin_shape"])
        trace = None
        if with_trace:
            trace = self._read_array(stem, "trace", meta["trace_shape"])
        return SampleChain(
            stem=stem, meta=meta, thin=thin, trace=trace,
            trace_rows=np.asarray(meta["trace_rows"], dtype=int),
        )

    def load_all(self) -> tuple[np.ndarray, pd.DataFrame]:
        """Concatenate every chain of this variant into one matrix."""
        blocks, records = [], []
        widths = set()
        for stem in self.stems:
            chain = self.load_chain(stem)
            widths.add(chain.thin.shape[1])
            if len(widths) > 1:
                raise RuntimeError(
                    f"{stem}: flux vectors of different length within one variant: {widths}"
                )
            blocks.append(chain.thin)
            info = parse_condition(stem)
            for k in range(chain.n_samples):
                records.append({
                    **info, "stem": stem, "sample_index": k,
                    "raw_index": k * chain.meta["thin"],
                    "sample_id": f"{stem}:{k:05d}",
                })
        V = np.vstack(blocks)
        metadata = pd.DataFrame.from_records(records)
        if len(metadata) != V.shape[0]:
            raise RuntimeError("metadata and flux matrix are out of alignment")
        return V, metadata


def build_metadata(store: SampleStore) -> pd.DataFrame:
    """Chain-level summary table (one row per variant x condition x chain)."""
    rows = []
    for stem in store.stems:
        meta = json.loads((store.directory / f"{stem}.json").read_text())
        rows.append({
            **parse_condition(stem), "stem": stem,
            "n_raw_samples": meta["n_samples_total"], "thinning": meta["thin"],
            "n_retained_samples": meta["n_samples_kept"], "n_reactions": meta["n_reactions"],
            "source_file": meta["file"],
        })
    return pd.DataFrame(rows).sort_values(
        ["model_variant", "state", "g6pd_level", "h2o2_level", "chain"], ignore_index=True
    )

## 8. Reaction annotation and admissibility

Classifies every reaction structurally — exchange, sink, transport, metabolic, pseudo-reaction — which is what defines the internal/external split. The external block is the 480 exchange reactions and nothing else. This cell also carries the curated admissibility record: biological role, whether the boundary is potentially modifiable, and a toxicity weight.

> Reaction-level annotation and the internal/external decomposition.

In [ ]:
import re
from dataclasses import dataclass

import numpy as np
import pandas as pd


__all__ = [
    "ReactionClass",
    "annotate_reactions",
    "check_pathway_map",
    "internal_mask",
    "external_mask",
    "EXCHANGE_ADMISSIBILITY",
]


class ReactionClass:
    """Mutually exclusive structural roles a reaction can take."""

    EXCHANGE = "exchange"          # EX_*: mass crossing the system boundary
    SINK = "sink"                  # SK_*: intracellular pseudo-pools
    TRANSPORT = "transport"        # moves a metabolite between compartments
    METABOLIC = "metabolic"        # genuine intracellular conversion
    PSEUDO = "pseudoreaction"      # lumped/pseudo processes that are not sinks


# --------------------------------------------------------------------------
# Biological admissibility of boundary reactions
# --------------------------------------------------------------------------
#
# ``role`` answers "what is this compound to an erythrocyte in plasma?" and
# ``modifiable`` answers "could a clinician plausibly change its availability?".
# ``toxicity`` is an ordinal penalty weight in [0, 1] used by the intervention
# utility function; it encodes the harm of *increasing* exposure to the
# metabolite, and is what prevents mathematically attractive but clinically
# absurd interventions from being proposed.

@dataclass(frozen=True)
class Admissibility:
    role: str
    modifiable: bool
    toxicity: float
    note: str = ""


#: Curated admissibility for the boundary metabolites that carry the biology of
#: G6PD deficiency. Everything not listed falls back to :func:`_default_admissibility`.
EXCHANGE_ADMISSIBILITY: dict[str, Admissibility] = {
    # --- primary carbon and energy ------------------------------------
    "glc__D_e": Admissibility("nutrient", True, 0.05, "principal RBC carbon source"),
    "lac__L_e": Admissibility("waste", True, 0.10, "glycolytic end product"),
    "pyr_e": Admissibility("nutrient", True, 0.10, "alternative carbon source"),
    "fru_e": Admissibility("nutrient", True, 0.10, ""),
    "gal_e": Admissibility("nutrient", True, 0.10, ""),
    "man_e": Admissibility("nutrient", True, 0.10, ""),
    "rib__D_e": Admissibility("nutrient", True, 0.10, "feeds non-oxidative PPP"),
    "adn_e": Admissibility("nutrient", True, 0.15, "adenosine salvage"),
    "ade_e": Admissibility("nutrient", True, 0.15, "adenine salvage"),
    "ins_e": Admissibility("nutrient", True, 0.15, "inosine salvage"),
    "pi_e": Admissibility("nutrient", True, 0.05, ""),
    # --- redox and oxidative stress -----------------------------------
    "h2o2_e": Admissibility("toxicant", True, 0.95, "oxidative insult"),
    "o2_e": Admissibility("nutrient", True, 0.20, "substrate for ROS generation"),
    "o2s_e": Admissibility("toxicant", False, 1.00, "superoxide"),
    "no_e": Admissibility("signalling", False, 0.60, ""),
    "gthrd_e": Admissibility("therapeutic", True, 0.05, "reduced glutathione"),
    "gthox_e": Admissibility("waste", True, 0.30, "oxidised glutathione"),
    "cys__L_e": Admissibility("therapeutic", True, 0.10, "glutathione precursor"),
    "gly_e": Admissibility("therapeutic", True, 0.05, "glutathione precursor"),
    "glu__L_e": Admissibility("therapeutic", True, 0.10, "glutathione precursor"),
    "glycys_e": Admissibility("therapeutic", True, 0.10, "gamma-glutamylcysteine"),
    "ascb__L_e": Admissibility("therapeutic", True, 0.10, "vitamin C"),
    "dhdascb_e": Admissibility("waste", True, 0.25, "dehydroascorbate"),
    "nac_e": Admissibility("therapeutic", True, 0.10, "nicotinate, NAD precursor"),
    "ncam_e": Admissibility("therapeutic", True, 0.10, "nicotinamide, NAD precursor"),
    "ribflv_e": Admissibility("therapeutic", True, 0.10, "riboflavin, FAD precursor"),
    "pydxn_e": Admissibility("therapeutic", True, 0.10, "vitamin B6"),
    "thm_e": Admissibility("therapeutic", True, 0.10, "thiamine, transketolase cofactor"),
    "avite1_e": Admissibility("therapeutic", True, 0.10, "alpha-tocopherol"),
    "selni_e": Admissibility("therapeutic", True, 0.55, "selenium, narrow therapeutic index"),
    # --- known haemolytic triggers in G6PD deficiency -------------------
    "pchlor_e": Admissibility("toxicant", False, 1.00, "primaquine-type oxidant"),
    "meoh_e": Admissibility("toxicant", False, 0.90, ""),
    "cyan_e": Admissibility("toxicant", False, 1.00, "cyanide"),
    "co_e": Admissibility("toxicant", False, 0.95, "carbon monoxide"),
    "nh4_e": Admissibility("waste", True, 0.45, ""),
    "urea_e": Admissibility("waste", True, 0.10, ""),
    "co2_e": Admissibility("waste", True, 0.05, ""),
    "h_e": Admissibility("ion", True, 0.20, "pH"),
    "h2o_e": Admissibility("solvent", False, 0.00, ""),
}

#: Metabolite name fragments that mark a boundary compound as pharmacological
#: or xenobiotic rather than nutritional. Increasing exposure to these is
#: penalised because they are drugs or environmental agents, not nutrients.
_XENOBIOTIC_PATTERNS = (
    "azathioprine", "mercaptopurine", "ribavirin", "fluorouracil",
    "estr", "andro", "steroid", "prostaglandin", "leukotriene",
    "primaquine", "dapsone", "sulfa", "nitro",
)


def _default_admissibility(met_id: str, met_name: str, formula: str) -> Admissibility:
    lowered = f"{met_id} {met_name}".lower()
    if any(pat in lowered for pat in _XENOBIOTIC_PATTERNS):
        return Admissibility("xenobiotic", False, 0.75, "pharmacological agent")
    if re.search(r"\bion\b|^na1_|^k_|^cl_|^ca2_|^mg2_|^fe2_|^fe3_|^zn2_|^cu2_", met_id):
        return Admissibility("ion", True, 0.30, "")
    # An uncharacterised organic boundary metabolite is treated as a possible
    # nutrient but is not considered clinically modifiable, so it can never be
    # promoted into an intervention without explicit curation.
    if formula and "C" in formula:
        return Admissibility("metabolite", False, 0.40, "uncurated boundary metabolite")
    return Admissibility("metabolite", False, 0.40, "uncurated boundary metabolite")


# --------------------------------------------------------------------------
# Annotation
# --------------------------------------------------------------------------

def _pathway_lookup() -> dict[str, str]:
    out: dict[str, str] = {}
    for pathway, subsystems in PATHWAY_GROUPS.items():
        for sub in subsystems:
            out[sub] = pathway
    return out


def check_pathway_map(gem: GEM) -> pd.DataFrame:
    """Report model subsystems that no curated pathway group claims.

    Unmapped subsystems are not an error, but they must be visible: they are
    reported in the supplementary material so that the pathway-level feature
    family can be audited for coverage.
    """
    lookup = _pathway_lookup()
    unmapped = sorted({s for s in gem.subsystems if s and s not in lookup})
    counts = pd.Series(gem.subsystems).value_counts()
    return pd.DataFrame(
        {"subsystem": unmapped, "n_reactions": [int(counts.get(s, 0)) for s in unmapped]}
    ).sort_values("n_reactions", ascending=False, ignore_index=True)


def annotate_reactions(gem: GEM) -> pd.DataFrame:
    """Build the reaction metadata table.

    Returns one row per reaction with structural role, compartment span,
    pathway assignment, bound information and — for boundary reactions — the
    biological admissibility record used by the intervention framework.
    """
    S = gem.S.tocsc()
    n_rxn = gem.n_reactions
    lookup = _pathway_lookup()

    met_comp = np.array([m.rsplit("_", 1)[-1] for m in gem.mets], dtype=object)

    rows = []
    for j in range(n_rxn):
        rxn = gem.rxns[j]
        col = S[:, [j]].tocoo()
        met_idx = col.row
        coeffs = col.data
        comps = sorted(set(met_comp[met_idx].tolist()))
        subsystem = gem.subsystems[j]

        if rxn.startswith("EX_"):
            cls = ReactionClass.EXCHANGE
        elif rxn.startswith(("SK_", "sink_", "DM_")):
            cls = ReactionClass.SINK
        elif len(met_idx) == 1:
            cls = ReactionClass.SINK
        elif len(comps) > 1:
            cls = ReactionClass.TRANSPORT
        elif subsystem in ("Pseudoreactions", "Miscellaneous"):
            cls = ReactionClass.PSEUDO
        else:
            cls = ReactionClass.METABOLIC

        boundary_met = ""
        boundary_name = ""
        boundary_formula = ""
        if cls == ReactionClass.EXCHANGE and len(met_idx) == 1:
            boundary_met = gem.mets[met_idx[0]]
            boundary_name = gem.met_names[met_idx[0]]
            boundary_formula = gem.met_formulas[met_idx[0]]

        pathway = lookup.get(subsystem, "unassigned")
        if rxn in RAPOPORT_LUEBERING_RXNS:
            pathway = "rapoport_luebering"

        rows.append(
            {
                "rxn_id": rxn,
                "rxn_name": gem.rxn_names[j],
                "subsystem": subsystem,
                "pathway": pathway,
                "reaction_class": cls,
                "n_metabolites": int(len(met_idx)),
                "n_compartments": len(comps),
                "compartments": "|".join(comps),
                "lb": float(gem.lb[j]),
                "ub": float(gem.ub[j]),
                "reversible": bool(gem.lb[j] < 0 < gem.ub[j]),
                "objective_coefficient": float(gem.c[j]),
                "boundary_metabolite": boundary_met,
                "boundary_metabolite_name": boundary_name,
                "boundary_formula": boundary_formula,
                "stoich_sum_abs": float(np.abs(coeffs).sum()),
                "gene_rule": gem.gr_rules[j] if j < len(gem.gr_rules) else "",
                "is_oxidative_ppp": rxn in OXIDATIVE_PPP_RXNS,
            }
        )

    table = pd.DataFrame(rows)

    # -- admissibility for boundary reactions ---------------------------
    roles, modifiable, toxicity, notes = [], [], [], []
    for _, row in table.iterrows():
        if row["reaction_class"] != ReactionClass.EXCHANGE:
            roles.append("")
            modifiable.append(False)
            toxicity.append(np.nan)
            notes.append("")
            continue
        met = row["boundary_metabolite"]
        adm = EXCHANGE_ADMISSIBILITY.get(met)
        if adm is None:
            adm = _default_admissibility(
                met, row["boundary_metabolite_name"], row["boundary_formula"]
            )
        roles.append(adm.role)
        modifiable.append(adm.modifiable)
        toxicity.append(adm.toxicity)
        notes.append(adm.note)

    table["boundary_role"] = roles
    table["clinically_modifiable"] = modifiable
    table["toxicity_weight"] = toxicity
    table["admissibility_note"] = notes

    # -- the internal/external decomposition ----------------------------
    #
    # "External" is deliberately narrow: only reactions that exchange mass with
    # the extracellular environment qualify, because only those correspond to
    # something a clinician could in principle manipulate. Intracellular sink
    # pseudo-reactions are boundary reactions in the linear-algebraic sense but
    # are not environmental, so they are kept on the internal side and flagged.
    table["is_external"] = table["reaction_class"] == ReactionClass.EXCHANGE
    table["is_internal"] = ~table["is_external"]
    table["carbon_count"] = [
        _carbon_count(f) for f in table["boundary_formula"].fillna("")
    ]
    return table


def _carbon_count(formula: str) -> float:
    if not isinstance(formula, str) or not formula:
        return np.nan
    match = re.search(r"C(\d*)(?![a-z])", formula)
    if not match:
        return 0.0
    return float(match.group(1)) if match.group(1) else 1.0


def internal_mask(reactions: pd.DataFrame) -> np.ndarray:
    return reactions["is_internal"].to_numpy(dtype=bool)


def external_mask(reactions: pd.DataFrame) -> np.ndarray:
    return reactions["is_external"].to_numpy(dtype=bool)

## 9. What a feature is

A feature is admitted only with a family, a block, a declaration of whether it consumes the signed flux vector or its magnitudes, units, and a closed-form definition. The registry is the object the engine iterates over, so a feature that cannot be defined cannot be computed.

> Feature registry and shared computation context.

In [ ]:
from dataclasses import dataclass, field
from typing import Callable, Iterable

import numpy as np
import pandas as pd
import scipy.sparse as sp


__all__ = ["FeatureSpec", "FeatureContext", "FeatureRegistry", "FeatureFamily"]


@dataclass(frozen=True)
class FeatureSpec:
    """Declarative description of one scalar feature of a flux vector."""

    name: str
    family: str
    block: str          # "internal" | "external" | "combined"
    description: str
    formula: str        # LaTeX, without surrounding delimiters
    units: str          # "flux", "dimensionless", "nats", "count", "ratio"
    signed: bool        # computed on v (True) or on |v| (False)
    model_dependent: bool = False

    def as_record(self) -> dict:
        return {
            "feature": self.name,
            "family": self.family,
            "block": self.block,
            "description": self.description,
            "formula": self.formula,
            "units": self.units,
            "signed_input": self.signed,
            "model_dependent": self.model_dependent,
        }


@dataclass
class FeatureContext:
    """Everything a feature family needs to know about the model."""

    gem: GEM
    reactions: pd.DataFrame

    # derived, filled in __post_init__
    internal_idx: np.ndarray = field(init=False)
    external_idx: np.ndarray = field(init=False)
    S_int: sp.csr_matrix = field(init=False)
    S_ext: sp.csr_matrix = field(init=False)

    def __post_init__(self) -> None:
        self.internal_idx = np.flatnonzero(internal_mask(self.reactions))
        self.external_idx = np.flatnonzero(external_mask(self.reactions))
        S = self.gem.S.tocsc()
        self.S_int = sp.csr_matrix(S[:, self.internal_idx])
        self.S_ext = sp.csr_matrix(S[:, self.external_idx])

    # -- convenience ----------------------------------------------------
    def split(self, V: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
        """Split a ``(n_samples, n_reactions)`` matrix into internal/external."""
        return V[:, self.internal_idx], V[:, self.external_idx]

    def internal_reactions(self) -> pd.DataFrame:
        return self.reactions.iloc[self.internal_idx].reset_index(drop=True)

    def external_reactions(self) -> pd.DataFrame:
        return self.reactions.iloc[self.external_idx].reset_index(drop=True)

    def pathway_groups(self, block: str = "internal") -> dict[str, np.ndarray]:
        """Map pathway label -> positional indices within the given block."""
        table = self.internal_reactions() if block == "internal" else self.external_reactions()
        return {
            pathway: np.asarray(idx, dtype=int)
            for pathway, idx in table.groupby("pathway").groups.items()
        }


class FeatureFamily:
    """Base class for a group of features computed together."""

    name: str = "unnamed"

    def __init__(self, ctx: FeatureContext):
        self.ctx = ctx

    def specs(self) -> list[FeatureSpec]:  # pragma: no cover - interface
        raise NotImplementedError

    def compute(self, V_int: np.ndarray, V_ext: np.ndarray) -> pd.DataFrame:  # pragma: no cover
        raise NotImplementedError


class FeatureRegistry:
    """Ordered collection of feature specifications."""

    def __init__(self) -> None:
        self._specs: dict[str, FeatureSpec] = {}

    def add(self, spec: FeatureSpec) -> None:
        if spec.name in self._specs:
            raise ValueError(f"duplicate feature name {spec.name!r}")
        self._specs[spec.name] = spec

    def extend(self, specs: Iterable[FeatureSpec]) -> None:
        for spec in specs:
            self.add(spec)

    def __contains__(self, name: str) -> bool:
        return name in self._specs

    def __len__(self) -> int:
        return len(self._specs)

    @property
    def names(self) -> list[str]:
        return list(self._specs)

    def get(self, name: str) -> FeatureSpec:
        return self._specs[name]

    def to_frame(self) -> pd.DataFrame:
        return pd.DataFrame([s.as_record() for s in self._specs.values()])

    def subset(self, *, block: str | None = None, family: str | None = None) -> list[str]:
        out = []
        for name, spec in self._specs.items():
            if block is not None and spec.block != block:
                continue
            if family is not None and spec.family != family:
                continue
            out.append(name)
        return out


# --------------------------------------------------------------------------
# Small numerical helpers shared by several families
# --------------------------------------------------------------------------

def safe_divide(num: np.ndarray, den: np.ndarray, fill: float = np.nan) -> np.ndarray:
    """Element-wise division that returns ``fill`` where the denominator vanishes."""
    num = np.asarray(num, dtype=float)
    den = np.asarray(den, dtype=float)
    out = np.full(np.broadcast(num, den).shape, fill, dtype=float)
    ok = np.abs(den) > 0
    np.divide(num, den, out=out, where=ok)
    return out


def positive_part(V: np.ndarray) -> np.ndarray:
    return np.where(V > 0, V, 0.0)


def negative_part(V: np.ndarray) -> np.ndarray:
    """Magnitude of the negative entries (returned as a non-negative array)."""
    return np.where(V < 0, -V, 0.0)


def gini(A: np.ndarray) -> np.ndarray:
    """Gini coefficient of each row of a non-negative matrix."""
    A = np.sort(np.abs(A), axis=1)
    n = A.shape[1]
    index = np.arange(1, n + 1)
    total = A.sum(axis=1)
    numerator = ((2 * index - n - 1) * A).sum(axis=1)
    return safe_divide(numerator, n * total, fill=0.0)


def hoyer_sparsity(A: np.ndarray) -> np.ndarray:
    """Hoyer sparsity in [0, 1]; 0 = fully dense, 1 = a single non-zero."""
    n = A.shape[1]
    l1 = np.abs(A).sum(axis=1)
    l2 = np.sqrt((A ** 2).sum(axis=1))
    root_n = np.sqrt(n)
    return safe_divide(root_n - safe_divide(l1, l2, fill=0.0), root_n - 1.0, fill=0.0)


def shannon_entropy(A: np.ndarray) -> np.ndarray:
    """Shannon entropy (nats) of the flux-magnitude distribution of each row."""
    P = np.abs(A)
    total = P.sum(axis=1, keepdims=True)
    P = safe_divide(P, total, fill=0.0)
    with np.errstate(divide="ignore", invalid="ignore"):
        terms = np.where(P > 0, P * np.log(P), 0.0)
    return -terms.sum(axis=1)


def renyi_entropy(A: np.ndarray, alpha: float = 2.0) -> np.ndarray:
    """Renyi entropy of order ``alpha`` (nats)."""
    P = np.abs(A)
    total = P.sum(axis=1, keepdims=True)
    P = safe_divide(P, total, fill=0.0)
    powered = np.power(P, alpha, where=P > 0, out=np.zeros_like(P)).sum(axis=1)
    with np.errstate(divide="ignore", invalid="ignore"):
        out = np.log(powered) / (1.0 - alpha)
    return np.where(powered > 0, out, 0.0)


def top_fraction(A: np.ndarray, fraction: float) -> np.ndarray:
    """Share of total flux magnitude carried by the top ``fraction`` reactions."""
    M = np.abs(A)
    n = M.shape[1]
    k = max(1, int(np.ceil(fraction * n)))
    part = np.partition(M, n - k, axis=1)[:, n - k:]
    return safe_divide(part.sum(axis=1), M.sum(axis=1), fill=0.0)

## 10. Mathematical feature families

Magnitude, distribution, sparsity, entropy and architecture: the five families that describe how flux is organised without reference to biochemistry.

> F1-F2: mathematical, statistical, sparsity, entropy and architecture features.

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats


__all__ = ["StructuralFamily", "structural_block"]

_QUANTILES = (0.05, 0.25, 0.50, 0.75, 0.95)


def structural_block(V: np.ndarray, prefix: str, *, reduced: bool = False) -> pd.DataFrame:
    """Compute the structural feature block for one flux matrix.

    Parameters
    ----------
    V
        ``(n_samples, n_reactions)`` signed flux matrix.
    prefix
        Prepended to every column name, e.g. ``"int"`` or ``"ext"``.
    reduced
        If True, only the subset of features that remain interpretable for a
        boundary flux vector is returned.
    """
    A = np.abs(V)
    n = V.shape[1]
    total = A.sum(axis=1)
    out: dict[str, np.ndarray] = {}

    # -- magnitude / norms ---------------------------------------------
    out["L0"] = (A > TOL.flux_zero).sum(axis=1).astype(float)
    out["L1"] = total
    out["L2"] = np.sqrt((V ** 2).sum(axis=1))
    out["Linf"] = A.max(axis=1)
    out["RMS"] = np.sqrt((V ** 2).mean(axis=1))

    # -- distribution of the magnitude vector ---------------------------
    out["mean_abs"] = A.mean(axis=1)
    out["median_abs"] = np.median(A, axis=1)
    out["sd_abs"] = A.std(axis=1, ddof=1)
    out["max_abs"] = out["Linf"]
    out["iqr_abs"] = np.subtract(*np.percentile(A, [75, 25], axis=1))
    out["mad_abs"] = np.median(np.abs(A - np.median(A, axis=1, keepdims=True)), axis=1)
    out["cv_abs"] = safe_divide(out["sd_abs"], out["mean_abs"])
    for q in _QUANTILES:
        out[f"q{int(q * 100):02d}_abs"] = np.quantile(A, q, axis=1)

    # -- distribution of the signed vector ------------------------------
    out["mean_signed"] = V.mean(axis=1)
    out["median_signed"] = np.median(V, axis=1)
    out["sd_signed"] = V.std(axis=1, ddof=1)
    out["skewness_signed"] = stats.skew(V, axis=1, bias=False)
    out["kurtosis_signed"] = stats.kurtosis(V, axis=1, bias=False)

    # -- sparsity -------------------------------------------------------
    active = A > TOL.flux_zero
    out["fraction_zero"] = 1.0 - active.mean(axis=1)
    out["fraction_active"] = active.mean(axis=1)
    out["hoyer_sparsity"] = hoyer_sparsity(V)
    out["gini"] = gini(V)

    # -- entropy of the flux-magnitude distribution ---------------------
    H = shannon_entropy(V)
    out["shannon_entropy"] = H
    out["normalised_entropy"] = H / np.log(n) if n > 1 else np.zeros_like(H)
    out["renyi_entropy_2"] = renyi_entropy(V, alpha=2.0)

    # -- flux architecture ----------------------------------------------
    pos = positive_part(V)
    neg = negative_part(V)
    out["fraction_positive"] = (V > TOL.flux_zero).mean(axis=1)
    out["fraction_negative"] = (V < -TOL.flux_zero).mean(axis=1)
    out["positive_flux"] = pos.sum(axis=1)
    out["negative_flux"] = neg.sum(axis=1)
    out["directionality"] = safe_divide(pos.sum(axis=1) - neg.sum(axis=1), total, fill=0.0)
    out["dominant_share"] = safe_divide(out["Linf"], total, fill=0.0)
    out["top1pct_share"] = top_fraction(V, 0.01)
    out["top5pct_share"] = top_fraction(V, 0.05)
    out["top10pct_share"] = top_fraction(V, 0.10)
    P = safe_divide(A, total[:, None], fill=0.0)
    out["hhi_concentration"] = (P ** 2).sum(axis=1)
    out["effective_reactions"] = np.exp(H)

    if reduced:
        keep = [
            "L0", "L1", "L2", "Linf", "mean_abs", "median_abs", "sd_abs",
            "cv_abs", "fraction_active", "gini", "shannon_entropy",
            "normalised_entropy", "fraction_positive", "fraction_negative",
            "positive_flux", "negative_flux", "directionality",
            "dominant_share", "hhi_concentration", "effective_reactions",
        ]
        out = {k: out[k] for k in keep}

    return pd.DataFrame({f"{prefix}_{k}": v for k, v in out.items()})


_DEFS: list[tuple[str, str, str, str, bool]] = [
    # (suffix, description, latex, units, signed)
    ("L0", "number of active reactions", r"\lVert v\rVert_0=\#\{i:|v_i|>\tau\}", "count", True),
    ("L1", "total flux magnitude", r"\lVert v\rVert_1=\sum_i|v_i|", "flux", False),
    ("L2", "Euclidean flux norm", r"\lVert v\rVert_2=\sqrt{\sum_i v_i^2}", "flux", True),
    ("Linf", "largest single flux", r"\lVert v\rVert_\infty=\max_i|v_i|", "flux", False),
    ("RMS", "root-mean-square flux", r"\sqrt{n^{-1}\sum_i v_i^2}", "flux", True),
    ("mean_abs", "mean flux magnitude", r"n^{-1}\sum_i|v_i|", "flux", False),
    ("median_abs", "median flux magnitude", r"\mathrm{med}_i|v_i|", "flux", False),
    ("sd_abs", "dispersion of flux magnitudes", r"\mathrm{sd}_i(|v_i|)", "flux", False),
    ("max_abs", "maximum flux magnitude", r"\max_i|v_i|", "flux", False),
    ("iqr_abs", "interquartile range of magnitudes", r"Q_{75}(|v|)-Q_{25}(|v|)", "flux", False),
    ("mad_abs", "median absolute deviation of magnitudes", r"\mathrm{med}\,||v|-\mathrm{med}|v||", "flux", False),
    ("cv_abs", "coefficient of variation of magnitudes", r"\mathrm{sd}(|v|)/\overline{|v|}", "dimensionless", False),
    ("mean_signed", "mean signed flux", r"n^{-1}\sum_i v_i", "flux", True),
    ("median_signed", "median signed flux", r"\mathrm{med}_i v_i", "flux", True),
    ("sd_signed", "dispersion of signed fluxes", r"\mathrm{sd}_i(v_i)", "flux", True),
    ("skewness_signed", "skewness of the signed flux distribution", r"\gamma_1(v)", "dimensionless", True),
    ("kurtosis_signed", "excess kurtosis of the signed flux distribution", r"\gamma_2(v)", "dimensionless", True),
    ("fraction_zero", "fraction of inactive reactions", r"n^{-1}\#\{i:|v_i|\le\tau\}", "ratio", False),
    ("fraction_active", "fraction of active reactions", r"n^{-1}\#\{i:|v_i|>\tau\}", "ratio", False),
    ("hoyer_sparsity", "Hoyer sparsity", r"\frac{\sqrt n-\lVert v\rVert_1/\lVert v\rVert_2}{\sqrt n-1}", "dimensionless", True),
    ("gini", "Gini coefficient of flux magnitudes", r"G(|v|)", "dimensionless", False),
    ("shannon_entropy", "entropy of the flux-magnitude distribution", r"H=-\sum_i p_i\ln p_i,\ p_i=|v_i|/\lVert v\rVert_1", "nats", False),
    ("normalised_entropy", "entropy normalised by its maximum", r"H/\ln n", "dimensionless", False),
    ("renyi_entropy_2", "Renyi entropy of order two", r"-\ln\sum_i p_i^2", "nats", False),
    ("fraction_positive", "fraction of reactions in the forward direction", r"n^{-1}\#\{i:v_i>\tau\}", "ratio", True),
    ("fraction_negative", "fraction of reactions in the reverse direction", r"n^{-1}\#\{i:v_i<-\tau\}", "ratio", True),
    ("positive_flux", "total forward flux", r"\sum_i v_i^+", "flux", True),
    ("negative_flux", "total reverse flux", r"\sum_i v_i^-", "flux", True),
    ("directionality", "net directional bias", r"\frac{\sum_i v_i^+-\sum_i v_i^-}{\sum_i|v_i|}", "dimensionless", True),
    ("dominant_share", "share carried by the single largest reaction", r"\lVert v\rVert_\infty/\lVert v\rVert_1", "ratio", False),
    ("top1pct_share", "share carried by the top 1% of reactions", r"\sum_{i\in T_{1\%}}|v_i|/\lVert v\rVert_1", "ratio", False),
    ("top5pct_share", "share carried by the top 5% of reactions", r"\sum_{i\in T_{5\%}}|v_i|/\lVert v\rVert_1", "ratio", False),
    ("top10pct_share", "share carried by the top 10% of reactions", r"\sum_{i\in T_{10\%}}|v_i|/\lVert v\rVert_1", "ratio", False),
    ("hhi_concentration", "Herfindahl-Hirschman flux concentration", r"\sum_i p_i^2", "dimensionless", False),
    ("effective_reactions", "effective number of flux-carrying reactions", r"\exp(H)", "count", False),
]
_DEFS += [
    (f"q{int(q * 100):02d}_abs", f"{int(q * 100)}th percentile of flux magnitudes",
     rf"Q_{{{int(q * 100)}}}(|v|)", "flux", False)
    for q in _QUANTILES
]

_REDUCED_KEEP = {
    "L0", "L1", "L2", "Linf", "mean_abs", "median_abs", "sd_abs", "cv_abs",
    "fraction_active", "gini", "shannon_entropy", "normalised_entropy",
    "fraction_positive", "fraction_negative", "positive_flux",
    "negative_flux", "directionality", "dominant_share",
    "hhi_concentration", "effective_reactions",
}


class StructuralFamily(FeatureFamily):
    """Mathematical, statistical, sparsity, entropy and architecture features."""

    name = "structural"

    def __init__(self, ctx: FeatureContext, *, external_reduced: bool = True):
        super().__init__(ctx)
        self.external_reduced = external_reduced

    def specs(self) -> list[FeatureSpec]:
        out: list[FeatureSpec] = []
        for suffix, desc, formula, units, signed in _DEFS:
            family = _family_of(suffix)
            out.append(FeatureSpec(f"int_{suffix}", family, "internal",
                                   f"internal fluxes: {desc}", formula, units, signed))
        for suffix, desc, formula, units, signed in _DEFS:
            if self.external_reduced and suffix not in _REDUCED_KEEP:
                continue
            family = _family_of(suffix)
            out.append(FeatureSpec(f"ext_{suffix}", family, "external",
                                   f"exchange fluxes: {desc}", formula, units, signed))
        return out

    def compute(self, V_int: np.ndarray, V_ext: np.ndarray) -> pd.DataFrame:
        left = structural_block(V_int, "int", reduced=False)
        right = structural_block(V_ext, "ext", reduced=self.external_reduced)
        return pd.concat([left, right], axis=1)


def _family_of(suffix: str) -> str:
    if suffix in {"L0", "L1", "L2", "Linf", "RMS"}:
        return "magnitude"
    if suffix.startswith(("q0", "q9")) or suffix in {
        "mean_abs", "median_abs", "sd_abs", "max_abs", "iqr_abs", "mad_abs",
        "cv_abs", "mean_signed", "median_signed", "sd_signed",
        "skewness_signed", "kurtosis_signed",
    }:
        return "distribution"
    if suffix in {"fraction_zero", "fraction_active", "hoyer_sparsity", "gini"}:
        return "sparsity"
    if "entropy" in suffix:
        return "entropy"
    return "architecture"

## 11. Biochemical feature families

Pathway activity, gross stoichiometric turnover, named reactions, efficiency ratios and network descriptors. One point governs this whole cell: because every sampled vector satisfies Sv = 0, production and consumption of any metabolite are equal by construction, so only gross turnover and its apportionment carry information.

> F3-F7: pathway, biochemical, turnover, efficiency and network features.

In [ ]:
import numpy as np
import pandas as pd
import scipy.sparse as sp


__all__ = ["PathwayFamily", "BiochemicalFamily", "NetworkFamily"]


# --------------------------------------------------------------------------
# F3 - pathway activity
# --------------------------------------------------------------------------

class PathwayFamily(FeatureFamily):
    """Absolute activity, relative share, directionality and engagement per pathway."""

    name = "pathway"

    #: Pathways carrying too few reactions to support a directionality statistic
    #: are still reported for activity, but flagged in the registry.
    MIN_REACTIONS = 2

    def __init__(self, ctx: FeatureContext):
        super().__init__(ctx)
        self.groups = {
            name: idx
            for name, idx in ctx.pathway_groups("internal").items()
            if name not in ("unassigned",)
        }

    def specs(self) -> list[FeatureSpec]:
        out = []
        for pathway, idx in self.groups.items():
            n = len(idx)
            out.append(FeatureSpec(
                f"path_{pathway}_activity", "pathway", "internal",
                f"total flux magnitude carried by {pathway} ({n} reactions)",
                r"F_P=\sum_{i\in P}|v_i|", "flux", False))
            out.append(FeatureSpec(
                f"path_{pathway}_share", "pathway", "internal",
                f"share of internal flux carried by {pathway}",
                r"R_P=F_P/\sum_i|v_i|", "ratio", False))
            out.append(FeatureSpec(
                f"path_{pathway}_engagement", "pathway", "internal",
                f"fraction of {pathway} reactions carrying flux",
                r"A_P=|P|^{-1}\#\{i\in P:|v_i|>\tau\}", "ratio", False))
            if n >= self.MIN_REACTIONS:
                out.append(FeatureSpec(
                    f"path_{pathway}_directionality", "pathway", "internal",
                    f"net directional bias of {pathway}",
                    r"D_P=\frac{\sum_{i\in P}v_i^+-\sum_{i\in P}v_i^-}{\sum_{i\in P}|v_i|}",
                    "dimensionless", True))
        return out

    def compute(self, V_int: np.ndarray, V_ext: np.ndarray) -> pd.DataFrame:
        A = np.abs(V_int)
        total = A.sum(axis=1)
        out: dict[str, np.ndarray] = {}
        for pathway, idx in self.groups.items():
            block = V_int[:, idx]
            mag = np.abs(block)
            activity = mag.sum(axis=1)
            out[f"path_{pathway}_activity"] = activity
            out[f"path_{pathway}_share"] = safe_divide(activity, total, fill=0.0)
            out[f"path_{pathway}_engagement"] = (mag > TOL.flux_zero).mean(axis=1)
            if len(idx) >= self.MIN_REACTIONS:
                pos = positive_part(block).sum(axis=1)
                neg = negative_part(block).sum(axis=1)
                out[f"path_{pathway}_directionality"] = safe_divide(pos - neg, activity, fill=0.0)
        return pd.DataFrame(out)


# --------------------------------------------------------------------------
# F4-F6 - biochemical phenotype, turnover and efficiency
# --------------------------------------------------------------------------

#: Reactions that define erythrocyte redox and energy handling. Each entry maps
#: a label to the RBC-GEM reaction identifier; missing identifiers are skipped
#: and reported, so the family degrades gracefully on other models.
KEY_REACTIONS = {
    "g6pd": "G6PDH2",             # glucose-6-phosphate dehydrogenase
    "pgl": "PGL",                 # 6-phosphogluconolactonase
    "gnd": "GND",                 # 6-phosphogluconate dehydrogenase
    "catalase": "CAT",            # H2O2 -> H2O + O2
    "gpx": "GTHP",                # glutathione peroxidase
    "sod": "SPODM",               # superoxide dismutase
    "hexokinase": "HEX1",
    "pfk": "PFK",
    "pyruvate_kinase": "PYK",
    "ldh": "LDH_L",
    "dpg_mutase": "DPGM",
    "dpg_phosphatase": "DPGase",
    "adenylate_kinase": "ADK1",
    "na_k_atpase": "NaKt",
}

#: Efficiency ratios, expressed as (label, numerator, denominator). Numerators
#: and denominators are feature names produced earlier in this family, so every
#: ratio is traceable to explicit reaction sets rather than being an arbitrary
#: quotient of two convenient numbers.
EFFICIENCY_RATIOS = [
    ("nadph_per_glucose", "turnover_nadph_c", "turnover_glc__D_c"),
    ("nadph_per_atp", "turnover_nadph_c", "turnover_atp_c"),
    ("atp_per_glucose", "turnover_atp_c", "turnover_glc__D_c"),
    ("nadph_per_h2o2", "turnover_nadph_c", "turnover_h2o2_c"),
    ("gsh_per_h2o2", "turnover_gthrd_c", "turnover_h2o2_c"),
    ("redox_buffer_ratio", "turnover_gthrd_c", "turnover_gthox_c"),
    ("nadph_per_nadh", "turnover_nadph_c", "turnover_nadh_c"),
    ("oxidative_ppp_per_glycolysis", "path_pentose_phosphate_activity", "path_glycolysis_activity"),
    ("ppp_flux_split", "rxn_g6pd", "rxn_hexokinase"),
    ("peroxide_clearance_split", "rxn_catalase", "rxn_gpx"),
]


class BiochemicalFamily(FeatureFamily):
    """Metabolite turnover, key-reaction fluxes, redox budget and efficiencies."""

    name = "biochemical"

    def __init__(self, ctx: FeatureContext):
        super().__init__(ctx)
        gem = ctx.gem
        self.metabolites = [m for m in TURNOVER_METABOLITES if gem.has_met(m)]
        self.missing_metabolites = [m for m in TURNOVER_METABOLITES if not gem.has_met(m)]
        met_rows = [gem.met_index(m) for m in self.metabolites]
        # |S| restricted to the internal block and the metabolites of interest
        self.absS = sp.csr_matrix(np.abs(ctx.S_int[met_rows, :].toarray()))

        self.key_rxns: dict[str, int] = {}
        self.missing_reactions: list[str] = []
        internal_positions = {gem.rxns[j]: p for p, j in enumerate(ctx.internal_idx)}
        for label, rxn in KEY_REACTIONS.items():
            if rxn in internal_positions:
                self.key_rxns[label] = internal_positions[rxn]
            else:
                self.missing_reactions.append(rxn)

        # positions of the oxidative PPP reactions inside the internal block
        self.ox_ppp = [internal_positions[r] for r in OXIDATIVE_PPP_RXNS
                       if r in internal_positions]

    # -- registry -------------------------------------------------------
    def specs(self) -> list[FeatureSpec]:
        out = []
        for met in self.metabolites:
            out.append(FeatureSpec(
                f"turnover_{met}", "turnover", "internal",
                f"gross intracellular turnover of {met}",
                r"T_m=\sum_i|S_{mi}v_i|", "flux", False, model_dependent=True))
        for label in self.key_rxns:
            out.append(FeatureSpec(
                f"rxn_{label}", "biochemical", "internal",
                f"flux through {KEY_REACTIONS[label]} ({label})",
                r"v_{" + KEY_REACTIONS[label] + r"}", "flux", True, model_dependent=True))
        out.append(FeatureSpec(
            "oxidative_ppp_flux", "biochemical", "internal",
            "mean flux through the oxidative branch of the pentose phosphate pathway",
            r"\tfrac13(v_{G6PDH2}+v_{PGL}+v_{GND})", "flux", True, model_dependent=True))
        out.append(FeatureSpec(
            "nadph_from_oxidative_ppp", "biochemical", "internal",
            "share of NADPH turnover attributable to the oxidative PPP",
            r"\frac{\sum_{i\in\mathrm{oxPPP}}|S_{\mathrm{nadph},i}v_i|}{T_{\mathrm{nadph}}}",
            "ratio", False, model_dependent=True))
        out.append(FeatureSpec(
            "redox_burden", "biochemical", "internal",
            "peroxide turnover relative to reducing-equivalent turnover",
            r"T_{\mathrm{h2o2}}/(T_{\mathrm{nadph}}+T_{\mathrm{gthrd}})",
            "dimensionless", False, model_dependent=True))
        for label, num, den in EFFICIENCY_RATIOS:
            if not self._ratio_available(num, den):
                continue
            out.append(FeatureSpec(
                f"eff_{label}", "efficiency", "internal",
                f"{num} divided by {den}",
                rf"\mathrm{{{num}}}/\mathrm{{{den}}}", "dimensionless",
                False, model_dependent=True))
        return out

    def _ratio_available(self, num: str, den: str) -> bool:
        known = (
            {f"turnover_{m}" for m in self.metabolites}
            | {f"rxn_{k}" for k in self.key_rxns}
            | {f"path_{p}_activity" for p in self.ctx.pathway_groups("internal")}
        )
        return num in known and den in known

    # -- computation ----------------------------------------------------
    def compute(self, V_int: np.ndarray, V_ext: np.ndarray) -> pd.DataFrame:
        out: dict[str, np.ndarray] = {}

        turnover = np.abs(V_int) @ self.absS.T.toarray()   # (n_samples, n_metabolites)
        for k, met in enumerate(self.metabolites):
            out[f"turnover_{met}"] = turnover[:, k]

        for label, pos in self.key_rxns.items():
            out[f"rxn_{label}"] = V_int[:, pos]

        if self.ox_ppp:
            out["oxidative_ppp_flux"] = V_int[:, self.ox_ppp].mean(axis=1)

        # NADPH sourced from the oxidative PPP, as a share of NADPH turnover
        if "nadph_c" in self.metabolites and self.ox_ppp:
            row = self.metabolites.index("nadph_c")
            coeffs = np.abs(self.absS[row].toarray()).ravel()
            contribution = (np.abs(V_int[:, self.ox_ppp]) * coeffs[self.ox_ppp]).sum(axis=1)
            out["nadph_from_oxidative_ppp"] = safe_divide(
                contribution, turnover[:, row], fill=np.nan
            )

        if all(m in self.metabolites for m in ("h2o2_c", "nadph_c", "gthrd_c")):
            i_h = self.metabolites.index("h2o2_c")
            i_n = self.metabolites.index("nadph_c")
            i_g = self.metabolites.index("gthrd_c")
            out["redox_burden"] = safe_divide(
                turnover[:, i_h], turnover[:, i_n] + turnover[:, i_g], fill=np.nan
            )

        frame = pd.DataFrame(out)
        return frame

    def add_efficiencies(self, features: pd.DataFrame) -> pd.DataFrame:
        """Append efficiency ratios once pathway features are also available."""
        extra = {}
        for label, num, den in EFFICIENCY_RATIOS:
            if num not in features.columns or den not in features.columns:
                continue
            extra[f"eff_{label}"] = safe_divide(
                features[num].to_numpy(), features[den].to_numpy(), fill=np.nan
            )
        return pd.DataFrame(extra, index=features.index)


# --------------------------------------------------------------------------
# F7 - flux-weighted network descriptors
# --------------------------------------------------------------------------

class NetworkFamily(FeatureFamily):
    """Descriptors of how metabolic activity is spread across the network.

    Purely topological quantities are identical for every sample and therefore
    carry no phenotypic information; only *flux-weighted* versions are
    computed. Currency metabolites are excluded, because their promiscuous
    participation would otherwise dominate every weighting.
    """

    name = "network"

    def __init__(self, ctx: FeatureContext):
        super().__init__(ctx)
        gem = ctx.gem
        keep = np.array([m not in CURRENCY_METABOLITES for m in gem.mets])
        S_int = ctx.S_int
        self.absS = sp.csr_matrix(np.abs(S_int[keep, :].toarray()))
        # structural degree of each internal reaction (metabolites touched)
        self.reaction_degree = np.asarray((S_int != 0).sum(axis=0)).ravel().astype(float)
        # structural degree of each retained metabolite (reactions touching it)
        self.metabolite_degree = np.asarray((self.absS != 0).sum(axis=1)).ravel().astype(float)

    def specs(self) -> list[FeatureSpec]:
        return [
            FeatureSpec("net_weighted_reaction_degree", "network", "internal",
                        "flux-weighted mean number of metabolites per active reaction",
                        r"\sum_i p_i d_i,\ p_i=|v_i|/\lVert v\rVert_1", "count", False),
            FeatureSpec("net_metabolite_activity_gini", "network", "internal",
                        "inequality of metabolite turnover across the network",
                        r"G(T),\ T_m=\sum_i|S_{mi}v_i|", "dimensionless", False),
            FeatureSpec("net_metabolite_activity_entropy", "network", "internal",
                        "entropy of the metabolite turnover distribution",
                        r"-\sum_m q_m\ln q_m,\ q_m=T_m/\sum_{m'}T_{m'}", "nats", False),
            FeatureSpec("net_active_metabolites", "network", "internal",
                        "number of metabolites with non-zero turnover",
                        r"\#\{m:T_m>\tau\}", "count", False),
            FeatureSpec("net_hub_turnover_share", "network", "internal",
                        "share of total turnover carried by the ten most connected metabolites",
                        r"\sum_{m\in H_{10}}T_m/\sum_m T_m", "ratio", False),
            FeatureSpec("net_weighted_metabolite_degree", "network", "internal",
                        "turnover-weighted mean metabolite connectivity",
                        r"\sum_m q_m k_m", "count", False),
        ]

    def compute(self, V_int: np.ndarray, V_ext: np.ndarray) -> pd.DataFrame:
        A = np.abs(V_int)
        total = A.sum(axis=1, keepdims=True)
        P = safe_divide(A, total, fill=0.0)
        T = A @ self.absS.T.toarray()      # (n_samples, n_metabolites)
        T_total = T.sum(axis=1, keepdims=True)
        Q = safe_divide(T, T_total, fill=0.0)

        hubs = np.argsort(-self.metabolite_degree)[:10]
        with np.errstate(divide="ignore", invalid="ignore"):
            entropy = -np.where(Q > 0, Q * np.log(Q), 0.0).sum(axis=1)

        return pd.DataFrame(
            {
                "net_weighted_reaction_degree": P @ self.reaction_degree,
                "net_metabolite_activity_gini": gini(T),
                "net_metabolite_activity_entropy": entropy,
                "net_active_metabolites": (T > TOL.flux_zero).sum(axis=1).astype(float),
                "net_hub_turnover_share": safe_divide(
                    T[:, hubs].sum(axis=1), T_total.ravel(), fill=0.0
                ),
                "net_weighted_metabolite_degree": Q @ self.metabolite_degree,
            }
        )

## 12. Boundary feature families

The external phenotype: individually tracked exchanges, role-level aggregates, exchange burden, carbon retention, and the oxidative and redox environment.

> F8: the external (boundary) phenotype.

In [ ]:
import numpy as np
import pandas as pd


__all__ = ["ExternalFamily", "TRACKED_EXCHANGES"]


#: Boundary metabolites reported individually because they carry the biology of
#: erythrocyte energy metabolism and of G6PD deficiency. Everything else
#: contributes only through the aggregate role-level features, which keeps the
#: external phenotype interpretable instead of 480-dimensional.
TRACKED_EXCHANGES = {
    "glc__D_e": "glucose",
    "lac__L_e": "lactate",
    "pyr_e": "pyruvate",
    "o2_e": "oxygen",
    "co2_e": "carbon_dioxide",
    "h2o2_e": "hydrogen_peroxide",
    "o2s_e": "superoxide",
    "gthrd_e": "reduced_glutathione",
    "gthox_e": "oxidised_glutathione",
    "cys__L_e": "cysteine",
    "gly_e": "glycine",
    "glu__L_e": "glutamate",
    "ascb__L_e": "ascorbate",
    "dhdascb_e": "dehydroascorbate",
    "adn_e": "adenosine",
    "ade_e": "adenine",
    "ins_e": "inosine",
    "rib__D_e": "ribose",
    "pi_e": "phosphate",
    "nac_e": "nicotinate",
    "ncam_e": "nicotinamide",
    "ribflv_e": "riboflavin",
    "thm_e": "thiamine",
    "h_e": "proton",
    "urea_e": "urea",
    "nh4_e": "ammonium",
}

_ROLES = ("nutrient", "waste", "toxicant", "therapeutic", "ion", "xenobiotic", "metabolite")


class ExternalFamily(FeatureFamily):
    """Nutrient, waste, oxidative, redox and boundary-efficiency features."""

    name = "external"

    def __init__(self, ctx: FeatureContext):
        super().__init__(ctx)
        table = ctx.external_reactions()
        self.table = table
        self.met_ids = table["boundary_metabolite"].to_numpy()
        self.roles = table["boundary_role"].to_numpy()
        self.carbon = table["carbon_count"].to_numpy(dtype=float)
        self.carbon = np.nan_to_num(self.carbon, nan=0.0)
        self.toxicity = np.nan_to_num(table["toxicity_weight"].to_numpy(dtype=float), nan=0.4)

        self.tracked: dict[str, int] = {}
        for position, met in enumerate(self.met_ids):
            if met in TRACKED_EXCHANGES:
                self.tracked[TRACKED_EXCHANGES[met]] = position
        self.role_positions = {
            role: np.flatnonzero(self.roles == role) for role in _ROLES
        }

    # -- registry -------------------------------------------------------
    def specs(self) -> list[FeatureSpec]:
        out: list[FeatureSpec] = []
        for label in self.tracked:
            out.append(FeatureSpec(
                f"ex_{label}_net", "boundary_metabolite", "external",
                f"net exchange flux of {label} (negative = uptake)",
                r"v_{\mathrm{EX}}", "flux", True, model_dependent=True))
            out.append(FeatureSpec(
                f"ex_{label}_uptake", "boundary_metabolite", "external",
                f"uptake flux of {label}, reported as a positive magnitude",
                r"\max(-v_{\mathrm{EX}},0)", "flux", True, model_dependent=True))
        for role in _ROLES:
            if self.role_positions[role].size == 0:
                continue
            out.append(FeatureSpec(
                f"exrole_{role}_uptake", "boundary_role", "external",
                f"total uptake across boundary metabolites classified as {role}",
                r"\sum_{i\in R}\max(-v_i,0)", "flux", True))
            out.append(FeatureSpec(
                f"exrole_{role}_secretion", "boundary_role", "external",
                f"total secretion across boundary metabolites classified as {role}",
                r"\sum_{i\in R}\max(v_i,0)", "flux", True))
        out += [
            FeatureSpec("ex_total_uptake", "exchange_burden", "external",
                        "total uptake flux across all boundary reactions",
                        r"U=\sum_i\max(-v_i,0)", "flux", True),
            FeatureSpec("ex_total_secretion", "exchange_burden", "external",
                        "total secretion flux across all boundary reactions",
                        r"E=\sum_i\max(v_i,0)", "flux", True),
            FeatureSpec("ex_net_balance", "exchange_burden", "external",
                        "net mass flow across the boundary",
                        r"E-U", "flux", True),
            FeatureSpec("ex_turnover_ratio", "exchange_burden", "external",
                        "secretion relative to uptake",
                        r"E/U", "dimensionless", True),
            FeatureSpec("ex_n_uptaken", "exchange_burden", "external",
                        "number of metabolites taken up",
                        r"\#\{i:v_i<-\tau\}", "count", True),
            FeatureSpec("ex_n_secreted", "exchange_burden", "external",
                        "number of metabolites secreted",
                        r"\#\{i:v_i>\tau\}", "count", True),
            FeatureSpec("ex_uptake_entropy", "exchange_burden", "external",
                        "entropy of the uptake distribution across metabolites",
                        r"-\sum_i u_i\ln u_i,\ u_i=\max(-v_i,0)/U", "nats", True),
            FeatureSpec("ex_carbon_uptake", "boundary_efficiency", "external",
                        "carbon atoms entering the system per unit time",
                        r"C_U=\sum_i c_i\max(-v_i,0)", "flux", True,
                        model_dependent=True),
            FeatureSpec("ex_carbon_secretion", "boundary_efficiency", "external",
                        "carbon atoms leaving the system per unit time",
                        r"C_E=\sum_i c_i\max(v_i,0)", "flux", True,
                        model_dependent=True),
            FeatureSpec("ex_carbon_retention", "boundary_efficiency", "external",
                        "fraction of incoming carbon not immediately re-exported",
                        r"(C_U-C_E)/C_U", "ratio", True, model_dependent=True),
            FeatureSpec("ex_oxidative_exposure", "oxidative_environment", "external",
                        "combined uptake of peroxide, superoxide and oxygen",
                        r"\sum_{i\in\mathrm{ROS}}\max(-v_i,0)", "flux", True,
                        model_dependent=True),
            FeatureSpec("ex_reductant_supply", "redox_environment", "external",
                        "combined uptake of glutathione, cysteine and ascorbate",
                        r"\sum_{i\in\mathrm{RED}}\max(-v_i,0)", "flux", True,
                        model_dependent=True),
            FeatureSpec("ex_redox_supply_balance", "redox_environment", "external",
                        "reductant supply relative to oxidative exposure",
                        r"\mathrm{RED}/(\mathrm{RED}+\mathrm{ROS})", "ratio", True,
                        model_dependent=True),
            FeatureSpec("ex_toxicity_load", "admissibility", "external",
                        "toxicity-weighted uptake, used only as a descriptor",
                        r"\sum_i w_i\max(-v_i,0)", "flux", True, model_dependent=True),
        ]
        return out

    # -- computation ----------------------------------------------------
    def compute(self, V_int: np.ndarray, V_ext: np.ndarray) -> pd.DataFrame:
        uptake = negative_part(V_ext)     # positive magnitudes
        secretion = positive_part(V_ext)
        out: dict[str, np.ndarray] = {}

        for label, position in self.tracked.items():
            out[f"ex_{label}_net"] = V_ext[:, position]
            out[f"ex_{label}_uptake"] = uptake[:, position]

        for role, positions in self.role_positions.items():
            if positions.size == 0:
                continue
            out[f"exrole_{role}_uptake"] = uptake[:, positions].sum(axis=1)
            out[f"exrole_{role}_secretion"] = secretion[:, positions].sum(axis=1)

        U = uptake.sum(axis=1)
        E = secretion.sum(axis=1)
        out["ex_total_uptake"] = U
        out["ex_total_secretion"] = E
        out["ex_net_balance"] = E - U
        out["ex_turnover_ratio"] = safe_divide(E, U, fill=np.nan)
        out["ex_n_uptaken"] = (V_ext < -TOL.flux_zero).sum(axis=1).astype(float)
        out["ex_n_secreted"] = (V_ext > TOL.flux_zero).sum(axis=1).astype(float)
        out["ex_uptake_entropy"] = shannon_entropy(uptake)

        C_U = uptake @ self.carbon
        C_E = secretion @ self.carbon
        out["ex_carbon_uptake"] = C_U
        out["ex_carbon_secretion"] = C_E
        out["ex_carbon_retention"] = safe_divide(C_U - C_E, C_U, fill=np.nan)

        ros = [self.tracked[k] for k in ("hydrogen_peroxide", "superoxide", "oxygen")
               if k in self.tracked]
        red = [self.tracked[k] for k in ("reduced_glutathione", "cysteine", "ascorbate")
               if k in self.tracked]
        ros_load = uptake[:, ros].sum(axis=1) if ros else np.zeros(len(V_ext))
        red_load = uptake[:, red].sum(axis=1) if red else np.zeros(len(V_ext))
        out["ex_oxidative_exposure"] = ros_load
        out["ex_reductant_supply"] = red_load
        out["ex_redox_supply_balance"] = safe_divide(
            red_load, red_load + ros_load, fill=np.nan
        )
        out["ex_toxicity_load"] = uptake @ self.toxicity

        return pd.DataFrame(out)

## 13. The feature engine

Applies the registry to a sample matrix in blocks, so tens of thousands of states × thousands of reactions can be turned into one row of descriptors per state without holding an intermediate the size of the machine.

> The feature engine: turn a matrix of flux vectors into a phenotype matrix.

## 14. Cofactors, capacity and coupling

Three families the first version did not have: how a state allocates
redox and adenylate turnover, where it sits relative to the model's own
bounds, and how much of its activity crosses the boundary. A feature whose
requirements the model does not meet returns `NaN`, never `0`.

In [ ]:
"""Three families the first version of the framework did not have.

They close the gaps the design review named: cofactor and redox allocation,
the relation between a state and the model's own capacity limits, and
descriptors that live *between* the internal and boundary blocks.

Two rules are enforced here and nowhere else in the code:

* a feature whose reactions or metabolites are missing from the model returns
  ``NaN``, never ``0``. A zero would be read downstream as a real biological
  value and would make two models look different for the wrong reason;
* no feature is a function of the reaction whose bound defines the
  perturbation. The saturation of G6PDH2 would recover the severity axis by
  construction, which is why the capacity family keeps only aggregate
  statistics over the whole vector.
"""
import numpy as np
import pandas as pd


__all__ = ["CofactorFamily", "CapacityFamily", "CouplingFamily"]

#: Cofactor pairs whose turnover allocation the redox family reports.
COFACTOR_PAIRS = {
    "nadph_nadp": ("nadph_c", "nadp_c"),
    "nadh_nad": ("nadh_c", "nad_c"),
    "glutathione": ("gthrd_c", "gthox_c"),
}

#: Adenylates, in the order the energy-charge proxy expects.
ADENYLATES = ("atp_c", "adp_c", "amp_c")


def _metabolite_turnover(S, V_block, idx) -> np.ndarray:
    """Gross stoichiometric turnover of one metabolite over a block of states."""
    row = S[idx, :]
    return np.abs(V_block) @ np.abs(np.asarray(row.todense()).ravel())


class _TurnoverMixin:
    def _index(self, met: str):
        try:
            return int(self.ctx.gem.mets.index(met))
        except (ValueError, AttributeError):
            mets = list(self.ctx.gem.mets)
            return int(mets.index(met)) if met in mets else None

    def _turnover(self, V_int, met: str):
        idx = self._index(met)
        if idx is None:
            return None
        return _metabolite_turnover(self.ctx.S_int, V_int, idx)


class CofactorFamily(_TurnoverMixin, FeatureFamily):
    """How a state allocates reducing equivalents and adenylate turnover.

    Every feature is a *share*: a turnover divided by the turnover of the pool
    it belongs to. Shares are used rather than absolute turnovers because the
    sampled polytope is not in physiological units, so only relative
    allocation is comparable between states and between models.
    """

    name = "cofactor"

    def specs(self) -> list[FeatureSpec]:
        out = []
        for key, (reduced, oxidised) in COFACTOR_PAIRS.items():
            out.append(FeatureSpec(
                name=f"cof_{key}_share", family=self.name, block="internal",
                description=f"turnover share of {reduced} within the {reduced}/{oxidised} pool",
                formula=rf"T_{{{reduced}}} / (T_{{{reduced}}} + T_{{{oxidised}}})",
                units="ratio", signed=False, model_dependent=True,
            ))
        out.append(FeatureSpec(
            name="cof_nadph_over_nadh", family=self.name, block="internal",
            description="share of reducing-equivalent turnover carried by NADPH rather than NADH",
            formula=r"T_{nadph} / (T_{nadph} + T_{nadh})",
            units="ratio", signed=False, model_dependent=True,
        ))
        out.append(FeatureSpec(
            name="cof_energy_charge_proxy", family=self.name, block="internal",
            description=("adenylate energy charge computed on turnover rather than on "
                         "concentration; a flux-level proxy, not a measured charge"),
            formula=r"(T_{atp} + 0.5\,T_{adp}) / (T_{atp} + T_{adp} + T_{amp})",
            units="ratio", signed=False, model_dependent=True,
        ))
        out.append(FeatureSpec(
            name="cof_redox_per_energy", family=self.name, block="internal",
            description="redox turnover relative to adenylate turnover",
            formula=r"(T_{nadph} + T_{gthrd}) / (T_{atp} + T_{adp} + T_{amp})",
            units="ratio", signed=False, model_dependent=True,
        ))
        return out

    def compute(self, V_int: np.ndarray, V_ext: np.ndarray) -> pd.DataFrame:
        n = V_int.shape[0]
        nan = np.full(n, np.nan)
        turnover = {}
        for met in {m for pair in COFACTOR_PAIRS.values() for m in pair} | set(ADENYLATES):
            turnover[met] = self._turnover(V_int, met)

        def share(a, b):
            ta, tb = turnover.get(a), turnover.get(b)
            if ta is None or tb is None:
                return nan
            total = ta + tb
            return np.where(total > 0, ta / np.where(total > 0, total, 1.0), np.nan)

        columns = {f"cof_{key}_share": share(red, ox) for key, (red, ox) in COFACTOR_PAIRS.items()}
        columns["cof_nadph_over_nadh"] = share("nadph_c", "nadh_c")

        atp, adp, amp = (turnover.get(m) for m in ADENYLATES)
        if atp is None or adp is None or amp is None:
            columns["cof_energy_charge_proxy"] = nan
            columns["cof_redox_per_energy"] = nan
        else:
            pool = atp + adp + amp
            columns["cof_energy_charge_proxy"] = np.where(
                pool > 0, (atp + 0.5 * adp) / np.where(pool > 0, pool, 1.0), np.nan)
            nadph, gthrd = turnover.get("nadph_c"), turnover.get("gthrd_c")
            if nadph is None or gthrd is None:
                columns["cof_redox_per_energy"] = nan
            else:
                columns["cof_redox_per_energy"] = np.where(
                    pool > 0, (nadph + gthrd) / np.where(pool > 0, pool, 1.0), np.nan)
        return pd.DataFrame(columns)


class CapacityFamily(FeatureFamily):
    """Where a state sits relative to the bounds the model imposes.

    These are the only features that depend on the constraint set rather than
    on the flux distribution alone, which is exactly why they are reported
    separately: a difference between two models in this family is a statement
    about the models, not about metabolism.
    """

    name = "capacity"

    def __init__(self, ctx):
        super().__init__(ctx)
        lb, ub = np.asarray(ctx.gem.lb, float), np.asarray(ctx.gem.ub, float)
        self.lb_int, self.ub_int = lb[ctx.internal_idx], ub[ctx.internal_idx]
        self.lb_ext, self.ub_ext = lb[ctx.external_idx], ub[ctx.external_idx]
        self._tol = 1e-6

    def specs(self) -> list[FeatureSpec]:
        common = dict(family=self.name, units="ratio", signed=True, model_dependent=True)
        return [
            FeatureSpec(name="cap_at_upper_internal", block="internal",
                        description="fraction of internal reactions within tolerance of their upper bound",
                        formula=r"|\{j : v_j \geq u_j - \epsilon\}| / n_{int}", **common),
            FeatureSpec(name="cap_at_lower_internal", block="internal",
                        description="fraction of internal reactions within tolerance of their lower bound",
                        formula=r"|\{j : v_j \leq l_j + \epsilon\}| / n_{int}", **common),
            FeatureSpec(name="cap_mean_saturation_internal", block="internal",
                        description="mean position within the admissible interval, over bounded internal reactions",
                        formula=r"\mathrm{mean}_j\ (v_j - l_j) / (u_j - l_j)", **common),
            FeatureSpec(name="cap_at_bound_boundary", block="external",
                        description="fraction of exchange reactions at either of their bounds",
                        formula=r"|\{j : v_j \leq l_j + \epsilon \lor v_j \geq u_j - \epsilon\}| / n_{bnd}",
                        **common),
            FeatureSpec(name="cap_mean_saturation_boundary", block="external",
                        description="mean position within the admissible interval, over bounded exchanges",
                        formula=r"\mathrm{mean}_j\ (v_j - l_j) / (u_j - l_j)", **common),
        ]

    @staticmethod
    def _saturation(V, lb, ub):
        span = ub - lb
        usable = np.isfinite(span) & (span > 0)
        if not usable.any():
            return np.full(V.shape[0], np.nan)
        position = (V[:, usable] - lb[usable]) / span[usable]
        return position.mean(axis=1)

    def compute(self, V_int: np.ndarray, V_ext: np.ndarray) -> pd.DataFrame:
        tol = self._tol
        finite_i = np.isfinite(self.ub_int) & np.isfinite(self.lb_int)
        finite_e = np.isfinite(self.ub_ext) & np.isfinite(self.lb_ext)
        n_i = max(int(finite_i.sum()), 1)
        n_e = max(int(finite_e.sum()), 1)
        at_upper = (V_int[:, finite_i] >= self.ub_int[finite_i] - tol).sum(axis=1) / n_i
        at_lower = (V_int[:, finite_i] <= self.lb_int[finite_i] + tol).sum(axis=1) / n_i
        at_bound_e = (
            (V_ext[:, finite_e] >= self.ub_ext[finite_e] - tol)
            | (V_ext[:, finite_e] <= self.lb_ext[finite_e] + tol)
        ).sum(axis=1) / n_e
        return pd.DataFrame({
            "cap_at_upper_internal": at_upper,
            "cap_at_lower_internal": at_lower,
            "cap_mean_saturation_internal": self._saturation(V_int, self.lb_int, self.ub_int),
            "cap_at_bound_boundary": at_bound_e,
            "cap_mean_saturation_boundary": self._saturation(V_ext, self.lb_ext, self.ub_ext),
        })


class CouplingFamily(FeatureFamily):
    """Descriptors that only exist because the two blocks are kept apart.

    They are state-level quantities, one number per sampled state, and they are
    not the same thing as the statistical coupling between blocks measured
    later by canonical correlation: these say how much of a state's activity
    crosses the boundary, those say how predictable one block is from the other.
    """

    name = "coupling"

    def __init__(self, ctx):
        super().__init__(ctx)
        groups = ctx.pathway_groups("internal")
        self.ppp_idx = groups.get("pentose_phosphate", np.array([], dtype=int))
        self.glycolysis_idx = groups.get("glycolysis", np.array([], dtype=int))
        external = ctx.external_reactions()
        self.glucose_idx = self._find(external, "EX_glc__D_e")
        self.oxygen_idx = self._find(external, "EX_o2_e")

    @staticmethod
    def _find(table: pd.DataFrame, rxn_id: str):
        hit = np.flatnonzero((table["rxn_id"] == rxn_id).to_numpy())
        return int(hit[0]) if hit.size else None

    def specs(self) -> list[FeatureSpec]:
        common = dict(family=self.name, block="combined", units="ratio",
                      signed=False, model_dependent=True)
        return [
            FeatureSpec(name="cpl_boundary_share", description=(
                "share of the total flux magnitude of a state that crosses the system boundary"),
                formula=r"\|v_{bnd}\|_1 / \|v\|_1", **common),
            FeatureSpec(name="cpl_secretion_over_uptake", description=(
                "secreted magnitude relative to taken-up magnitude at the boundary"),
                formula=r"\sum_j \max(v_j, 0) / \sum_j \max(-v_j, 0)", **common),
            FeatureSpec(name="cpl_internal_per_boundary", description=(
                "internal activity supported per unit of boundary exchange"),
                formula=r"\|v_{int}\|_1 / \|v_{bnd}\|_1", **common),
            FeatureSpec(name="cpl_ppp_per_glucose_uptake", description=(
                "pentose-phosphate activity per unit of glucose taken up; NaN when the "
                "model has no glucose exchange"),
                formula=r"\sum_{i \in PPP} |v_i| / |v_{EX\_glc}|", **common),
            FeatureSpec(name="cpl_ppp_per_oxygen_uptake", description=(
                "pentose-phosphate activity per unit of oxygen taken up; NaN when the "
                "model has no oxygen exchange"),
                formula=r"\sum_{i \in PPP} |v_i| / |v_{EX\_o2}|", **common),
        ]

    def compute(self, V_int: np.ndarray, V_ext: np.ndarray) -> pd.DataFrame:
        n = V_int.shape[0]
        nan = np.full(n, np.nan)
        internal_l1 = np.abs(V_int).sum(axis=1)
        boundary_l1 = np.abs(V_ext).sum(axis=1)
        total = internal_l1 + boundary_l1
        uptake = np.maximum(-V_ext, 0).sum(axis=1)
        secretion = np.maximum(V_ext, 0).sum(axis=1)
        ppp = np.abs(V_int[:, self.ppp_idx]).sum(axis=1) if self.ppp_idx.size else nan

        def ratio(numerator, denominator):
            if numerator is nan or denominator is None:
                return nan
            safe = np.where(np.abs(denominator) > 0, denominator, np.nan)
            return numerator / safe

        glucose = np.abs(V_ext[:, self.glucose_idx]) if self.glucose_idx is not None else None
        oxygen = np.abs(V_ext[:, self.oxygen_idx]) if self.oxygen_idx is not None else None

        return pd.DataFrame({
            "cpl_boundary_share": ratio(boundary_l1, total),
            "cpl_secretion_over_uptake": ratio(secretion, uptake),
            "cpl_internal_per_boundary": ratio(internal_l1, boundary_l1),
            "cpl_ppp_per_glucose_uptake": ratio(ppp, glucose),
            "cpl_ppp_per_oxygen_uptake": ratio(ppp, oxygen),
        })

In [ ]:
import numpy as np
import pandas as pd


__all__ = ["FeatureEngine"]


class FeatureEngine:
    """Compute the full candidate feature space for a set of flux vectors."""

    def __init__(self, gem: GEM, reactions: pd.DataFrame):
        self.ctx = FeatureContext(gem=gem, reactions=reactions)
        self.structural = StructuralFamily(self.ctx)
        self.pathway = PathwayFamily(self.ctx)
        self.biochemical = BiochemicalFamily(self.ctx)
        self.network = NetworkFamily(self.ctx)
        self.external = ExternalFamily(self.ctx)
        self.cofactor = CofactorFamily(self.ctx)
        self.capacity = CapacityFamily(self.ctx)
        self.coupling = CouplingFamily(self.ctx)
        self.families = [
            self.structural, self.pathway, self.biochemical,
            self.network, self.external,
            self.cofactor, self.capacity, self.coupling,
        ]
        self.registry = FeatureRegistry()
        for family in self.families:
            self.registry.extend(family.specs())

    # ------------------------------------------------------------------
    def transform(self, V: np.ndarray, *, block_size: int = 4000) -> pd.DataFrame:
        """Map ``(n_samples, n_reactions)`` fluxes to ``(n_samples, n_features)``.

        Computation is chunked over samples so that memory stays bounded for
        the full 300,000-sample dataset.
        """
        chunks = []
        for start in range(0, V.shape[0], block_size):
            block = V[start:start + block_size]
            V_int, V_ext = self.ctx.split(block)
            parts = [family.compute(V_int, V_ext) for family in self.families]
            frame = pd.concat(parts, axis=1)
            frame = pd.concat([frame, self.biochemical.add_efficiencies(frame)], axis=1)
            chunks.append(frame)
        out = pd.concat(chunks, ignore_index=True)
        return out.loc[:, ~out.columns.duplicated()]

    def transform_vector(self, v: np.ndarray) -> pd.Series:
        """Phenotype of a single flux vector (e.g. an FBA reference point)."""
        return self.transform(np.asarray(v, dtype=float).reshape(1, -1)).iloc[0]

    # ------------------------------------------------------------------
    def registry_frame(self) -> pd.DataFrame:
        """Registry restricted to features the engine actually produces."""
        return self.registry.to_frame()

    def coverage_report(self) -> pd.DataFrame:
        """Model entities the engine expected but could not find."""
        rows = [
            {"kind": "metabolite", "identifier": m, "consequence": "turnover feature omitted"}
            for m in self.biochemical.missing_metabolites
        ] + [
            {"kind": "reaction", "identifier": r, "consequence": "key-reaction feature omitted"}
            for r in self.biochemical.missing_reactions
        ]
        return pd.DataFrame(rows, columns=["kind", "identifier", "consequence"])

## 15. Sampling diagnostics

Integrated autocorrelation time by Sokal windowing, effective sample size, split-R̂, cross-chain Kolmogorov–Smirnov and running-mean drift. These must be computed on the unthinned traces, which is why the traces are kept separately from the analysis set.

> MCMC diagnostics for sampled flux spaces.

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats


__all__ = [
    "autocorrelation",
    "integrated_autocorr_time",
    "effective_sample_size",
    "split_rhat",
    "chain_diagnostics",
    "cross_chain_agreement",
    "running_mean_stability",
]


def autocorrelation(x: np.ndarray, max_lag: int | None = None) -> np.ndarray:
    """Normalised autocorrelation function of a 1-D trace, computed by FFT."""
    x = np.asarray(x, dtype=float)
    n = x.size
    x = x - x.mean()
    var = np.dot(x, x) / n
    if var <= 0:
        return np.zeros(max_lag or n)
    size = 1 << (2 * n - 1).bit_length()
    f = np.fft.rfft(x, size)
    acf = np.fft.irfft(f * np.conjugate(f), size)[:n].real
    acf /= acf[0]
    return acf[: max_lag] if max_lag else acf


def integrated_autocorr_time(x: np.ndarray, c: float = 5.0, max_lag: int = 2000) -> float:
    """Integrated autocorrelation time using Sokal's automatic windowing.

    Returns ``nan`` for a constant trace, for which the quantity is undefined.
    """
    x = np.asarray(x, dtype=float)
    if x.size < 10 or np.ptp(x) <= TOL.invariant_range:
        return float("nan")
    acf = autocorrelation(x, max_lag=min(max_lag, x.size))
    taus = 2.0 * np.cumsum(acf) - 1.0
    window = np.arange(len(taus)) < c * taus
    idx = int(np.argmin(window)) if not window.all() else len(taus) - 1
    return float(max(taus[idx], 1.0))


def effective_sample_size(x: np.ndarray, **kwargs) -> float:
    r"""ESS :math:`= N/\tau_{\mathrm{int}}`."""
    tau = integrated_autocorr_time(x, **kwargs)
    if not np.isfinite(tau):
        return float("nan")
    return float(len(x) / tau)


def split_rhat(chains: np.ndarray) -> float:
    """Gelman-Rubin split-:math:`\\hat R` for a ``(n_chains, n_draws)`` array.

    Each chain is split in half first, which makes the statistic sensitive to
    within-chain drift as well as between-chain disagreement.
    """
    chains = np.asarray(chains, dtype=float)
    m, n = chains.shape
    if n < 4:
        return float("nan")
    half = n // 2
    split = np.vstack([chains[:, :half], chains[:, half: 2 * half]])
    m2, n2 = split.shape
    if np.ptp(split) <= TOL.invariant_range:
        return float("nan")
    chain_means = split.mean(axis=1)
    chain_vars = split.var(axis=1, ddof=1)
    W = chain_vars.mean()
    B = n2 * chain_means.var(ddof=1)
    if W <= 0:
        return float("nan")
    var_hat = ((n2 - 1) / n2) * W + B / n2
    return float(np.sqrt(var_hat / W))


def chain_diagnostics(
    traces: dict[int, np.ndarray],
    reaction_ids: np.ndarray,
    *,
    max_lag: int = 2000,
) -> pd.DataFrame:
    """Per-reaction diagnostics for one condition.

    Parameters
    ----------
    traces
        Mapping ``chain number -> (n_reactions, n_draws)`` trace array.
    reaction_ids
        Identifiers aligned with the rows of the trace arrays.
    """
    chain_ids = sorted(traces)
    stacked = np.stack([traces[c] for c in chain_ids])   # (n_chains, n_rxn, n_draws)
    n_chains, n_rxn, n_draws = stacked.shape

    rows = []
    for i in range(n_rxn):
        per_chain = stacked[:, i, :]
        taus, esss = [], []
        for c in range(n_chains):
            tau = integrated_autocorr_time(per_chain[c], max_lag=max_lag)
            taus.append(tau)
            esss.append(n_draws / tau if np.isfinite(tau) else np.nan)
        rows.append(
            {
                "rxn_id": reaction_ids[i],
                "n_chains": n_chains,
                "n_draws": n_draws,
                "range": float(np.ptp(per_chain)),
                "invariant": bool(np.ptp(per_chain) <= TOL.invariant_range),
                "tau_int_mean": float(np.nanmean(taus)),
                "tau_int_max": float(np.nanmax(taus)) if np.any(np.isfinite(taus)) else np.nan,
                "ess_mean": float(np.nanmean(esss)),
                "ess_min": float(np.nanmin(esss)) if np.any(np.isfinite(esss)) else np.nan,
                "ess_total": float(np.nansum(esss)),
                "split_rhat": split_rhat(per_chain),
            }
        )
    return pd.DataFrame(rows)


def cross_chain_agreement(traces: dict[int, np.ndarray], reaction_ids: np.ndarray) -> pd.DataFrame:
    """Pairwise two-sample distributional comparison between chains.

    Uses the Kolmogorov-Smirnov statistic, which is reported as an effect size
    rather than a p-value: with 10,000 draws per chain any p-value is
    essentially zero, whereas the statistic itself quantifies how far apart the
    empirical distributions are.
    """
    chain_ids = sorted(traces)
    rows = []
    for i, rxn in enumerate(reaction_ids):
        stats_ks = []
        for a in range(len(chain_ids)):
            for b in range(a + 1, len(chain_ids)):
                xa = traces[chain_ids[a]][i]
                xb = traces[chain_ids[b]][i]
                if np.ptp(np.concatenate([xa, xb])) <= TOL.invariant_range:
                    continue
                stats_ks.append(stats.ks_2samp(xa, xb).statistic)
        rows.append(
            {
                "rxn_id": rxn,
                "ks_mean": float(np.mean(stats_ks)) if stats_ks else np.nan,
                "ks_max": float(np.max(stats_ks)) if stats_ks else np.nan,
                "n_pairs": len(stats_ks),
            }
        )
    return pd.DataFrame(rows)


def running_mean_stability(trace: np.ndarray, checkpoints: tuple[float, ...] = (0.25, 0.5, 0.75, 1.0)) -> dict:
    """Relative change of the running mean at successive fractions of a chain."""
    trace = np.asarray(trace, dtype=float)
    n = trace.size
    means = {f: float(trace[: max(1, int(f * n))].mean()) for f in checkpoints}
    final = means[checkpoints[-1]]
    scale = np.abs(final) if abs(final) > TOL.flux_zero else 1.0
    return {
        **{f"mean_at_{int(f * 100)}pct": v for f, v in means.items()},
        "relative_drift": float(abs(means[checkpoints[-2]] - final) / scale),
    }

## 16. Variance stabilisation

Chooses identity, asinh or log per feature, **fitted on the healthy samples only** so that the transform cannot be tuned to the disease contrast it is later used to measure. Without this step two ratio features contribute 99.9% of the squared distance between any two phenotypes.

> Variance-stabilising transforms for the phenotype space.

In [ ]:
from dataclasses import dataclass

import numpy as np
import pandas as pd

__all__ = ["PhenotypeTransform", "tail_diagnostics"]

_SPAN_THRESHOLD = 100.0     # q99.9 / q00.1 above which a positive feature is logged
_TAIL_THRESHOLD = 50.0      # (q99.9 - median) / IQR above which tails are stabilised


def tail_diagnostics(features: pd.DataFrame) -> pd.DataFrame:
    """Per-feature tail behaviour, used both for QC and to choose transforms."""
    rows = []
    for name in features.columns:
        x = features[name].to_numpy(dtype=float)
        x = x[np.isfinite(x)]
        if x.size == 0:
            rows.append({"feature": name, "strictly_positive": False, "span_ratio": np.nan,
                         "upper_tail_ratio": np.nan, "lower_tail_ratio": np.nan})
            continue
        lo, med, hi = np.percentile(x, [0.1, 50, 99.9])
        q1, q3 = np.percentile(x, [25, 75])
        iqr = q3 - q1
        rows.append(
            {
                "feature": name,
                "strictly_positive": bool(x.min() > 0),
                # only meaningful for a strictly positive feature; for a
                # sign-changing one the tail ratios below are the test that
                # applies, and an undefined span must not force a transform
                "span_ratio": float(hi / lo) if lo > 0 else np.nan,
                "upper_tail_ratio": float((hi - med) / iqr) if iqr > 0 else np.inf,
                "lower_tail_ratio": float((med - lo) / iqr) if iqr > 0 else np.inf,
            }
        )
    return pd.DataFrame(rows)


@dataclass
class PhenotypeTransform:
    """Chooses and applies a per-feature variance-stabilising transform."""

    decisions: pd.DataFrame

    @classmethod
    def fit(cls, healthy: pd.DataFrame) -> "PhenotypeTransform":
        diagnostics = tail_diagnostics(healthy)
        transforms, scales, offsets = [], [], []
        for _, row in diagnostics.iterrows():
            x = healthy[row["feature"]].to_numpy(dtype=float)
            x = x[np.isfinite(x)]
            span = row["span_ratio"]
            heavy = (
                max(row["upper_tail_ratio"], row["lower_tail_ratio"]) > _TAIL_THRESHOLD
                or (np.isfinite(span) and span > _SPAN_THRESHOLD)
            )
            if not heavy or x.size == 0:
                transforms.append("identity")
                scales.append(1.0)
                offsets.append(0.0)
            elif row["strictly_positive"]:
                transforms.append("log")
                scales.append(1.0)
                offsets.append(0.0)
            else:
                iqr = float(np.subtract(*np.percentile(x, [75, 25])))
                scale = iqr / 1.349 if iqr > 0 else (float(np.std(x)) or 1.0)
                transforms.append("asinh")
                scales.append(scale)
                offsets.append(0.0)
        diagnostics["transform"] = transforms
        diagnostics["scale"] = scales
        diagnostics["offset"] = offsets
        return cls(decisions=diagnostics.set_index("feature"))

    def apply(self, features: pd.DataFrame) -> pd.DataFrame:
        out = {}
        for name in features.columns:
            x = features[name].to_numpy(dtype=float)
            if name not in self.decisions.index:
                out[name] = x
                continue
            kind = self.decisions.loc[name, "transform"]
            if kind == "log":
                # the floor keeps the transform finite for values that reach zero
                # in disease states even though the healthy set is strictly positive
                floor = np.nextafter(0, 1)
                out[name] = np.log(np.maximum(x, floor))
            elif kind == "asinh":
                scale = float(self.decisions.loc[name, "scale"])
                out[name] = np.arcsinh(x / scale)
            else:
                out[name] = x
        return pd.DataFrame(out, index=features.index)

    def summary(self) -> pd.DataFrame:
        return self.decisions.reset_index()[
            ["feature", "transform", "strictly_positive", "span_ratio",
             "upper_tail_ratio", "lower_tail_ratio", "scale"]
        ]

## 17. The quality-control cascade

Constancy, missingness, outliers, chain dependence and redundancy — and nothing else. Discrimination between conditions is deliberately not a selection criterion: no step of this cascade ever sees a condition label.

> Feature quality control and construction of the core phenotype space.

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
from scipy.cluster.hierarchy import fcluster, linkage
from scipy.spatial.distance import squareform


__all__ = [
    "numerical_quality",
    "chain_reproducibility",
    "redundancy_clusters",
    "group_discrimination",
    "benjamini_hochberg",
    "select_core_features",
]


def benjamini_hochberg(pvalues: np.ndarray) -> np.ndarray:
    """Benjamini-Hochberg adjusted p-values."""
    p = np.asarray(pvalues, dtype=float)
    finite = np.isfinite(p)
    out = np.full_like(p, np.nan)
    q = p[finite]
    if q.size == 0:
        return out
    order = np.argsort(q)
    ranked = q[order]
    n = ranked.size
    adjusted = ranked * n / np.arange(1, n + 1)
    adjusted = np.minimum.accumulate(adjusted[::-1])[::-1]
    result = np.empty_like(adjusted)
    result[order] = np.clip(adjusted, 0, 1)
    out[finite] = result
    return out


def numerical_quality(features: pd.DataFrame) -> pd.DataFrame:
    """Per-feature numerical health: missingness, constancy, tail behaviour."""
    rows = []
    for name in features.columns:
        x = features[name].to_numpy(dtype=float)
        finite = np.isfinite(x)
        xf = x[finite]
        if xf.size == 0:
            rows.append({"feature": name, "n_finite": 0, "fraction_missing": 1.0,
                         "mean": np.nan, "sd": np.nan, "cv": np.nan, "range": np.nan,
                         "constant": True, "fraction_outliers": np.nan})
            continue
        mean, sd = float(xf.mean()), float(xf.std(ddof=1)) if xf.size > 1 else 0.0
        scale = abs(mean) if abs(mean) > TOL.flux_zero else 1.0
        q1, q3 = np.percentile(xf, [25, 75])
        iqr = q3 - q1
        outliers = np.mean((xf < q1 - 3 * iqr) | (xf > q3 + 3 * iqr)) if iqr > 0 else 0.0
        rows.append(
            {
                "feature": name,
                "n_finite": int(finite.sum()),
                "fraction_missing": float(1.0 - finite.mean()),
                "mean": mean,
                "sd": sd,
                "cv": float(sd / scale),
                "range": float(np.ptp(xf)),
                "constant": bool(np.ptp(xf) <= TOL.invariant_range or sd / scale < TOL.min_cv),
                "fraction_outliers": float(outliers),
            }
        )
    return pd.DataFrame(rows)


def chain_reproducibility(
    features: pd.DataFrame, metadata: pd.DataFrame, *, group: str = "condition"
) -> pd.DataFrame:
    """How much of a feature's variance is chain identity rather than biology.

    For every condition the variance is decomposed into a between-chain and a
    within-chain component. A feature whose value depends appreciably on which
    chain produced the sample is measuring the sampler, not the metabolism, and
    is removed from the core space.
    """
    rows = []
    keys = metadata[group].to_numpy()
    chains = metadata["chain"].to_numpy()
    for name in features.columns:
        x = features[name].to_numpy(dtype=float)
        ratios, icc_values = [], []
        for key in np.unique(keys):
            mask = keys == key
            xv, cv = x[mask], chains[mask]
            if not np.isfinite(xv).all() or np.ptp(xv[np.isfinite(xv)]) <= 0:
                continue
            groups = [xv[cv == c] for c in np.unique(cv)]
            groups = [g[np.isfinite(g)] for g in groups]
            groups = [g for g in groups if g.size > 1]
            if len(groups) < 2:
                continue
            grand = np.concatenate(groups)
            n_per = np.array([g.size for g in groups])
            means = np.array([g.mean() for g in groups])
            ss_between = float((n_per * (means - grand.mean()) ** 2).sum())
            ss_within = float(sum(((g - g.mean()) ** 2).sum() for g in groups))
            total = ss_between + ss_within
            if total <= 0:
                continue
            ratios.append(ss_between / total)
            df_between = len(groups) - 1
            df_within = grand.size - len(groups)
            ms_between = ss_between / df_between
            ms_within = ss_within / df_within if df_within > 0 else np.nan
            n0 = (grand.size - (n_per ** 2).sum() / grand.size) / df_between
            var_between = max((ms_between - ms_within) / n0, 0.0)
            icc_values.append(var_between / (var_between + ms_within)
                              if (var_between + ms_within) > 0 else 0.0)
        rows.append(
            {
                "feature": name,
                "between_chain_variance_share": float(np.mean(ratios)) if ratios else np.nan,
                "between_chain_variance_share_max": float(np.max(ratios)) if ratios else np.nan,
                "chain_icc": float(np.mean(icc_values)) if icc_values else np.nan,
            }
        )
    return pd.DataFrame(rows)


def redundancy_clusters(
    features: pd.DataFrame, *, threshold: float = TOL.redundancy_rho, sample_size: int = 6000,
    seed: int = 0,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Cluster features by absolute Spearman correlation and pick a representative.

    Returns the cluster assignment and the pairwise correlation matrix used.
    Representatives are chosen as the feature with the highest mean absolute
    correlation to the rest of its cluster, i.e. the most central member.
    """
    rng = np.random.default_rng(seed)
    n = len(features)
    idx = rng.choice(n, size=min(sample_size, n), replace=False)
    X = features.iloc[idx].to_numpy(dtype=float)
    keep = np.isfinite(X).all(axis=0) & (np.nanstd(X, axis=0) > 0)
    names = features.columns[keep].to_numpy()
    X = X[:, keep]

    rho = stats.spearmanr(X).statistic
    rho = np.atleast_2d(rho)
    if rho.shape[0] != len(names):  # spearmanr returns a scalar for 2 columns
        rho = np.corrcoef(stats.rankdata(X, axis=0), rowvar=False)
    distance = 1.0 - np.abs(np.nan_to_num(rho, nan=0.0))
    np.fill_diagonal(distance, 0.0)
    distance = (distance + distance.T) / 2.0

    linkage_matrix = linkage(squareform(distance, checks=False), method="average")
    labels = fcluster(linkage_matrix, t=1.0 - threshold, criterion="distance")

    rows = []
    for cluster in np.unique(labels):
        members = np.flatnonzero(labels == cluster)
        if members.size == 1:
            representative = names[members[0]]
        else:
            sub = np.abs(rho[np.ix_(members, members)])
            np.fill_diagonal(sub, 0.0)
            representative = names[members[np.argmax(sub.mean(axis=1))]]
        for m in members:
            rows.append({"feature": names[m], "cluster": int(cluster),
                         "cluster_size": int(members.size),
                         "representative": representative,
                         "is_representative": names[m] == representative})
    correlation = pd.DataFrame(rho, index=names, columns=names)
    return pd.DataFrame(rows), correlation


def _epsilon_squared(groups: list[np.ndarray]) -> float:
    """Effect size for the Kruskal-Wallis test."""
    n = sum(g.size for g in groups)
    if n <= len(groups):
        return np.nan
    H = stats.kruskal(*groups).statistic
    return float((H - len(groups) + 1) / (n - len(groups)))


def group_discrimination(
    features: pd.DataFrame, labels: np.ndarray, *, label_name: str = "group"
) -> pd.DataFrame:
    """Non-parametric test of whether a feature separates the given groups.

    Kruskal-Wallis is used rather than one-way ANOVA because most flux-derived
    features are strongly non-normal, and the accompanying epsilon-squared
    effect size is reported alongside the FDR-adjusted p-value so that
    significance driven purely by the large sample size can be recognised.
    """
    rows = []
    unique = np.unique(labels)
    for name in features.columns:
        x = features[name].to_numpy(dtype=float)
        groups = [x[labels == u] for u in unique]
        groups = [g[np.isfinite(g)] for g in groups]
        groups = [g for g in groups if g.size > 1 and np.ptp(g) > 0]
        if len(groups) < 2:
            rows.append({"feature": name, "axis": label_name, "statistic": np.nan,
                         "p_value": np.nan, "epsilon_squared": np.nan})
            continue
        result = stats.kruskal(*groups)
        rows.append(
            {
                "feature": name,
                "axis": label_name,
                "statistic": float(result.statistic),
                "p_value": float(result.pvalue),
                "epsilon_squared": _epsilon_squared(groups),
            }
        )
    frame = pd.DataFrame(rows)
    frame["p_adjusted"] = benjamini_hochberg(frame["p_value"].to_numpy())
    return frame


def select_core_features(
    quality: pd.DataFrame,
    reproducibility: pd.DataFrame,
    clusters: pd.DataFrame,
    *,
    max_missing: float = 0.01,
    max_chain_share: float = 0.02,
    max_outliers: float = 0.10,
    without_healthy_unit: frozenset = frozenset(),
) -> pd.DataFrame:
    """Apply the QC cascade and return the per-feature decision table."""
    table = quality.merge(reproducibility, on="feature", how="left")
    table = table.merge(
        clusters[["feature", "cluster", "cluster_size", "representative", "is_representative"]],
        on="feature", how="left",
    )
    table["fail_missing"] = (table["fraction_missing"] > max_missing).astype(bool)
    table["fail_constant"] = table["constant"].fillna(True).astype(bool)
    table["fail_outliers"] = (table["fraction_outliers"] > max_outliers).astype(bool)
    table["fail_chain_dependence"] = (
        table["between_chain_variance_share"] > max_chain_share
    ).astype(bool)
    # The core phenotype is the space in which distance from health is
    # measured, and that distance is expressed in units of healthy variation.
    # A descriptor whose spread over the healthy states is negligible against
    # its own range has no such unit: dividing by it makes one coordinate
    # carry the whole distance, and the shrunk precision matrix then gives it
    # a weight no amount of rescaling removes. It is not constant, so it is
    # not caught above, and it remains in the registry and in the effect
    # sizes, which need no healthy unit — it is excluded from the core.
    table["fail_no_healthy_unit"] = table["feature"].isin(without_healthy_unit)
    # A feature absent from the clustering (constant or missing) is not a
    # cluster representative, so it fails redundancy as well; the earlier flags
    # record the primary reason.
    table["fail_redundant"] = ~table["is_representative"].fillna(False).astype(bool)

    fail_columns = [c for c in table.columns if c.startswith("fail_")]
    table["retained"] = ~table[fail_columns].any(axis=1)
    table["exclusion_reason"] = table[fail_columns].apply(
        lambda row: ";".join(c.removeprefix("fail_") for c in fail_columns if row[c]) or "",
        axis=1,
    )
    return table

## 18. Health distance

The healthy reference and the candidate distances computed against it: Euclidean, Manhattan, Chebyshev, Mahalanobis with Ledoit–Wolf shrinkage, cosine, a PCA latent distance, and the multivariate energy distance.

> Healthy references and the health distance.

In [ ]:
from dataclasses import dataclass, field

import numpy as np
import pandas as pd
from scipy import stats
from sklearn.covariance import LedoitWolf
from sklearn.decomposition import PCA
from sklearn.metrics import roc_auc_score

__all__ = ["HealthyReference", "DistanceSet", "evaluate_metrics", "energy_distance"]


@dataclass
class HealthyReference:
    """Robust standardisation and covariance structure of the healthy phenotype."""

    features: list[str]
    centre: np.ndarray = field(init=False)
    scale: np.ndarray = field(init=False)
    precision: np.ndarray = field(init=False)
    pca: PCA = field(init=False)
    _healthy_z: np.ndarray = field(init=False, repr=False)

    def __init__(self, healthy: pd.DataFrame, *, n_components: int = 10, seed: int = 0):
        self.features = list(healthy.columns)
        X = healthy.to_numpy(dtype=float)
        self.centre = np.median(X, axis=0)
        iqr = np.subtract(*np.percentile(X, [75, 25], axis=0))
        sd = X.std(axis=0, ddof=1)
        # A descriptor constant in health still varies in disease, so the
        # standard deviation is the right fallback — but only while that
        # standard deviation means something. A healthy sd of 1.8e-15 on a
        # descriptor whose own range is 3.6e-4 is a rounding error, and
        # dividing by it gave that one coordinate 100% of every squared health
        # distance and collapsed every partial correlation to zero. When the
        # healthy spread is negligible against the descriptor's own range, the
        # range is used as the unit instead: the descriptor keeps its place in
        # the geometry and contributes at most one unit of distance, rather
        # than deciding it alone.
        spread = np.nanmax(X, axis=0) - np.nanmin(X, axis=0)
        self.scale = np.where(iqr > 0, iqr / 1.349, np.where(sd > 0, sd, np.nan))
        floor = 1e-9 * np.where(spread > 0, spread, 1.0)
        unusable = ~(np.isfinite(self.scale) & (self.scale >= floor))
        if unusable.any():
            names = [f for f, bad in zip(self.features, unusable) if bad]
            print(f"  healthy reference: {len(names)} descriptors have no usable spread "
                  f"in the healthy states and are scaled by their own range instead: "
                  f"{', '.join(names)}")
            fallback = np.where(spread > 0, spread, 1.0)
            self.scale = np.where(unusable, fallback, self.scale)
        Z = (X - self.centre) / self.scale
        self._healthy_z = Z
        self.precision = LedoitWolf(assume_centered=True).fit(Z).precision_
        self.pca = PCA(n_components=min(n_components, Z.shape[1]), random_state=seed).fit(Z)

    # -- transforms -----------------------------------------------------
    def standardise(self, features: pd.DataFrame) -> np.ndarray:
        X = features[self.features].to_numpy(dtype=float)
        return (X - self.centre) / self.scale

    def project(self, features: pd.DataFrame) -> np.ndarray:
        return self.pca.transform(self.standardise(features))

    # -- distances ------------------------------------------------------
    def euclidean(self, features: pd.DataFrame) -> np.ndarray:
        Z = self.standardise(features)
        return np.sqrt((Z ** 2).sum(axis=1))

    def manhattan(self, features: pd.DataFrame) -> np.ndarray:
        return np.abs(self.standardise(features)).sum(axis=1)

    def chebyshev(self, features: pd.DataFrame) -> np.ndarray:
        return np.abs(self.standardise(features)).max(axis=1)

    def mahalanobis(self, features: pd.DataFrame) -> np.ndarray:
        Z = self.standardise(features)
        return np.sqrt(np.einsum("ij,jk,ik->i", Z, self.precision, Z, optimize=True))

    def pca_distance(self, features: pd.DataFrame) -> np.ndarray:
        P = self.project(features)
        healthy_centre = self.pca.transform(self._healthy_z).mean(axis=0)
        return np.sqrt(((P - healthy_centre) ** 2).sum(axis=1))

    def cosine(self, features: pd.DataFrame) -> np.ndarray:
        Z = self.standardise(features)
        reference = self._healthy_z.mean(axis=0)
        norms = np.linalg.norm(Z, axis=1) * np.linalg.norm(reference)
        with np.errstate(divide="ignore", invalid="ignore"):
            similarity = np.where(norms > 0, Z @ reference / norms, 0.0)
        return 1.0 - similarity

    def to_point(self, point: pd.Series) -> dict[str, float]:
        """Distances of a single reference phenotype (e.g. the FBA point)."""
        frame = point[self.features].to_frame().T.astype(float)
        return {
            "euclidean": float(self.euclidean(frame)[0]),
            "mahalanobis": float(self.mahalanobis(frame)[0]),
            "pca": float(self.pca_distance(frame)[0]),
            "cosine": float(self.cosine(frame)[0]),
        }


def point_distances(
    features: pd.DataFrame, reference_point: pd.Series, scale: np.ndarray
) -> dict[str, np.ndarray]:
    """Distances of every sample to one fixed reference phenotype."""
    X = features.to_numpy(dtype=float)
    r = reference_point[features.columns].to_numpy(dtype=float)
    Z = (X - r) / scale
    norms = np.linalg.norm(X, axis=1) * np.linalg.norm(r)
    with np.errstate(divide="ignore", invalid="ignore"):
        cosine = 1.0 - np.where(norms > 0, X @ r / norms, 0.0)
    return {
        "euclidean": np.sqrt((Z ** 2).sum(axis=1)),
        "manhattan": np.abs(Z).sum(axis=1),
        "cosine": cosine,
    }


def raw_flux_distances(V: np.ndarray, reference: np.ndarray) -> dict[str, np.ndarray]:
    """Distances computed directly in the 3275-dimensional flux space."""
    D = V - reference
    norms = np.linalg.norm(V, axis=1) * np.linalg.norm(reference)
    with np.errstate(divide="ignore", invalid="ignore"):
        cosine = 1.0 - np.where(norms > 0, V @ reference / norms, 0.0)
    return {
        "flux_euclidean": np.sqrt((D ** 2).sum(axis=1)),
        "flux_manhattan": np.abs(D).sum(axis=1),
        "flux_cosine": cosine,
    }


def energy_distance(a: np.ndarray, b: np.ndarray, *, seed: int = 0, n_max: int = 1500) -> float:
    """Multivariate energy distance between two samples of feature vectors."""
    rng = np.random.default_rng(seed)
    A = a[rng.choice(len(a), size=min(n_max, len(a)), replace=False)]
    B = b[rng.choice(len(b), size=min(n_max, len(b)), replace=False)]

    def mean_pairwise(X, Y):
        return float(np.sqrt(((X[:, None, :] - Y[None, :, :]) ** 2).sum(axis=2)).mean())

    return 2 * mean_pairwise(A, B) - mean_pairwise(A, A) - mean_pairwise(B, B)


@dataclass
class DistanceSet:
    """A named collection of per-sample health distances."""

    values: pd.DataFrame
    metadata: pd.DataFrame


def evaluate_metrics(distances: pd.DataFrame, metadata: pd.DataFrame) -> pd.DataFrame:
    """Score each candidate distance against the four criteria of stage 7.

    The criteria are, in order of importance:

    1. **severity recovery** - Spearman correlation with the known G6PD
       perturbation magnitude, which should be strongly negative (lower enzyme
       capacity, larger distance);
    2. **health separation** - AUC for healthy versus disease;
    3. **null-axis inertness** - effect size on the H2O2 axis, which should be
       indistinguishable from zero;
    4. **reproducibility** - share of within-condition variance attributable to
       chain identity.
    """
    severity = metadata["g6pd_level"].to_numpy(dtype=float)
    is_disease = (metadata["state"] == "disease").to_numpy()
    rows = []
    for name in distances.columns:
        d = distances[name].to_numpy(dtype=float)
        ok = np.isfinite(d)
        # A metric anchored on a reference point with undefined features (a
        # ratio with a vanishing denominator, say) cannot be scored; it is
        # reported as unusable rather than silently dropped.
        if ok.sum() < 0.5 * len(d) or not (is_disease[ok].any() and (~is_disease[ok]).any()):
            rows.append({"metric": name, "usable": False,
                         "n_finite": int(ok.sum()), "selection_score": -np.inf})
            continue
        spearman = stats.spearmanr(severity[ok], d[ok]).statistic
        auc = roc_auc_score(is_disease[ok], d[ok])

        # null axis: Kruskal-Wallis across H2O2 levels within disease states
        groups = [
            d[is_disease & (metadata["h2o2_level"].to_numpy() == level) & ok]
            for level in np.unique(metadata.loc[is_disease, "h2o2_level"])
        ]
        kruskal = stats.kruskal(*groups)
        n_total = sum(g.size for g in groups)
        epsilon_sq = (kruskal.statistic - len(groups) + 1) / (n_total - len(groups))

        # chain reproducibility within condition
        shares = []
        for condition in metadata["condition"].unique():
            mask = (metadata["condition"] == condition).to_numpy() & ok
            sub, chains = d[mask], metadata.loc[mask, "chain"].to_numpy()
            grand = sub.mean()
            between = sum(
                (chains == c).sum() * (sub[chains == c].mean() - grand) ** 2
                for c in np.unique(chains)
            )
            within = sum(((sub[chains == c] - sub[chains == c].mean()) ** 2).sum()
                         for c in np.unique(chains))
            if between + within > 0:
                shares.append(between / (between + within))

        # monotonicity across the four ordered severity levels
        level_means = [d[ok & (severity == lv)].mean() for lv in sorted(np.unique(severity))]
        monotone = bool(all(np.diff(level_means) < 0))

        rows.append(
            {
                "metric": name,
                "usable": True,
                "n_finite": int(ok.sum()),
                "spearman_with_g6pd_level": float(spearman),
                "healthy_vs_disease_auc": float(auc),
                "null_axis_epsilon_squared": float(epsilon_sq),
                "null_axis_p_value": float(kruskal.pvalue),
                "between_chain_variance_share": float(np.mean(shares)) if shares else np.nan,
                "severity_monotone": monotone,
                "level_means": ";".join(f"{m:.4g}" for m in level_means),
            }
        )
    frame = pd.DataFrame(rows)
    scored = frame["usable"].fillna(False).astype(bool)
    # a single composite score, stated explicitly rather than chosen by eye
    frame.loc[scored, "selection_score"] = (
        frame["spearman_with_g6pd_level"].abs()
        + (frame["healthy_vs_disease_auc"] - 0.5).abs() * 2
        - frame["null_axis_epsilon_squared"] * 10
        - frame["between_chain_variance_share"].fillna(1.0) * 10
        + frame["severity_monotone"].astype(float) * 0.5
    )[scored]
    return frame.sort_values("selection_score", ascending=False, ignore_index=True)

## 19. The model ladder

Six regressors from a mean baseline to gradient boosting, always with folds grouped by MCMC chain. Machine learning appears in exactly one place in this study, and its job there is to fail to find a relationship.

> Interpretable models linking the external phenotype to the health distance.

In [ ]:
from dataclasses import dataclass

import numpy as np
import pandas as pd
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.inspection import permutation_importance
from sklearn.linear_model import ElasticNetCV, RidgeCV
from sklearn.model_selection import GroupKFold, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeRegressor, export_text

__all__ = ["build_models", "evaluate_models", "fit_best", "surrogate_rules"]


def build_models(seed: int = 0) -> dict[str, object]:
    """The model ladder, from fully transparent to fully flexible."""
    return {
        "mean_baseline": DummyRegressor(strategy="mean"),
        "ridge": Pipeline([("scale", StandardScaler()),
                           ("model", RidgeCV(alphas=np.logspace(-3, 3, 13)))]),
        "elastic_net": Pipeline([("scale", StandardScaler()),
                                 ("model", ElasticNetCV(l1_ratio=[0.1, 0.5, 0.9, 1.0],
                                                        alphas=20, max_iter=2000, tol=1e-3,
                                                        random_state=seed))]),
        "decision_tree": DecisionTreeRegressor(max_depth=4, min_samples_leaf=200,
                                               random_state=seed),
        "random_forest": RandomForestRegressor(n_estimators=200, min_samples_leaf=10,
                                               n_jobs=-1, random_state=seed),
        "gradient_boosting": HistGradientBoostingRegressor(max_depth=6, max_iter=300,
                                                           learning_rate=0.08,
                                                           random_state=seed),
    }


@dataclass
class ModelScore:
    name: str
    r2: float
    spearman: float
    rmse: float


def _score(y: np.ndarray, pred: np.ndarray) -> dict:
    from scipy import stats

    residual = y - pred
    ss_res = float((residual ** 2).sum())
    ss_tot = float(((y - y.mean()) ** 2).sum())
    return {
        "r2": 1.0 - ss_res / ss_tot if ss_tot > 0 else np.nan,
        "spearman": float(stats.spearmanr(y, pred).statistic),
        "rmse": float(np.sqrt((residual ** 2).mean())),
    }


def evaluate_models(
    X: pd.DataFrame,
    y: np.ndarray,
    groups: np.ndarray,
    *,
    seed: int = 0,
    n_splits: int = 5,
) -> pd.DataFrame:
    """Grouped cross-validation of the whole model ladder.

    Folds are grouped so that all samples from one MCMC chain stay together.
    Samples within a chain are not independent, and an ungrouped split would
    leak that dependence into the test fold and inflate every score.
    """
    n_groups = len(np.unique(groups))
    splitter = GroupKFold(n_splits=min(n_splits, n_groups))
    rows = []
    for name, model in build_models(seed).items():
        pred = cross_val_predict(model, X, y, cv=splitter, groups=groups, n_jobs=1)
        rows.append({"model": name, "n_samples": len(y), "n_features": X.shape[1],
                     "n_groups": n_groups, **_score(y, pred)})
    return pd.DataFrame(rows).sort_values("r2", ascending=False, ignore_index=True)


def fit_best(
    X: pd.DataFrame, y: np.ndarray, groups: np.ndarray, *, name: str, seed: int = 0
) -> object:
    model = build_models(seed)[name]
    model.fit(X, y)
    return model


def model_importances(
    model, X: pd.DataFrame, y: np.ndarray, *, seed: int = 0, n_repeats: int = 10,
    subsample: int = 4000,
) -> pd.DataFrame:
    """Permutation importance, computed on a held-out subsample."""
    rng = np.random.default_rng(seed)
    idx = rng.choice(len(X), size=min(subsample, len(X)), replace=False)
    result = permutation_importance(
        model, X.iloc[idx], y[idx], n_repeats=n_repeats, random_state=seed, n_jobs=-1
    )
    return pd.DataFrame(
        {
            "feature": X.columns,
            "permutation_importance": result.importances_mean,
            "permutation_importance_sd": result.importances_std,
        }
    ).sort_values("permutation_importance", ascending=False, ignore_index=True)


def surrogate_rules(X: pd.DataFrame, y: np.ndarray, *, max_depth: int = 3, seed: int = 0) -> str:
    """A shallow decision tree fitted to the predictions, printed as rules."""
    tree = DecisionTreeRegressor(max_depth=max_depth, min_samples_leaf=300, random_state=seed)
    tree.fit(X, y)
    return export_text(tree, feature_names=list(X.columns), decimals=3)

## 20. Shapley values

A model-agnostic Monte-Carlo interventional Shapley implementation, used only to interpret the null result of the boundary analysis.

> Monte-Carlo estimation of Shapley values for any fitted regressor.

In [ ]:
import numpy as np
import pandas as pd

__all__ = ["shapley_values", "global_shapley_importance"]


def shapley_values(
    model,
    X: pd.DataFrame,
    background: pd.DataFrame,
    *,
    n_permutations: int = 60,
    n_background: int = 40,
    seed: int = 0,
) -> tuple[np.ndarray, np.ndarray]:
    """Interventional Shapley values for each row of ``X``.

    Returns ``(values, standard_errors)``, both of shape ``X.shape``.

    The cost is ``n_permutations * n_background * n_features`` model
    evaluations per explained instance, so ``X`` should be a modest sample of
    the data rather than the whole dataset.
    """
    rng = np.random.default_rng(seed)
    features = list(X.columns)
    n_features = len(features)
    Xv = X.to_numpy(dtype=float)
    B = background.to_numpy(dtype=float)
    background_idx = rng.choice(len(B), size=min(n_background, len(B)), replace=False)
    B = B[background_idx]

    totals = np.zeros((len(Xv), n_features))
    squares = np.zeros((len(Xv), n_features))

    for _ in range(n_permutations):
        order = rng.permutation(n_features)
        b = B[rng.integers(len(B))]
        # start from the background instance and reveal features in `order`
        current = np.tile(b, (len(Xv), 1))
        previous = model.predict(pd.DataFrame(current, columns=features))
        for position in order:
            current[:, position] = Xv[:, position]
            updated = model.predict(pd.DataFrame(current, columns=features))
            delta = updated - previous
            totals[:, position] += delta
            squares[:, position] += delta ** 2
            previous = updated

    mean = totals / n_permutations
    variance = np.maximum(squares / n_permutations - mean ** 2, 0.0)
    stderr = np.sqrt(variance / n_permutations)
    return mean, stderr


def global_shapley_importance(
    model,
    X: pd.DataFrame,
    background: pd.DataFrame,
    *,
    n_explain: int = 250,
    seed: int = 0,
    **kwargs,
) -> pd.DataFrame:
    """Mean absolute Shapley value per feature, with Monte-Carlo error."""
    rng = np.random.default_rng(seed)
    idx = rng.choice(len(X), size=min(n_explain, len(X)), replace=False)
    subset = X.iloc[idx]
    values, stderr = shapley_values(model, subset, background, seed=seed, **kwargs)
    frame = pd.DataFrame(
        {
            "feature": X.columns,
            "mean_abs_shapley": np.abs(values).mean(axis=0),
            "mean_shapley": values.mean(axis=0),
            "monte_carlo_se": stderr.mean(axis=0),
        }
    )
    frame["signal_to_noise"] = frame["mean_abs_shapley"] / frame["monte_carlo_se"].replace(0, np.nan)
    return frame.sort_values("mean_abs_shapley", ascending=False, ignore_index=True)

## 21. Linear programming

FBA, parsimonious FBA and flux variability through HiGHS, used to construct the candidate healthy references and to test whether an imposed bound is attainable.

> Linear programming over the constraint-based model.

In [ ]:
from dataclasses import dataclass

import numpy as np
import scipy.sparse as sp
from scipy.optimize import linprog


__all__ = ["LPResult", "fba", "pfba", "flux_variability", "bound_is_binding"]


@dataclass
class LPResult:
    status: int
    success: bool
    objective: float
    v: np.ndarray | None
    message: str = ""


def _bounds(gem: GEM, overrides: dict[str, tuple[float | None, float | None]] | None):
    lb, ub = gem.lb.copy(), gem.ub.copy()
    for rxn, (low, high) in (overrides or {}).items():
        j = gem.rxn_index(rxn)
        if low is not None:
            lb[j] = low
        if high is not None:
            ub[j] = high
    return lb, ub


def _solve(A_eq, b_eq, c, lb, ub, A_ub=None, b_ub=None) -> LPResult:
    res = linprog(
        c=c,
        A_eq=A_eq,
        b_eq=b_eq,
        A_ub=A_ub,
        b_ub=b_ub,
        bounds=np.column_stack([lb, ub]),
        method="highs",
    )
    return LPResult(
        status=res.status,
        success=bool(res.success),
        objective=float(res.fun) if res.success else float("nan"),
        v=np.asarray(res.x, dtype=float) if res.success else None,
        message=res.message,
    )


def fba(
    gem: GEM,
    *,
    objective: str | None = None,
    maximize: bool = True,
    bounds: dict[str, tuple[float | None, float | None]] | None = None,
) -> LPResult:
    """Standard flux balance analysis."""
    lb, ub = _bounds(gem, bounds)
    c = np.zeros(gem.n_reactions)
    if objective is None:
        c[:] = gem.c
    else:
        c[gem.rxn_index(objective)] = 1.0
    sign = -1.0 if maximize else 1.0
    result = _solve(gem.S, gem.b, sign * c, lb, ub)
    if result.success:
        result.objective *= sign
    return result


def pfba(
    gem: GEM,
    *,
    objective: str | None = None,
    fraction_of_optimum: float = 1.0,
    bounds: dict[str, tuple[float | None, float | None]] | None = None,
) -> LPResult:
    """Parsimonious FBA: minimise total absolute flux at (near-)optimal objective.

    The plain FBA optimum of a model with generic bounds is a highly degenerate
    vertex whose flux values are dominated by thermodynamically infeasible
    cycles running at the arbitrary bound. Minimising the L1 norm subject to the
    optimal objective removes those cycles and returns a reference state that is
    both unique in practice and interpretable, which is what makes it a usable
    definition of "the healthy state".
    """
    base = fba(gem, objective=objective, bounds=bounds)
    if not base.success:
        return base

    lb, ub = _bounds(gem, bounds)
    n = gem.n_reactions
    c_obj = np.zeros(n)
    if objective is None:
        c_obj[:] = gem.c
    else:
        c_obj[gem.rxn_index(objective)] = 1.0

    # split v = v+ - v-, minimise sum(v+ + v-)
    S2 = sp.hstack([sp.csr_matrix(gem.S), -sp.csr_matrix(gem.S)], format="csr")
    b_eq = np.concatenate([np.atleast_1d(gem.b).ravel()])
    obj_row = sp.csr_matrix(np.concatenate([c_obj, -c_obj]).reshape(1, -1))
    A_eq = sp.vstack([S2, obj_row], format="csr")
    b_eq = np.concatenate([b_eq, [fraction_of_optimum * base.objective]])

    lb2 = np.concatenate([np.maximum(lb, 0.0), np.maximum(-ub, 0.0)])
    ub2 = np.concatenate([np.maximum(ub, 0.0), np.maximum(-lb, 0.0)])
    c2 = np.ones(2 * n)

    res = _solve(A_eq, b_eq, c2, lb2, ub2)
    if not res.success:
        return LPResult(res.status, False, float("nan"), None, "pFBA infeasible: " + res.message)
    v = res.v[:n] - res.v[n:]
    return LPResult(res.status, True, base.objective, v, "pFBA")


def flux_variability(
    gem: GEM,
    reactions: list[str],
    *,
    bounds: dict[str, tuple[float | None, float | None]] | None = None,
    fraction_of_optimum: float = 0.0,
) -> dict[str, tuple[float, float]]:
    """Minimum and maximum flux of selected reactions over the feasible set."""
    lb, ub = _bounds(gem, bounds)
    A_eq, b_eq = gem.S, gem.b
    if fraction_of_optimum > 0:
        base = fba(gem, bounds=bounds)
        A_eq = sp.vstack([sp.csr_matrix(gem.S), sp.csr_matrix(gem.c.reshape(1, -1))], format="csr")
        b_eq = np.concatenate([np.atleast_1d(gem.b).ravel(),
                               [fraction_of_optimum * base.objective]])

    out = {}
    for rxn in reactions:
        j = gem.rxn_index(rxn)
        c = np.zeros(gem.n_reactions)
        c[j] = 1.0
        lo = _solve(A_eq, b_eq, c, lb, ub)
        hi = _solve(A_eq, b_eq, -c, lb, ub)
        out[rxn] = (
            lo.objective if lo.success else float("nan"),
            -hi.objective if hi.success else float("nan"),
        )
    return out


def bound_is_binding(
    gem: GEM,
    reaction: str,
    *,
    which: str = "upper",
    bounds: dict[str, tuple[float | None, float | None]] | None = None,
    tolerance: float = 1e-6,
) -> dict:
    """Test whether a bound can be attained anywhere in the feasible set.

    A constraint that no feasible point can reach does not perturb the model at
    all. Establishing this by optimisation converts an empirical observation
    ("the sampler never got close") into a statement about the polytope.
    """
    lb, ub = _bounds(gem, bounds)
    j = gem.rxn_index(reaction)
    relaxed = dict(bounds or {})
    # relax the bound under test so the attainable extreme is measured, not the cap
    relaxed[reaction] = (None, 1e6) if which == "upper" else (-1e6, None)
    lb_r, ub_r = _bounds(gem, relaxed)

    c = np.zeros(gem.n_reactions)
    c[j] = -1.0 if which == "upper" else 1.0
    res = _solve(gem.S, gem.b, c, lb_r, ub_r)
    if not res.success:
        return {"reaction": reaction, "which": which, "feasible": False}
    extreme = -res.objective if which == "upper" else res.objective
    limit = ub[j] if which == "upper" else lb[j]
    binding = (extreme >= limit - tolerance) if which == "upper" else (extreme <= limit + tolerance)
    return {
        "reaction": reaction,
        "which": which,
        "feasible": True,
        "bound": float(limit),
        "attainable_extreme": float(extreme),
        "slack": float(limit - extreme) if which == "upper" else float(extreme - limit),
        "binding": bool(binding),
    }

## 22. The intervention catalogue

Supplementation, restriction and export block at three quantile levels of the healthy flux, over every exchange reaction that supports a level — and the utility function that penalises toxicity and non-modifiability.

> Construction and scoring of a biologically admissible intervention catalogue.

In [ ]:
from dataclasses import dataclass, field

import numpy as np
import pandas as pd

__all__ = ["Intervention", "build_catalogue", "utility_score"]


@dataclass(frozen=True)
class Intervention:
    """A named tightening of the exchange bounds of a model."""

    intervention_id: str
    kind: str                  # supplementation | restriction | export_block | combination
    targets: tuple[str, ...]   # reaction identifiers
    description: str
    bounds: dict[str, tuple[float | None, float | None]]
    metabolites: tuple[str, ...] = ()
    roles: tuple[str, ...] = ()
    max_toxicity: float = 0.0
    magnitude: float = 0.0

    def apply(self, lb: np.ndarray, ub: np.ndarray, index: dict[str, int]
              ) -> tuple[np.ndarray, np.ndarray]:
        """Return tightened copies of the bound vectors."""
        lb, ub = lb.copy(), ub.copy()
        for rxn, (low, high) in self.bounds.items():
            j = index[rxn]
            if low is not None:
                lb[j] = max(lb[j], low)
            if high is not None:
                ub[j] = min(ub[j], high)
        return lb, ub


def build_catalogue(
    reactions: pd.DataFrame,
    healthy_samples: np.ndarray,
    *,
    quantiles: tuple[float, ...] = (0.25, 0.50, 0.75),
    include_non_modifiable: bool = True,
) -> list[Intervention]:
    """Enumerate candidate interventions on clinically relevant exchanges.

    Bound levels are taken from quantiles of the *healthy* sampled flux
    distribution for the same reaction, so that every candidate is expressed in
    units the model itself produces and none of them demands a flux the healthy
    cell never achieves. Non-modifiable and toxic exchanges are included by
    default but flagged, because the value of the admissibility layer can only
    be demonstrated if inadmissible candidates are actually generated and then
    seen to be rejected.
    """
    catalogue: list[Intervention] = []
    exchanges = reactions[reactions["reaction_class"] == "exchange"]

    for _, row in exchanges.iterrows():
        rxn = row["rxn_id"]
        met = row["boundary_metabolite"]
        role = row["boundary_role"]
        toxicity = float(row["toxicity_weight"]) if np.isfinite(row["toxicity_weight"]) else 0.4
        if not include_non_modifiable and not row["clinically_modifiable"]:
            continue

        j = int(row.name)
        column = healthy_samples[:, j]
        if np.ptp(column) <= 0:
            continue

        uptake = -column[column < 0]
        secretion = column[column > 0]

        for q in quantiles:
            if uptake.size > 50:
                level = float(np.quantile(uptake, q))
                if level > 0:
                    # force at least `level` of uptake: v <= -level
                    catalogue.append(Intervention(
                        intervention_id=f"supplement::{met}::q{int(q*100)}",
                        kind="supplementation",
                        targets=(rxn,),
                        description=f"force uptake of {met} to at least {level:.3g}",
                        bounds={rxn: (None, -level)},
                        metabolites=(met,), roles=(role,),
                        max_toxicity=toxicity, magnitude=level,
                    ))
                    # cap uptake at `level`: v >= -level
                    catalogue.append(Intervention(
                        intervention_id=f"restrict::{met}::q{int(q*100)}",
                        kind="restriction",
                        targets=(rxn,),
                        description=f"cap uptake of {met} at {level:.3g}",
                        bounds={rxn: (-level, None)},
                        metabolites=(met,), roles=(role,),
                        max_toxicity=0.0, magnitude=level,
                    ))
            if secretion.size > 50:
                level = float(np.quantile(secretion, q))
                if level > 0:
                    catalogue.append(Intervention(
                        intervention_id=f"blockexport::{met}::q{int(q*100)}",
                        kind="export_block",
                        targets=(rxn,),
                        description=f"cap secretion of {met} at {level:.3g}",
                        bounds={rxn: (None, level)},
                        metabolites=(met,), roles=(role,),
                        max_toxicity=0.0, magnitude=level,
                    ))
    return catalogue


def combine(a: Intervention, b: Intervention) -> Intervention:
    """Two-target combination therapy."""
    bounds = dict(a.bounds)
    for rxn, (low, high) in b.bounds.items():
        if rxn in bounds:
            existing_low, existing_high = bounds[rxn]
            low = max(x for x in (low, existing_low) if x is not None) \
                if (low is not None and existing_low is not None) else (low or existing_low)
            high = min(x for x in (high, existing_high) if x is not None) \
                if (high is not None and existing_high is not None) else (high or existing_high)
        bounds[rxn] = (low, high)
    return Intervention(
        intervention_id=f"{a.intervention_id}+{b.intervention_id}",
        kind="combination",
        targets=tuple(sorted(set(a.targets) | set(b.targets))),
        description=f"{a.description}; {b.description}",
        bounds=bounds,
        metabolites=tuple(sorted(set(a.metabolites) | set(b.metabolites))),
        roles=tuple(sorted(set(a.roles) | set(b.roles))),
        max_toxicity=max(a.max_toxicity, b.max_toxicity),
        magnitude=a.magnitude + b.magnitude,
    )


def utility_score(
    results: pd.DataFrame,
    *,
    lambda_toxicity: float = 1.0,
    mu_non_modifiable: float = 0.5,
    nu_magnitude: float = 0.1,
) -> pd.DataFrame:
    """Combine phenotypic recovery with biological admissibility.

    .. math::
        U = R - \\lambda\\,\\tau - \\mu\\,\\mathbb{1}[\\text{not modifiable}]
            - \\nu\\,\\tilde m

    where :math:`R` is the recovery of the health distance, :math:`\\tau` the
    toxicity weight of the metabolite whose exposure is increased, and
    :math:`\\tilde m` the intervention magnitude scaled to its own range. The
    penalties apply only to interventions that *increase* exposure: capping an
    uptake or a secretion cannot raise toxicity, so its penalty is zero by
    construction in :func:`build_catalogue`.
    """
    out = results.copy()
    magnitude = out["magnitude"].to_numpy(dtype=float)
    span = np.ptp(magnitude[np.isfinite(magnitude)]) if np.isfinite(magnitude).any() else 0.0
    scaled = (magnitude - np.nanmin(magnitude)) / span if span > 0 else np.zeros_like(magnitude)
    out["utility"] = (
        out["recovery"]
        - lambda_toxicity * out["max_toxicity"]
        - mu_non_modifiable * (~out["clinically_modifiable"]).astype(float)
        - nu_magnitude * scaled
    )
    out["admissible"] = out["clinically_modifiable"] & (out["max_toxicity"] <= 0.5)
    return out.sort_values("utility", ascending=False, ignore_index=True)

## 23. Sampling a restricted polytope

Rejection screening, which is exact because a tightening defines a sub-polytope, and a hit-and-run sampler for the confirmatory tier. The comment inside `_direction` records a defect worth knowing about: the textbook artificially-centred variant drifts out of the null space of S through accumulated floating-point error.

> Hit-and-run sampling of a restricted flux polytope.

In [ ]:
from dataclasses import dataclass

import numpy as np

__all__ = ["rejection_sample", "ACHRSampler", "RetentionReport"]


@dataclass
class RetentionReport:
    n_total: int
    n_retained: int
    retention: float
    sufficient: bool


def rejection_sample(
    V: np.ndarray, lb: np.ndarray, ub: np.ndarray, *, tolerance: float = 1e-6,
    min_retained: int = 200,
) -> tuple[np.ndarray, RetentionReport]:
    """Select the samples of ``V`` that satisfy tightened bounds."""
    keep = np.all((V >= lb - tolerance) & (V <= ub + tolerance), axis=1)
    n_retained = int(keep.sum())
    return V[keep], RetentionReport(
        n_total=len(V),
        n_retained=n_retained,
        retention=float(keep.mean()),
        sufficient=n_retained >= min_retained,
    )


class ACHRSampler:
    """Hit-and-run over ``{v : Sv = 0, lb <= v <= ub}`` with null-space directions.

    Directions are differences between warm-up points, which satisfy the
    steady-state constraint exactly, so every generated point satisfies
    :math:`Sv = 0` to machine precision without solving a projection problem
    at any step. See :meth:`_direction` for why the adaptive centre of the
    textbook artificially-centred variant is deliberately not used.
    """

    def __init__(
        self,
        warmup: np.ndarray,
        lb: np.ndarray,
        ub: np.ndarray,
        *,
        seed: int = 0,
        tolerance: float = 1e-9,
    ):
        if warmup.shape[0] < 2:
            raise ValueError("at least two warm-up points are required")
        self.warmup = np.asarray(warmup, dtype=float)
        self.lb = np.asarray(lb, dtype=float)
        self.ub = np.asarray(ub, dtype=float)
        self.rng = np.random.default_rng(seed)
        self.tolerance = tolerance
        self.centre = self.warmup.mean(axis=0)
        self.n_generated = 0

    def _direction(self) -> np.ndarray:
        """Draw a search direction that lies exactly in the null space of S.

        Directions are differences between *warm-up* points, or between a
        warm-up point and their fixed mean. Both are exact, because the warm-up
        set is never modified.

        An adaptive centre updated with the generated points -- the usual
        artificially-centred variant -- was tried first and had to be
        abandoned: the running-mean update accumulates floating-point error, so
        the centre slowly acquires a component outside the null space, every
        subsequent direction inherits it, and the constraint violation grows
        without bound. The feedback is easy to miss because each individual
        update is correct to machine precision.
        """
        i, j = self.rng.integers(len(self.warmup), size=2)
        if i == j:
            return self.warmup[i] - self.centre
        return self.warmup[i] - self.warmup[j]

    def _step(self, point: np.ndarray) -> np.ndarray:
        direction = self._direction()
        norm = np.linalg.norm(direction)
        if norm < self.tolerance:
            return point
        direction = direction / norm

        # largest step in each sense that keeps every coordinate within bounds
        with np.errstate(divide="ignore", invalid="ignore"):
            upper = np.where(direction > self.tolerance, (self.ub - point) / direction, np.inf)
            lower = np.where(direction > self.tolerance, (self.lb - point) / direction, -np.inf)
            upper = np.where(direction < -self.tolerance, (self.lb - point) / direction, upper)
            lower = np.where(direction < -self.tolerance, (self.ub - point) / direction, lower)
        max_step = float(np.min(upper))
        min_step = float(np.max(lower))
        if not np.isfinite(max_step) or not np.isfinite(min_step) or max_step <= min_step:
            return point
        step = self.rng.uniform(min_step, max_step)
        return point + step * direction

    def sample(self, n_samples: int, *, thinning: int = 100, burn_in: int = 1000) -> np.ndarray:
        """Draw ``n_samples`` points from the polytope."""
        point = self.warmup[self.rng.integers(len(self.warmup))].copy()
        for _ in range(burn_in):
            point = self._step(point)
        out = np.empty((n_samples, self.warmup.shape[1]))
        for i in range(n_samples):
            for _ in range(thinning):
                point = self._step(point)
            self.n_generated += 1
            out[i] = point
        # No clipping: the step construction already guarantees the bounds, and
        # clipping would move the point off the null space of S and so break the
        # steady-state constraint the whole scheme is built to preserve.
        return out

## 24. The visual system

One palette with fixed meaning for every figure: severity is always the same warm ramp, the internal block always teal-blue, the external block always violet, and any null, control or raw-flux quantity always neutral grey.

> One visual system for every figure in the paper.

In [ ]:
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

__all__ = [
    "INK", "MUTED", "RULE", "PAPER",
    "SEVERITY", "SEVERITY_ORDER", "BLOCK", "FAMILY", "ACCENT",
    "SEQUENTIAL", "DIVERGING",
    "apply_theme", "panel_label", "save", "soft_box",
]

# -- neutrals ---------------------------------------------------------------
INK = "#1B2A38"        # text and axes
MUTED = "#6B7A8C"      # secondary text
RULE = "#DCE2E8"       # gridlines and hairlines
PAPER = "#FFFFFF"

# -- the two structural blocks ---------------------------------------------
BLOCK = {
    "internal": "#1F5B7A",     # deep teal-blue: what the network does
    "internal_soft": "#E7EFF4",
    "external": "#6E4E8F",     # violet: what crosses the boundary
    "external_soft": "#F0EAF6",
    "neutral": "#8A94A6",      # raw flux space, nulls, controls
    "neutral_soft": "#EEF1F4",
}

# -- severity ramp ----------------------------------------------------------
SEVERITY_ORDER = ["healthy", "mild", "moderate", "severe"]
SEVERITY = {
    "healthy": "#3E7D5A",
    "mild": "#E3AE55",
    "moderate": "#D97B41",
    "severe": "#B04430",
}

ACCENT = {
    "signal": "#B04430",     # thresholds, the thing to look at
    "ok": "#3E7D5A",
    "warn": "#E3AE55",
    "info": "#1F5B7A",
}

# -- feature families -------------------------------------------------------
# Mathematical families share the neutral end of the scale; biochemical
# families share the saturated end. The grouping is the message of Figure 3.
FAMILY = {
    "magnitude": "#A9B4C2",
    "distribution": "#98A5B5",
    "sparsity": "#8795A8",
    "entropy": "#75859B",
    "architecture": "#64758E",
    "network": "#536681",
    "pathway": "#2E7D8F",
    "turnover": "#1F5B7A",
    "biochemical": "#17475F",
    "efficiency": "#0F3346",
    "boundary_metabolite": "#6E4E8F",
    "boundary_role": "#84679E",
    "exchange_burden": "#9A82AF",
    "boundary_efficiency": "#B09DC0",
    "oxidative_environment": "#C2A6D3",
    "redox_environment": "#5B3F78",
    "admissibility": "#6B7A8C",
}

SEQUENTIAL = LinearSegmentedColormap.from_list(
    "mfp_sequential", ["#F2F6F8", "#B8CEDA", "#6C9CB4", "#2E7D8F", "#17475F"])
DIVERGING = LinearSegmentedColormap.from_list(
    "mfp_diverging", ["#1F5B7A", "#8FB3C4", "#F2F1EE", "#E0A98A", "#B04430"])


def apply_theme() -> None:
    mpl.rcParams.update({
        "figure.dpi": 160,
        "savefig.dpi": 450,
        "savefig.bbox": "tight",
        "savefig.facecolor": PAPER,
        "figure.facecolor": PAPER,
        "axes.facecolor": PAPER,
        "font.family": "DejaVu Sans",
        "font.size": 7.5,
        "text.color": INK,
        "axes.titlesize": 8,
        "axes.titleweight": "bold",
        "axes.titlecolor": INK,
        "axes.titlepad": 6,
        "axes.labelsize": 7.5,
        "axes.labelcolor": INK,
        "axes.edgecolor": "#B9C3CE",
        "axes.linewidth": 0.6,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.grid": True,
        "axes.axisbelow": True,
        "grid.color": RULE,
        "grid.linewidth": 0.55,
        "xtick.labelsize": 6.8,
        "ytick.labelsize": 6.8,
        "xtick.color": MUTED,
        "ytick.color": MUTED,
        "xtick.major.width": 0.6,
        "ytick.major.width": 0.6,
        "xtick.major.size": 2.5,
        "ytick.major.size": 2.5,
        "legend.frameon": False,
        "legend.fontsize": 6.6,
        "legend.handlelength": 1.2,
        "legend.handletextpad": 0.5,
        "legend.labelspacing": 0.35,
        "lines.linewidth": 1.3,
        "lines.solid_capstyle": "round",
        "patch.linewidth": 0.5,
        "patch.edgecolor": PAPER,
    })


def panel_label(ax, letter: str, title: str = "", *, pad: float = 7.0,
                offset: float = -17.0, gap: float = 11.0):
    """Bold panel letter plus a light title, positioned consistently.

    The offsets are in typographic points rather than axes fractions, so the
    letter sits the same physical distance from the corner in every panel of
    every figure regardless of how wide that panel happens to be.
    """
    ax.annotate(letter, xy=(0, 1), xycoords="axes fraction",
                xytext=(offset, pad), textcoords="offset points",
                fontsize=9, fontweight="bold", color=INK, ha="left", va="baseline",
                annotation_clip=False)
    if title:
        ax.annotate(title, xy=(0, 1), xycoords="axes fraction",
                    xytext=(offset + gap, pad), textcoords="offset points",
                    fontsize=7.8, color=INK, ha="left", va="baseline",
                    annotation_clip=False)


def soft_box(ax, x, y, w, h, *, face, edge, radius=0.02, lw=1.0, alpha=1.0):
    from matplotlib.patches import FancyBboxPatch

    patch = FancyBboxPatch((x, y), w, h, boxstyle=f"round,pad=0,rounding_size={radius}",
                           facecolor=face, edgecolor=edge, linewidth=lw, alpha=alpha)
    ax.add_patch(patch)
    return patch


def save(fig, path, *, formats=("png", "pdf")):
    from pathlib import Path

    path = Path(path)
    written = []
    for suffix in formats:
        target = path.with_suffix(f".{suffix}")
        fig.savefig(target)
        written.append(target)
    plt.close(fig)
    return written

Figures are written as vector PDF and as JPG. The PDF is the
version to submit; the JPG is there so the folder can be browsed.

In [ ]:
def save(fig, path, *, formats=("pdf", "jpg")):
    """Write a figure in every format the project keeps, then close it."""
    path = Path(path)
    written = []
    for suffix in formats:
        target = path.with_suffix(f".{suffix}")
        fig.savefig(target, dpi=450 if suffix != "pdf" else None)
        written.append(target)
    plt.close(fig)
    return written

---

# Part II — the analysis

From here the notebook *runs* the machinery, one stage per section, in the
order of the plan: the models first, then the samples, then the
representation, then the comparison that the paper is built on, then the
applications.

## 3. The two models, compared before anything else

In [ ]:
"""Stage — the two models, compared before anything is said about phenotypes.

Nothing in this stage looks at a feature or a sampled state. It answers the
question the design review insisted comes first: *how do the two
reconstructions differ, and is the same experiment even posed in both?*

Writing this report before comparing phenotypes is what keeps a later
difference from being attributed to thermodynamics when it is really the
consequence of 695 missing reactions or of a bound that moved.
"""
import numpy as np
import pandas as pd


def run_00_model_comparison() -> None:
    config.ensure_dirs()
    out = METADATA_DIR
    out.mkdir(parents=True, exist_ok=True)
    report_dir = Path(REPORT_DIR)
    report_dir.mkdir(parents=True, exist_ok=True)

    models, tables = {}, {}
    for variant, spec in MODEL_VARIANTS.items():
        path = Path(spec["model_file"])
        if not path.exists():
            print(f"{variant}: {path.name} not found — skipped")
            continue
        gem = load_gem(path, spec["model_name"])
        models[variant] = gem
        tables[variant] = annotate_reactions(gem)
        print(f"{variant:9s} {gem.n_reactions:5d} reactions  {gem.n_metabolites:5d} metabolites")

    if len(models) < 2:
        print("only one model available; the comparison needs both")
        return

    a, b = models["original"], models["thermo"]
    ta, tb = tables["original"], tables["thermo"]
    ra, rb = list(a.rxns), list(b.rxns)
    sa, sb = set(ra), set(rb)
    shared = sorted(sa & sb)

    # ----------------------------------------------------------- summary
    summary = pd.DataFrame([
        {"quantity": "reactions", "original": a.n_reactions, "thermo": b.n_reactions},
        {"quantity": "metabolites", "original": a.n_metabolites, "thermo": b.n_metabolites},
        {"quantity": "genes", "original": int(np.size(a.genes)),
         "thermo": int(np.size(b.genes))},
        {"quantity": "exchange reactions",
         "original": int((ta["reaction_class"] == ReactionClass.EXCHANGE).sum()),
         "thermo": int((tb["reaction_class"] == ReactionClass.EXCHANGE).sum())},
        {"quantity": "reversible reactions",
         "original": int(ta["reversible"].sum()), "thermo": int(tb["reversible"].sum())},
        {"quantity": "shared reactions", "original": len(shared), "thermo": len(shared)},
        {"quantity": "only in this model",
         "original": len(sa - sb), "thermo": len(sb - sa)},
    ])
    summary["difference"] = summary["thermo"] - summary["original"]
    summary.to_csv(out / "model_comparison_summary.csv", index=False)
    print()
    print(summary.to_string(index=False))

    # -------------------------------------------------- reaction-level diff
    index_a = {r: j for j, r in enumerate(ra)}
    index_b = {r: j for j, r in enumerate(rb)}
    rows = []
    for rxn in shared:
        ja, jb = index_a[rxn], index_b[rxn]
        lb_a, ub_a, lb_b, ub_b = float(a.lb[ja]), float(a.ub[ja]), float(b.lb[jb]), float(b.ub[jb])
        rows.append({
            "rxn_id": rxn,
            "pathway": ta.loc[ja, "pathway"],
            "lb_original": lb_a, "ub_original": ub_a,
            "lb_thermo": lb_b, "ub_thermo": ub_b,
            "lb_changed": not np.isclose(lb_a, lb_b),
            "ub_changed": not np.isclose(ub_a, ub_b),
            "reversibility_changed": bool((lb_a < 0) != (lb_b < 0)),
        })
    for rxn in sorted(sa - sb):
        ja = index_a[rxn]
        rows.append({"rxn_id": rxn, "pathway": ta.loc[ja, "pathway"],
                     "lb_original": float(a.lb[ja]), "ub_original": float(a.ub[ja]),
                     "lb_thermo": np.nan, "ub_thermo": np.nan,
                     "lb_changed": False, "ub_changed": False,
                     "reversibility_changed": False})
    diff = pd.DataFrame(rows)
    diff["status"] = np.where(diff["lb_thermo"].isna(), "removed_in_thermo", "shared")
    diff.to_csv(out / "model_reaction_diff.csv", index=False)

    removed = diff[diff["status"] == "removed_in_thermo"]
    print()
    print(f"{len(removed)} reactions are not in the thermodynamic subset. By pathway:")
    print(removed["pathway"].value_counts().head(12).to_string())
    print()
    print(f"bounds changed on {int(diff['ub_changed'].sum())} upper and "
          f"{int(diff['lb_changed'].sum())} lower limits of the shared reactions; "
          f"reversibility changed on {int(diff['reversibility_changed'].sum())}")

    # --------------------------------------------- key reactions and design
    key = list(OXIDATIVE_PPP_RXNS) + [G6PD_REACTION, H2O2_EXCHANGE, "HEX1", "PGI", "PYK",
                                      "GTHPi", "CAT", "EX_glc__D_e", "EX_o2_e"]
    key_rows = []
    for rxn in dict.fromkeys(key):
        row = {"rxn_id": rxn,
               "in_original": rxn in sa, "in_thermo": rxn in sb}
        for variant, gem, index in (("original", a, index_a), ("thermo", b, index_b)):
            if rxn in index:
                j = index[rxn]
                row[f"lb_{variant}"] = float(gem.lb[j])
                row[f"ub_{variant}"] = float(gem.ub[j])
        key_rows.append(row)
    key_frame = pd.DataFrame(key_rows)
    key_frame.to_csv(out / "model_key_reactions.csv", index=False)
    print()
    print(key_frame.to_string(index=False))

    # ------------------------------------------- is the design posed in both?
    design = []
    for variant, gem, index in (("original", a, index_a), ("thermo", b, index_b)):
        reference_ub = {
            rxn: float(gem.ub[index[rxn]]) if rxn in index else np.nan
            for rxn in (G6PD_REACTION, H2O2_EXCHANGE)
        }
        levels = [(1.0, 1.0)] + [(g, h) for g in G6PD_LEVELS for h in H2O2_LEVELS]
        for g6pd, h2o2 in levels:
            bounds = None if g6pd == 1.0 else {
                G6PD_REACTION: (None, reference_ub[G6PD_REACTION] * g6pd),
                H2O2_EXCHANGE: (None, reference_ub[H2O2_EXCHANGE] * h2o2),
            }
            solution = fba(gem, bounds=bounds)
            design.append({
                "model_variant": variant, "g6pd_level": g6pd, "h2o2_level": h2o2,
                "feasible": bool(solution.success),
                "objective": float(solution.objective) if solution.success else np.nan,
                "g6pdh2_flux": float(solution.v[index[G6PD_REACTION]])
                if solution.success and G6PD_REACTION in index else np.nan,
                "reference_ub_g6pdh2": reference_ub[G6PD_REACTION],
                "reference_ub_h2o2": reference_ub[H2O2_EXCHANGE],
            })
    design_frame = pd.DataFrame(design)
    design_frame.to_csv(out / "model_condition_feasibility.csv", index=False)
    print()
    print(design_frame.to_string(index=False))

    infeasible = design_frame[~design_frame["feasible"]]
    if len(infeasible):
        print(f"\n{len(infeasible)} condition(s) are infeasible and must be excluded "
              "from the comparison rather than silently analysed")
    else:
        print("\nevery condition of the factorial design is feasible in both models")

## 25. Does every chain carry the constraint it is named after? (`run_00b_constraint_integrity`)

A chain's name says which bounds were imposed before sampling; the archive does not prove it. Each chain is checked against its own claim, and a chain that fails is excluded from every later stage with its reason on record.

In [ ]:
"""Stage 00b — does every chain carry the constraint its name says it carries?

A chain is named after the bounds that were meant to be imposed on the model
before sampling. Nothing in the archive proves they were: a sampler handed a
rounded polytope built from another model samples *that* polytope, whatever
the model next to it says. So each chain is checked against its own claim.

For every perturbed reaction the check asks two things of the retained states:

* **the bound holds** — no state exceeds the imposed upper bound
  (up to a numerical tolerance);
* **the bound binds** — when the healthy distribution of the same
  reconstruction lies above the imposed bound, the chain must reach it: its
  maximum must be within 5 % of the bound. A chain capped well below its bound
  was sampled under a tighter constraint than the one it is named after.

A bound that the healthy distribution never reaches cannot be checked this
way — imposing it removes no state the sampler would visit — and is reported
as *not binding* rather than passed or failed. That is the situation of the
peroxide exchange in both reconstructions, and it is why the peroxide axis is
a negative control: the constraint is real (flux variability analysis shows
the exchange can reach +1000), but the region it removes holds no sampled
state.

Chains that fail are written to ``results/processed/excluded_chains.json``
and every later stage skips them; the table records why. Nothing is relabelled:
a chain that was sampled under the wrong constraint is not reassigned to the
condition it happens to resemble, because its provenance is unknown.
"""
import json

import numpy as np
import pandas as pd


__all__ = ["BOUND_TOLERANCE", "BINDING_FRACTION", "constraint_integrity",
           "run_00b_constraint_integrity"]

#: A retained state may exceed its bound by this relative amount (float32 export).
BOUND_TOLERANCE = 1e-4
#: A binding bound must be reached to within this fraction of its value.
BINDING_FRACTION = 0.95


def _perturbations():
    """(reaction, level key in parse_condition) for every axis of the design."""
    return [(G6PD_REACTION, "g6pd_level"), (H2O2_EXCHANGE, "h2o2_level")]


def flux_range(model, reaction: str) -> tuple[float, float]:
    """Flux variability of one reaction in the unperturbed model (LP, HiGHS)."""
    import scipy.sparse as sp
    from scipy.optimize import linprog

    S = sp.csr_matrix(model.S)
    k = list(model.rxns).index(reaction)
    bounds = list(zip(np.asarray(model.lb, float), np.asarray(model.ub, float)))
    extremes = []
    for sense in (-1.0, 1.0):
        c = np.zeros(S.shape[1])
        c[k] = sense
        result = linprog(c, A_eq=S, b_eq=np.zeros(S.shape[0]), bounds=bounds, method="highs")
        extremes.append(float(result.x[k]) if result.status == 0 else np.nan)
    return extremes[1], extremes[0]


def constraint_integrity(variant: str) -> pd.DataFrame:
    """One row per chain and perturbed reaction, with a verdict."""
    spec = MODEL_VARIANTS[variant]
    model = load_gem(spec["model_file"], spec["model_name"])
    reactions = list(model.rxns)
    store = SampleStore(SAMPLE_DIR, variant, include_excluded=True)

    chains = {stem: store.load_chain(stem) for stem in store.stems}
    healthy = [s for s in store.stems if parse_condition(s)["state"] == "healthy"]

    rows = []
    for reaction, level_key in _perturbations():
        k = reactions.index(reaction)
        reference_ub = float(model.ub[k])
        healthy_max = max(float(chains[s].thin[:, k].max()) for s in healthy)
        feasible_min, feasible_max = flux_range(model, reaction)
        for stem, chain in chains.items():
            info = parse_condition(stem)
            if info["state"] == "healthy":
                continue
            imposed = reference_ub * info[level_key]
            v = chain.thin[:, k]
            observed_max = float(v.max())
            binding = healthy_max > imposed
            holds = observed_max <= imposed * (1 + BOUND_TOLERANCE) + 1e-9
            reaches = observed_max >= BINDING_FRACTION * imposed
            if not holds:
                verdict, why = "fail", f"exceeds its bound ({observed_max:.1f} > {imposed:.1f})"
            elif binding and not reaches:
                verdict, why = "fail", (f"capped at {observed_max:.1f}, well below its bound "
                                        f"of {imposed:.1f}: sampled under a tighter constraint")
            elif not binding:
                verdict, why = "not_binding", (f"healthy maximum {healthy_max:.1f} never reaches "
                                               f"the bound of {imposed:.1f}")
            else:
                verdict, why = "pass", ""
            rows.append({
                "model_variant": variant, "stem": stem, "condition": info["condition"],
                "chain": info["chain"], "reaction": reaction, "level": info[level_key],
                "reference_ub": reference_ub, "imposed_ub": imposed,
                "healthy_max": healthy_max, "feasible_max": feasible_max,
                "feasible_min": feasible_min, "chain_median": float(np.median(v)),
                "chain_max": observed_max, "max_over_imposed": observed_max / imposed,
                "binding": binding, "verdict": verdict, "reason": why,
                "n_raw_points": chain.meta["n_samples_total"],
                "n_retained": chain.meta["n_samples_kept"], "thinning": chain.meta["thin"],
            })
    return pd.DataFrame(rows)


def run_00b_constraint_integrity() -> None:
    """Check every chain of both reconstructions and record the exclusions."""
    tables = [constraint_integrity(v) for v in MODEL_VARIANTS]
    table = pd.concat(tables, ignore_index=True)

    failed = table[table["verdict"] == "fail"]
    excluded = sorted(failed["stem"].unique())
    reasons = (failed.groupby("stem")["reason"].first()).to_dict()
    (Path(SAMPLE_DIR).parent / "excluded_chains.json").write_text(json.dumps(
        {"rule": (f"a perturbed reaction must stay at or below its imposed bound and, "
                  f"when the bound binds, reach at least {BINDING_FRACTION:.0%} of it"),
         "excluded": [{"stem": s, "reason": reasons[s]} for s in excluded]}, indent=1))

    for variant in MODEL_VARIANTS:
        out = TABLE_ROOT / variant / "sampling"
        out.mkdir(parents=True, exist_ok=True)
        table[table["model_variant"] == variant].to_csv(out / "constraint_integrity.csv", index=False)

    summary = (table.groupby(["model_variant", "reaction", "verdict"]).size()
               .unstack(fill_value=0))
    print(summary.to_string())
    print()
    for variant in MODEL_VARIANTS:
        lost = [s for s in excluded if s.startswith(variant)]
        conditions = sorted({parse_condition(s)["condition"] for s in lost})
        print(f"{variant}: {len(lost)} chains excluded"
              + (f" ({', '.join(conditions)})" if lost else ""))
        for s in lost:
            print(f"   {s}: {reasons[s]}")
    if len(excluded):
        print("\nRe-sample those chains with generateModelsAndSamples.m and re-run stage 00;"
              " they are picked up automatically once they pass.")

## 26. Annotation, perturbation magnitude, references (`run_01_prepare`)

Before a single feature is computed: classify the reactions, measure how much of the healthy sampled measure each imposed constraint actually excludes, and build the three candidate healthy references. The second of these is what turns the H₂O₂ axis into a built-in null.

Writes: `reaction_metadata.csv`, `chain_inventory.csv`, `constraint_diagnostics.csv`, `reference_flux_vectors.csv`

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd


RAW = config.RAW_DIR
OUT = config.RESULTS_DIR


def run_01_prepare() -> None:
    config.ensure_dirs()
    gem = load_gem(config.MODEL_FILE, config.MODEL_NAME)
    print(f"model: {gem.n_metabolites} metabolites x {gem.n_reactions} reactions")

    # ---------------------------------------------------------------- annotation
    reactions = annotate_reactions(gem)
    reactions.to_csv(OUT / "reaction_metadata.csv", index=False)
    unmapped = check_pathway_map(gem)
    unmapped.to_csv(OUT / "unmapped_subsystems.csv", index=False)
    print(reactions["reaction_class"].value_counts().to_string())
    print(f"internal {int(reactions.is_internal.sum())} / external {int(reactions.is_external.sum())}")

    # ---------------------------------------------------------------- inventory
    store = SampleStore(RAW)
    inventory = build_metadata(store)
    inventory.to_csv(OUT / "sampling" / "chain_inventory.csv", index=False)
    print(f"{len(inventory)} chains, {inventory.n_retained_samples.sum()} retained samples")

    # ------------------------------------------------- perturbation diagnostics
    #
    # Two questions are asked of each imposed constraint. (1) Is it binding on
    # the polytope at all? (2) How much of the *healthy sampled measure* does it
    # exclude? A constraint can be formally binding yet exclude a region the
    # sampler never visits, in which case it cannot produce a detectable
    # phenotypic effect however many samples are drawn. Answering (2) before
    # looking at any feature gives an a-priori expected effect size per axis.
    healthy = np.vstack([store.load_chain(s).thin for s in store.stems if parse_condition(s)["state"] == "healthy"])
    j_g6pd = gem.rxn_index(config.G6PD_REACTION)
    j_h2o2 = gem.rxn_index(config.H2O2_EXCHANGE)

    rows = []
    for level in config.G6PD_LEVELS:
        limit = gem.ub[j_g6pd] * level
        diag = bound_is_binding(gem, config.G6PD_REACTION, which="upper",
                                bounds={config.G6PD_REACTION: (None, limit)})
        excluded = float((healthy[:, j_g6pd] > limit).mean())
        rows.append({**diag, "axis": "G6PD", "level": level,
                     "healthy_measure_excluded": excluded,
                     "healthy_max_observed": float(healthy[:, j_g6pd].max())})
    for level in config.H2O2_LEVELS:
        limit = gem.ub[j_h2o2] * level
        diag = bound_is_binding(gem, config.H2O2_EXCHANGE, which="upper",
                                bounds={config.H2O2_EXCHANGE: (None, limit)})
        excluded = float((healthy[:, j_h2o2] > limit).mean())
        rows.append({**diag, "axis": "H2O2", "level": level,
                     "healthy_measure_excluded": excluded,
                     "healthy_max_observed": float(healthy[:, j_h2o2].max())})

    diagnostics = pd.DataFrame(rows)
    diagnostics.to_csv(OUT / "sampling" / "constraint_diagnostics.csv", index=False)
    print()
    print(diagnostics[["axis", "level", "bound", "attainable_extreme", "binding",
                       "healthy_max_observed", "healthy_measure_excluded"]].to_string(index=False))

    # ---------------------------------------------------------------- references
    v_fba_raw = load_flux_vector(config.FBA_FILE)
    solved = fba(gem)
    parsimonious = pfba(gem)
    healthy_mean = healthy.mean(axis=0)
    healthy_median = np.median(healthy, axis=0)

    references = pd.DataFrame(
        {
            "rxn_id": gem.rxns,
            "subsystem": gem.subsystems,
            "is_external": reactions["is_external"].to_numpy(),
            "v_healthy_fba_matlab": v_fba_raw,
            "v_healthy_fba_python": solved.v,
            "v_healthy_pfba": parsimonious.v,
            "v_healthy_sampled_mean": healthy_mean,
            "v_healthy_sampled_median": healthy_median,
        }
    )
    references.to_csv(OUT / "phenotype" / "reference_flux_vectors.csv", index=False)

    summary = pd.DataFrame(
        [
            {"reference": name,
             "objective_NaKt": float(v[gem.rxn_index('NaKt')]),
             "L1": float(np.abs(v).sum()),
             "active_reactions": int((np.abs(v) > config.TOL.flux_zero).sum()),
             "G6PDH2": float(v[j_g6pd]),
             "EX_h2o2_e": float(v[j_h2o2])}
            for name, v in [
                ("healthy FBA (MATLAB)", v_fba_raw),
                ("healthy FBA (Python)", solved.v),
                ("healthy pFBA", parsimonious.v),
                ("healthy sampled mean", healthy_mean),
                ("healthy sampled median", healthy_median),
            ]
        ]
    )
    summary.to_csv(OUT / "phenotype" / "reference_summary.csv", index=False)
    print()
    print(summary.to_string(index=False))

## 27. Is the sampled space usable? (`run_02_sampling_qc`)

Five diagnostics against five thresholds fixed in advance. This is a go/no-go gate: if it fails, nothing downstream is worth computing.

Writes: `reaction_diagnostics.csv`, `condition_diagnostics.csv`, `running_mean_stability.csv`, `qc_decision.csv`

In [ ]:
from collections import defaultdict
from pathlib import Path

import numpy as np
import pandas as pd



#: A chain is considered usable when the reactions that actually vary reach
#: these thresholds. They are the conventional values used for MCMC output.
ESS_TARGET = 400.0
RHAT_TARGET = 1.05


def run_02_sampling_qc() -> None:
    OUT = config.RESULTS_DIR / "sampling"
    config.ensure_dirs()
    gem = load_gem(config.MODEL_FILE, config.MODEL_NAME)
    store = SampleStore(config.RAW_DIR)

    by_condition: dict[str, dict[int, np.ndarray]] = defaultdict(dict)
    trace_rows = None
    for stem in store.stems:
        chain = store.load_chain(stem, with_trace=True)
        info = parse_condition(stem)
        by_condition[info["condition"]][info["chain"]] = chain.trace
        trace_rows = chain.trace_rows
    reaction_ids = gem.rxns[trace_rows]

    per_reaction, per_condition, stability = [], [], []
    for condition, traces in by_condition.items():
        diag = chain_diagnostics(traces, reaction_ids)
        agree = cross_chain_agreement(traces, reaction_ids)
        diag = diag.merge(agree, on="rxn_id")
        diag.insert(0, "condition", condition)
        per_reaction.append(diag)

        varying = diag[~diag["invariant"]]
        per_condition.append(
            {
                "condition": condition,
                "n_chains": len(traces),
                "n_draws_per_chain": next(iter(traces.values())).shape[1],
                "n_reactions_monitored": len(diag),
                "n_invariant": int(diag["invariant"].sum()),
                "n_varying": len(varying),
                "tau_median": float(varying["tau_int_mean"].median()),
                "tau_p95": float(varying["tau_int_mean"].quantile(0.95)),
                "ess_per_chain_median": float(varying["ess_mean"].median()),
                "ess_per_chain_min": float(varying["ess_mean"].min()),
                "ess_total_median": float(varying["ess_total"].median()),
                "frac_ess_below_target": float((varying["ess_total"] < ESS_TARGET).mean()),
                "rhat_median": float(varying["split_rhat"].median()),
                "rhat_max": float(varying["split_rhat"].max()),
                "frac_rhat_above_target": float((varying["split_rhat"] > RHAT_TARGET).mean()),
                "ks_median": float(varying["ks_mean"].median()),
                "ks_max": float(varying["ks_max"].max()),
            }
        )

        for chain_id, trace in traces.items():
            for row, rxn in enumerate(reaction_ids):
                if np.ptp(trace[row]) <= config.TOL.invariant_range:
                    continue
                stability.append(
                    {"condition": condition, "chain": chain_id, "rxn_id": rxn,
                     **running_mean_stability(trace[row])}
                )

    reaction_frame = pd.concat(per_reaction, ignore_index=True)
    condition_frame = pd.DataFrame(per_condition)
    stability_frame = pd.DataFrame(stability)

    reaction_frame.to_csv(OUT / "reaction_diagnostics.csv", index=False)
    condition_frame.to_csv(OUT / "condition_diagnostics.csv", index=False)
    stability_frame.to_csv(OUT / "running_mean_stability.csv", index=False)

    decision = pd.DataFrame(
        [
            {
                "criterion": "median total ESS across varying reactions",
                "target": f">= {ESS_TARGET:.0f}",
                "observed": f"{condition_frame['ess_total_median'].min():.0f} (worst condition)",
                "passed": bool(condition_frame["ess_total_median"].min() >= ESS_TARGET),
            },
            {
                "criterion": "fraction of reactions with total ESS below target",
                "target": "<= 0.05",
                "observed": f"{condition_frame['frac_ess_below_target'].max():.3f} (worst condition)",
                "passed": bool(condition_frame["frac_ess_below_target"].max() <= 0.05),
            },
            {
                "criterion": "split R-hat across chains",
                "target": f"<= {RHAT_TARGET}",
                "observed": f"{condition_frame['rhat_max'].max():.3f} (worst reaction)",
                "passed": bool(condition_frame["rhat_max"].max() <= RHAT_TARGET),
            },
            {
                "criterion": "fraction of reactions with R-hat above target",
                "target": "<= 0.05",
                "observed": f"{condition_frame['frac_rhat_above_target'].max():.3f} (worst condition)",
                "passed": bool(condition_frame["frac_rhat_above_target"].max() <= 0.05),
            },
            {
                "criterion": "median running-mean drift over the final quarter",
                "target": "<= 0.05",
                "observed": f"{stability_frame['relative_drift'].median():.4f}",
                "passed": bool(stability_frame["relative_drift"].median() <= 0.05),
            },
        ]
    )
    decision.to_csv(OUT / "qc_decision.csv", index=False)

    print(condition_frame.to_string(index=False))
    print()
    print(decision.to_string(index=False))

## 28. Compute every declared descriptor (`run_03_features`)

The longest stage: every retained state in, one row of every declared descriptor out.

Writes: `feature_registry.csv`, `reference_phenotypes.csv`

In [ ]:
import time
from pathlib import Path

import numpy as np
import pandas as pd


def run_03_features() -> None:
    config.ensure_dirs()
    gem = load_gem(config.MODEL_FILE, config.MODEL_NAME)
    reactions = annotate_reactions(gem)
    engine = FeatureEngine(gem, reactions)
    print(f"{len(engine.registry)} candidate features declared")

    store = SampleStore(config.RAW_DIR)
    V, metadata = store.load_all()
    print(f"flux matrix: {V.shape[0]} samples x {V.shape[1]} reactions")

    started = time.time()
    phenotypes = engine.transform(V)
    print(f"phenotypes computed in {time.time() - started:.1f}s -> {phenotypes.shape}")

    phenotypes.to_csv(config.PROCESSED_DIR / "phenotypes.csv.gz", index=False, compression="gzip")
    metadata.to_csv(config.PROCESSED_DIR / "sample_metadata.csv.gz", index=False, compression="gzip")

    registry = engine.registry_frame()
    registry.to_csv(config.RESULTS_DIR / "features" / "feature_registry.csv", index=False)
    engine.coverage_report().to_csv(
        config.RESULTS_DIR / "features" / "feature_coverage_gaps.csv", index=False
    )

    # ------------------------------------------------------------ references
    references = {
        "healthy_fba_matlab": load_flux_vector(config.FBA_FILE),
        "healthy_fba_python": fba(gem).v,
        "healthy_pfba": pfba(gem).v,
    }
    healthy_mask = (metadata["state"] == "healthy").to_numpy()
    references["healthy_sampled_mean"] = V[healthy_mask].mean(axis=0)
    references["healthy_sampled_median"] = np.median(V[healthy_mask], axis=0)

    rows = {name: engine.transform_vector(v) for name, v in references.items()}
    reference_frame = pd.DataFrame(rows).T
    reference_frame.index.name = "reference"
    reference_frame.to_csv(config.RESULTS_DIR / "features" / "reference_phenotypes.csv")

    print()
    print(registry.groupby(["block", "family"]).size().to_string())
    print()
    print("samples per condition:")
    print(metadata.groupby(["state", "g6pd_level", "h2o2_level"]).size().to_string())

## 29. The quality-control cascade (`run_04_feature_qc`)

The counts of what each criterion removes, and the size of the core that remains, are printed when the stage runs and written to `features/feature_qc_decision.csv`; a descriptor that fails more than one criterion is counted under each.

Writes: `feature_numerical_qc.csv`, `feature_chain_reproducibility.csv`, `feature_redundancy_clusters.csv`, `feature_discrimination.csv`, `feature_qc_decision.csv`, `core_features.csv`

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd



ALPHA = 0.05
MIN_EFFECT = 0.01   # epsilon-squared floor: below this a difference is not material


def run_04_feature_qc() -> None:
    OUT = config.RESULTS_DIR / "features"
    config.ensure_dirs()
    features = pd.read_csv(config.PROCESSED_DIR / "phenotypes.csv.gz")
    metadata = pd.read_csv(config.PROCESSED_DIR / "sample_metadata.csv.gz")
    registry = pd.read_csv(OUT / "feature_registry.csv")
    print(f"{features.shape[1]} candidate features over {features.shape[0]} samples")

    # ------------------------------------------------- variance stabilisation
    #
    # Chosen from the healthy samples alone, so the transform cannot be tuned
    # to the disease contrast it is later used to measure.
    healthy_mask = (metadata["state"] == "healthy").to_numpy()
    transform = PhenotypeTransform.fit(features.loc[healthy_mask])
    transform.summary().to_csv(OUT / "feature_transforms.csv", index=False)
    features = transform.apply(features)
    features.to_csv(
        config.PROCESSED_DIR / "phenotypes_transformed.csv.gz", index=False, compression="gzip"
    )
    print(transform.summary()["transform"].value_counts().to_string())

    # ------------------------------------------------------------ numerical QC
    quality = numerical_quality(features)
    quality.to_csv(OUT / "feature_numerical_qc.csv", index=False)

    # --------------------------------------------------- chain reproducibility
    reproducibility = chain_reproducibility(features, metadata)
    reproducibility.to_csv(OUT / "feature_chain_reproducibility.csv", index=False)

    # ------------------------------------------------------------- redundancy
    usable = quality.loc[~quality["constant"] & (quality["fraction_missing"] <= 0.01), "feature"]
    clusters, correlation = redundancy_clusters(features[usable], seed=config.RANDOM_SEED)
    clusters.to_csv(OUT / "feature_redundancy_clusters.csv", index=False)
    correlation.to_csv(OUT / "feature_correlation_matrix.csv")

    # ----------------------------------------------------------- discrimination
    disease = metadata["state"].to_numpy() == "disease"
    tests = []

    # positive-control axis: healthy plus the three G6PD levels
    tests.append(group_discrimination(
        features, metadata["g6pd_level"].to_numpy(), label_name="g6pd_including_healthy"))
    # positive-control axis restricted to disease states only
    tests.append(group_discrimination(
        features[disease], metadata.loc[disease, "g6pd_level"].to_numpy(),
        label_name="g6pd_disease_only"))
    # null axis: H2O2 level, which stage 1 proved excludes no sampled measure
    tests.append(group_discrimination(
        features[disease], metadata.loc[disease, "h2o2_level"].to_numpy(),
        label_name="h2o2_null_axis"))
    # health status
    tests.append(group_discrimination(
        features, metadata["state"].to_numpy(), label_name="healthy_vs_disease"))

    discrimination = pd.concat(tests, ignore_index=True)
    discrimination.to_csv(OUT / "feature_discrimination.csv", index=False)

    # ------------------------------------------------------ null calibration
    calibration_rows = []
    for axis, block in discrimination.groupby("axis"):
        block = block.dropna(subset=["p_value"])
        calibration_rows.append(
            {
                "axis": axis,
                "n_features_tested": len(block),
                "n_p_below_alpha": int((block["p_value"] < ALPHA).sum()),
                "n_fdr_significant": int((block["p_adjusted"] < ALPHA).sum()),
                "fraction_fdr_significant": float((block["p_adjusted"] < ALPHA).mean()),
                "n_significant_and_material": int(
                    ((block["p_adjusted"] < ALPHA) & (block["epsilon_squared"] >= MIN_EFFECT)).sum()
                ),
                "median_epsilon_squared": float(block["epsilon_squared"].median()),
                "max_epsilon_squared": float(block["epsilon_squared"].max()),
            }
        )
    calibration = pd.DataFrame(calibration_rows)
    calibration.to_csv(OUT / "null_axis_calibration.csv", index=False)

    # ------------------------------------------------------------- core space
    #
    # Chain-dependence is judged *within* condition, so a feature that varies
    # between chains only because the chains sit in different conditions is not
    # penalised. Redundancy pruning keeps one representative per correlation
    # cluster. Discrimination is deliberately NOT a selection criterion: the
    # core space must be defined before the disease comparison, otherwise the
    # later separation results would be circular.
    healthy_values = features.loc[healthy_mask]
    healthy_spread = healthy_values.std(axis=0, ddof=1).to_numpy(dtype=float)
    own_range = (features.max(axis=0) - features.min(axis=0)).to_numpy(dtype=float)
    no_unit = frozenset(
        f for f, spread, extent in zip(features.columns, healthy_spread, own_range)
        if np.isfinite(spread) and np.isfinite(extent) and extent > 0
        and spread < 1e-9 * extent
    )
    if no_unit:
        print(f"{len(no_unit)} descriptors have no usable spread in the healthy states: "
              f"{', '.join(sorted(no_unit))}")
    decision = select_core_features(quality, reproducibility, clusters,
                                    without_healthy_unit=no_unit)
    decision = decision.merge(
        registry[["feature", "family", "block", "units", "description"]],
        on="feature", how="left",
    )
    decision.to_csv(OUT / "feature_qc_decision.csv", index=False)

    core = decision[decision["retained"]].copy()
    core = core.merge(
        discrimination[discrimination["axis"] == "g6pd_including_healthy"]
        [["feature", "epsilon_squared", "p_adjusted"]],
        on="feature", how="left",
    )
    core.to_csv(OUT / "core_features.csv", index=False)

    print()
    print("QC cascade:")
    fail_columns = [c for c in decision.columns if c.startswith("fail_")]
    print(decision[fail_columns].sum().to_string())
    print(f"\nretained {int(decision.retained.sum())} of {len(decision)} candidate features")
    print(core.groupby(["block", "family"]).size().to_string())
    print()
    print("null-axis calibration:")
    print(calibration.to_string(index=False))

## 30. Which distance, against which reference? (`run_05_health_distance`)

Seventeen candidate distances scored against four criteria stated in advance. The FBA references fail here, and that failure is a reported result.

Writes: `health_distances.csv.gz`, `metric_comparison.csv`, `reference_position.csv`, `condition_distance_summary.csv`

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd



def run_05_health_distance() -> None:
    OUT = config.RESULTS_DIR / "distance"
    config.ensure_dirs()
    features = pd.read_csv(config.PROCESSED_DIR / "phenotypes_transformed.csv.gz")
    metadata = pd.read_csv(config.PROCESSED_DIR / "sample_metadata.csv.gz")
    core = pd.read_csv(config.RESULTS_DIR / "features" / "core_features.csv")
    references = pd.read_csv(
        config.RESULTS_DIR / "features" / "reference_phenotypes.csv", index_col=0
    )

    internal = [f for f in core.loc[core["block"] == "internal", "feature"] if f in features]
    print(f"internal core phenotype: {len(internal)} features")

    healthy_mask = (metadata["state"] == "healthy").to_numpy()
    healthy_features = features.loc[healthy_mask, internal]
    reference = HealthyReference(healthy_features, seed=config.RANDOM_SEED)

    # ------------------------------------------------------ distance families
    distances: dict[str, np.ndarray] = {}

    # (A) distance to the healthy *distribution*, in the internal phenotype space
    distances["phenotype_euclidean"] = reference.euclidean(features[internal])
    distances["phenotype_manhattan"] = reference.manhattan(features[internal])
    distances["phenotype_chebyshev"] = reference.chebyshev(features[internal])
    distances["phenotype_mahalanobis"] = reference.mahalanobis(features[internal])
    distances["phenotype_pca10"] = reference.pca_distance(features[internal])
    distances["phenotype_cosine"] = reference.cosine(features[internal])

    # (B) distance to a single healthy reference *point*
    for label in ("healthy_fba_matlab", "healthy_pfba", "healthy_sampled_median"):
        point = references.loc[label]
        # The reference may have excluded descriptors that have no healthy
        # unit, so the point distances are taken on the coordinates the
        # reference actually kept.
        block = point_distances(features[reference.features], point[reference.features],
                                reference.scale)
        for kind, values in block.items():
            distances[f"point_{label}_{kind}"] = values

    # (C) distance in the raw flux space, for comparison with the phenotype space
    store = SampleStore(config.RAW_DIR)
    V, meta_check = store.load_all()
    assert (meta_check["sample_id"].to_numpy() == metadata["sample_id"].to_numpy()).all()
    internal_idx = np.flatnonzero(
        ~pd.read_csv(config.RESULTS_DIR / "reaction_metadata.csv")["is_external"].to_numpy()
    )
    v_fba = pd.read_csv(config.RESULTS_DIR / "phenotype" / "reference_flux_vectors.csv")
    for label, column in [
        ("fba", "v_healthy_fba_matlab"),
        ("sampled_mean", "v_healthy_sampled_mean"),
    ]:
        block = raw_flux_distances(V[:, internal_idx], v_fba[column].to_numpy()[internal_idx])
        for kind, values in block.items():
            distances[f"raw_{label}_{kind}"] = values

    distance_frame = pd.DataFrame(distances)
    output = pd.concat([metadata.reset_index(drop=True), distance_frame], axis=1)
    output.to_csv(OUT / "health_distances.csv.gz", index=False, compression="gzip")

    # ---------------------------------------------------------- metric choice
    comparison = evaluate_metrics(distance_frame, metadata)
    comparison.to_csv(OUT / "metric_comparison.csv", index=False)
    selected = comparison.iloc[0]["metric"]
    print()
    print(comparison.head(12).to_string(index=False))
    print(f"\nselected metric: {selected}")

    # ------------------------------------- where does the FBA reference sit?
    #
    # If the healthy FBA point were representative of the healthy metabolic
    # space, its phenotype would fall inside the healthy sampled distribution.
    # This is tested rather than assumed.
    rows = []
    healthy_distances = distance_frame.loc[healthy_mask]
    for label in references.index:
        point = references.loc[label]
        if point[internal].isna().any():
            continue
        values = reference.to_point(point)
        rows.append(
            {
                "reference": label,
                **values,
                "euclidean_percentile_in_healthy": float(
                    (healthy_distances["phenotype_euclidean"] < values["euclidean"]).mean() * 100
                ),
                "mahalanobis_percentile_in_healthy": float(
                    (healthy_distances["phenotype_mahalanobis"] < values["mahalanobis"]).mean() * 100
                ),
            }
        )
    position = pd.DataFrame(rows)
    position.to_csv(OUT / "reference_position.csv", index=False)
    print()
    print(position.to_string(index=False))

    # ------------------------------------------------- per-condition summary
    summary = (
        output.groupby(["state", "g6pd_level", "h2o2_level"])[selected]
        .agg(["mean", "std", "median",
              lambda s: s.quantile(0.05), lambda s: s.quantile(0.95)])
        .rename(columns={"<lambda_0>": "q05", "<lambda_1>": "q95"})
        .reset_index()
    )
    # distribution-level distance of each condition to the healthy set
    Z_healthy = reference.standardise(healthy_features)
    energies = []
    for (state, g6pd, h2o2), block in output.groupby(["state", "g6pd_level", "h2o2_level"]):
        Z = reference.standardise(features.loc[block.index, internal])
        energies.append(
            {"state": state, "g6pd_level": g6pd, "h2o2_level": h2o2,
             "energy_distance_to_healthy": energy_distance(Z, Z_healthy, seed=config.RANDOM_SEED)}
        )
    summary = summary.merge(pd.DataFrame(energies), on=["state", "g6pd_level", "h2o2_level"])
    summary.to_csv(OUT / "condition_distance_summary.csv", index=False)
    print()
    print(summary.to_string(index=False))

## 31. The disease landscape (`run_06_landscape`)

Where the conditions sit relative to the healthy distribution, and which diseased states are nonetheless healthy-like.

Writes: `pca_embedding.csv.gz`, `pca_loadings.csv`, `state_clusters.csv`, `cluster_composition.csv`, `healthy_like_states.csv.gz`, `healthy_like_external_contrast.csv`

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import stats
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import adjusted_rand_score, silhouette_score



METRIC = "phenotype_mahalanobis"


def stratified_subsample(labels, size, seed):
    """A subsample that keeps every cluster, so the silhouette is defined.

    Uniform sampling can miss a small cluster completely, which either
    raises or quietly scores a coarser partition than the one reported.
    """
    rng = np.random.default_rng(seed)
    labels = np.asarray(labels)
    groups = np.unique(labels)
    per_group = max(2, size // max(len(groups), 1))
    picked = []
    for group in groups:
        members = np.flatnonzero(labels == group)
        picked.append(rng.choice(members, size=min(len(members), per_group),
                                 replace=False))
    return np.sort(np.concatenate(picked))


def run_06_landscape() -> None:
    OUT = config.RESULTS_DIR / "landscape"
    config.ensure_dirs()
    features = pd.read_csv(config.PROCESSED_DIR / "phenotypes_transformed.csv.gz")
    metadata = pd.read_csv(config.PROCESSED_DIR / "sample_metadata.csv.gz")
    core = pd.read_csv(config.RESULTS_DIR / "features" / "core_features.csv")
    distances = pd.read_csv(config.RESULTS_DIR / "distance" / "health_distances.csv.gz")

    internal = [f for f in core.loc[core["block"] == "internal", "feature"] if f in features]
    external = [f for f in core.loc[core["block"] == "external", "feature"] if f in features]
    print(f"internal core {len(internal)}, external core {len(external)}")

    healthy_mask = (metadata["state"] == "healthy").to_numpy()
    Xi = features[internal].to_numpy(dtype=float)
    centre = np.median(Xi[healthy_mask], axis=0)
    iqr = np.subtract(*np.percentile(Xi[healthy_mask], [75, 25], axis=0))
    sd = Xi[healthy_mask].std(axis=0, ddof=1)
    # A descriptor with no usable spread in the healthy reference cannot be
    # expressed in units of that spread. "No spread" is judged against the
    # descriptor's own range rather than against zero: a healthy standard
    # deviation of 1.8e-15 on a descriptor that moves by 0.7 across the
    # conditions is floating-point noise, and dividing by it turns one
    # coordinate into 1e14 units and lets it decide every partition by itself.
    # Such a descriptor is excluded from this one space, and the exclusion is
    # reported; it stays in the registry and in every other analysis.
    spread = Xi.max(axis=0) - Xi.min(axis=0)
    scale = np.where(iqr > 0, iqr / 1.349, np.where(sd > 0, sd, np.nan))
    scalable = np.isfinite(scale) & (scale >= 1e-9 * np.where(spread > 0, spread, 1.0))
    if not scalable.all():
        dropped = [f for f, ok in zip(internal, scalable) if not ok]
        print(f"  {len(dropped)} descriptors have no usable spread in the healthy "
              f"reference and are excluded from the healthy-scaled embedding: "
              f"{', '.join(dropped)}")
        internal = [f for f, ok in zip(internal, scalable) if ok]
        Xi, centre, scale = Xi[:, scalable], centre[scalable], scale[scalable]
    Z = (Xi - centre) / scale

    # ------------------------------------------------------------- embedding
    pca = PCA(n_components=10, random_state=config.RANDOM_SEED).fit(Z[healthy_mask])
    embedding = pca.transform(Z)
    embed_frame = pd.concat(
        [metadata.reset_index(drop=True),
         pd.DataFrame(embedding, columns=[f"PC{i+1}" for i in range(embedding.shape[1])])],
        axis=1,
    )
    embed_frame[METRIC] = distances[METRIC].to_numpy()
    embed_frame.to_csv(OUT / "pca_embedding.csv.gz", index=False, compression="gzip")

    loadings = pd.DataFrame(
        pca.components_.T, index=internal,
        columns=[f"PC{i+1}" for i in range(pca.n_components_)],
    )
    loadings["explained_variance_ratio_of_pc1"] = pca.explained_variance_ratio_[0]
    loadings.to_csv(OUT / "pca_loadings.csv")
    print("explained variance:", np.round(pca.explained_variance_ratio_[:5], 4))

    # ------------------------------------------------------------ clustering
    #
    # The question is whether unsupervised structure in the phenotype space
    # recovers the imposed severity levels. Agreement is measured against the
    # G6PD level and, as a control, against the null H2O2 level.
    rows = []
    assignments = {}
    for k in range(2, 9):
        km = KMeans(n_clusters=k, n_init=10, random_state=config.RANDOM_SEED).fit(Z)
        labels = km.labels_
        assignments[k] = labels
        subsample = stratified_subsample(labels, 5000, config.RANDOM_SEED)
        rows.append(
            {
                "k": k,
                "inertia": float(km.inertia_),
                "silhouette": float(silhouette_score(Z[subsample], labels[subsample])),
                "ari_with_g6pd_level": float(
                    adjusted_rand_score(metadata["g6pd_level"], labels)),
                "ari_with_h2o2_level_null": float(
                    adjusted_rand_score(metadata["h2o2_level"], labels)),
                "ari_with_chain_null": float(
                    adjusted_rand_score(metadata["chain"], labels)),
            }
        )
    cluster_scores = pd.DataFrame(rows)
    cluster_scores.to_csv(OUT / "state_clusters.csv", index=False)
    print()
    print(cluster_scores.to_string(index=False))

    best_k = int(cluster_scores.loc[cluster_scores["ari_with_g6pd_level"].idxmax(), "k"])
    composition = (
        pd.crosstab(assignments[best_k], metadata["condition"])
        .rename_axis(index="cluster", columns="condition")
    )
    composition.to_csv(OUT / "cluster_composition.csv")

    # --------------------------------------------------- healthy-like states
    #
    # Within each disease condition, rank the sampled states by their internal
    # health distance and mark the most healthy-like quantiles. Ranking within
    # condition, rather than pooling, prevents the mildest condition from
    # supplying every "healthy-like" state and keeps the question biological:
    # given this enzyme capacity, which states come closest to health?
    distances = distances.copy()
    distances["health_distance"] = distances[METRIC]
    distances["healthy_like_rank"] = distances.groupby("condition")["health_distance"].rank(pct=True)
    for q in config.HEALTHY_LIKE_QUANTILES:
        distances[f"healthy_like_top{int(q*100)}pct"] = distances["healthy_like_rank"] <= q
    distances["pooled_rank"] = distances["health_distance"].rank(pct=True)
    distances.to_csv(OUT / "healthy_like_states.csv.gz", index=False, compression="gzip")

    healthy_reference_band = np.quantile(
        distances.loc[healthy_mask, "health_distance"], [0.05, 0.5, 0.95]
    )
    disease = (metadata["state"] == "disease").to_numpy()
    overlap = pd.DataFrame(
        [
            {
                "condition": condition,
                "n_samples": int(mask.sum()),
                "median_health_distance": float(distances.loc[mask, "health_distance"].median()),
                "fraction_inside_healthy_90pct_band": float(
                    (distances.loc[mask, "health_distance"] <= healthy_reference_band[2]).mean()
                ),
                "fraction_below_healthy_median": float(
                    (distances.loc[mask, "health_distance"] <= healthy_reference_band[1]).mean()
                ),
            }
            for condition, mask in (
                (c, (metadata["condition"] == c).to_numpy()) for c in metadata["condition"].unique()
            )
        ]
    )
    overlap.to_csv(OUT / "healthy_like_overlap.csv", index=False)
    print()
    print(overlap.to_string(index=False))

    # ------------------------- what distinguishes healthy-like external states?
    #
    # For every external core feature, compare the healthy-like tail of each
    # condition with the rest of that condition. This is the association step:
    # it identifies boundary conditions that accompany healthy-like internal
    # states, and makes no claim that they cause them.
    contrast_rows = []
    for condition in metadata.loc[disease, "condition"].unique():
        mask = (metadata["condition"] == condition).to_numpy()
        top = mask & distances["healthy_like_top5pct"].to_numpy()
        rest = mask & ~distances["healthy_like_top5pct"].to_numpy()
        for feature in external:
            a = features.loc[top, feature].to_numpy(dtype=float)
            b = features.loc[rest, feature].to_numpy(dtype=float)
            a, b = a[np.isfinite(a)], b[np.isfinite(b)]
            if a.size < 10 or b.size < 10 or np.ptp(np.concatenate([a, b])) == 0:
                continue
            u = stats.mannwhitneyu(a, b, alternative="two-sided")
            # rank-biserial correlation: a bounded, interpretable effect size
            effect = 2 * u.statistic / (a.size * b.size) - 1
            contrast_rows.append(
                {"condition": condition, "feature": feature,
                 "median_healthy_like": float(np.median(a)),
                 "median_rest": float(np.median(b)),
                 "rank_biserial": float(effect), "p_value": float(u.pvalue)}
            )
    contrast = pd.DataFrame(contrast_rows)
    contrast["p_adjusted"] = benjamini_hochberg(contrast["p_value"].to_numpy())
    contrast["consistent_sign"] = contrast.groupby("feature")["rank_biserial"].transform(
        lambda s: bool(np.all(np.sign(s) == np.sign(s.iloc[0])))
    )
    contrast.to_csv(OUT / "healthy_like_external_contrast.csv", index=False)

    summary = (
        contrast[contrast["p_adjusted"] < 0.05]
        .groupby("feature")
        .agg(mean_effect=("rank_biserial", "mean"),
             min_abs_effect=("rank_biserial", lambda s: float(np.abs(s).min())),
             n_conditions=("condition", "nunique"),
             consistent=("consistent_sign", "first"))
        .sort_values("mean_effect", key=lambda s: s.abs(), ascending=False)
    )
    summary.to_csv(OUT / "healthy_like_external_summary.csv")
    print()
    print("external features associated with healthy-like internal states:")
    print(summary.head(15).to_string())

## 32. Can the boundary predict internal health? (`run_07_external_inference`)

The framework's central applied question, with a permuted-label control and a power bound. The answer is no, and it is bounded.

Writes: `model_comparison.csv`, `permutation_importance.csv`, `shapley_importance.csv`, `surrogate_tree.txt`, `negative_result_power.csv`

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import stats



METRIC = "phenotype_mahalanobis"


def run_07_external_inference() -> None:
    OUT = config.RESULTS_DIR / "ml"
    config.ensure_dirs()
    features = pd.read_csv(config.PROCESSED_DIR / "phenotypes_transformed.csv.gz")
    metadata = pd.read_csv(config.PROCESSED_DIR / "sample_metadata.csv.gz")
    core = pd.read_csv(config.RESULTS_DIR / "features" / "core_features.csv")
    distances = pd.read_csv(config.RESULTS_DIR / "landscape" / "healthy_like_states.csv.gz")

    external = [f for f in core.loc[core["block"] == "external", "feature"] if f in features]
    X_all = features[external].replace([np.inf, -np.inf], np.nan)
    X_all = X_all.fillna(X_all.median())
    y_all = distances["health_distance"].to_numpy(dtype=float)
    disease = (metadata["state"] == "disease").to_numpy()
    # Cross-validating six models on every experiment is the most expensive
    # step of the pipeline and buys little: stage 2 puts the integrated
    # autocorrelation time near two *before* thinning, so consecutive retained
    # samples are already near-independent and a fixed stride keeps the design
    # balanced while making the comparison tractable.
    stride = 3
    disease = disease & (np.arange(len(metadata)) % stride == 0)
    print(f"external predictors: {len(external)}; "
          f"{int(disease.sum())} disease samples used for model comparison")

    # ------------------------------------------------------------ experiments
    comparisons = []

    pooled = evaluate_models(
        X_all[disease], y_all[disease],
        groups=metadata.loc[disease, "stem"].to_numpy(),
        seed=config.RANDOM_SEED,
    )
    pooled.insert(0, "experiment", "pooled_disease")
    comparisons.append(pooled)

    for level in sorted(metadata.loc[disease, "g6pd_level"].unique()):
        mask = disease & (metadata["g6pd_level"].to_numpy() == level)
        block = evaluate_models(
            X_all[mask], y_all[mask],
            groups=metadata.loc[mask, "stem"].to_numpy(),
            seed=config.RANDOM_SEED,
        )
        block.insert(0, "experiment", f"within_g6pd_{level:g}")
        comparisons.append(block)

    # a permuted-label control quantifies the score a model achieves on noise
    rng = np.random.default_rng(config.RANDOM_SEED)
    permuted = y_all[disease].copy()
    rng.shuffle(permuted)
    control = evaluate_models(
        X_all[disease], permuted,
        groups=metadata.loc[disease, "stem"].to_numpy(), seed=config.RANDOM_SEED,
    )
    control.insert(0, "experiment", "permuted_label_control")
    comparisons.append(control)

    comparison = pd.concat(comparisons, ignore_index=True)
    comparison.to_csv(OUT / "model_comparison.csv", index=False)
    print()
    print(comparison.pivot(index="model", columns="experiment", values="r2").round(4).to_string())

    # ------------------------------------------------- explain the best model
    pooled_best = pooled.loc[pooled["model"] != "mean_baseline"].iloc[0]["model"]
    model = fit_best(X_all[disease], y_all[disease],
                     metadata.loc[disease, "stem"].to_numpy(),
                     name=pooled_best, seed=config.RANDOM_SEED)
    print(f"\nbest pooled model: {pooled_best}")

    importance = model_importances(model, X_all[disease], y_all[disease], seed=config.RANDOM_SEED)
    importance.insert(0, "experiment", "pooled_disease")
    importance.to_csv(OUT / "permutation_importance.csv", index=False)
    print(importance.head(12).to_string(index=False))

    shapley = global_shapley_importance(
        model, X_all[disease], X_all[disease],
        n_explain=200, n_permutations=40, n_background=25, seed=config.RANDOM_SEED,
    )
    shapley.to_csv(OUT / "shapley_importance.csv", index=False)
    print()
    print(shapley.head(12).to_string(index=False))

    rules = surrogate_rules(X_all[disease], model.predict(X_all[disease]), seed=config.RANDOM_SEED)
    (OUT / "surrogate_tree.txt").write_text(rules)

    # ------------------------------------------- rigour for a negative result
    #
    # If the within-severity models fail, the claim "no association" must be
    # accompanied by the effect size the design could have detected. The bound
    # below is the smallest correlation detectable at 80% power with the
    # available number of independent chains, which is the true unit of
    # replication.
    rows = []
    for level in sorted(metadata.loc[disease, "g6pd_level"].unique()):
        mask = disease & (metadata["g6pd_level"].to_numpy() == level)
        block = comparison[comparison["experiment"] == f"within_g6pd_{level:g}"]
        best = block.loc[block["model"] != "mean_baseline", "r2"].max()
        n = int(mask.sum())
        n_groups = int(metadata.loc[mask, "stem"].nunique())
        # detectable |rho| at alpha = 0.05, power = 0.80, using the number of
        # samples but acknowledging the chain structure in the reported groups
        z = stats.norm.ppf(0.975) + stats.norm.ppf(0.80)
        detectable_rho = float(np.tanh(z / np.sqrt(n - 3)))
        rows.append(
            {
                "g6pd_level": level,
                "n_samples": n,
                "n_independent_chains": n_groups,
                "best_cv_r2": float(best),
                "detectable_spearman_at_80pct_power": detectable_rho,
                "implied_detectable_r2": float(detectable_rho ** 2),
            }
        )
    power = pd.DataFrame(rows)
    power.to_csv(OUT / "negative_result_power.csv", index=False)
    print()
    print(power.to_string(index=False))

## 33. Screen every boundary intervention (`run_08_interventions`)

2,304 candidates screened exactly by rejection, because a tightening defines a sub-polytope of the space already sampled.

Writes: `catalogue.csv`, `screening.csv`, `validation.csv`, `ranking.csv`, `mechanism_clusters.csv`

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.cluster import AgglomerativeClustering



MIN_RETAINED = 300
N_VALIDATE = 12
N_VALIDATION_SAMPLES = 1500

#: Recovery is measured on the condition where there is most to recover.
TARGET_CONDITION = "G6PD0.1_H2O21"

#: Feature families summarising the mechanism of an intervention.
MECHANISM_FEATURES = [
    "turnover_nadph_c", "turnover_atp_c", "turnover_gthrd_c", "turnover_h2o2_c",
    "turnover_nadh_c", "path_pentose_phosphate_activity", "path_glycolysis_activity",
    "path_glutathione_activity", "path_redox_detox_activity", "ex_total_uptake",
]


def run_08_interventions() -> None:
    OUT = config.RESULTS_DIR / "interventions"
    config.ensure_dirs()
    gem = load_gem(config.MODEL_FILE, config.MODEL_NAME)
    reactions = annotate_reactions(gem)
    engine = FeatureEngine(gem, reactions)
    index = {r: i for i, r in enumerate(gem.rxns)}

    store = SampleStore(config.RAW_DIR)
    V, metadata = store.load_all()
    raw_features = pd.read_csv(config.PROCESSED_DIR / "phenotypes.csv.gz")
    core = pd.read_csv(config.RESULTS_DIR / "features" / "core_features.csv")
    internal = [f for f in core.loc[core["block"] == "internal", "feature"] if f in raw_features]

    healthy_mask = (metadata["state"] == "healthy").to_numpy()
    transform = PhenotypeTransform.fit(raw_features.loc[healthy_mask])
    healthy_phenotype = transform.apply(raw_features.loc[healthy_mask])[internal]
    reference = HealthyReference(healthy_phenotype, seed=config.RANDOM_SEED)

    def health_distance(flux: np.ndarray) -> np.ndarray:
        phenotype = transform.apply(engine.transform(flux))[internal]
        return reference.mahalanobis(phenotype)

    target_mask = (metadata["condition"] == TARGET_CONDITION).to_numpy()
    V_target = V[target_mask]
    baseline = health_distance(V_target)
    baseline_median = float(np.median(baseline))
    healthy_distances = reference.mahalanobis(healthy_phenotype)
    healthy_median = float(np.median(healthy_distances))
    healthy_upper_band = float(np.quantile(healthy_distances, 0.95))
    print(f"baseline median health distance ({TARGET_CONDITION}): {baseline_median:.3f}")
    print(f"healthy median health distance: {healthy_median:.3f}")

    # -------------------------------------------------------------- catalogue
    catalogue = build_catalogue(reactions, V[healthy_mask])
    print(f"{len(catalogue)} candidate interventions")
    pd.DataFrame(
        [
            {"intervention_id": c.intervention_id, "kind": c.kind,
             "targets": "|".join(c.targets), "metabolites": "|".join(c.metabolites),
             "roles": "|".join(c.roles), "description": c.description,
             "max_toxicity": c.max_toxicity, "magnitude": c.magnitude}
            for c in catalogue
        ]
    ).to_csv(OUT / "catalogue.csv", index=False)

    # -------------------------------------------------------------- screening
    modifiable = dict(zip(reactions["rxn_id"], reactions["clinically_modifiable"]))
    rows = []
    for candidate in catalogue:
        lb, ub = candidate.apply(gem.lb, gem.ub, index)
        keep = np.all((V_target >= lb - 1e-6) & (V_target <= ub + 1e-6), axis=1)
        report = RetentionReport(len(V_target), int(keep.sum()),
                                 float(keep.mean()), int(keep.sum()) >= MIN_RETAINED)
        record = {
            "intervention_id": candidate.intervention_id,
            "kind": candidate.kind,
            "targets": "|".join(candidate.targets),
            "metabolites": "|".join(candidate.metabolites),
            "roles": "|".join(candidate.roles),
            "max_toxicity": candidate.max_toxicity,
            "magnitude": candidate.magnitude,
            "clinically_modifiable": all(modifiable.get(t, False) for t in candidate.targets),
            "n_retained": report.n_retained,
            "retention": report.retention,
            "estimable": report.sufficient,
        }
        if report.sufficient:
            # The retained points are a subset of samples whose phenotype has
            # already been computed, so the screening tier costs one boolean
            # test per candidate rather than a fresh pass of the feature engine.
            d = baseline[keep]
            record.update(
                {
                    "median_distance": float(np.median(d)),
                    "mean_distance": float(np.mean(d)),
                    "recovery": float((baseline_median - np.median(d))
                                      / max(baseline_median - healthy_median, 1e-9)),
                    "fraction_healthy_like": float((d <= healthy_upper_band).mean()),
                }
            )
        else:
            record.update({"median_distance": np.nan, "mean_distance": np.nan,
                           "recovery": np.nan, "fraction_healthy_like": np.nan})
        rows.append(record)

    screening = pd.DataFrame(rows)
    screening.to_csv(OUT / "screening.csv", index=False)
    estimable = screening[screening["estimable"]].copy()
    print(f"{len(estimable)} of {len(screening)} candidates estimable at "
          f">= {MIN_RETAINED} retained samples")

    scored = utility_score(estimable)
    scored.to_csv(OUT / "ranking.csv", index=False)
    print()
    print(scored.head(15)[["intervention_id", "kind", "roles", "n_retained", "recovery",
                           "max_toxicity", "admissible", "utility"]].to_string(index=False))

    # ------------------------------------------------------------- validation
    #
    # The rejection estimate reuses points drawn from the unrestricted
    # polytope, so a tight intervention is estimated from a small and possibly
    # unrepresentative corner of it. The top candidates are therefore re-sampled
    # with a fresh chain before any of them is reported as effective.
    top = scored.head(N_VALIDATE)
    lookup = {c.intervention_id: c for c in catalogue}
    validation_rows = []
    mechanism_profiles = {}
    for _, row in top.iterrows():
        candidate = lookup[row["intervention_id"]]
        lb, ub = candidate.apply(gem.lb, gem.ub, index)
        warmup, report = rejection_sample(V_target, lb, ub, min_retained=2)
        if report.n_retained < 2:
            continue
        sampler = ACHRSampler(warmup, lb, ub, seed=config.RANDOM_SEED)
        fresh = sampler.sample(N_VALIDATION_SAMPLES, thinning=200, burn_in=2000)
        residual = float(np.abs(gem.S @ fresh.T).max())
        d = health_distance(fresh)
        phenotype = engine.transform(fresh)
        mechanism_profiles[candidate.intervention_id] = phenotype[
            [f for f in MECHANISM_FEATURES if f in phenotype]
        ].median()
        validation_rows.append(
            {
                "intervention_id": candidate.intervention_id,
                "kind": candidate.kind,
                "roles": "|".join(candidate.roles),
                "n_warmup": report.n_retained,
                "n_validation_samples": len(fresh),
                "max_steady_state_residual": residual,
                "screening_recovery": float(row["recovery"]),
                "validated_median_distance": float(np.median(d)),
                "validated_recovery": float((baseline_median - np.median(d))
                                            / max(baseline_median - healthy_median, 1e-9)),
                "admissible": bool(row["admissible"]),
                "max_toxicity": float(row["max_toxicity"]),
            }
        )
    validation = pd.DataFrame(validation_rows)
    validation.to_csv(OUT / "validation.csv", index=False)
    print()
    print(validation.to_string(index=False))

    # ------------------------------------------------------ mechanism clusters
    if len(mechanism_profiles) >= 3:
        profiles = pd.DataFrame(mechanism_profiles).T
        baseline_profile = engine.transform(V_target)[profiles.columns].median()
        relative = np.log2(
            (profiles + 1e-9).div(baseline_profile + 1e-9, axis=1).clip(lower=1e-6)
        )
        n_clusters = min(4, len(relative))
        labels = AgglomerativeClustering(n_clusters=n_clusters).fit_predict(relative.to_numpy())
        relative["mechanism_cluster"] = labels
        relative.to_csv(OUT / "mechanism_clusters.csv")
        print()
        print(relative.round(3).to_string())

## 34. The sampler control (`run_08b_sampler_control`)

The same confirmatory procedure run on the *unmodified* polytope, where the answer is known. This is what shows the confirmatory tier is not yet validated.

Writes: `sampler_control.csv`

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import stats



TARGET_CONDITION = "G6PD0.1_H2O21"
BUDGETS = [(800, 10), (800, 50), (1500, 50), (1500, 200)]


def run_08b_sampler_control() -> None:
    OUT = config.RESULTS_DIR / "interventions"
    config.ensure_dirs()
    gem = load_gem(config.MODEL_FILE, config.MODEL_NAME)
    reactions = annotate_reactions(gem)
    engine = FeatureEngine(gem, reactions)

    store = SampleStore(config.RAW_DIR)
    V, metadata = store.load_all()
    raw_features = pd.read_csv(config.PROCESSED_DIR / "phenotypes.csv.gz")
    core = pd.read_csv(config.RESULTS_DIR / "features" / "core_features.csv")
    internal = [f for f in core.loc[core["block"] == "internal", "feature"] if f in raw_features]

    healthy_mask = (metadata["state"] == "healthy").to_numpy()
    transform = PhenotypeTransform.fit(raw_features.loc[healthy_mask])
    reference = HealthyReference(
        transform.apply(raw_features.loc[healthy_mask])[internal], seed=config.RANDOM_SEED
    )

    def health_distance(flux):
        return reference.mahalanobis(transform.apply(engine.transform(flux))[internal])

    target = (metadata["condition"] == TARGET_CONDITION).to_numpy()
    V_target = V[target]
    lb = gem.lb.copy()
    ub = gem.ub.copy()
    j = gem.rxn_index(config.G6PD_REACTION)
    ub[j] = gem.ub[j] * 0.10
    reference_distance = health_distance(V_target)

    rows = []
    for n_samples, thinning in BUDGETS:
        sampler = ACHRSampler(V_target, lb, ub, seed=config.RANDOM_SEED)
        fresh = sampler.sample(n_samples, thinning=thinning, burn_in=10 * thinning)
        d = health_distance(fresh)
        rows.append(
            {
                "n_samples": n_samples,
                "thinning": thinning,
                "steps_per_sample": thinning,
                "total_steps": n_samples * thinning,
                "reference_median": float(np.median(reference_distance)),
                "achr_median": float(np.median(d)),
                "median_ratio": float(np.median(d) / np.median(reference_distance)),
                "ks_statistic": float(stats.ks_2samp(d, reference_distance).statistic),
                "max_steady_state_residual": float(np.abs(gem.S @ fresh.T).max()),
                "bounds_respected": bool(
                    np.all(fresh >= lb - 1e-6) and np.all(fresh <= ub + 1e-6)
                ),
            }
        )
        print(rows[-1])

    control = pd.DataFrame(rows)
    control["reproduces_reference"] = control["ks_statistic"] < 0.10
    control.to_csv(OUT / "sampler_control.csv", index=False)
    print()
    print(control.to_string(index=False))

## 35. Robustness (`run_09_robustness`)

Sample size, leave-one-chain-out, and ablation of each feature family.

Writes: `sample_size.csv`, `leave_one_chain_out.csv`, `feature_family_ablation.csv`, `internal_external_coupling.csv`

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import stats
from sklearn.cross_decomposition import CCA
from sklearn.metrics import roc_auc_score



def severity_scores(distance: np.ndarray, metadata: pd.DataFrame) -> dict:
    ok = np.isfinite(distance)
    is_disease = (metadata["state"] == "disease").to_numpy()[ok]
    level = metadata["g6pd_level"].to_numpy(dtype=float)[ok]
    d = distance[ok]
    means = [d[level == lv].mean() for lv in sorted(np.unique(level))]
    return {
        "spearman_with_g6pd_level": float(stats.spearmanr(level, d).statistic),
        "healthy_vs_disease_auc": float(roc_auc_score(is_disease, d)),
        "severity_monotone": bool(all(np.diff(means) < 0)),
    }


def run_09_robustness() -> None:
    OUT = config.RESULTS_DIR / "robustness"
    config.ensure_dirs()
    features = pd.read_csv(config.PROCESSED_DIR / "phenotypes_transformed.csv.gz")
    metadata = pd.read_csv(config.PROCESSED_DIR / "sample_metadata.csv.gz")
    core = pd.read_csv(config.RESULTS_DIR / "features" / "core_features.csv")

    internal = [f for f in core.loc[core["block"] == "internal", "feature"] if f in features]
    external = [f for f in core.loc[core["block"] == "external", "feature"] if f in features]
    healthy_mask = (metadata["state"] == "healthy").to_numpy()
    families = core.set_index("feature")["family"].to_dict()

    def build_and_score(rows: np.ndarray, columns: list[str]) -> dict:
        sub_meta = metadata.loc[rows].reset_index(drop=True)
        sub_features = features.loc[rows, columns].reset_index(drop=True)
        healthy_rows = (sub_meta["state"] == "healthy").to_numpy()
        if healthy_rows.sum() < 50 or (~healthy_rows).sum() < 50:
            return {}
        ref = HealthyReference(sub_features.loc[healthy_rows], seed=config.RANDOM_SEED)
        return severity_scores(ref.mahalanobis(sub_features), sub_meta)

    # ------------------------------------------------------------ sample size
    rng = np.random.default_rng(config.RANDOM_SEED)
    rows = []
    for n_per_condition in config.ROBUSTNESS_SAMPLE_SIZES:
        for replicate in range(3):
            selected = []
            for condition, block in metadata.groupby("condition"):
                take = min(n_per_condition, len(block))
                selected.append(rng.choice(block.index.to_numpy(), size=take, replace=False))
            idx = np.sort(np.concatenate(selected))
            scores = build_and_score(idx, internal)
            if scores:
                rows.append({"n_per_condition": n_per_condition, "replicate": replicate,
                             "n_total": len(idx), **scores})
    sample_size = pd.DataFrame(rows)
    sample_size.to_csv(OUT / "sample_size.csv", index=False)
    print(sample_size.groupby("n_per_condition")[
        ["spearman_with_g6pd_level", "healthy_vs_disease_auc"]].agg(["mean", "std"]).to_string())

    # ------------------------------------------------- leave one chain out
    rows = []
    for chain in sorted(metadata["chain"].unique()):
        idx = metadata.index[metadata["chain"] != chain].to_numpy()
        scores = build_and_score(idx, internal)
        if scores:
            rows.append({"chain_removed": int(chain), "n_total": len(idx), **scores})
    loco = pd.DataFrame(rows)
    loco.to_csv(OUT / "leave_one_chain_out.csv", index=False)
    print()
    print(loco.to_string(index=False))

    # --------------------------------------------------- feature family ablation
    rows = []
    all_idx = metadata.index.to_numpy()
    for family in sorted({families[f] for f in internal}):
        columns = [f for f in internal if families[f] == family]
        if len(columns) < 2:
            continue
        rows.append({"feature_set": f"only_{family}", "n_features": len(columns),
                     **build_and_score(all_idx, columns)})
        remaining = [f for f in internal if families[f] != family]
        rows.append({"feature_set": f"without_{family}", "n_features": len(remaining),
                     **build_and_score(all_idx, remaining)})
    rows.append({"feature_set": "full_internal_core", "n_features": len(internal),
                 **build_and_score(all_idx, internal)})
    rows.append({"feature_set": "external_core_only", "n_features": len(external),
                 **build_and_score(all_idx, external)})
    ablation = pd.DataFrame(rows).sort_values(
        "spearman_with_g6pd_level", key=lambda s: s.abs(), ascending=False, ignore_index=True
    )
    ablation.to_csv(OUT / "feature_family_ablation.csv", index=False)
    print()
    print(ablation.to_string(index=False))

    # ------------------------------------------- internal / external coupling
    #
    # Canonical correlation measures the strongest linear relationship that
    # exists between any combination of internal features and any combination
    # of external features. Comparing it with a permutation null separates a
    # real dependence from the correlation two random blocks of this size and
    # sample count would show anyway.
    rows = []
    for condition, block in metadata.groupby("condition"):
        idx = block.index.to_numpy()
        Xi = features.loc[idx, internal].to_numpy(dtype=float)
        Xe = features.loc[idx, external].to_numpy(dtype=float)
        ok = np.isfinite(Xi).all(axis=1) & np.isfinite(Xe).all(axis=1)
        Xi, Xe = Xi[ok], Xe[ok]
        Xi = (Xi - Xi.mean(0)) / np.where(Xi.std(0) > 0, Xi.std(0), 1)
        Xe = (Xe - Xe.mean(0)) / np.where(Xe.std(0) > 0, Xe.std(0), 1)

        cca = CCA(n_components=1, max_iter=1000).fit(Xi, Xe)
        u, v = cca.transform(Xi, Xe)
        observed = float(abs(np.corrcoef(u[:, 0], v[:, 0])[0, 1]))

        null = []
        rng_local = np.random.default_rng(config.RANDOM_SEED)
        for _ in range(10):
            permutation = rng_local.permutation(len(Xe))
            cca_null = CCA(n_components=1, max_iter=500).fit(Xi, Xe[permutation])
            un, vn = cca_null.transform(Xi, Xe[permutation])
            null.append(abs(np.corrcoef(un[:, 0], vn[:, 0])[0, 1]))
        rows.append(
            {
                "condition": condition,
                "n_samples": int(ok.sum()),
                "first_canonical_correlation": observed,
                "permutation_null_mean": float(np.mean(null)),
                "permutation_null_p95": float(np.quantile(null, 0.95)),
                "excess_over_null": observed - float(np.mean(null)),
                "shared_variance": observed ** 2,
            }
        )
    coupling = pd.DataFrame(rows)
    coupling.to_csv(OUT / "internal_external_coupling.csv", index=False)
    print()
    print(coupling.to_string(index=False))

## 36. Per-feature coupling (`run_09b_coupling_detail`)

How much of each internal feature the boundary can predict, which is what reconciles a canonical correlation of 0.94 with an R² of zero.

Writes: `internal_feature_predictability.csv`

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import stats
from sklearn.linear_model import RidgeCV
from sklearn.model_selection import GroupKFold, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler



STRIDE = 5


def run_09b_coupling_detail() -> None:
    OUT = config.RESULTS_DIR / "robustness"
    config.ensure_dirs()
    features = pd.read_csv(config.PROCESSED_DIR / "phenotypes_transformed.csv.gz")
    metadata = pd.read_csv(config.PROCESSED_DIR / "sample_metadata.csv.gz")
    core = pd.read_csv(config.RESULTS_DIR / "features" / "core_features.csv")
    discrimination = pd.read_csv(config.RESULTS_DIR / "features" / "feature_discrimination.csv")

    internal = [f for f in core.loc[core["block"] == "internal", "feature"] if f in features]
    external = [f for f in core.loc[core["block"] == "external", "feature"] if f in features]
    families = core.set_index("feature")["family"].to_dict()

    rows_idx = np.flatnonzero(np.arange(len(metadata)) % STRIDE == 0)
    X = features.iloc[rows_idx][external].replace([np.inf, -np.inf], np.nan)
    X = X.fillna(X.median())
    groups = metadata.iloc[rows_idx]["stem"].to_numpy()
    splitter = GroupKFold(n_splits=5)
    model = Pipeline([("scale", StandardScaler()),
                      ("ridge", RidgeCV(alphas=np.logspace(-2, 4, 13)))])

    severity = (
        discrimination[discrimination["axis"] == "g6pd_including_healthy"]
        .set_index("feature")["epsilon_squared"].to_dict()
    )

    rows = []
    for feature in internal:
        y = features.iloc[rows_idx][feature].to_numpy(dtype=float)
        if not np.isfinite(y).all() or np.ptp(y) == 0:
            continue
        pred = cross_val_predict(model, X, y, cv=splitter, groups=groups)
        ss_res = float(((y - pred) ** 2).sum())
        ss_tot = float(((y - y.mean()) ** 2).sum())
        rows.append(
            {
                "feature": feature,
                "family": families.get(feature, ""),
                "predictability_r2_from_external": 1.0 - ss_res / ss_tot if ss_tot > 0 else np.nan,
                "severity_epsilon_squared": severity.get(feature, np.nan),
            }
        )
    table = pd.DataFrame(rows)
    table.to_csv(OUT / "internal_feature_predictability.csv", index=False)

    valid = table.dropna(subset=["predictability_r2_from_external", "severity_epsilon_squared"])
    rho = stats.spearmanr(
        valid["predictability_r2_from_external"], valid["severity_epsilon_squared"]
    )
    print(f"internal core features regressed on the external block: {len(valid)}")
    print(f"median predictability R2 = {valid['predictability_r2_from_external'].median():.3f}")
    print(f"features with R2 > 0.5: {(valid['predictability_r2_from_external'] > 0.5).sum()}")
    print(f"Spearman(predictability, severity effect) = {rho.statistic:.3f} (p = {rho.pvalue:.2g})")
    print()
    print(table.groupby("family")[
        ["predictability_r2_from_external", "severity_epsilon_squared"]
    ].median().sort_values("severity_epsilon_squared", ascending=False).to_string())
    print()
    print("most severity-informative internal features and their predictability:")
    print(valid.sort_values("severity_epsilon_squared", ascending=False).head(12).to_string(index=False))

## 37. Do the QC thresholds matter? (`run_11_threshold_sensitivity`)

Each of the four thresholds tightened and loosened by a factor of two.

Writes: `threshold_sensitivity.csv`

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import spearmanr



BASE = {"missing": 0.01, "outlier": 0.10, "chain": 0.02, "rho": 0.95}
VARIANTS = [
    ("baseline", BASE),
    ("missing tightened", {**BASE, "missing": 0.005}),
    ("missing loosened", {**BASE, "missing": 0.02}),
    ("outlier tightened", {**BASE, "outlier": 0.05}),
    ("outlier loosened", {**BASE, "outlier": 0.20}),
    ("chain tightened", {**BASE, "chain": 0.01}),
    ("chain loosened", {**BASE, "chain": 0.04}),
    ("redundancy tightened", {**BASE, "rho": 0.90}),
    ("redundancy loosened", {**BASE, "rho": 0.975}),
]


def redundancy_representatives(corr: pd.DataFrame, rho_cut: float,
                               candidates: list[str]) -> list[str]:
    """Average-linkage clustering on 1 - |rho|, one representative per cluster.

    Implemented directly rather than through scipy so that the linkage is the
    same average-linkage rule used in stage 4 and the tie-breaking is explicit.
    """
    from scipy.cluster.hierarchy import fcluster, linkage
    from scipy.spatial.distance import squareform

    # A descriptor whose correlation with another is undefined — one of them is
    # constant over the states being clustered — cannot be placed in the
    # dendrogram. It is dropped from the clustering and kept as its own
    # representative, which is the same thing the cascade does with a constant
    # descriptor, rather than being given an imputed correlation of zero.
    block = corr.loc[candidates, candidates]
    usable = [c for c in candidates if np.isfinite(block.loc[c]).all()]
    undefined = [c for c in candidates if c not in usable]
    if undefined:
        print(f"  {len(undefined)} descriptors have an undefined correlation and are "
              f"kept outside the redundancy clustering: {', '.join(undefined[:6])}"
              + (" ..." if len(undefined) > 6 else ""))
    if len(usable) < 2:
        return list(candidates)
    candidates = usable
    sub = corr.loc[candidates, candidates].to_numpy()
    distance = 1.0 - np.abs(sub)
    np.fill_diagonal(distance, 0.0)
    distance = (distance + distance.T) / 2.0
    labels = fcluster(linkage(squareform(distance, checks=False), method="average"),
                      t=1.0 - rho_cut, criterion="distance")
    representatives = []
    for label in np.unique(labels):
        members = [c for c, l in zip(candidates, labels) if l == label]
        if len(members) == 1:
            representatives.append(members[0])
            continue
        block = corr.loc[members, members].abs()
        representatives.append(block.mean(axis=1).idxmax())
    return representatives


def run_11_threshold_sensitivity() -> None:
    OUT = config.RESULTS_DIR / "features"
    config.ensure_dirs()
    decision = pd.read_csv(OUT / "feature_qc_decision.csv")
    corr = pd.read_csv(OUT / "feature_correlation_matrix.csv", index_col=0)
    phenotypes = pd.read_csv(config.PROCESSED_DIR / "phenotypes.csv.gz")
    metadata = pd.read_csv(config.RESULTS_DIR / "landscape" / "pca_embedding.csv.gz",
                           usecols=["state", "g6pd_level"])

    healthy = (metadata["state"] == "healthy").to_numpy()
    transform = PhenotypeTransform.fit(phenotypes.loc[healthy])
    transformed = transform.apply(phenotypes)

    rows = []
    for name, t in VARIANTS:
        keep = (~decision["constant"].astype(bool)
                & (decision["fraction_missing"] <= t["missing"])
                & (decision["fraction_outliers"] <= t["outlier"])
                & (decision["between_chain_variance_share"].fillna(0.0) <= t["chain"]))
        candidates = [f for f in decision.loc[keep, "feature"] if f in corr.index]
        representatives = redundancy_representatives(corr, t["rho"], candidates)
        core = decision[decision["feature"].isin(representatives)]
        internal = [f for f in core.loc[core["block"] == "internal", "feature"]
                    if f in transformed]

        reference = HealthyReference(transformed.loc[healthy, internal],
                                     seed=config.RANDOM_SEED)
        distance = reference.mahalanobis(transformed[internal])
        rho = spearmanr(distance, metadata["g6pd_level"]).statistic
        disease = metadata["state"].to_numpy() == "disease"
        auc = float((distance[disease][:, None] > distance[~disease][None, :]).mean())

        rows.append({"variant": name, **t, "n_core": len(core),
                     "n_internal": len(internal),
                     "spearman_with_g6pd_level": float(rho),
                     "healthy_vs_disease_auc": auc})
        print(f"{name:24s} core={len(core):4d} internal={len(internal):4d} "
              f"rho={rho:+.3f} auc={auc:.3f}")

    frame = pd.DataFrame(rows)
    frame.to_csv(OUT / "threshold_sensitivity.csv", index=False)
    base = frame.iloc[0]
    print()
    print(f"core size range: {frame['n_core'].min()}-{frame['n_core'].max()} "
          f"(baseline {int(base['n_core'])})")
    print(f"largest change in |rho|: "
          f"{(frame['spearman_with_g6pd_level'] - base['spearman_with_g6pd_level']).abs().max():.4f}")

## 38. The same samples in two representations (`run_12_raw_vs_phenotype`)

Raw flux and engineered phenotype through one identical unsupervised analysis, with no condition label shown to either.

Writes: `representation_comparison.csv`, `representation_clusters.csv`, `representation_embedding.csv.gz`

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import adjusted_rand_score, silhouette_score



K_VALUES = (2, 3, 4, 5, 6)
N_SILHOUETTE = 4000      # silhouette is O(n^2); evaluated on a fixed subsample
RAW_VS_PHENOTYPE_BOOTSTRAP = 5
N_COMPONENTS = 50        # clustering happens in the same reduced space for both
SEED = config.RANDOM_SEED


def standardise(X: np.ndarray, healthy: np.ndarray) -> np.ndarray:
    """Centre and scale on the healthy samples, dropping degenerate columns.

    Fitting the scaling on healthy samples only keeps the two representations
    on the same footing: it is the rule already used for the phenotype, and it
    means neither representation is scaled using the diseased samples it will
    later be asked to separate.
    """
    centre = X[healthy].mean(axis=0)
    scale = X[healthy].std(axis=0, ddof=1)
    # A column whose healthy standard deviation is a rounding error relative to
    # its own range is not "varying in health"; dividing by it would make that
    # one coordinate worth 1e14 units and decide the geometry on its own. The
    # test is relative to the column's range, not against zero.
    spread = X.max(axis=0) - X.min(axis=0)
    keep = scale >= 1e-9 * np.where(spread > 0, spread, 1.0)
    dropped = int((~keep).sum())
    if dropped:
        print(f"  {dropped} columns have no usable spread in the healthy samples "
              f"and are excluded from this comparison")
    return (X[:, keep] - centre[keep]) / scale[keep]


def continuum_score(Z: np.ndarray, labels: np.ndarray, rng) -> float:
    """Separation of a partition, as a standardised gap between distances.

    Zero means the within-cluster and between-cluster distance distributions
    coincide -- a continuum cut arbitrarily. Values near one mean the clusters
    are separated by a gap, that is, genuinely discrete states.
    """
    from scipy.spatial.distance import pdist

    # pdist keeps the condensed upper triangle only; the dense difference
    # tensor this replaces needed several gigabytes at these dimensions
    idx = rng.choice(len(Z), size=min(2500, len(Z)), replace=False)
    Zs, ls = Z[idx], labels[idx]
    d = pdist(Zs)
    same = pdist(ls.reshape(-1, 1), metric=lambda a, b: float(a[0] == b[0])).astype(bool)
    within = d[same]
    between = d[~same]
    if not len(within) or not len(between):
        return float("nan")
    pooled = np.sqrt((within.var() + between.var()) / 2)
    return float((between.mean() - within.mean()) / pooled) if pooled > 0 else float("nan")


def analyse(name: str, Z: np.ndarray, metadata: pd.DataFrame) -> tuple[list[dict], dict]:
    """Reduce, cluster and score one representation.

    Both representations are clustered in the space of their own leading 50
    principal components rather than in their native dimension. This is not a
    convenience: clustering a 2,166-dimensional flux matrix directly against a
    98-dimensional feature matrix would confound the representation with the
    dimension, which is the confound the comparison exists to remove.
    """
    rng = np.random.default_rng(SEED)
    pca = PCA(n_components=min(N_COMPONENTS, Z.shape[1]), random_state=SEED).fit(Z)
    full = pca.transform(Z)
    embedding = full[:, :3]
    Z = full                                   # cluster in the reduced space
    # The silhouette subsample is drawn per partition, inside the loop over k,
    # so that it always contains every cluster of the partition it scores.

    rows = []
    for k in K_VALUES:
        labels = KMeans(n_clusters=k, n_init=6, random_state=SEED).fit_predict(Z)
        stability = []
        for b in range(RAW_VS_PHENOTYPE_BOOTSTRAP):
            boot = np.random.default_rng(SEED + b).choice(len(Z), size=len(Z), replace=True)
            relabel = KMeans(n_clusters=k, n_init=2,
                             random_state=SEED + b).fit(Z[boot]).predict(Z)
            stability.append(adjusted_rand_score(labels, relabel))
        rows.append({
            "representation": name,
            "k": k,
            "silhouette": _silhouette_of(Z, labels, N_SILHOUETTE, SEED),
            "ari_with_severity": float(adjusted_rand_score(metadata["g6pd_level"], labels)),
            "ari_with_h2o2_null": float(adjusted_rand_score(metadata["h2o2_level"], labels)),
            "ari_with_chain_null": float(adjusted_rand_score(metadata["chain"], labels)),
            "bootstrap_stability_ari": float(np.mean(stability)),
            "separation_score": continuum_score(Z, labels, rng),
        })
        print(f"  {name:11s} k={k}  silhouette={rows[-1]['silhouette']:+.3f}  "
              f"ARI(severity)={rows[-1]['ari_with_severity']:.3f}  "
              f"stability={rows[-1]['bootstrap_stability_ari']:.3f}  "
              f"separation={rows[-1]['separation_score']:.3f}")

    summary = {
        "representation": name,
        "n_dimensions": Z.shape[1],
        "pc1_variance": float(pca.explained_variance_ratio_[0]),
        "variance_in_10_components": float(pca.explained_variance_ratio_[:10].sum()),
        "variance_in_50_components": float(pca.explained_variance_ratio_.sum()),
        "best_silhouette": max(r["silhouette"] for r in rows),
        "best_ari_with_severity": max(r["ari_with_severity"] for r in rows),
        "max_ari_with_nulls": max(max(r["ari_with_h2o2_null"], r["ari_with_chain_null"])
                                  for r in rows),
        "mean_bootstrap_stability": float(np.mean([r["bootstrap_stability_ari"] for r in rows])),
    }
    return rows, {"summary": summary, "embedding": embedding}


def _silhouette_of(Z, labels, size, seed):
    """Silhouette on a subsample that contains every cluster, or NaN."""
    subsample = stratified_subsample(labels, size, seed)
    if len(np.unique(labels[subsample])) < 2:
        return float("nan")
    return float(silhouette_score(Z[subsample], labels[subsample]))


def run_12_raw_vs_phenotype() -> None:
    OUT = config.RESULTS_DIR / "landscape"
    config.ensure_dirs()
    metadata = pd.read_csv(config.PROCESSED_DIR / "sample_metadata.csv.gz")
    healthy = (metadata["state"] == "healthy").to_numpy()

    # -- representation A: the raw internal flux vectors -------------------
    gem = load_gem(config.MODEL_FILE, config.MODEL_NAME)
    reactions = annotate_reactions(gem)
    internal_rxns = reactions["is_internal"].to_numpy().astype(bool)
    store = SampleStore(config.RAW_DIR)
    V, meta_check = store.load_all()
    assert len(meta_check) == len(metadata)
    Z_raw = standardise(V[:, internal_rxns], healthy)
    print(f"raw internal flux: {Z_raw.shape[1]} varying reactions of {internal_rxns.sum()}")

    # -- representation B: the internal core phenotype ---------------------
    features = pd.read_csv(config.PROCESSED_DIR / "phenotypes_transformed.csv.gz")
    core = pd.read_csv(config.RESULTS_DIR / "features" / "core_features.csv")
    internal = [f for f in core.loc[core["block"] == "internal", "feature"] if f in features]
    Z_phi = standardise(features[internal].to_numpy(dtype=float), healthy)
    print(f"internal core phenotype: {Z_phi.shape[1]} features")

    rows, embeddings = [], {}
    for name, Z in [("raw flux", Z_raw), ("phenotype", Z_phi)]:
        part, extra = analyse(name, Z, metadata)
        rows.extend(part)
        embeddings[name] = extra

    clusters = pd.DataFrame(rows)
    clusters.to_csv(OUT / "representation_clusters.csv", index=False)
    comparison = pd.DataFrame([embeddings[n]["summary"] for n in embeddings])
    comparison.to_csv(OUT / "representation_comparison.csv", index=False)

    frames = []
    for name, extra in embeddings.items():
        frame = pd.DataFrame(extra["embedding"], columns=["PC1", "PC2", "PC3"])
        frame["representation"] = name
        frame["g6pd_level"] = metadata["g6pd_level"].to_numpy()
        frame["state"] = metadata["state"].to_numpy()
        frame["chain"] = metadata["chain"].to_numpy()
        frames.append(frame)
    pd.concat(frames, ignore_index=True).to_csv(
        OUT / "representation_embedding.csv.gz", index=False)

    print()
    print(comparison.to_string(index=False))

## 39. Contribution without a fitted model (`run_13_feature_contribution`)

Six measures that involve no learned model, and the finding that they disagree.

Writes: `feature_contribution.csv`, `contribution_agreement.csv`

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import spearmanr
from sklearn.decomposition import PCA
from sklearn.feature_selection import mutual_info_regression



MI_SUBSAMPLE = 6000     # mutual information is the slowest measure by far


def run_13_feature_contribution() -> None:
    OUT = config.RESULTS_DIR / "features"
    config.ensure_dirs()
    features = pd.read_csv(config.PROCESSED_DIR / "phenotypes_transformed.csv.gz")
    metadata = pd.read_csv(config.PROCESSED_DIR / "sample_metadata.csv.gz")
    core = pd.read_csv(config.RESULTS_DIR / "features" / "core_features.csv")

    internal = [f for f in core.loc[core["block"] == "internal", "feature"] if f in features]
    healthy = (metadata["state"] == "healthy").to_numpy()
    X = features[internal]
    reference = HealthyReference(X.loc[healthy], seed=config.RANDOM_SEED)
    distance = reference.mahalanobis(X)
    severity = metadata["g6pd_level"].to_numpy()
    base_rho = float(spearmanr(distance, severity).statistic)
    print(f"full internal core: {len(internal)} features, rho = {base_rho:+.3f}")

    # -- the cheap measures, computed on everything ------------------------
    Xv = X.to_numpy(dtype=float)
    centre = np.median(Xv[healthy], axis=0)
    scale = np.subtract(*np.percentile(Xv[healthy], [75, 25], axis=0)) / 1.349
    scale = np.where(scale > 0, scale, np.where(Xv[healthy].std(axis=0) > 0,
                                                Xv[healthy].std(axis=0), 1.0))
    Z = (Xv - centre) / scale

    rho = np.array([spearmanr(Xv[:, k], distance).statistic for k in range(Xv.shape[1])])
    covariance = np.array([np.cov(Z[:, k], distance)[0, 1] for k in range(Z.shape[1])])

    rng = np.random.default_rng(config.RANDOM_SEED)
    sub = rng.choice(len(Z), size=min(MI_SUBSAMPLE, len(Z)), replace=False)
    mi = mutual_info_regression(Z[sub], distance[sub], random_state=config.RANDOM_SEED)

    pca = PCA(n_components=3, random_state=config.RANDOM_SEED).fit(Z[healthy])
    loading = np.abs(pca.components_[0])

    # Closed-form sensitivity of the Mahalanobis distance to coordinate k,
    # evaluated at the mean diseased phenotype: dD/dz = (P z) / D.
    precision = reference.precision
    z_disease = ((Xv[~healthy].mean(axis=0) - reference.centre) / reference.scale)
    d0 = float(np.sqrt(z_disease @ precision @ z_disease))
    sensitivity = np.abs(precision @ z_disease) / max(d0, 1e-12)

    # -- ablation: remove one feature, recompute the distance --------------
    ablation = np.full(len(internal), np.nan)
    for k, feature in enumerate(internal):
        kept = [f for f in internal if f != feature]
        ref_k = HealthyReference(X.loc[healthy, kept], seed=config.RANDOM_SEED)
        rho_k = float(spearmanr(ref_k.mahalanobis(X[kept]), severity).statistic)
        ablation[k] = abs(base_rho) - abs(rho_k)
        if (k + 1) % 20 == 0:
            print(f"  ablated {k + 1}/{len(internal)}")

    contribution = pd.DataFrame({
        "feature": internal,
        "family": core.set_index("feature").loc[internal, "family"].to_numpy(),
        "spearman_with_health_distance": rho,
        "covariance_with_health_distance": covariance,
        "mutual_information": mi,
        "pca1_loading": loading,
        "sensitivity": sensitivity,
        "ablation_delta_rho": ablation,
        "severity_epsilon_squared":
            core.set_index("feature").loc[internal, "epsilon_squared"].to_numpy(),
    })
    contribution.to_csv(OUT / "feature_contribution.csv", index=False)

    # -- do the measures agree, and do they agree with the model? ----------
    shapley = pd.read_csv(config.RESULTS_DIR / "ml" / "shapley_importance.csv")
    measures = ["spearman_with_health_distance", "covariance_with_health_distance",
                "mutual_information", "pca1_loading", "sensitivity",
                "ablation_delta_rho", "severity_epsilon_squared"]
    ranks = contribution[measures].abs()
    agreement = ranks.corr(method="spearman")
    agreement.to_csv(OUT / "contribution_agreement.csv")

    print()
    print("agreement between contribution measures (Spearman of |value|):")
    print(agreement.round(2).to_string())
    print()
    top = contribution.reindex(contribution["ablation_delta_rho"].abs()
                               .sort_values(ascending=False).index).head(12)
    print(top[["feature", "family", "ablation_delta_rho", "mutual_information",
               "severity_epsilon_squared"]].to_string(index=False))
    print()
    print(f"largest single-feature ablation effect: "
          f"{contribution['ablation_delta_rho'].abs().max():.4f} in |rho| "
          f"(baseline {abs(base_rho):.3f}); "
          f"{int((contribution['ablation_delta_rho'].abs() > 0.01).sum())} features "
          f"change |rho| by more than 0.01")
    if len(shapley):
        print(f"model-based importances available for {len(shapley)} external features "
              f"(stage 7); the two sets are disjoint by construction.")

## 40. Family combinations and the minimal phenotype (`run_14_minimal_phenotype`)

Whether the families combine, and how compact the phenotype can be made.

Writes: `family_combinations.csv`, `minimal_phenotype.csv`

In [ ]:
import itertools
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import spearmanr



#: The families, grouped as the review groups them.
GROUPS = {
    "mathematical": ["magnitude", "distribution", "sparsity", "entropy", "architecture"],
    "pathway": ["pathway"],
    "biochemical": ["biochemical", "turnover", "efficiency"],
    "network": ["network"],
}
MAX_FEATURES = 25


def score(X: pd.DataFrame, healthy: np.ndarray, severity: np.ndarray,
          columns: list[str]) -> tuple[float, float]:
    """Severity recovery and healthy-versus-disease AUC for a feature subset."""
    if not columns:
        return float("nan"), float("nan")
    reference = HealthyReference(X.loc[healthy, columns], seed=config.RANDOM_SEED)
    d = reference.mahalanobis(X[columns])
    rho = float(spearmanr(d, severity).statistic)
    disease = severity < 1.0
    auc = float((d[disease][:, None] > d[~disease][None, :]).mean())
    return rho, auc


def run_14_minimal_phenotype() -> None:
    OUT = config.RESULTS_DIR / "robustness"
    config.ensure_dirs()
    features = pd.read_csv(config.PROCESSED_DIR / "phenotypes_transformed.csv.gz")
    metadata = pd.read_csv(config.PROCESSED_DIR / "sample_metadata.csv.gz")
    core = pd.read_csv(config.RESULTS_DIR / "features" / "core_features.csv")

    internal = core[core["block"] == "internal"]
    internal = internal[internal["feature"].isin(features.columns)]
    columns = internal["feature"].tolist()
    family_of = dict(zip(internal["feature"], internal["family"]))
    X = features[columns]
    healthy = (metadata["state"] == "healthy").to_numpy()
    severity = metadata["g6pd_level"].to_numpy()

    members = {group: [f for f in columns if family_of[f] in fams]
               for group, fams in GROUPS.items()}

    # ------------------------------------------------ family combinations
    rows = []
    names = list(GROUPS)
    for size in range(1, len(names) + 1):
        for combination in itertools.combinations(names, size):
            selected = sorted({f for g in combination for f in members[g]})
            if not selected:
                continue
            rho, auc = score(X, healthy, severity, selected)
            rows.append({
                "combination": " + ".join(combination),
                "n_groups": size,
                "n_features": len(selected),
                "spearman_with_g6pd_level": rho,
                "healthy_vs_disease_auc": auc,
            })
            print(f"{rows[-1]['combination']:48s} n={len(selected):3d} "
                  f"rho={rho:+.3f} auc={auc:.3f}")

    combinations = pd.DataFrame(rows).sort_values("spearman_with_g6pd_level")
    combinations.to_csv(OUT / "family_combinations.csv", index=False)

    # Does adding the mathematical families to a biochemical set help?
    def get(name):
        row = combinations[combinations["combination"] == name]
        return float(row["spearman_with_g6pd_level"].iloc[0]) if len(row) else float("nan")

    print()
    print(f"biochemical alone                : rho = {get('biochemical'):+.3f}")
    print(f"biochemical + mathematical       : rho = {get('mathematical + biochemical'):+.3f}")
    print(f"pathway + biochemical            : rho = {get('pathway + biochemical'):+.3f}")
    print(f"all four groups                  : "
          f"rho = {get('mathematical + pathway + biochemical + network'):+.3f}")

    # -------------------------------------------------- minimal phenotype
    print()
    selected: list[str] = []
    remaining = list(columns)
    curve = []
    while len(selected) < MAX_FEATURES and remaining:
        best, best_rho, best_auc = None, 0.0, np.nan
        for candidate in remaining:
            rho, auc = score(X, healthy, severity, selected + [candidate])
            if abs(rho) > abs(best_rho):
                best, best_rho, best_auc = candidate, rho, auc
        if best is None:
            break
        selected.append(best)
        remaining.remove(best)
        curve.append({"step": len(selected), "feature": best,
                      "family": family_of[best],
                      "spearman_with_g6pd_level": best_rho,
                      "healthy_vs_disease_auc": best_auc})
        print(f"  {len(selected):2d}  {best:42s} {family_of[best]:14s} "
              f"rho={best_rho:+.3f} auc={best_auc:.3f}")

    minimal = pd.DataFrame(curve)
    full_rho, full_auc = score(X, healthy, severity, columns)
    minimal["fraction_of_full_rho"] = minimal["spearman_with_g6pd_level"] / full_rho
    minimal.to_csv(OUT / "minimal_phenotype.csv", index=False)

    reached = minimal[minimal["fraction_of_full_rho"] >= 0.95]
    print()
    print(f"full internal core ({len(columns)} features): rho = {full_rho:+.3f}, "
          f"auc = {full_auc:.3f}")
    if len(reached):
        k = int(reached["step"].iloc[0])
        print(f"95% of the full severity recovery is reached with {k} features")
        print("minimal phenotype:", ", ".join(minimal["feature"].head(k)))

## 41. Recovery profiles per intervention (`run_15_recovery_profiles`)

What each intervention changes, not only how much.

Writes: `recovery_profiles.csv`, `recovery_clusters.csv`

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.cluster import AgglomerativeClustering



TARGET_CONDITION = "G6PD0.1_H2O21"
MIN_RETAINED = 300
N_CLUSTERS = 4

#: The five biochemical axes, each named by the feature that measures it.
AXES = {
    "redox (NADPH)": "turnover_nadph_c",
    "glutathione": "turnover_gthrd_c",
    "energy (ATP)": "turnover_atp_c",
    "oxidative (H2O2)": "turnover_h2o2_c",
    "pentose phosphate": "path_pentose_phosphate_activity",
}
#: An axis needs a healthy/disease gap of at least this fraction of the diseased
#: median before a ratio of differences means anything.
MIN_GAP = 0.02


def run_15_recovery_profiles() -> None:
    OUT = config.RESULTS_DIR / "interventions"
    config.ensure_dirs()
    gem = load_gem(config.MODEL_FILE, config.MODEL_NAME)
    reactions = annotate_reactions(gem)
    index = {r: i for i, r in enumerate(gem.rxns)}

    features = pd.read_csv(config.PROCESSED_DIR / "phenotypes.csv.gz")
    metadata = pd.read_csv(config.PROCESSED_DIR / "sample_metadata.csv.gz")
    distances = pd.read_csv(config.RESULTS_DIR / "distance" / "health_distances.csv.gz",
                            usecols=["condition", "phenotype_mahalanobis"])
    store = SampleStore(config.RAW_DIR)
    V, _ = store.load_all()

    healthy = (metadata["state"] == "healthy").to_numpy()
    target = (metadata["condition"] == TARGET_CONDITION).to_numpy()
    V_target = V[target]

    available = {name: column for name, column in AXES.items() if column in features}
    missing = set(AXES) - set(available)
    if missing:
        print(f"axes not present in the feature table and therefore skipped: {sorted(missing)}")

    healthy_median = features.loc[healthy, list(available.values())].median()
    disease_median = features.loc[target, list(available.values())].median()
    gap = healthy_median - disease_median
    usable = {name: column for name, column in available.items()
              if abs(gap[column]) > MIN_GAP * max(abs(disease_median[column]), 1e-9)}
    print("axes with room to recover:", ", ".join(usable) or "none")

    d_target = distances.loc[(distances["condition"] == TARGET_CONDITION).to_numpy(),
                             "phenotype_mahalanobis"].to_numpy()
    d_healthy = distances.loc[(distances["condition"] == "healthy").to_numpy(),
                              "phenotype_mahalanobis"].to_numpy()
    d_disease_median, d_healthy_median = float(np.median(d_target)), float(np.median(d_healthy))

    target_features = features.loc[target].reset_index(drop=True)
    catalogue = build_catalogue(reactions, V[healthy])
    print(f"{len(catalogue)} candidates; profiling those that retain "
          f">= {MIN_RETAINED} samples")

    rows = []
    for candidate in catalogue:
        lb, ub = candidate.apply(gem.lb, gem.ub, index)
        keep = np.all((V_target >= lb - 1e-6) & (V_target <= ub + 1e-6), axis=1)
        if int(keep.sum()) < MIN_RETAINED:
            continue
        record = {
            "intervention_id": candidate.intervention_id,
            "kind": candidate.kind,
            "roles": "|".join(candidate.roles),
            "max_toxicity": candidate.max_toxicity,
            "n_retained": int(keep.sum()),
            "recovery_health_distance":
                float((d_disease_median - np.median(d_target[keep]))
                      / max(d_disease_median - d_healthy_median, 1e-9)),
        }
        for name, column in usable.items():
            intervened = float(target_features.loc[keep, column].median())
            record[f"recovery::{name}"] = float(
                (intervened - disease_median[column]) / gap[column])
        rows.append(record)

    profiles = pd.DataFrame(rows)
    profiles.to_csv(OUT / "recovery_profiles.csv", index=False)
    print(f"{len(profiles)} recovery profiles over {len(usable)} axes")

    axis_columns = [f"recovery::{name}" for name in usable]
    P = profiles[axis_columns].to_numpy()
    # clip before clustering: a handful of candidates have enormous ratios on an
    # axis with a small gap, and they would otherwise define the clusters
    P = np.clip(P, -2.0, 2.0)
    labels = AgglomerativeClustering(n_clusters=N_CLUSTERS).fit_predict(P)
    profiles["mechanism_cluster"] = labels

    summary = profiles.groupby("mechanism_cluster").agg(
        n=("intervention_id", "size"),
        median_health_recovery=("recovery_health_distance", "median"),
        best_health_recovery=("recovery_health_distance", "max"),
        **{f"median{c[8:]}": (c, "median") for c in axis_columns},
    )
    top = (profiles.sort_values("recovery_health_distance", ascending=False)
           .groupby("mechanism_cluster")["intervention_id"].first())
    summary["exemplar"] = top
    summary.to_csv(OUT / "recovery_clusters.csv")
    profiles.to_csv(OUT / "recovery_profiles.csv", index=False)

    print()
    print(summary.round(3).to_string())
    print()
    best = profiles.nlargest(8, "recovery_health_distance")
    print(best[["intervention_id", "mechanism_cluster", "recovery_health_distance"]
               + axis_columns].round(3).to_string(index=False))

## 42. The geometry of the representation (`run_16_geometry`)

The exact additive decomposition of the squared Mahalanobis distance, partial correlations, and the separation between redundancy and complementarity.

Writes: `mahalanobis_decomposition.csv`, `partial_correlations.csv`, `feature_interactions.csv`

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.covariance import LedoitWolf



N_INTERACTIONS = 40


def partial_correlation_matrix(Z: np.ndarray) -> np.ndarray:
    """Partial correlations from the precision matrix: -P_ij / sqrt(P_ii P_jj)."""
    precision = LedoitWolf(assume_centered=False).fit(Z).precision_
    d = np.sqrt(np.diag(precision))
    partial = -precision / np.outer(d, d)
    np.fill_diagonal(partial, 1.0)
    return partial


def run_16_geometry() -> None:
    OUT = config.RESULTS_DIR / "features"
    config.ensure_dirs()
    features = pd.read_csv(config.PROCESSED_DIR / "phenotypes_transformed.csv.gz")
    metadata = pd.read_csv(config.PROCESSED_DIR / "sample_metadata.csv.gz")
    core = pd.read_csv(config.RESULTS_DIR / "features" / "core_features.csv")

    internal = [f for f in core.loc[core["block"] == "internal", "feature"] if f in features]
    family = dict(zip(core["feature"], core["family"]))
    healthy = (metadata["state"] == "healthy").to_numpy()
    X = features[internal]
    reference = HealthyReference(X.loc[healthy], seed=config.RANDOM_SEED)

    # ------------------------------------------- exact distance decomposition
    Z = reference.standardise(X)
    P = reference.precision
    PZ = Z @ P
    terms = Z * PZ                       # rows sum exactly to D^2
    d2 = terms.sum(axis=1)
    assert np.allclose(d2, reference.mahalanobis(X) ** 2, rtol=1e-8)

    disease = ~healthy
    share_disease = terms[disease].mean(axis=0) / terms[disease].mean(axis=0).sum()
    share_healthy = terms[healthy].mean(axis=0) / terms[healthy].mean(axis=0).sum()

    decomposition = pd.DataFrame({
        "feature": internal,
        "family": [family.get(f, "") for f in internal],
        "mean_term_disease": terms[disease].mean(axis=0),
        "mean_term_healthy": terms[healthy].mean(axis=0),
        "share_of_squared_distance_disease": share_disease,
        "share_of_squared_distance_healthy": share_healthy,
        "excess_share": share_disease - share_healthy,
    }).sort_values("share_of_squared_distance_disease", ascending=False)
    decomposition.to_csv(OUT / "mahalanobis_decomposition.csv", index=False)

    top = decomposition.head(10)
    print("top contributors to the squared health distance in disease states")
    print(top[["feature", "family", "share_of_squared_distance_disease",
               "excess_share"]].round(4).to_string(index=False))
    print(f"\nthe 10 largest contributors carry "
          f"{top['share_of_squared_distance_disease'].sum() * 100:.1f}% of the squared distance; "
          f"{int((decomposition['share_of_squared_distance_disease'] > 0.01).sum())} features "
          f"carry more than 1% each")

    # ------------------------------------- partial correlations and redundancy
    distance = reference.mahalanobis(X)
    joint = np.column_stack([Z, (distance - distance.mean()) / distance.std()])
    partial = partial_correlation_matrix(joint)
    partial_with_distance = partial[:-1, -1]
    marginal = np.array([np.corrcoef(Z[:, k], distance)[0, 1] for k in range(Z.shape[1])])

    with np.errstate(divide="ignore", invalid="ignore"):
        ratio = np.abs(partial_with_distance) / np.where(np.abs(marginal) > 1e-9,
                                                         np.abs(marginal), np.nan)
    role = np.where(~np.isfinite(ratio), "undetermined",
                    np.where(ratio < 0.5, "redundant",
                             np.where(ratio > 1.5, "complementary", "independent")))

    partials = pd.DataFrame({
        "feature": internal,
        "family": [family.get(f, "") for f in internal],
        "marginal_correlation_with_distance": marginal,
        "partial_correlation_with_distance": partial_with_distance,
        "partial_over_marginal": ratio,
        "role": role,
    }).sort_values("partial_correlation_with_distance", key=np.abs, ascending=False)
    partials.to_csv(OUT / "partial_correlations.csv", index=False)

    print()
    print(partials["role"].value_counts().to_string())
    print()
    print("features whose information only appears once the others are held fixed:")
    print(partials[partials["role"] == "complementary"]
          .head(8)[["feature", "family", "marginal_correlation_with_distance",
                    "partial_correlation_with_distance"]].round(3).to_string(index=False))

    # -------------------------------------------------- strongest interactions
    feature_partial = partial[:-1, :-1]
    iu = np.triu_indices(len(internal), k=1)
    strength = np.abs(feature_partial[iu])
    order = np.argsort(strength)[::-1][:N_INTERACTIONS]
    interactions = pd.DataFrame({
        "feature_a": [internal[iu[0][k]] for k in order],
        "feature_b": [internal[iu[1][k]] for k in order],
        "family_a": [family.get(internal[iu[0][k]], "") for k in order],
        "family_b": [family.get(internal[iu[1][k]], "") for k in order],
        "partial_correlation": feature_partial[iu][order],
        "marginal_correlation": [np.corrcoef(Z[:, iu[0][k]], Z[:, iu[1][k]])[0, 1]
                                 for k in order],
    })
    interactions["same_family"] = interactions["family_a"] == interactions["family_b"]
    interactions.to_csv(OUT / "feature_interactions.csv", index=False)

    print()
    print(f"of the {N_INTERACTIONS} strongest partial correlations between features, "
          f"{int(interactions['same_family'].sum())} are within a family and "
          f"{int((~interactions['same_family']).sum())} cross families")
    print(interactions.head(8).round(3).to_string(index=False))

## 43. Three sensitivity analyses (`run_17_sensitivity`)

The pathway partition, the admissibility weights, and a bootstrap of the family-combination result.

Writes: `pathway_sensitivity.csv`, `admissibility_sensitivity.csv`, `combination_bootstrap.csv`

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import kendalltau, spearmanr



SENSITIVITY_BOOTSTRAP = 200
TOP_K = 20

GROUPS = {
    "mathematical": ["magnitude", "distribution", "sparsity", "entropy", "architecture"],
    "pathway": ["pathway"],
    "biochemical": ["biochemical", "turnover", "efficiency"],
    "network": ["network"],
}


def recovery(X, healthy, severity, columns):
    if len(columns) < 2:
        return float("nan")
    reference = HealthyReference(X.loc[healthy, columns], seed=config.RANDOM_SEED)
    return float(spearmanr(reference.mahalanobis(X[columns]), severity).statistic)


def run_17_sensitivity() -> None:
    OUT = config.RESULTS_DIR / "robustness"
    config.ensure_dirs()
    features = pd.read_csv(config.PROCESSED_DIR / "phenotypes_transformed.csv.gz")
    metadata = pd.read_csv(config.PROCESSED_DIR / "sample_metadata.csv.gz")
    core = pd.read_csv(config.RESULTS_DIR / "features" / "core_features.csv")
    internal = core[(core["block"] == "internal") & core["feature"].isin(features.columns)]
    columns = internal["feature"].tolist()
    family = dict(zip(internal["feature"], internal["family"]))
    X = features[columns]
    healthy = (metadata["state"] == "healthy").to_numpy()
    severity = metadata["g6pd_level"].to_numpy()
    chain_id = (metadata["condition"].astype(str) + ":" + metadata["chain"].astype(str)).to_numpy()

    base = recovery(X, healthy, severity, columns)
    print(f"baseline: {len(columns)} internal core features, rho = {base:+.3f}")

    # ------------------------------------------------- 1. pathway partition
    rows = []
    pathway_features = [f for f in columns if family[f] == "pathway"]
    groups = sorted({f.rsplit("_", 1)[0] for f in pathway_features})
    for group in groups:
        dropped = [f for f in pathway_features if f.startswith(group)]
        kept = [f for f in columns if f not in dropped]
        rho = recovery(X, healthy, severity, kept)
        rows.append({"pathway_group": group.replace("path_", ""),
                     "n_features_dropped": len(dropped),
                     "spearman_with_g6pd_level": rho,
                     "delta_vs_baseline": abs(rho) - abs(base)})
    pathway = pd.DataFrame(rows).sort_values("delta_vs_baseline")
    pathway.to_csv(OUT / "pathway_sensitivity.csv", index=False)
    print()
    print("effect of removing each pathway group from the phenotype")
    print(pathway.head(6).round(4).to_string(index=False))
    print(f"largest effect of any single pathway group: "
          f"{pathway['delta_vs_baseline'].abs().max():.4f} in |rho|")

    # -------------------------------------------- 2. admissibility weights
    screening = pd.read_csv(config.RESULTS_DIR / "interventions" / "screening.csv")
    estimable = screening[screening["estimable"]].copy()
    baseline_lambda, baseline_cut = 0.5, 0.5

    def rank(lmbda, cut, mu=0.25, nu=0.05):
        u = (estimable["recovery"]
             - lmbda * estimable["max_toxicity"]
             - mu * (~estimable["clinically_modifiable"].astype(bool))
             - nu * estimable["magnitude"].clip(upper=1.0))
        admissible = (estimable["clinically_modifiable"].astype(bool)
                      & (estimable["max_toxicity"] <= cut))
        out = estimable.assign(utility=u, admissible=admissible)
        out = out[out["admissible"]].sort_values("utility", ascending=False)
        return out["intervention_id"].tolist()

    reference_rank = rank(baseline_lambda, baseline_cut)
    rows = []
    for lmbda in [0.1, 0.25, 0.5, 1.0, 2.0]:
        for cut in [0.2, 0.35, 0.5, 0.65, 0.8]:
            candidate = rank(lmbda, cut)
            shared = [c for c in candidate[:TOP_K] if c in reference_rank[:TOP_K]]
            common = [c for c in candidate if c in reference_rank[:TOP_K]]
            tau = np.nan
            if len(common) > 2:
                a = [reference_rank.index(c) for c in common]
                b = [candidate.index(c) for c in common]
                tau = float(kendalltau(a, b).statistic)
            rows.append({"toxicity_penalty_lambda": lmbda, "admissibility_cut": cut,
                         "n_admissible": len(candidate),
                         "top_candidate": candidate[0] if candidate else "",
                         "top_candidate_unchanged": bool(candidate and reference_rank
                                                         and candidate[0] == reference_rank[0]),
                         "overlap_in_top20": len(shared) / TOP_K,
                         "kendall_tau_top20": tau})
    admissibility = pd.DataFrame(rows)
    admissibility.to_csv(OUT / "admissibility_sensitivity.csv", index=False)
    print()
    print(f"admissibility grid: top candidate unchanged in "
          f"{int(admissibility['top_candidate_unchanged'].sum())} of {len(admissibility)} "
          f"weight settings; median overlap in the top {TOP_K} is "
          f"{admissibility['overlap_in_top20'].median():.2f}")

    # ------------------------------------------- 3. bootstrap the combinations
    members = {g: [f for f in columns if family[f] in fams] for g, fams in GROUPS.items()}
    contrasts = {
        "biochemical": members["biochemical"],
        "mathematical + biochemical": sorted(set(members["biochemical"]) | set(members["mathematical"])),
        "all four groups": columns,
    }
    chains = np.unique(chain_id)
    rng = np.random.default_rng(config.RANDOM_SEED)
    draws = {name: [] for name in contrasts}
    for b in range(SENSITIVITY_BOOTSTRAP):
        picked = rng.choice(chains, size=len(chains), replace=True)
        idx = np.concatenate([np.flatnonzero(chain_id == c) for c in picked])
        Xb, hb, sb = X.iloc[idx], healthy[idx], severity[idx]
        if hb.sum() < 200:
            continue
        for name, cols in contrasts.items():
            draws[name].append(recovery(Xb.reset_index(drop=True), hb, sb, cols))
        if (b + 1) % 50 == 0:
            print(f"  bootstrap {b + 1}/{SENSITIVITY_BOOTSTRAP}")

    rows = []
    for name, values in draws.items():
        v = np.abs(np.array([x for x in values if np.isfinite(x)]))
        rows.append({"feature_set": name, "n_bootstrap": len(v),
                     "mean_abs_rho": float(v.mean()),
                     "ci_low": float(np.quantile(v, 0.025)),
                     "ci_high": float(np.quantile(v, 0.975))})
    combination = pd.DataFrame(rows)

    bio = np.abs(np.array(draws["biochemical"]))
    both = np.abs(np.array(draws["mathematical + biochemical"]))
    n = min(len(bio), len(both))
    difference = bio[:n] - both[:n]
    combination.loc[len(combination)] = {
        "feature_set": "biochemical minus (mathematical + biochemical)",
        "n_bootstrap": n, "mean_abs_rho": float(difference.mean()),
        "ci_low": float(np.quantile(difference, 0.025)),
        "ci_high": float(np.quantile(difference, 0.975)),
    }
    combination.to_csv(OUT / "combination_bootstrap.csv", index=False)
    print()
    print(combination.round(4).to_string(index=False))
    print(f"\nadding the mathematical families changes |rho| by "
          f"{difference.mean():+.4f} (95% CI {np.quantile(difference, 0.025):+.4f} to "
          f"{np.quantile(difference, 0.975):+.4f})")

## 44. Inference with the chain as the unit (`run_18_chain_level`)

Thirty thousand states are not thirty thousand independent observations. Every headline number is recomputed with the sampling chain, and then the condition, as the unit of analysis — and the canonical correlation between the two blocks is regularised and validated on chains its directions never saw.

Writes: `chain_level_inference.csv`, `condition_level_summary.csv`, `cca_validation.csv`

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import kruskal, spearmanr
from sklearn.covariance import LedoitWolf



CHAIN_BOOTSTRAP = 2000
CHAIN_SEVERITY = {1.0: "healthy", 0.5: "mild", 0.28: "moderate", 0.1: "severe"}


def epsilon_squared(groups: list[np.ndarray]) -> float:
    """Kruskal-Wallis effect size, on however few observations there are."""
    n = sum(len(g) for g in groups)
    if n <= len(groups) or min(len(g) for g in groups) == 0:
        return float("nan")
    h = kruskal(*groups).statistic
    return float((h - len(groups) + 1) / (n - len(groups)))


def auc(positive: np.ndarray, negative: np.ndarray) -> float:
    if not len(positive) or not len(negative):
        return float("nan")
    return float((positive[:, None] > negative[None, :]).mean()
                 + 0.5 * (positive[:, None] == negative[None, :]).mean())


def regularised_cca(X: np.ndarray, Y: np.ndarray, *, n_components: int = 1):
    """First canonical correlation with Ledoit-Wolf shrinkage on both blocks.

    Returns the correlation and the two canonical weight vectors. Shrinkage is
    what keeps the estimate from being a statement about dimensionality: with
    98 and 63 variables, the unregularised solution can align almost any pair of
    blocks if the effective sample size is small.
    """
    Xc = X - X.mean(axis=0)
    Yc = Y - Y.mean(axis=0)
    Sxx = LedoitWolf().fit(Xc).covariance_
    Syy = LedoitWolf().fit(Yc).covariance_
    Sxy = Xc.T @ Yc / (len(Xc) - 1)

    def inv_sqrt(S):
        w, V = np.linalg.eigh(S)
        w = np.clip(w, 1e-10, None)
        return V @ np.diag(w ** -0.5) @ V.T

    K = inv_sqrt(Sxx) @ Sxy @ inv_sqrt(Syy)
    U, s, Vt = np.linalg.svd(K, full_matrices=False)
    a = inv_sqrt(Sxx) @ U[:, :n_components]
    b = inv_sqrt(Syy) @ Vt[:n_components].T
    return float(s[0]), a, b


def run_18_chain_level() -> None:
    OUT = config.RESULTS_DIR / "robustness"
    config.ensure_dirs()
    distances = pd.read_csv(config.RESULTS_DIR / "distance" / "health_distances.csv.gz")
    features = pd.read_csv(config.PROCESSED_DIR / "phenotypes_transformed.csv.gz")
    metadata = pd.read_csv(config.PROCESSED_DIR / "sample_metadata.csv.gz")
    core = pd.read_csv(config.RESULTS_DIR / "features" / "core_features.csv")

    distances["unit"] = (distances["condition"].astype(str) + ":"
                         + distances["chain"].astype(str))
    distances["severity"] = distances["g6pd_level"].map(
        lambda x: CHAIN_SEVERITY.get(round(float(x), 2), "severe"))

    # ------------------------------------------------------------ chain level
    chain = (distances.groupby(["condition", "chain", "g6pd_level", "h2o2_level",
                                "severity", "state"])
             .agg(median_health_distance=("phenotype_mahalanobis", "median"),
                  median_raw_flux_distance=("raw_sampled_mean_flux_euclidean", "median"),
                  n_states=("phenotype_mahalanobis", "size"))
             .reset_index())
    chain.to_csv(OUT / "chain_level_inference.csv", index=False)
    print(f"{len(chain)} sampling units (condition x chain), "
          f"{chain['n_states'].iloc[0]} states each")

    rho_chain = spearmanr(chain["median_health_distance"], chain["g6pd_level"])
    groups = [chain.loc[chain["severity"] == s, "median_health_distance"].to_numpy()
              for s in ["healthy", "mild", "moderate", "severe"]]
    medians = [float(np.median(g)) for g in groups]
    monotone = all(medians[i] <= medians[i + 1] for i in range(len(medians) - 1))
    auc_chain = auc(chain.loc[chain["state"] == "disease", "median_health_distance"].to_numpy(),
                    chain.loc[chain["state"] == "healthy", "median_health_distance"].to_numpy())

    print(f"\nchain-level Spearman rho = {rho_chain.statistic:+.3f} "
          f"(p = {rho_chain.pvalue:.2g}, n = {len(chain)} chains)")
    print(f"chain-level medians healthy -> severe: "
          f"{', '.join(f'{m:.2f}' for m in medians)}  (monotone: {monotone})")
    print(f"chain-level healthy vs disease AUC = {auc_chain:.3f} "
          f"(3 healthy chains vs 27 disease chains)")
    print(f"chain-level epsilon^2 across the four levels = "
          f"{epsilon_squared(groups):.3f}")

    # the same comparison in the raw flux space, at the same level
    rho_raw_chain = spearmanr(chain["median_raw_flux_distance"], chain["g6pd_level"])
    print(f"raw flux space, same units: rho = {rho_raw_chain.statistic:+.3f} "
          f"(p = {rho_raw_chain.pvalue:.2g})")

    # ---------------------------------------------------- bootstrap over chains
    rng = np.random.default_rng(config.RANDOM_SEED)
    units = chain.index.to_numpy()
    draws = []
    for _ in range(CHAIN_BOOTSTRAP):
        picked = rng.choice(units, size=len(units), replace=True)
        block = chain.loc[picked]
        if block["g6pd_level"].nunique() < 2:
            continue
        draws.append(spearmanr(block["median_health_distance"],
                               block["g6pd_level"]).statistic)
    draws = np.array([d for d in draws if np.isfinite(d)])
    ci = (float(np.quantile(draws, 0.025)), float(np.quantile(draws, 0.975)))
    print(f"bootstrap over chains: rho = {draws.mean():+.3f} "
          f"(95% CI {ci[0]:+.3f} to {ci[1]:+.3f}, {len(draws)} resamples)")

    # -------------------------------------------------------- condition level
    condition = (chain.groupby(["condition", "g6pd_level", "h2o2_level", "severity", "state"])
                 .agg(median_of_chain_medians=("median_health_distance", "median"),
                      min_chain=("median_health_distance", "min"),
                      max_chain=("median_health_distance", "max"),
                      n_chains=("median_health_distance", "size"))
                 .reset_index()
                 .sort_values("g6pd_level", ascending=False))
    condition["chain_range"] = condition["max_chain"] - condition["min_chain"]
    condition.to_csv(OUT / "condition_level_summary.csv", index=False)
    rho_condition = spearmanr(condition["median_of_chain_medians"], condition["g6pd_level"])
    print(f"\ncondition-level Spearman rho = {rho_condition.statistic:+.3f} "
          f"(n = {len(condition)} conditions)")
    print(condition[["condition", "severity", "median_of_chain_medians",
                     "chain_range"]].round(3).to_string(index=False))
    separation = (condition.loc[condition["severity"] == "severe", "min_chain"].min()
                  - condition.loc[condition["severity"] == "healthy", "max_chain"].max())
    print(f"\ngap between the worst healthy chain and the best severe chain: "
          f"{separation:+.3f} health-distance units")

    # ------------------------------------------------------- CCA, validated
    internal = [f for f in core.loc[core["block"] == "internal", "feature"] if f in features]
    external = [f for f in core.loc[core["block"] == "external", "feature"] if f in features]
    unit = (metadata["condition"].astype(str) + ":" + metadata["chain"].astype(str)).to_numpy()

    rows = []
    for condition_name, block in metadata.groupby("condition"):
        idx = block.index.to_numpy()
        X = features.loc[idx, internal].to_numpy(dtype=float)
        Y = features.loc[idx, external].to_numpy(dtype=float)
        chains = np.unique(unit[idx])
        in_sample, _, _ = regularised_cca(X, Y)

        # fit the canonical directions on two chains, score them on the third
        held_out = []
        for chain_id in chains:
            train = unit[idx] != chain_id
            test = ~train
            if train.sum() < 100 or test.sum() < 100:
                continue
            _, a, b = regularised_cca(X[train], Y[train])
            u = (X[test] - X[train].mean(axis=0)) @ a[:, 0]
            v = (Y[test] - Y[train].mean(axis=0)) @ b[:, 0]
            if u.std() > 0 and v.std() > 0:
                held_out.append(abs(float(np.corrcoef(u, v)[0, 1])))

        # a permutation null that breaks the pairing but keeps both marginals
        null = []
        for k in range(20):
            perm = np.random.default_rng(config.RANDOM_SEED + k).permutation(len(X))
            null.append(regularised_cca(X, Y[perm])[0])

        rows.append({
            "condition": condition_name,
            "n_states": len(idx),
            "n_chains": len(chains),
            "regularised_in_sample": in_sample,
            "held_out_mean": float(np.mean(held_out)) if held_out else np.nan,
            "held_out_min": float(np.min(held_out)) if held_out else np.nan,
            "permutation_null_mean": float(np.mean(null)),
            "excess_over_null": (float(np.mean(held_out)) - float(np.mean(null))
                                 if held_out else np.nan),
        })
        print(f"  {condition_name:20s} in-sample {in_sample:.3f}  "
              f"held-out {rows[-1]['held_out_mean']:.3f}  "
              f"null {rows[-1]['permutation_null_mean']:.3f}")

    cca = pd.DataFrame(rows)
    cca.to_csv(OUT / "cca_validation.csv", index=False)
    print(f"\nregularised canonical correlation: in-sample "
          f"{cca['regularised_in_sample'].min():.3f}-{cca['regularised_in_sample'].max():.3f}, "
          f"held out on an unseen chain "
          f"{cca['held_out_mean'].min():.3f}-{cca['held_out_mean'].max():.3f}, "
          f"permutation null {cca['permutation_null_mean'].mean():.3f}")

## 45. Supervised separability, block by block (`run_19_supervised_separability`)

The unsupervised comparison above asks whether the representation
organises states on its own. This asks the complementary question: can a
classifier name the experimental condition of a single state, and from
which part of the representation? Folds are whole chains, so no state
shares a chain with its own training set.

The second half of the stage is the incremental-value analysis: what each
family adds once the others are present, and what it achieves alone.

In [ ]:
"""Stage — the comparison the whole paper is organised around.

Raw flux against the engineered phenotype, on the *same* states, the *same*
folds and the *same* metrics. Three questions are kept apart because they are
three different claims:

A. does the representation organise states without ever seeing a label?
B. can a model identify the experimental condition from a state, and which
   block of the representation carries that ability?
C. what does each feature family add that the others did not already have?

Every split holds out whole chains. States from one chain are not independent,
so a random row split would put near-duplicates on both sides of the fold and
return an accuracy that says more about the sampler than about the
representation.

The fold is the *chain index*: the first independent chain of every condition
is held out together, then the second, then the third. Holding out one chain
at a time instead would leave a test fold containing a single condition, where
accuracy can only be 0 or 1 and a balanced score means nothing — which is
exactly what the first version of this stage did, and what its degenerate
baseline revealed.
"""
import itertools

import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.decomposition import PCA
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier


#: The model ladder stays deliberately short and interpretable. A deeper
#: network would not change what is being tested, which is the representation.
def _classifiers(seed: int) -> dict:
    return {
        "majority": DummyClassifier(strategy="most_frequent"),
        "logistic": make_pipeline(
            StandardScaler(),
            LogisticRegression(max_iter=2000, random_state=seed),
        ),
        "tree": DecisionTreeClassifier(max_depth=6, random_state=seed),
        "forest": RandomForestClassifier(
            n_estimators=300, min_samples_leaf=5, n_jobs=-1, random_state=seed),
    }


def _chain_grouped_scores(X, y, groups, seed, label, transform=None) -> list[dict]:
    """Chain-held-out scores for every classifier on one representation.

    ``transform``, when given, is an unsupervised reducer fitted **inside** each
    fold on the training chains only. Fitting it on all the states first would
    let the held-out chain inform its own representation.
    """
    rows = []
    splitter = LeaveOneGroupOut()
    for name, model in _classifiers(seed).items():
        accuracy, balanced, macro_f1 = [], [], []
        for train, test in splitter.split(X, y, groups):
            if len(np.unique(y[train])) < 2:
                continue
            X_train, X_test = X[train], X[test]
            if transform is not None:
                reducer = clone(transform).fit(X_train)
                X_train, X_test = reducer.transform(X_train), reducer.transform(X_test)
            fitted = model.fit(X_train, y[train])
            predicted = fitted.predict(X_test)
            accuracy.append(accuracy_score(y[test], predicted))
            balanced.append(balanced_accuracy_score(y[test], predicted))
            macro_f1.append(f1_score(y[test], predicted, average="macro", zero_division=0))
        rows.append({
            "representation": label, "model": name,
            "accuracy_mean": float(np.mean(accuracy)), "accuracy_sd": float(np.std(accuracy)),
            "balanced_accuracy_mean": float(np.mean(balanced)),
            "balanced_accuracy_sd": float(np.std(balanced)),
            "macro_f1_mean": float(np.mean(macro_f1)),
            "n_folds": len(accuracy),
            "n_features": int(X.shape[1]) if transform is None else int(X_train.shape[1]),
        })
    return rows


def run_19_supervised_separability(*, max_states_per_chain: int = 400) -> None:
    """Identify the experimental condition from a single sampled state."""
    config.ensure_dirs()
    OUT = config.RESULTS_DIR / "benchmark"
    Path(OUT).mkdir(parents=True, exist_ok=True)
    seed = config.RANDOM_SEED
    rng = np.random.default_rng(seed)

    phenotypes = pd.read_csv(config.PROCESSED_DIR / "phenotypes.csv.gz")
    metadata = pd.read_csv(config.PROCESSED_DIR / "sample_metadata.csv.gz")
    core = pd.read_csv(config.RESULTS_DIR / "features" / "core_features.csv")
    registry = pd.read_csv(config.RESULTS_DIR / "features" / "feature_registry.csv")
    store = SampleStore(config.RAW_DIR)
    V, flux_metadata = store.load_all()
    assert len(flux_metadata) == len(metadata), "flux and phenotype tables disagree"

    # -- one subsample per chain, used by every representation alike ------
    keep = []
    for _, block in metadata.groupby("stem").groups.items():
        index = np.asarray(block)
        take = min(max_states_per_chain, len(index))
        keep.append(rng.choice(index, size=take, replace=False))
    keep = np.sort(np.concatenate(keep))
    meta = metadata.iloc[keep].reset_index(drop=True)
    groups = meta["chain"].to_numpy()       # a fold is one chain of every condition
    y = meta["condition"].to_numpy()

    core_names = core["feature"].tolist()
    internal = [f for f in core_names if _registry_block(registry, f) == "internal"]
    boundary = [f for f in core_names if _registry_block(registry, f) == "external"]
    combined = [f for f in core_names if _registry_block(registry, f) == "combined"]

    flux = V[keep]
    flux = flux[:, flux.std(axis=0) > config.TOL.invariant_range]

    # A learned, uninterpreted low-dimensional representation of the same states.
    # Without it the comparison would only be "engineered descriptors against the
    # raw vector", and the obvious objection — that any compression would do — is
    # left unanswered. The arm is fitted on the training chains inside each fold
    # by the pipeline below, never on the held-out chain.
    n_components = min(32, flux.shape[1] - 1, flux.shape[0] - 1)
    learned = make_pipeline(StandardScaler(), PCA(n_components=n_components,
                                                  random_state=seed))

    representations = {
        "raw flux": flux,
        f"learned embedding ({n_components}d)": ("fit", learned, flux),
        "phenotype (all)": phenotypes.iloc[keep][core_names].to_numpy(),
        "phenotype (internal)": phenotypes.iloc[keep][internal].to_numpy(),
        "phenotype (boundary)": phenotypes.iloc[keep][boundary].to_numpy(),
    }
    if combined:
        representations["phenotype (coupling)"] = phenotypes.iloc[keep][combined].to_numpy()

    rows = []
    for label, entry in representations.items():
        transform = None
        if isinstance(entry, tuple):
            _, transform, X = entry
        else:
            X = entry
        X = np.nan_to_num(X, nan=np.nanmedian(X) if np.isnan(X).any() else 0.0)
        print(f"{label:28s} {X.shape[1]:5d} variables"
              f"{' (reduced inside each fold)' if transform is not None else ''}")
        rows.extend(_chain_grouped_scores(X, y, groups, seed, label, transform=transform))
    scores = pd.DataFrame(rows)
    scores.to_csv(OUT / "supervised_condition.csv", index=False)
    print()
    print(scores.pivot(index="representation", columns="model",
                       values="balanced_accuracy_mean").round(3).to_string())

    # -- C. what does each family add? ------------------------------------
    families = sorted(registry[registry["feature"].isin(core_names)]["family"].unique())
    increments = []
    base = phenotypes.iloc[keep][core_names].to_numpy()
    base = np.nan_to_num(base)
    full = _chain_grouped_scores(base, y, groups, seed, "full")
    full_score = max(r["balanced_accuracy_mean"] for r in full if r["model"] == "forest")
    for family in families:
        without = [f for f in core_names if _registry_family(registry, f) != family]
        only = [f for f in core_names if _registry_family(registry, f) == family]
        X_without = np.nan_to_num(phenotypes.iloc[keep][without].to_numpy())
        X_only = np.nan_to_num(phenotypes.iloc[keep][only].to_numpy())
        drop = [r for r in _chain_grouped_scores(X_without, y, groups, seed, f"-{family}")
                if r["model"] == "forest"][0]
        alone = [r for r in _chain_grouped_scores(X_only, y, groups, seed, f"only {family}")
                 if r["model"] == "forest"][0]
        increments.append({
            "family": family, "n_features": len(only),
            "balanced_accuracy_full": full_score,
            "balanced_accuracy_without": drop["balanced_accuracy_mean"],
            "balanced_accuracy_alone": alone["balanced_accuracy_mean"],
            "incremental_value": full_score - drop["balanced_accuracy_mean"],
        })
        print(f"{family:22s} alone {alone['balanced_accuracy_mean']:.3f}   "
              f"without {drop['balanced_accuracy_mean']:.3f}   "
              f"increment {full_score - drop['balanced_accuracy_mean']:+.3f}")
    pd.DataFrame(increments).sort_values("incremental_value", ascending=False).to_csv(
        OUT / "family_incremental_value.csv", index=False)

    print()
    print("A family with no incremental value is not useless: it may still describe a "
          "property of the state. It is only shown not to add predictive information "
          "the other families did not already carry.")


def _registry_block(registry: pd.DataFrame, feature: str) -> str:
    hit = registry.loc[registry["feature"] == feature, "block"]
    return str(hit.iloc[0]) if len(hit) else "internal"


def _registry_family(registry: pd.DataFrame, feature: str) -> str:
    hit = registry.loc[registry["feature"] == feature, "family"]
    return str(hit.iloc[0]) if len(hit) else "unknown"

## 46. Thermodynamic robustness (`run_20_cross_model`)

Run this once both variants have been analysed. It compares the
*conclusions*, claim by claim, and reports where they differ instead of
averaging the difference away.

In [ ]:
"""Stage — do the conclusions survive the change of model?

This runs after both variants have been analysed. It does not recompute
anything: it reads the tables each variant wrote and asks, one claim at a
time, whether the claim holds in both. A claim that holds only in the
original model is reported as such; that is a result about the framework, not
a failure to be hidden.

The comparison is always *within* a model and then across the two sets of
conclusions. Flux vectors of different length are never compared directly,
because 3,275 and 2,580 coordinates do not form a common space.
"""
import numpy as np
import pandas as pd


def _read(variant: str, *parts):
    path = TABLE_ROOT / variant
    for part in parts:
        path = path / part
    return pd.read_csv(path) if path.exists() else None


def run_20_cross_model() -> None:
    config.ensure_dirs()
    out = Path(REPORT_DIR)
    out.mkdir(parents=True, exist_ok=True)
    variants = [v for v in MODEL_VARIANTS if (TABLE_ROOT / v / "features").exists()]
    if len(variants) < 2:
        print(f"only {variants} analysed so far; run the pipeline for both variants first")
        return

    rows = []

    # 1. the representation itself -----------------------------------------
    for variant in variants:
        registry = _read(variant, "features", "feature_registry.csv")
        core = _read(variant, "features", "core_features.csv")
        rows.append({"claim": "declared candidate features", "variant": variant,
                     "value": len(registry) if registry is not None else np.nan})
        rows.append({"claim": "core features after QC", "variant": variant,
                     "value": len(core) if core is not None else np.nan})

    # 2. severity recovery --------------------------------------------------
    for variant in variants:
        metrics = _read(variant, "distance", "metric_comparison.csv")
        if metrics is not None and "phenotype_mahalanobis" in set(metrics["metric"]):
            row = metrics.set_index("metric").loc["phenotype_mahalanobis"]
            rows.append({"claim": "severity recovery, phenotype (Spearman rho)",
                         "variant": variant, "value": float(row["spearman_with_g6pd_level"])})
            rows.append({"claim": "healthy vs disease AUC, phenotype",
                         "variant": variant, "value": float(row["healthy_vs_disease_auc"])})
            raw = metrics[metrics["metric"].str.startswith("raw_")]
            if len(raw):
                best = raw.iloc[raw["spearman_with_g6pd_level"].abs().argmax()]
                rows.append({"claim": "severity recovery, raw flux (best |rho|)",
                             "variant": variant,
                             "value": float(best["spearman_with_g6pd_level"])})

    # 3. chain-level inference ---------------------------------------------
    for variant in variants:
        chain = _read(variant, "robustness", "chain_level_inference.csv")
        if chain is not None and "median_health_distance" in chain:
            rho = stats.spearmanr(chain["median_health_distance"], chain["g6pd_level"]).statistic
            rows.append({"claim": "chain-level severity recovery (rho)",
                         "variant": variant, "value": float(rho)})

    # 4. supervised identification -----------------------------------------
    for variant in variants:
        scores = _read(variant, "benchmark", "supervised_condition.csv")
        if scores is not None:
            for representation in ("raw flux", "phenotype (all)"):
                block = scores[(scores["representation"] == representation)
                               & (scores["model"] == "forest")]
                if len(block):
                    rows.append({
                        "claim": f"condition identification, {representation} (balanced acc.)",
                        "variant": variant,
                        "value": float(block["balanced_accuracy_mean"].iloc[0])})

    # 5. which families carry the signal ------------------------------------
    rankings = {}
    for variant in variants:
        increments = _read(variant, "benchmark", "family_incremental_value.csv")
        if increments is not None:
            rankings[variant] = increments.set_index("family")["incremental_value"]
    if len(rankings) == 2:
        joined = pd.DataFrame(rankings).dropna()
        if len(joined) > 2:
            tau = kendalltau(joined.iloc[:, 0], joined.iloc[:, 1]).statistic
            rows.append({"claim": "agreement of family ranking between models (Kendall tau)",
                         "variant": "both", "value": float(tau)})
        joined.to_csv(out / "family_value_by_model.csv")

    # 6. the intervention conclusion ----------------------------------------
    for variant in variants:
        screening = _read(variant, "interventions", "screening.csv")
        if screening is not None and "recovery" in screening:
            estimable = screening[screening.get("estimable", True)]
            rows.append({"claim": "best phenotypic recovery in the catalogue (%)",
                         "variant": variant,
                         "value": float(estimable["recovery"].max() * 100)})

    table = pd.DataFrame(rows)
    wide = table.pivot_table(index="claim", columns="variant", values="value")
    if set(variants).issubset(wide.columns):
        wide["difference"] = wide[variants[1]] - wide[variants[0]]
    wide.to_csv(out / "cross_model_conclusions.csv")
    print(wide.round(4).to_string())

    # 7. feature-level agreement on the shared vocabulary --------------------
    shared_features = None
    effects = {}
    for variant in variants:
        discrimination = _read(variant, "features", "feature_discrimination.csv")
        if discrimination is None:
            continue
        if "axis" in discrimination:
            discrimination = discrimination[
                discrimination["axis"] == "g6pd_including_healthy"]
        series = (discrimination.groupby("feature")["epsilon_squared"]
                  .max().dropna())
        effects[variant] = series
        shared_features = set(series.index) if shared_features is None else shared_features & set(series.index)
    if len(effects) == 2 and shared_features:
        frame = pd.DataFrame({v: s.loc[sorted(shared_features)] for v, s in effects.items()})
        frame["rank_original"] = frame[variants[0]].rank(ascending=False)
        frame["rank_thermo"] = frame[variants[1]].rank(ascending=False)
        frame.to_csv(out / "feature_effect_by_model.csv")
        tau = kendalltau(frame[variants[0]], frame[variants[1]]).statistic
        print(f"\n{len(frame)} features are computable in both models; "
              f"their effect-size ranking agrees at Kendall tau = {tau:.3f}")
        print("Features that exist in only one model are listed, never imputed as zero.")

## 47. The registry and the manifests (`run_21_publish_registry`)

Publishes `data/metadata/`: every declared feature with its meaning,
requirements, non-applicability rule and transferability scope, plus the
model, condition and sampling manifests.

In [ ]:
"""Stage — publish the registry and the manifests.

The registry is the object the engine iterates over, so it is not
documentation written after the fact. This stage adds to each entry the
fields a reader needs in order to reuse a descriptor in another model:
what it means biologically, what it requires from the reconstruction, what
happens when that requirement is absent, and which test checks it.

The three manifests record the experiment itself: which models, which
conditions, which chains.
"""
import json

import numpy as np
import pandas as pd


#: Per-family documentation. ``scope`` is the property that matters for
#: transferring the representation to another GEM:
#:
#:   common         computable, with the same meaning, in any reconstruction
#:   conditional    needs a pathway, metabolite or reaction that may be absent
#:   model_specific describes a property of one particular reconstruction
FAMILY_DOC = {
    "magnitude": dict(
        interpretation="overall scale of the metabolic state",
        requires="none", scope="common", na_rule="not applicable",
        sign_convention="computed on |v|", validated_by="test_magnitude_scales_with_flux"),
    "distribution": dict(
        interpretation="shape of the flux distribution across reactions",
        requires="none", scope="common", na_rule="not applicable",
        sign_convention="moments on v, quantiles on v and |v| separately",
        validated_by="test_distribution_moments"),
    "sparsity": dict(
        interpretation="how concentrated activity is in few reactions",
        requires="none", scope="common", na_rule="not applicable",
        sign_convention="computed on |v|", validated_by="test_sparsity_bounds"),
    "entropy": dict(
        interpretation="evenness of the flux shares over reactions",
        requires="none", scope="common", na_rule="not applicable",
        sign_convention="computed on |v| normalised to a probability vector",
        validated_by="test_entropy_uniform_case"),
    "architecture": dict(
        interpretation="directional balance and dominance structure of the state",
        requires="none", scope="common", na_rule="not applicable",
        sign_convention="signed where direction is the quantity of interest",
        validated_by="test_architecture_direction"),
    "network": dict(
        interpretation="how activity spreads over metabolites rather than reactions",
        requires="stoichiometric matrix; currency metabolites excluded",
        scope="common", na_rule="not applicable",
        sign_convention="computed on |v|", validated_by="test_network_weighting"),
    "pathway": dict(
        interpretation="functional state of a curated group of reactions",
        requires="pathway map over model subsystems", scope="conditional",
        na_rule="NaN when the model has no reaction in the group; never zero",
        sign_convention="activity on |v|, directionality on v",
        validated_by="test_pathway_activity_and_direction"),
    "turnover": dict(
        interpretation=("gross stoichiometric throughput attributed to a metabolite; "
                        "not an experimentally measurable turnover"),
        requires="the metabolite must exist in the model", scope="conditional",
        na_rule="NaN when the metabolite is absent; never zero",
        sign_convention="|S_mi v_i| summed, so production and consumption both count",
        validated_by="test_turnover_known_stoichiometry"),
    "biochemical": dict(
        interpretation="flux through a named reaction of erythrocyte metabolism",
        requires="the named reaction must exist", scope="conditional",
        na_rule="NaN when the reaction is absent; never zero",
        sign_convention="signed, in the model's own direction convention",
        validated_by="test_named_reaction_lookup"),
    "efficiency": dict(
        interpretation="ratio between two functional quantities of the same state",
        requires="both reaction sets of the ratio", scope="conditional",
        na_rule="NaN when the denominator set is absent or carries no flux",
        sign_convention="ratio of magnitudes unless stated",
        validated_by="test_efficiency_ratio_guard"),
    "cofactor": dict(
        interpretation="allocation of redox and adenylate turnover within its own pool",
        requires="the cofactor pair must exist", scope="conditional",
        na_rule="NaN when either member of the pair is absent",
        sign_convention="share of gross turnover, bounded in [0, 1]",
        validated_by="test_cofactor_share_bounds"),
    "capacity": dict(
        interpretation="position of the state relative to the model's own bounds",
        requires="finite bounds", scope="model_specific",
        na_rule="reactions with infinite bounds are excluded from the statistic",
        sign_convention="signed position within [l, u]",
        validated_by="test_capacity_saturation_bounds"),
    "coupling": dict(
        interpretation="how much of the state's activity crosses the boundary",
        requires="both blocks non-empty", scope="conditional",
        na_rule="NaN when the denominator is zero or the exchange is absent",
        sign_convention="ratios of magnitudes", validated_by="test_coupling_ratios"),
    "boundary_metabolite": dict(
        interpretation="exchange flux of one named boundary metabolite",
        requires="the exchange reaction must exist", scope="conditional",
        na_rule="NaN when the exchange is absent; never zero",
        sign_convention="negative is uptake, positive is secretion (COBRA)",
        validated_by="test_exchange_sign_convention"),
    "boundary_role": dict(
        interpretation="aggregate exchange by biological role of the metabolite",
        requires="admissibility record", scope="conditional",
        na_rule="roles with no exchange in the model are reported as NaN",
        sign_convention="negative is uptake", validated_by="test_role_aggregates"),
    "exchange_burden": dict(
        interpretation="total traffic across the boundary and how it is spread",
        requires="at least one exchange reaction", scope="common",
        na_rule="not applicable", sign_convention="magnitudes",
        validated_by="test_exchange_burden"),
    "boundary_efficiency": dict(
        interpretation="carbon economy of the boundary",
        requires="metabolite formulas", scope="conditional",
        na_rule="NaN when formulas are missing", sign_convention="magnitudes",
        validated_by="test_carbon_balance"),
    "oxidative_environment": dict(
        interpretation="oxidative exposure imposed through the boundary",
        requires="peroxide and oxygen exchanges", scope="conditional",
        na_rule="NaN when absent", sign_convention="negative is uptake",
        validated_by="test_oxidative_environment"),
    "redox_environment": dict(
        interpretation="redox-relevant exchange traffic",
        requires="redox-relevant exchanges", scope="conditional",
        na_rule="NaN when absent", sign_convention="negative is uptake",
        validated_by="test_redox_environment"),
    "admissibility": dict(
        interpretation="toxicity exposure implied by the boundary state",
        requires="curated admissibility record", scope="model_specific",
        na_rule="metabolites without a record take the documented default",
        sign_convention="weights in [0, 1], judgement not measurement",
        validated_by="test_admissibility_defaults"),
}


def run_21_publish_registry() -> None:
    """Write data/metadata/: the registry and the three manifests."""
    config.ensure_dirs()
    out = Path(METADATA_DIR)
    out.mkdir(parents=True, exist_ok=True)

    # -- the registry, enriched -------------------------------------------
    frames = []
    for variant in MODEL_VARIANTS:
        path = TABLE_ROOT / variant / "features" / "feature_registry.csv"
        if not path.exists():
            continue
        frame = pd.read_csv(path)
        frame["model_variant"] = variant
        core_path = TABLE_ROOT / variant / "features" / "core_features.csv"
        core = set(pd.read_csv(core_path)["feature"]) if core_path.exists() else set()
        frame["retained_in_core"] = frame["feature"].isin(core)
        frames.append(frame)
    if not frames:
        print("no registry written yet; run the feature stage first")
        return

    registry = pd.concat(frames, ignore_index=True)
    documentation = pd.DataFrame(FAMILY_DOC).T.rename_axis("family").reset_index()
    registry = registry.merge(documentation, on="family", how="left")
    missing = sorted(set(registry.loc[registry["interpretation"].isna(), "family"]))
    if missing:
        print(f"families with no documentation entry: {missing}")

    # The index is the descriptor, not its prose. A pathway descriptor's
    # description names how many reactions the group has, which differs
    # between the two reconstructions, so indexing on the description would
    # list the same descriptor twice and inflate the registry.
    identity = [c for c in ("feature", "family", "block", "units", "signed_input",
                            "model_dependent", "interpretation", "requires", "scope",
                            "na_rule", "sign_convention", "validated_by")
                if c in registry.columns]
    prose = registry.drop_duplicates("feature").set_index("feature")
    wide = registry.pivot_table(
        index=identity, columns="model_variant", values="retained_in_core",
        aggfunc="first",
    ).reset_index()
    for column in ("description", "formula"):
        if column in prose.columns:
            wide[column] = wide["feature"].map(prose[column])
    for variant in MODEL_VARIANTS:
        if variant in wide.columns:
            wide = wide.rename(columns={variant: f"core_in_{variant}"})
    wide.to_csv(out / "feature_registry.csv", index=False)
    print(f"feature_registry.csv: {len(wide)} declared features")
    print(wide.groupby(["scope"]).size().to_string())

    # -- the condition manifest -------------------------------------------
    conditions = [{
        "condition": "healthy", "g6pd_level": 1.0, "h2o2_level": 1.0,
        "g6pd_severity": "none", "h2o2_load": "reference",
        "description": "unconstrained reference model",
    }]
    for g6pd in G6PD_LEVELS:
        for h2o2 in H2O2_LEVELS:
            conditions.append({
                "condition": f"G6PD{g6pd:g}_H2O2{h2o2:g}",
                "g6pd_level": g6pd, "h2o2_level": h2o2,
                "g6pd_severity": G6PD_SEVERITY[g6pd], "h2o2_load": H2O2_LOAD[h2o2],
                "description": (f"upper bound of {G6PD_REACTION} scaled to {g6pd:g} and of "
                                f"{H2O2_EXCHANGE} to {h2o2:g} of the reference value"),
            })
    pd.DataFrame(conditions).to_csv(out / "condition_manifest.csv", index=False)

    # -- the sampling manifest --------------------------------------------
    rows = []
    for path in sorted(Path(SAMPLE_DIR).glob("*.json")):
        meta = json.loads(path.read_text())
        info = parse_condition(path.stem)
        rows.append({
            "stem": path.stem, **info,
            "sampler": config.SAMPLER,
            "n_raw_samples": meta["n_samples_total"], "thinning": meta["thin"],
            "n_retained": meta["n_samples_kept"], "n_reactions": meta["n_reactions"],
            "source_file": meta["file"],
        })
    sampling = pd.DataFrame(rows)
    sampling.to_csv(out / "sampling_manifest.csv", index=False)
    print()
    print(sampling.groupby(["model_variant", "state"])
          .agg(chains=("stem", "size"), reactions=("n_reactions", "first"),
               states=("n_retained", "sum")).to_string())

    # -- the model manifest ------------------------------------------------
    models = []
    for variant, spec in MODEL_VARIANTS.items():
        path = Path(spec["model_file"])
        row = {"model_variant": variant, "label": spec["label"], "role": spec["role"],
               "file": path.name, "available": path.exists()}
        if path.exists():
            gem = load_gem(path, spec["model_name"])
            row.update({"n_reactions": gem.n_reactions, "n_metabolites": gem.n_metabolites,
                        "n_genes": int(len(gem.genes))})
        models.append(row)
    pd.DataFrame(models).to_csv(out / "model_manifest.csv", index=False)
    print()
    print(pd.DataFrame(models).to_string(index=False))

## 48. Figures 1–6 of the paper (`run_22_figures`)

One figure per step of the argument (Figure 7, the method, is drawn after the numbers exist, in stage 31b): what the reduced reconstruction keeps and how much of each descriptor survives (1), the response to the imposed gradient beside the control axis (2), the raw flux vector against the representation on equal terms (3), the full representation against a reduced panel (4), what survives the structural reduction (5), and the secondary validations (6). Every panel reads a table; nothing is drawn by hand.

In [ ]:
"""Stage 22 — Figures 1–6 and S1 of the paper, drawn from the tables.

Figure 7, the method, is drawn by stage 31b from ``numbers.json``. One figure
per step of the argument:

1. **Two structures, one registry** — what the reduced reconstruction keeps,
   and how much of each descriptor's support survives.
2. **Response and negative control** — the gradient both models were given,
   read at the level of the chain, beside the axis that should show nothing.
3. **Same samples, same reference, same metric** — the raw flux vector and
   Φ(v) through one identical procedure; which families carry the signal; what
   a classifier uses.
4. **Full representation against a reduced panel** — accuracy and the
   stability of the selection, with their spread over held-out chains.
5. **What survives the reduction** — the response of every comparable
   descriptor in one model against the other, by how much of its support is
   left, and what happens to the panel and to prediction across models.
6. **Secondary validations** — unsupervised structure, the boundary against
   the interior, and the screening of boundary interventions with the control
   of the confirmatory sampler.

Every panel reads a file under ``results/``; a panel whose table does not
exist says so on the canvas instead of failing. The figures are written to
``results/figures/paper`` and copied to ``reports/paper/figures``, which is
where the manuscript is built from.
"""
import shutil
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.gridspec import GridSpec
from matplotlib.lines import Line2D
from matplotlib.patches import FancyBboxPatch, Patch
from matplotlib.ticker import NullFormatter, NullLocator


__all__ = ["run_22_figures", "PAPER_FIGURES"]

FIGURE_WIDTH = 7.2                                   # inches, two-column width
VARIANT_COLOUR = {"original": "#1F5B7A", "thermo": "#C26B3A"}
VARIANT_LABEL = {"original": "Original", "thermo": "Thermo"}
SUPPORT_COLOUR = {
    "identical support": "#1F5B7A",
    "reduced support": "#8FB3C4",
    "largely lost": "#E0A98A",
    "not computable": "#B04430",
    "not resolved": "#BBBBBB",
}
SUPPORT_ORDER = list(SUPPORT_COLOUR)
CLASS_COLOUR = {
    "conserved": "#1F5B7A",
    "attenuated": "#8FB3C4",
    "amplified": "#5E93AE",
    "reversed": "#B04430",
    "gained in reduced model": "#E3AE55",
    "unresponsive in both": "#DCE2E8",
    "not comparable": "#BBBBBB",
    "not computable": "#6B7A8C",
}
CLASS_ORDER = list(CLASS_COLOUR)
LEVEL_LABEL = {1.0: "healthy", 0.5: "0.50", 0.28: "0.28", 0.1: "0.10"}
#: Descriptors drawn in Figure 3a: the oxidative branch, one of its partners
#: upstream, and one shape statistic of the whole state.
TRACKED = ("eff_ppp_flux_split", "turnover_6pgc_c", "path_pentose_phosphate_activity",
           "rxn_hexokinase")


def _read(*parts) -> pd.DataFrame | None:
    path = Path(TABLE_ROOT).joinpath(*parts)
    return pd.read_csv(path) if path.exists() else None


def _read_json(*parts) -> dict | None:
    import json
    path = Path(TABLE_ROOT).joinpath(*parts)
    return json.loads(path.read_text()) if path.exists() else None


def _missing(ax, message: str) -> None:
    ax.set_axis_off()
    ax.text(0.5, 0.5, message, ha="center", va="center", fontsize=6.2,
            color=MUTED, style="italic", wrap=True, transform=ax.transAxes)


def _box(ax, x, y, w, h, title, body, face, edge, *, title_size=7.0, body_size=5.6):
    ax.add_patch(FancyBboxPatch((x, y), w, h, boxstyle="round,pad=0,rounding_size=0.018",
                                facecolor=face, edgecolor=edge, linewidth=1.0, zorder=2))
    ax.text(x + w / 2, y + h * 0.70, title, ha="center", va="center",
            fontsize=title_size, fontweight="bold", color=INK, zorder=3)
    ax.text(x + w / 2, y + h * 0.32, body, ha="center", va="center",
            fontsize=body_size, color=MUTED, zorder=3, linespacing=1.25)


def _arrow_between(ax, x0, y0, x1, y1, colour=None):
    ax.annotate("", xy=(x1, y1), xytext=(x0, y0),
                arrowprops=dict(arrowstyle="-|>", lw=1.0, color=colour or MUTED,
                                shrinkA=0, shrinkB=0, mutation_scale=8), zorder=1)


# ==========================================================================
# Figure 1
# ==========================================================================
def paper_figure2(out: Path) -> None:
    summary = (pd.read_csv(Path(METADATA_DIR) / "model_comparison_summary.csv")
               if (Path(METADATA_DIR) / "model_comparison_summary.csv").exists() else None)
    diff = (pd.read_csv(Path(METADATA_DIR) / "model_reaction_diff.csv")
            if (Path(METADATA_DIR) / "model_reaction_diff.csv").exists() else None)
    structural = _read("transfer", "A_structural.csv")

    fig = plt.figure(figsize=(FIGURE_WIDTH, 5.0))
    gs = GridSpec(2, 2, figure=fig, wspace=0.42, hspace=0.62,
                  left=0.09, right=0.985, top=0.93, bottom=0.09)

    # a. what each model has
    ax = fig.add_subplot(gs[0, 0])
    if summary is None:
        _missing(ax, "model comparison not run")
    else:
        rows = summary.set_index("quantity")
        keys = [k for k in ("reactions", "metabolites", "exchange reactions",
                            "reversible reactions", "genes") if k in rows.index]
        x = np.arange(len(keys))
        for k, variant in enumerate(("original", "thermo")):
            ax.bar(x + (k - 0.5) * 0.38, rows.loc[keys, variant].astype(float), width=0.36,
                   color=VARIANT_COLOUR[variant], edgecolor=PAPER, linewidth=0.4,
                   label=VARIANT_LABEL[variant])
        for xi, key in zip(x, keys):
            o, t = float(rows.loc[key, "original"]), float(rows.loc[key, "thermo"])
            ax.text(xi, max(o, t) * 1.07, f"{t / o:.0%}", ha="center", fontsize=5.6,
                    color=INK)
        ax.set_xticks(x)
        ax.set_xticklabels([k.replace(" ", "\n") for k in keys], fontsize=5.8)
        ax.set_ylabel("count")
        ax.set_ylim(0, rows.loc[keys, "original"].astype(float).max() * 1.25)
        ax.legend(fontsize=5.8, loc="upper right", ncol=2)
    panel_label(ax, "a", "What the reduced model keeps")

    # b. where the removals are, as a fraction of each curated pathway group
    ax = fig.add_subplot(gs[0, 1])
    if diff is None or "status" not in diff.columns or "pathway" not in diff.columns:
        _missing(ax, "reaction difference table not available")
    else:
        groups = (diff.assign(removed=diff["status"] != "shared")
                  .groupby(diff["pathway"].fillna("unassigned"))["removed"]
                  .agg(["sum", "size"]))
        groups = groups[groups["size"] >= 5]
        groups["fraction"] = groups["sum"] / groups["size"]
        groups = groups.sort_values("fraction")
        perturbed = {"pentose_phosphate", "glycolysis", "glutathione", "redox_detox",
                     "rapoport_luebering"}
        colours = [ACCENT["signal"] if g in perturbed else BLOCK["neutral"] for g in groups.index]
        y = np.arange(len(groups))
        ax.barh(y, groups["fraction"], color=colours, height=0.66)
        for yi, (_, row) in zip(y, groups.iterrows()):
            ax.text(row["fraction"] + 0.01, yi, f"{int(row['sum'])}/{int(row['size'])}",
                    va="center", fontsize=4.8, color=MUTED)
        ax.set_yticks(y)
        ax.set_yticklabels([g.replace("_", " ") for g in groups.index], fontsize=5.2)
        ax.set_xlim(0, 1)
        ax.set_xlabel("fraction of the group's reactions removed")
        ax.grid(axis="y", visible=False)
        ax.legend(handles=[Patch(color=ACCENT["signal"], label="where the perturbation acts")],
                  fontsize=5.2, loc="lower right")
    panel_label(ax, "b", "Where the removed reactions are", offset=-62)

    # c. retained support, descriptor by descriptor
    ax = fig.add_subplot(gs[1, 0])
    if structural is None:
        _missing(ax, "stage 29 not run")
    else:
        rng = np.random.default_rng(0)
        blocks = [("internal", "internal"), ("external", "boundary"), ("combined", "coupling")]
        present = [b for b in blocks if (structural["block"] == b[0]).any()]
        for k, (block, label) in enumerate(present):
            sub = structural[structural["block"] == block]
            values = sub["retained_support"].fillna(0).to_numpy()
            colours = [SUPPORT_COLOUR[c] for c in sub["support_category"]]
            ax.scatter(k + rng.uniform(-0.28, 0.28, len(values)), values, s=6,
                       c=colours, linewidths=0, alpha=0.9)
        ax.axhline(0.5, color=RULE, lw=0.8, ls="--")
        ax.set_xticks(range(len(present)))
        ax.set_xticklabels([f"{label}\n(n={(structural['block'] == b).sum()})"
                            for b, label in present], fontsize=6.0)
        ax.set_ylabel("fraction of support retained")
        ax.set_ylim(-0.05, 1.05)
        ax.grid(axis="x", visible=False)
    panel_label(ax, "c", "How much of what each descriptor reads survives")

    # d. the categories, for the registry and for the core
    ax = fig.add_subplot(gs[1, 1])
    if structural is None:
        _missing(ax, "stage 29 not run")
    else:
        groups = [("registry", structural), ("core (Original)", structural[structural["core_reference"]])]
        for k, (label, frame) in enumerate(groups):
            counts = frame["support_category"].value_counts()
            left = 0
            for category in SUPPORT_ORDER:
                n = int(counts.get(category, 0))
                if not n:
                    continue
                ax.barh(k, n, left=left, color=SUPPORT_COLOUR[category], height=0.55,
                        edgecolor=PAPER, linewidth=0.5)
                if n >= 8:
                    ax.text(left + n / 2, k, str(n), ha="center", va="center",
                            fontsize=5.6, color=PAPER if category != "reduced support" else INK)
                left += n
        ax.set_yticks(range(len(groups)))
        ax.set_yticklabels([g[0] for g in groups], fontsize=6.2)
        ax.set_xlabel("descriptors")
        ax.invert_yaxis()
        ax.grid(axis="y", visible=False)
        handles = [Patch(color=SUPPORT_COLOUR[c], label=c) for c in SUPPORT_ORDER
                   if (structural["support_category"] == c).any()]
        ax.legend(handles=handles, fontsize=5.4, loc="upper center",
                  bbox_to_anchor=(0.5, -0.32), ncol=2)
    panel_label(ax, "d", "Computable is not the same as equivalent")
    save(fig, out / "Figure1_two_structures")


# ==========================================================================
# Figure 2
# ==========================================================================
def _standardised_chain_medians(variant: str, features: list[str]) -> pd.DataFrame | None:
    raw = _read(variant, "phenotypes.csv.gz")
    meta = _read(variant, "sample_metadata.csv.gz")
    if raw is None or meta is None:
        return None
    present = [f for f in features if f in raw.columns]
    healthy = meta["state"].to_numpy() == "healthy"
    centre = raw.loc[healthy, present].median()
    q = raw.loc[healthy, present].quantile([0.25, 0.75])
    spread = ((q.loc[0.75] - q.loc[0.25]) / 1.349).replace(0, np.nan)
    spread = spread.fillna(raw.loc[healthy, present].std(ddof=1).replace(0, np.nan))
    z = (raw[present] - centre) / spread
    frame = pd.concat([meta[["condition", "chain", "g6pd_level", "h2o2_level"]], z], axis=1)
    return frame.groupby(["condition", "chain", "g6pd_level", "h2o2_level"],
                         as_index=False)[present].median()


def paper_figure3(out: Path) -> None:
    fig = plt.figure(figsize=(FIGURE_WIDTH, 5.2))
    gs = GridSpec(2, 4, figure=fig, wspace=0.55, hspace=0.70,
                  left=0.08, right=0.985, top=0.93, bottom=0.09)

    # a. four descriptors, chain medians, both models, in healthy-sd units
    medians = {v: _standardised_chain_medians(v, list(TRACKED)) for v in ("original", "thermo")}
    for k, feature in enumerate(TRACKED):
        ax = fig.add_subplot(gs[0, k])
        drawn = False
        for variant, frame in medians.items():
            if frame is None or feature not in frame.columns:
                continue
            levels = sorted(frame["g6pd_level"].unique())
            x = {level: i for i, level in enumerate(sorted({0.1, 0.28, 0.5, 1.0}))}
            jitter = -0.12 if variant == "original" else 0.12
            for level in levels:
                values = frame.loc[frame["g6pd_level"] == level, feature]
                ax.scatter(np.full(len(values), x[level] + jitter), values, s=5,
                           color=VARIANT_COLOUR[variant], alpha=0.75, linewidths=0)
            ax.plot([x[l] + jitter for l in levels],
                    [frame.loc[frame["g6pd_level"] == l, feature].median() for l in levels],
                    color=VARIANT_COLOUR[variant], lw=1.0, label=VARIANT_LABEL[variant])
            drawn = True
        if not drawn:
            _missing(ax, f"{feature}\nnot available")
            continue
        ax.set_xticks(range(4))
        ax.set_xticklabels(["0.10", "0.28", "0.50", "1"], fontsize=5.8)
        ax.set_xlabel("G6PDH2 capacity", fontsize=6.0)
        ax.set_title(feature.replace("_", " "), fontsize=6.0, fontweight="normal", pad=3)
        if k == 0:
            ax.set_ylabel("chain median\n(healthy robust SD units)", fontsize=6.2)
            panel_label(ax, "a", "")
        if k == len(TRACKED) - 1:
            ax.legend(fontsize=5.4, loc="best")
        ax.axhline(0, color=RULE, lw=0.6)

    # b. the gradient against the control, every core descriptor, both models
    ax = fig.add_subplot(gs[1, 0:2])
    any_drawn = False
    for variant in ("original", "thermo"):
        effects = _read(variant, "design", "factorial_effects.csv")
        if effects is None:
            continue
        ax.scatter(effects["gradient_rho"].abs(), effects["h2o2_eta_squared"], s=7,
                   color=VARIANT_COLOUR[variant], alpha=0.7, linewidths=0,
                   label=VARIANT_LABEL[variant])
        any_drawn = True
    if any_drawn:
        ax.axvline(RESPONSIVE_RHO, color=ACCENT["signal"], lw=0.8, ls="--")
        ax.set_xlabel("|Spearman ρ| with imposed G6PDH2 capacity (chain medians)")
        ax.set_ylabel("variance share of the\nH$_2$O$_2$ axis at fixed capacity")
        ax.set_xlim(0, 1)
        ax.legend(fontsize=5.6, loc="upper left")
    else:
        _missing(ax, "stage 27 not run")
    panel_label(ax, "b", "Responds to the gradient, not to the control")

    # c. counts
    ax = fig.add_subplot(gs[1, 2:4])
    rows = []
    for variant in ("original", "thermo"):
        effects = _read(variant, "design", "factorial_effects.csv")
        if effects is None:
            continue
        strong = effects[(effects["gradient_p_adjusted"] < 0.05)
                         & (effects["gradient_rho"].abs() >= RESPONSIVE_RHO)]
        rows.append((variant, len(effects), len(strong), int(strong["monotone"].sum()),
                     int((effects["h2o2_p_adjusted"] < 0.05).sum())))
    if rows:
        labels = ["track the\ngradient", "monotone", "respond to\nH$_2$O$_2$ axis"]
        x = np.arange(len(labels))
        for k, (variant, n, strong, monotone, control) in enumerate(rows):
            values = np.array([strong, monotone, control]) / n
            bars = ax.bar(x + (k - 0.5) * 0.36, values, width=0.34,
                          color=VARIANT_COLOUR[variant], label=f"{VARIANT_LABEL[variant]} (n={n})")
            for b, count in zip(bars, (strong, monotone, control)):
                ax.text(b.get_x() + b.get_width() / 2, b.get_height() + 0.015, str(count),
                        ha="center", fontsize=5.6)
        ax.set_xticks(x)
        ax.set_xticklabels(labels, fontsize=6.0)
        ax.set_ylabel("fraction of core descriptors")
        ax.set_ylim(0, 1)
        ax.legend(fontsize=5.6, loc="upper right")
    else:
        _missing(ax, "stage 27 not run")
    panel_label(ax, "c", "Counted at the level of the chain")
    save(fig, out / "Figure2_response_and_control")


# ==========================================================================
# Figure 3
# ==========================================================================
REPRESENTATION_COLOUR = {"raw internal flux": "#8A94A6", "internal core": None,
                         "biochemical families": None, "generic families": "#C9D1DA"}
H2O2_MARKER = {1.0: "o", 0.75: "s", 0.5: "^"}


def _noise_units(chains: pd.DataFrame) -> pd.DataFrame:
    """Chain medians as a distance from health in units of chain-to-chain noise."""
    within = chains.groupby("condition")["d"].std(ddof=1)
    pooled = float(np.sqrt((within ** 2).mean()))
    healthy = chains.loc[chains["g6pd_level"] == 1.0, "d"].mean()
    return chains.assign(z=(chains["d"] - healthy) / pooled)


def paper_figure4(out: Path) -> None:
    fig = plt.figure(figsize=(FIGURE_WIDTH, 5.0))
    gs = GridSpec(2, 3, figure=fig, wspace=0.55, hspace=0.62, left=0.085, right=0.985,
                  top=0.92, bottom=0.10, height_ratios=[1, 1.05])
    # the lower row gets its own widths, so that panel e has room for descriptor names
    gs_low = GridSpec(2, 2, figure=fig, wspace=0.95, hspace=0.62, left=0.085, right=0.985,
                      top=0.92, bottom=0.10, height_ratios=[1, 1.05], width_ratios=[1.55, 1])
    levels = [1.0, 0.5, 0.28, 0.1]
    pos = {lvl: k for k, lvl in enumerate(levels)}

    # a, b: the same chains, measured the same way, in two representations
    for k, variant in enumerate(("original", "thermo")):
        ax = fig.add_subplot(gs[0, k])
        chains = _read(variant, "evidence", "distance_chain_medians.csv")
        if chains is None:
            _missing(ax, "stage 29b not run")
            continue
        chains = chains[chains["metric"] == "euclidean"]
        for name, colour, shift in (("raw internal flux", BLOCK["neutral"], -0.13),
                                    ("internal core", VARIANT_COLOUR[variant], 0.13)):
            frame = _noise_units(chains[chains["representation"] == name].copy())
            x = frame["g6pd_level"].round(2).map(pos).to_numpy(float) + shift
            for h2o2, marker in H2O2_MARKER.items():
                sel = np.isclose(frame["h2o2_level"], h2o2)
                ax.scatter(x[sel], frame.loc[sel, "z"], s=9, marker=marker, color=colour,
                           alpha=0.75, linewidths=0, zorder=3)
            means = frame.groupby(frame["g6pd_level"].round(2))["z"].mean().reindex(levels)
            ax.plot(np.arange(4) + shift, means, color=colour, lw=1.2, zorder=4,
                    label="raw flux $v$" if name.startswith("raw") else "$\\Phi_{int}(v)$")
        ax.axhline(0, color=RULE, lw=0.7, zorder=1)
        ax.set_xticks(range(4), ["1", ".50", ".28", ".10"])
        ax.set_xlabel("G6PDH2 capacity")
        if k == 0:
            ax.set_ylabel("distance from health\n(chain-to-chain s.d.)")
        ax.legend(fontsize=5.4, loc="upper left")
        panel_label(ax, "ab"[k], f"{VARIANT_LABEL[variant]}: same chains, same metric")

    # c: how far each level sits from health, against how far chains sit from each other
    ax = fig.add_subplot(gs[0, 2])
    width = 0.19
    drawn = False
    for j, variant in enumerate(("original", "thermo")):
        energy = _read(variant, "evidence", "energy_over_noise.csv")
        if energy is None:
            continue
        drawn = True
        for i, (name, colour) in enumerate((("raw internal flux", BLOCK["neutral"]),
                                            ("internal core", VARIANT_COLOUR[variant]))):
            e = energy[energy["representation"] == name].set_index("g6pd_level")["energy_over_noise"]
            e = e.reindex([0.5, 0.28, 0.1])
            x = np.arange(3) + (j * 2 + i - 1.5) * width
            ax.bar(x, e.to_numpy(), width=width * 0.92, color=colour,
                   alpha=1.0 if i else 0.75, hatch=None if j == 0 else "////",
                   edgecolor="white", lw=0.4)
    if drawn:
        ax.axhline(1, color=ACCENT["signal"], lw=0.8, ls="--")

        ax.set_yscale("log")
        ax.set_xticks(range(3), [".50", ".28", ".10"])
        ax.set_xlabel("G6PDH2 capacity")
        ax.set_ylabel("energy distance to health\n/ between healthy chains")
        handles = [Patch(color=BLOCK["neutral"], alpha=0.75, label="raw flux"),
                   Patch(color=VARIANT_COLOUR["original"], label="$\\Phi_{int}$ Original"),
                   Patch(facecolor=VARIANT_COLOUR["thermo"], hatch="////", edgecolor="white",
                         label="$\\Phi_{int}$ Thermo"),
                   Line2D([], [], color=ACCENT["signal"], lw=0.8, ls="--",
                          label="healthy chain vs chain")]
        ax.set_ylim(0.8, 900)
        ax.legend(handles=handles, fontsize=5.0, loc="upper left", ncol=2, columnspacing=0.8)
    else:
        _missing(ax, "stage 29b not run")
    panel_label(ax, "c", "Signal over sampling noise")

    # d: which part of the registry carries the signal
    ax = fig.add_subplot(gs_low[1, 0])
    rows = []
    for variant in ("original", "thermo"):
        bench = _read(variant, "benchmark", "reduced_panel.csv")
        groups = _read(variant, "evidence", "family_groups_classification.csv")
        if bench is None or groups is None:
            continue
        def get(rep, model, n=None):
            sub = bench[(bench["representation"] == rep) & (bench["model"] == model)]
            sub = sub[sub["selection"] == "none"] if n is None else sub[sub["n_features"] == n]
            return (float(sub["balanced_accuracy_mean"].iloc[0]),
                    int(sub["n_features"].iloc[0])) if len(sub) else (np.nan, 0)
        for model in ("forest", "logistic"):
            for label, (value, n) in (
                    ("raw flux", get("raw internal flux", model)),
                    ("raw flux\nPCs", tuple(groups.loc[groups["representation"].str.startswith("raw flux, "),
                                                            [f"{model}_balanced_accuracy", "n_features"]].iloc[0])
                     if groups["representation"].str.startswith("raw flux, ").any() else (np.nan, 0)),
                    ("$\\Phi_{int}$ core", get("phenotype (internal core)", model)),
                    ("biochemical\nfamilies", tuple(groups.loc[groups["representation"] == "biochemical families",
                                                                [f"{model}_balanced_accuracy", "n_features"]].iloc[0])),
                    ("generic\nfamilies", tuple(groups.loc[groups["representation"] == "generic families",
                                                            [f"{model}_balanced_accuracy", "n_features"]].iloc[0])),
                    ("3-descriptor\npanel", get("phenotype (internal core)", "logistic", 3) if model == "logistic"
                     else (np.nan, 3))):
                rows.append({"variant": variant, "model": model, "label": label,
                             "value": float(value), "n": int(n)})
    if rows:
        frame = pd.DataFrame(rows)
        labels = list(dict.fromkeys(frame["label"]))
        width = 0.2
        for j, variant in enumerate(("original", "thermo")):
            for i, model in enumerate(("forest", "logistic")):
                sub = frame[(frame["variant"] == variant) & (frame["model"] == model)].set_index("label")
                x = np.arange(len(labels)) + (j * 2 + i - 1.5) * width
                ax.bar(x, sub.reindex(labels)["value"].to_numpy(), width=width * 0.92,
                       color=VARIANT_COLOUR[variant], alpha=1.0 if model == "forest" else 0.55,
                       hatch=None if j == 0 else "////", edgecolor="white", lw=0.4)
        dims = frame[frame["variant"] == "original"].drop_duplicates("label").set_index("label")["n"]
        ax.set_xticks(range(len(labels)),
                      [f"{l}\n({dims[l]:,})" + ("\nlogistic only" if "panel" in l else "")
                       for l in labels], fontsize=5.2)
        ax.axhline(0.25, color=MUTED, lw=0.7, ls=":")
        ax.set_ylim(0, 1.13)
        ax.set_yticks(np.arange(0, 1.01, 0.2))
        ax.set_ylabel("balanced accuracy\n(chains held out)")
        handles = [Patch(color=VARIANT_COLOUR["original"], label="Original, forest"),
                   Patch(color=VARIANT_COLOUR["original"], alpha=0.55, label="Original, logistic"),
                   Patch(facecolor=VARIANT_COLOUR["thermo"], hatch="////", edgecolor="white",
                         label="Thermo, forest"),
                   Patch(facecolor=VARIANT_COLOUR["thermo"], alpha=0.55, hatch="////",
                         edgecolor="white", label="Thermo, logistic")]
        ax.legend(handles=handles, fontsize=5.0, ncol=4, loc="upper center",
                  bbox_to_anchor=(0.5, 1.01))
    else:
        _missing(ax, "stage 29b not run")
    panel_label(ax, "d", "Biochemical descriptors carry the signal; generic statistics do not")

    # e: which descriptors the forest uses
    ax = fig.add_subplot(gs_low[1, 1])
    imp = {v: _read(v, "evidence", "permutation_importance.csv") for v in ("original", "thermo")}
    if imp["original"] is not None:
        top = imp["original"].head(8)
        y = np.arange(len(top))[::-1]
        ax.barh(y, top["importance"].clip(lower=1e-5), color=VARIANT_COLOUR["original"],
                height=0.6)
        if imp["thermo"] is not None:
            t = imp["thermo"].set_index("feature")["importance"]
            ax.scatter(t.reindex(top["feature"]).clip(lower=1e-5), y, marker="D", s=12,
                       color=VARIANT_COLOUR["thermo"], zorder=4, label="Thermo")
        ax.set_yticks(y, [f.replace("path_pentose_phosphate", "path_ppp") for f in top["feature"]],
                      fontsize=5.2, family="DejaVu Sans Mono")
        ax.set_xscale("log")
        ax.set_xlabel("drop in balanced accuracy\nwhen permuted (held-out chain)")
        ax.legend(fontsize=5.0, loc="lower right")
    else:
        _missing(ax, "stage 29b not run")
    panel_label(ax, "e", "What the forest uses")
    save(fig, out / "Figure3_same_samples")



# ==========================================================================
# Figure 4
# ==========================================================================
def paper_figure5(out: Path) -> None:
    fig = plt.figure(figsize=(FIGURE_WIDTH, 2.9))
    gs = GridSpec(1, 3, figure=fig, wspace=0.48, left=0.08, right=0.985, top=0.86, bottom=0.18)
    tables = {v: _read(v, "benchmark", "reduced_panel.csv") for v in ("original", "thermo")}

    for k, variant in enumerate(("original", "thermo")):
        ax = fig.add_subplot(gs[0, k])
        table = tables[variant]
        if table is None:
            _missing(ax, f"stage 28 not run for {variant}")
            continue
        meta = _read(variant, "sample_metadata.csv.gz")
        levels = int(meta["g6pd_level"].nunique()) if meta is not None else 4
        ax.axhline(1 / levels, color=MUTED, lw=0.7, ls=":")
        ax.text(80, 1 / levels + 0.012, "majority baseline", fontsize=5.2, color=MUTED, ha="right")
        reduced = table[(table["selection"] != "none")
                        & (table["representation"] == "phenotype (internal core)")]
        ax.errorbar(reduced["n_features"], reduced["balanced_accuracy_mean"],
                    yerr=reduced["balanced_accuracy_sd"], color=VARIANT_COLOUR[variant],
                    marker="o", ms=3, lw=1.0, capsize=2, label="panel (selected in fold)")
        full = table[table["selection"] == "none"]
        styles = {"raw internal flux": (BLOCK["neutral"], "--"),
                  "phenotype (internal core)": (VARIANT_COLOUR[variant], "-")}
        for representation, (colour, ls) in styles.items():
            row = full[(full["representation"] == representation) & (full["model"] == "logistic")]
            if len(row):
                ax.axhline(float(row["balanced_accuracy_mean"].iloc[0]), color=colour,
                           lw=0.9, ls=ls, alpha=0.9,
                           label=f"{representation.replace('phenotype (internal core)', 'full internal core')}"
                                 f" ({int(row['n_features'].iloc[0])})")
        ax.set_xscale("log")
        ax.xaxis.set_minor_locator(NullLocator())
        ax.xaxis.set_minor_formatter(NullFormatter())
        ax.set_xticks([3, 5, 10, 20, 40, 80])
        ax.set_xticklabels(["3", "5", "10", "20", "40", "80"])
        ax.set_xlabel("descriptors in the panel")
        ax.set_ylim(0.2, 1.0)
        if k == 0:
            ax.set_ylabel("balanced accuracy\n(chains held out, logistic)")
        ax.legend(fontsize=5.0, loc="center right")
        panel_label(ax, "ab"[k], f"{VARIANT_LABEL[variant]}: {levels} capacity levels")

    ax = fig.add_subplot(gs[0, 2])
    drawn = False
    for variant, table in tables.items():
        if table is None:
            continue
        reduced = table[(table["selection"] != "none")
                        & (table["representation"] == "phenotype (internal core)")]
        ax.plot(reduced["n_features"], reduced["selection_jaccard"], marker="o", ms=3,
                color=VARIANT_COLOUR[variant], lw=1.0, label=VARIANT_LABEL[variant])
        drawn = True
    if drawn:
        ax.set_xscale("log")
        ax.xaxis.set_minor_locator(NullLocator())
        ax.xaxis.set_minor_formatter(NullFormatter())
        ax.set_xticks([3, 5, 10, 20, 40, 80])
        ax.set_xticklabels(["3", "5", "10", "20", "40", "80"])
        ax.set_ylim(0, 1.05)
        ax.set_xlabel("descriptors in the panel")
        ax.set_ylabel("overlap of the sets\nchosen in different folds")
        ax.legend(fontsize=5.6)
    else:
        _missing(ax, "stage 28 not run")
    panel_label(ax, "c", "Is the same panel chosen?")
    save(fig, out / "Figure4_full_versus_panel")


# ==========================================================================
# Figure 5
# ==========================================================================
def paper_figure6(out: Path) -> None:
    response = _read("transfer", "B_response.csv")
    panel = _read("transfer", "C_panel.csv")
    predictive = _read("transfer", "D_predictive.csv")

    fig = plt.figure(figsize=(FIGURE_WIDTH, 5.4))
    gs = GridSpec(2, 2, figure=fig, wspace=0.42, hspace=0.66,
                  left=0.09, right=0.985, top=0.93, bottom=0.11)

    # a. response in one model against the other
    ax = fig.add_subplot(gs[0, 0])
    if response is None:
        _missing(ax, "stage 29 not run")
    else:
        column = sorted(c for c in response.columns if c.startswith("delta_reference_"))[0]
        level = column[len("delta_reference_"):]
        x = response[column].to_numpy(float)
        y = response[f"delta_reduced_{level}"].to_numpy(float)
        ok = np.isfinite(x) & np.isfinite(y)
        tx, ty = np.arcsinh(x[ok]), np.arcsinh(y[ok])
        colours = [SUPPORT_COLOUR.get(c, "#BBBBBB") for c in response.loc[ok, "support_category"]]
        ax.scatter(tx, ty, s=7, c=colours, linewidths=0, alpha=0.85)
        lim = max(np.abs(tx).max(), np.abs(ty).max()) * 1.05
        ax.plot([-lim, lim], [-lim, lim], color=MUTED, lw=0.6, ls="--")
        ax.axhline(0, color=RULE, lw=0.6)
        ax.axvline(0, color=RULE, lw=0.6)
        ax.set_xlim(-lim, lim)
        ax.set_ylim(-lim, lim)
        ticks = [-100, -30, -10, -3, -1, 0, 1, 3, 10, 30, 100]
        ticks = [t for t in ticks if abs(np.arcsinh(t)) <= lim * 1.001]
        for setter in (ax.set_xticks, ax.set_yticks):
            setter([np.arcsinh(t) for t in ticks])
        ax.set_xticklabels([str(t) for t in ticks], fontsize=5.8)
        ax.set_yticklabels([str(t) for t in ticks], fontsize=5.8)
        ax.set_xlabel(f"response in Original at G6PDH2 {level}\n(healthy robust SD units, asinh scale)",
                      fontsize=6.0)
        ax.set_ylabel("response in Thermo", fontsize=6.0)
        handles = [Patch(color=SUPPORT_COLOUR[c], label=c) for c in SUPPORT_ORDER
                   if (response.loc[ok, "support_category"] == c).any()]
        ax.legend(handles=handles, fontsize=5.0, loc="upper left")
    panel_label(ax, "a", "The same response, descriptor by descriptor")

    # b. what happens to the descriptors that respond in the original model
    ax = fig.add_subplot(gs[0, 1])
    if response is None:
        _missing(ax, "stage 29 not run")
    else:
        responsive = response[response["responsive_reference"].astype(bool)]
        categories = [c for c in SUPPORT_ORDER if (responsive["support_category"] == c).any()]
        shown = ["conserved", "attenuated", "amplified", "reversed", "not comparable"]
        for k, category in enumerate(categories):
            frame = responsive[responsive["support_category"] == category]
            counts = frame["response_class"].value_counts()
            left = 0.0
            for cls in shown:
                share = counts.get(cls, 0) / len(frame)
                if share:
                    ax.barh(k, share, left=left, color=CLASS_COLOUR[cls], height=0.6,
                            edgecolor=PAPER, linewidth=0.4)
                    if share >= 0.12:
                        ax.text(left + share / 2, k, f"{counts[cls]}", ha="center",
                                va="center", fontsize=5.6,
                                color=PAPER if cls in ("conserved", "reversed") else INK)
                    left += share
            ax.text(1.02, k, f"n={len(frame)}", va="center", fontsize=5.6, color=MUTED)
        ax.set_yticks(range(len(categories)))
        ax.set_yticklabels(categories, fontsize=6.0)
        ax.set_xlim(0, 1)
        ax.set_xlabel("share of the descriptors that respond in Original")
        ax.invert_yaxis()
        ax.grid(axis="y", visible=False)
        handles = [Patch(color=CLASS_COLOUR[c], label=c) for c in shown
                   if (responsive["response_class"] == c).any()]
        ax.legend(handles=handles, fontsize=5.2, loc="upper center",
                  bbox_to_anchor=(0.45, -0.28), ncol=4)
    panel_label(ax, "b", "What the reduction does to a response", offset=-60)

    # c. the panel, carried over without reselection
    ax = fig.add_subplot(gs[1, 0])
    if panel is None:
        _missing(ax, "stage 29 not run")
    else:
        order = [("panel selected in original", "original"), ("panel selected in original", "thermo"),
                 ("panel selected in thermo", "thermo"), ("full internal core (original)", "original"),
                 ("full internal core (thermo)", "thermo")]
        labels = ["Original\npanel in\nOriginal", "Original\npanel in\nThermo",
                  "Thermo\npanel in\nThermo", "full core\nin\nOriginal", "full core\nin\nThermo"]
        values, errors, colours = [], [], []
        for representation, evaluated in order:
            row = panel[(panel["representation"] == representation)
                        & (panel["evaluated_in"] == evaluated)]
            values.append(float(row["balanced_accuracy_mean"].iloc[0]) if len(row) else np.nan)
            errors.append(float(row["balanced_accuracy_sd"].iloc[0]) if len(row) else 0)
            colours.append(VARIANT_COLOUR[evaluated])
        x = np.arange(len(order))
        ax.bar(x, values, yerr=errors, color=colours, width=0.62, capsize=2,
               edgecolor=PAPER, linewidth=0.4, error_kw={"lw": 0.7})
        ax.axhline(float(panel["baseline"].iloc[0]), color=MUTED, lw=0.7, ls=":")
        for xi, value in zip(x, values):
            if np.isfinite(value):
                ax.text(xi, value + 0.035, f"{value:.2f}", ha="center", fontsize=5.6)
        ax.set_xticks(x)
        ax.set_xticklabels(labels, fontsize=5.4)
        ax.set_ylim(0, 1.1)
        ax.set_ylabel("balanced accuracy\n(shared design, chains held out)")
    panel_label(ax, "c", "The panel, carried over without reselection")

    # d. prediction within and across models
    ax = fig.add_subplot(gs[1, 1])
    if predictive is None:
        _missing(ax, "stage 29 not run")
    else:
        representations = list(dict.fromkeys(predictive["representation"]))
        short = {"internal descriptors with identical support": "identical\nsupport",
                 "all comparable internal core descriptors": "all\ncomparable",
                 "panel selected in original": "Original\npanel",
                 "raw internal fluxes (shared reactions)": "raw fluxes\n(shared rxns)"}
        settings = [("strict", "within model", "#9AA5B1", "within model"),
                    ("strict", "across models", "#B04430", "across, strict"),
                    ("reference-relative", "across models", "#1F5B7A", "across, reference-relative")]
        width = 0.26
        x = np.arange(len(representations))
        for k, (regime, kind, colour, label) in enumerate(settings):
            values = []
            for representation in representations:
                rows = predictive[(predictive["preprocessing"] == regime)
                                  & (predictive["kind"] == kind)
                                  & (predictive["representation"] == representation)]
                values.append(rows["balanced_accuracy"].mean() if len(rows) else np.nan)
            ax.bar(x + (k - 1) * width, values, width=width * 0.94, color=colour, label=label)
        ax.axhline(float(predictive["baseline"].iloc[0]), color=MUTED, lw=0.7, ls=":")
        ax.set_xticks(x)
        ax.set_xticklabels([short.get(r, r) for r in representations], fontsize=5.6)
        ax.set_ylim(0, 1.05)
        ax.set_ylabel("balanced accuracy\n(mean of both directions)")
        ax.legend(fontsize=5.2, loc="upper center", bbox_to_anchor=(0.5, -0.3), ncol=3)
    panel_label(ax, "d", "Within one model, and across the two", offset=-60)
    save(fig, out / "Figure5_what_survives")


# ==========================================================================
# Supplementary Figure S1 — is each sampled space usable, and is it the one it claims to be?
# ==========================================================================
# Figure 6
# ==========================================================================
def paper_figure7(out: Path) -> None:
    fig = plt.figure(figsize=(FIGURE_WIDTH, 4.7))
    gs = GridSpec(2, 2, figure=fig, wspace=0.42, hspace=0.62, left=0.09, right=0.975,
                  top=0.92, bottom=0.11)

    # a: does the representation organise states by condition, not by chain?
    ax = fig.add_subplot(gs[0, 0])
    groups = ("capacity", "H$_2$O$_2$\n(null)", "chain\n(null)")
    width, drawn = 0.19, False
    for j, variant in enumerate(("original", "thermo")):
        table = _read(variant, "landscape", "representation_clusters.csv")
        if table is None:
            continue
        drawn = True
        for i, (name, colour) in enumerate((("raw flux", BLOCK["neutral"]),
                                            ("phenotype", VARIANT_COLOUR[variant]))):
            sub = table[table["representation"] == name]
            best = sub.loc[sub["ari_with_severity"].idxmax()]
            values = [best["ari_with_severity"], best["ari_with_h2o2_null"],
                      best["ari_with_chain_null"]]
            x = np.arange(3) + (j * 2 + i - 1.5) * width
            ax.bar(x, values, width=width * 0.92, color=colour, alpha=1.0 if i else 0.75,
                   hatch=None if j == 0 else "////", edgecolor="white", lw=0.4)
    if drawn:
        ax.set_xticks(range(3), groups)
        ax.set_ylabel("adjusted Rand index\n(k-means, best k)")
        ax.set_ylim(0, 0.5)
        handles = [Patch(color=BLOCK["neutral"], alpha=0.75, label="raw flux"),
                   Patch(color=VARIANT_COLOUR["original"], label="$\\Phi_{int}$ Original"),
                   Patch(facecolor=VARIANT_COLOUR["thermo"], hatch="////", edgecolor="white",
                         label="$\\Phi_{int}$ Thermo")]
        ax.legend(handles=handles, fontsize=5.0, loc="upper right")
    else:
        _missing(ax, "stage 12 not run")
    panel_label(ax, "a", "Unsupervised structure follows the condition")

    # b: what the boundary can and cannot say about the interior
    ax = fig.add_subplot(gs[0, 1])
    pred = _read("original", "robustness", "internal_feature_predictability.csv")
    coupling = _read("original", "robustness", "internal_external_coupling.csv")
    if pred is not None:
        x = pred["severity_epsilon_squared"].clip(lower=1e-4)
        ax.scatter(x, pred["predictability_r2_from_external"], s=8, color=BLOCK["internal"],
                   alpha=0.55, linewidths=0)
        for feature, label, offset in (("turnover_6pgc_c", "6PGC turnover", (-6, 12)),
                                       ("path_pentose_phosphate_activity", "PPP activity", (-8, -10)),
                                       ("eff_ppp_flux_split", "G6PDH2/HK split", (-6, 5))):
            row = pred[pred["feature"] == feature]
            if len(row):
                ax.scatter(row["severity_epsilon_squared"], row["predictability_r2_from_external"],
                           s=22, color=ACCENT["signal"], zorder=4, linewidths=0)
                ax.annotate(label, (float(row["severity_epsilon_squared"].iloc[0]),
                                    float(row["predictability_r2_from_external"].iloc[0])),
                            xytext=offset, textcoords="offset points", fontsize=5.0,
                            color=ACCENT["signal"], ha="right", va="center",
                            arrowprops=dict(arrowstyle="-", lw=0.4, color=ACCENT["signal"]))
        ax.set_xscale("log")
        ax.set_xlabel("response to capacity ($\\varepsilon^2$)")
        ax.set_ylabel("$R^2$ predicted from the\nboundary block")
        ax.set_ylim(-0.02, max(0.6, float(pred["predictability_r2_from_external"].max()) + 0.05))
        if coupling is not None:
            cc = coupling["first_canonical_correlation"]
            ax.text(0.98, 0.95, f"first canonical correlation\n{cc.min():.2f}–{cc.max():.2f}",
                    transform=ax.transAxes, fontsize=5.2, color=MUTED, ha="right", va="top")
    else:
        _missing(ax, "stage 09b not run")
    panel_label(ax, "b", "Coupled overall, uninformative where it matters")

    # c: every boundary intervention, screened exactly
    ax = fig.add_subplot(gs[1, 0])
    drawn = False
    for variant in ("original", "thermo"):
        screening = _read(variant, "interventions", "screening.csv")
        if screening is None:
            continue
        drawn = True
        r = 100 * screening.loc[screening["estimable"], "recovery"].dropna()
        ax.hist(r, bins=np.linspace(-12, 12, 49), histtype="step" if variant == "thermo" else "stepfilled",
                color=VARIANT_COLOUR[variant], alpha=0.85 if variant == "thermo" else 0.55,
                lw=1.0, label=f"{VARIANT_LABEL[variant]} ({len(r):,})")
    if drawn:
        ax.axvline(0, color=INK, lw=0.6)
        ax.set_xlabel("recovery of the health distance (%)")
        ax.set_ylabel("candidate interventions")
        ax.legend(fontsize=5.2, loc="upper left")
    else:
        _missing(ax, "stage 08 not run")
    panel_label(ax, "c", "Boundary interventions barely move the interior")

    # d: the confirmatory sampler against a case where the answer is known
    ax = fig.add_subplot(gs[1, 1])
    drawn = False
    for variant in ("original", "thermo"):
        control = _read(variant, "interventions", "sampler_control.csv")
        if control is None:
            continue
        drawn = True
        ax.plot(control["total_steps"], control["ks_statistic"], marker="o", ms=3,
                color=VARIANT_COLOUR[variant], lw=1.0, label=VARIANT_LABEL[variant])
    if drawn:
        ax.axhline(0.10, color=ACCENT["ok"], lw=0.8, ls="--")
        ax.text(float(control["total_steps"].min()), 0.115, "agreement threshold", fontsize=5.0,
                color=ACCENT["ok"], va="bottom")
        ax.set_xscale("log")
        ax.set_ylim(0, 0.8)
        ax.set_xlabel("sampler steps")
        ax.set_ylabel("KS distance to the\nreference distribution")
        ax.legend(fontsize=5.2, loc="upper right")
    else:
        _missing(ax, "stage 08b not run")
    panel_label(ax, "d", "A confirmatory sampler that fails its own control")
    save(fig, out / "Figure6_secondary_validations")


# ==========================================================================
def supplementary_figure1(out: Path) -> None:
    fig = plt.figure(figsize=(FIGURE_WIDTH, 4.6))
    gs = GridSpec(2, 3, figure=fig, wspace=0.45, hspace=0.62,
                  left=0.08, right=0.985, top=0.93, bottom=0.10)
    diagnostics = {v: _read(v, "sampling", "reaction_diagnostics.csv") for v in ("original", "thermo")}
    gates = {v: _read(v, "sampling", "qc_decision.csv") for v in ("original", "thermo")}

    def per_variant(ax, column, transform, label, gate, letter, title, log=False):
        drawn = False
        for k, variant in enumerate(("original", "thermo")):
            frame = diagnostics[variant]
            if frame is None or column not in frame.columns:
                continue
            values = frame.loc[~frame["invariant"].astype(bool), column].dropna().to_numpy(float)
            values = transform(values)
            parts = ax.violinplot([values], positions=[k], widths=0.7, showextrema=False)
            for body in parts["bodies"]:
                body.set_facecolor(VARIANT_COLOUR[variant])
                body.set_alpha(0.55)
                body.set_edgecolor("none")
            ax.plot([k - 0.18, k + 0.18], [np.median(values)] * 2, color=INK, lw=1.0)
            drawn = True
        if not drawn:
            _missing(ax, "sampling diagnostics not run")
            return
        ax.axhline(transform(np.array([gate]))[0], color=ACCENT["signal"], lw=0.8, ls="--")
        ax.set_xticks([0, 1])
        ax.set_xticklabels(["Original", "Thermo"])
        ax.set_ylabel(label)
        ax.grid(axis="x", visible=False)
        panel_label(ax, letter, title)

    per_variant(fig.add_subplot(gs[0, 0]), "ess_total", np.log10,
                "$\\log_{10}$ total ESS per reaction", 400, "a", "Effective sample size")
    ax_b = fig.add_subplot(gs[0, 1])
    per_variant(ax_b, "split_rhat", lambda x: x,
                "split $\\hat{R}$ per reaction", 1.05, "b", "Agreement between chains")
    worst = [diagnostics[v]["split_rhat"].max() for v in ("original", "thermo")
             if diagnostics[v] is not None]
    if worst:
        ax_b.set_ylim(0.99, 1.30)
        for k, value in enumerate(worst):
            ax_b.text(k, 1.285, f"max {value:.2f}", ha="center", va="top", fontsize=5.4,
                      color=INK)

    ax = fig.add_subplot(gs[0, 2])
    ax.set_axis_off()
    rows = []
    for variant in ("original", "thermo"):
        table = gates[variant]
        if table is None:
            continue
        for _, row in table.iterrows():
            rows.append((VARIANT_LABEL[variant], str(row["criterion"]), str(row["target"]),
                         str(row["observed"]), bool(row["passed"])))
    if rows:
        y = 0.97
        for model, criterion, target, observed, passed in rows:
            short = (criterion.replace("across varying reactions", "")
                     .replace("fraction of reactions with ", "frac. ")
                     .replace("median running-mean drift over the final quarter", "running-mean drift")
                     .replace("split R-hat across chains", "split R-hat"))
            ax.text(0.0, y, f"{model}", fontsize=5.0, color=MUTED, va="top")
            ax.text(0.22, y, short[:34], fontsize=5.0, color=INK, va="top")
            ax.text(1.02, y, observed.split(" (")[0], fontsize=5.0, va="top", ha="right",
                    color=ACCENT["ok"] if passed else ACCENT["signal"], fontweight="bold")
            y -= 0.095
        panel_label(ax, "c", "Criteria fixed in advance", offset=0.0)
    else:
        _missing(ax, "no criteria table")

    # d. every chain against the bound it claims to carry
    ax = fig.add_subplot(gs[1, :])
    frames = [t for t in (_read(v, "sampling", "constraint_integrity.csv") for v in ("original", "thermo"))
              if t is not None]
    if not frames:
        _missing(ax, "stage 00b not run")
    else:
        table = pd.concat(frames)
        table = table[table["reaction"] == G6PD_REACTION].copy()
        table["x"] = np.arange(len(table))
        for verdict, marker in (("pass", "o"), ("fail", "X")):
            sub = table[table["verdict"] == verdict]
            ax.scatter(sub["x"], sub["max_over_imposed"],
                       c=[VARIANT_COLOUR[v] for v in sub["model_variant"]],
                       marker=marker, s=18 if verdict == "pass" else 34, linewidths=0,
                       label=f"chain {verdict}es" if verdict == "pass" else "chain fails")
        ax.axhspan(BINDING_FRACTION, 1 + 1e-3, color=ACCENT["ok"], alpha=0.12, lw=0)
        ax.axhline(1, color=ACCENT["ok"], lw=0.8)
        ax.set_ylim(0, max(3.4, float(table["max_over_imposed"].max()) * 1.1))
        ax.set_ylabel("max G6PDH2 flux / imposed bound")
        boundaries = table.groupby("model_variant")["x"].agg(["min", "max"])
        for variant, row in boundaries.iterrows():
            ax.text((row["min"] + row["max"]) / 2, 0.97, VARIANT_LABEL[variant],
                    transform=ax.get_xaxis_transform(), ha="center", va="top",
                    fontsize=6.4, color=VARIANT_COLOUR[variant], fontweight="bold")
        ax.axvline(boundaries["max"].min() + 0.5, color=RULE, lw=0.8)
        ticks = table.groupby(["model_variant", "level"])["x"].mean()
        ax.set_xticks(ticks.to_numpy())
        ax.set_xticklabels([f"{level:g}" for (_, level) in ticks.index], fontsize=6.0)
        ax.set_xlabel("imposed G6PDH2 capacity (each point one chain; nine chains per level)")
        ax.grid(axis="x", visible=False)
        ax.legend(handles=[Line2D([], [], marker="o", ls="", color=MUTED, label="carries its bound"),
                           Line2D([], [], marker="X", ls="", color=MUTED, ms=6,
                                  label="does not: excluded")], fontsize=5.6, loc="center left")
        panel_label(ax, "d", "Every perturbed chain against the bound it is named after")
    save(fig, out / "FigureS1_sampling_and_integrity")


PAPER_FIGURES = (
    ("Figure 1  two structures, one registry", paper_figure2),
    ("Figure 2  response and negative control", paper_figure3),
    ("Figure 3  same samples, same reference, different representation", paper_figure4),
    ("Figure 4  full representation against a reduced panel", paper_figure5),
    ("Figure 5  what survives the reduction", paper_figure6),
    ("Figure 6  secondary validations", paper_figure7),
    ("Figure S1 sampling diagnostics and constraint integrity", supplementary_figure1),
)


def run_22_figures() -> None:
    """Draw Figures 2–7 and S1 and copy them to reports/paper/figures."""
    apply_theme()
    out = Path(FIGURE_ROOT) / "paper"
    out.mkdir(parents=True, exist_ok=True)
    Path(PAPER_FIGURE_DIR).mkdir(parents=True, exist_ok=True)
    for name, draw in PAPER_FIGURES:
        draw(out)
        print(f"{name} written")
    for path in sorted(out.glob("Figure*")):
        shutil.copy2(path, Path(PAPER_FIGURE_DIR) / path.name)
    print(f"\nfigures in {out.relative_to(ROOT)}, copied to "
          f"{Path(PAPER_FIGURE_DIR).relative_to(ROOT)}")

## 49. Supplementary Table S1 — every declared descriptor (`run_23_supplementary_tables`)

The registry as the paper publishes it: definition, requirement, scope,
non-applicability rule, what quality control decided and why, and the
effect size on the imposed axis beside the effect size on the axis that
cannot respond. Descriptors that were dropped are listed with the reason.

In [ ]:
"""Stage — the supplementary tables, built from the registry and the stages.

Supplementary Table S1 is the object the paper promises in the Methods: every
descriptor the engine ever considered, with its definition, what it requires
from a reconstruction, what it returns when that requirement is absent, what
the quality-control cascade decided about it and why, and how large an effect
it shows on the imposed axis and on the axis that cannot respond.

It is not a summary of the retained features. A descriptor that was dropped is
listed with the reason it was dropped, because the reason is the part a reader
cannot reconstruct and the part a reviewer will ask for. Nothing in it is
written by hand: every column is read from a file an earlier stage wrote, so
the table cannot drift from the analysis it documents.
"""
import numpy as np
import pandas as pd


__all__ = ["S1_COLUMNS", "S1_NOTES", "build_supplementary_table_s1",
           "write_table_s1_workbook", "run_23_supplementary_tables"]


#: The published column order, and the heading each column carries. Order is
#: narrative: what the descriptor is, what it needs, what was decided, what it
#: showed.
S1_COLUMNS = [
    ("feature", "Descriptor"),
    ("family", "Family"),
    ("block", "Block"),
    ("description", "Definition"),
    ("formula", "Closed form"),
    ("units", "Units"),
    ("signed_input", "Input is signed v"),
    ("interpretation", "Biological reading"),
    ("requires", "Requires"),
    ("scope", "Transferability scope"),
    ("na_rule", "When the requirement is absent"),
    ("sign_convention", "Sign convention"),
    ("validated_by", "Checked by"),
    ("transform", "Transform applied"),
    ("n_finite", "Finite values"),
    ("fraction_missing", "Fraction missing"),
    ("mean", "Mean"),
    ("sd", "SD"),
    ("cv", "CV"),
    ("fraction_outliers", "Fraction outlying"),
    ("chain_icc", "Chain ICC"),
    ("cluster", "Redundancy cluster"),
    ("representative", "Cluster representative"),
    ("retained", "In the core phenotype"),
    ("exclusion_reason", "Reason for exclusion"),
    ("epsilon_squared_active", "Effect size on the G6PDH2 axis"),
    ("p_adjusted_active", "FDR-adjusted p, G6PDH2 axis"),
    ("epsilon_squared_null", "Effect size on the H2O2 null axis"),
    ("p_adjusted_null", "FDR-adjusted p, H2O2 null axis"),
]

#: The two axes the table reports side by side. The second one was shown in
#: advance to exclude none of the healthy sampled measure, so what a descriptor
#: scores there is the false-positive scale for what it scores on the first.
S1_AXES = {"active": "g6pd_including_healthy", "null": "h2o2_null_axis"}


def build_supplementary_table_s1(variant: str) -> pd.DataFrame:
    """Join registry, documentation, quality control and effect sizes for one model."""
    features = TABLE_ROOT / variant / "features"
    registry_path = features / "feature_registry.csv"
    if not registry_path.exists():
        raise FileNotFoundError(f"no registry for {variant}: run the feature stage first")

    table = pd.read_csv(registry_path)

    documentation = pd.DataFrame(FAMILY_DOC).T.rename_axis("family").reset_index()
    table = table.merge(documentation, on="family", how="left")
    undocumented = sorted(set(table.loc[table["scope"].isna(), "family"]))
    if undocumented:
        print(f"  families with no documentation entry: {undocumented}")

    decision_path = features / "feature_qc_decision.csv"
    if decision_path.exists():
        decision = pd.read_csv(decision_path)
        keep = ["feature", "n_finite", "fraction_missing", "mean", "sd", "cv",
                "fraction_outliers", "chain_icc", "cluster", "representative",
                "retained", "exclusion_reason"]
        table = table.merge(decision[[c for c in keep if c in decision.columns]],
                            on="feature", how="left")

    transform_path = features / "feature_transforms.csv"
    if transform_path.exists():
        transforms = pd.read_csv(transform_path)
        table = table.merge(transforms[["feature", "transform"]], on="feature", how="left")

    discrimination_path = features / "feature_discrimination.csv"
    if discrimination_path.exists():
        discrimination = pd.read_csv(discrimination_path)
        for suffix, axis in S1_AXES.items():
            block = (discrimination[discrimination["axis"] == axis]
                     .set_index("feature")[["epsilon_squared", "p_adjusted"]]
                     .rename(columns={"epsilon_squared": f"epsilon_squared_{suffix}",
                                      "p_adjusted": f"p_adjusted_{suffix}"}))
            table = table.merge(block, left_on="feature", right_index=True, how="left")

    # A descriptor that was never tested is missing, not zero: the distinction
    # is the whole point of the non-applicability rule, so it is kept here too.
    for column, heading in S1_COLUMNS:
        if column not in table.columns:
            table[column] = np.nan
    table = table[[column for column, _ in S1_COLUMNS]]

    order = {"internal": 0, "combined": 1, "external": 2}
    table = (table
             .assign(_block=table["block"].map(order).fillna(9),
                     _effect=table["epsilon_squared_active"].fillna(-1.0))
             .sort_values(["_block", "family", "_effect"], ascending=[True, True, False])
             .drop(columns=["_block", "_effect"])
             .reset_index(drop=True))
    table.insert(0, "n", np.arange(1, len(table) + 1))
    return table


#: The legend that travels with the workbook. A supplementary table nobody can
#: read without the paper beside it is not supplementary information.
S1_NOTES = [
    ("What this table is",
     "Every descriptor the engine declared for this reconstruction, whether or not it "
     "survived quality control. A descriptor that was dropped is listed with the reason, "
     "because the reason is the part that cannot be reconstructed from the retained set."),
    ("Block",
     "internal: computed from the internal reactions of the state. external: computed from "
     "the exchange reactions. combined: a ratio between the two."),
    ("Transferability scope",
     "common — computable with the same meaning in any reconstruction. conditional — needs a "
     "pathway, metabolite or reaction that may be absent. model_specific — describes a "
     "property of one particular reconstruction."),
    ("When the requirement is absent",
     "The descriptor returns a missing value and keeps its scope. It is never imputed as "
     "zero: zero is a measurement, absence is not."),
    ("In the core phenotype / Reason for exclusion",
     "The quality-control cascade removes descriptors that are constant, dominated by "
     "outliers, dependent on the sampling chain rather than the condition, or redundant with "
     "a retained representative. No step of the cascade sees a condition label."),
    ("Redundancy cluster / Cluster representative",
     "Descriptors in one cluster are mutually redundant above the declared correlation "
     "threshold; the representative is the one kept. The others remain declared and remain "
     "computable — they are excluded from the core, not from the registry."),
    ("Effect size on the G6PDH2 axis",
     "Epsilon squared over the conditions of the imposed enzyme-capacity axis, with the "
     "healthy reference included. The material-effect floor used in the paper is 0.01; "
     "statistical significance alone is not treated as a finding."),
    ("Effect size on the H2O2 null axis",
     "The same statistic on an axis shown in advance to exclude none of the healthy sampled "
     "measure, which therefore cannot carry a real effect. Whatever a descriptor scores here "
     "is the false-positive scale on which its score on the imposed axis must be read."),
    ("Empty cells",
     "A descriptor that was constant across every state was not tested, so its effect sizes "
     "and chain statistics are empty rather than zero."),
]


def write_table_s1_workbook(table: pd.DataFrame, path, variant: str) -> bool:
    """Write the published workbook: the table, formatted, plus its legend.

    Returns False, without raising, when openpyxl is not installed: the CSV is
    the table of record and the workbook is a convenience for the submission.
    """
    try:
        from openpyxl import Workbook
        from openpyxl.styles import Alignment, Border, Font, PatternFill, Side
        from openpyxl.utils import get_column_letter
    except ImportError:
        print("  openpyxl is not installed; the .xlsx was not written")
        return False

    headings = dict(S1_COLUMNS)
    body = Font(name="Arial", size=9)
    head = Font(name="Arial", size=9, bold=True, color="FFFFFF")
    fill = PatternFill("solid", fgColor="1F5B7A")
    hairline = Side(style="thin", color="DCE2E8")

    workbook = Workbook()
    sheet = workbook.active
    sheet.title = "Table S1"

    columns = ["n"] + [column for column, _ in S1_COLUMNS]
    columns = [c for c in columns if c in table.columns]
    sheet.append(["n" if c == "n" else headings.get(c, c) for c in columns])
    for _, row in table[columns].iterrows():
        sheet.append([None if pd.isna(value) else
                      (bool(value) if isinstance(value, (bool, np.bool_)) else
                       (value.item() if hasattr(value, "item") else value))
                      for value in row])

    numeric = {"mean": "0.000E+00", "sd": "0.000E+00", "cv": "0.0000",
               "fraction_missing": "0.0000", "fraction_outliers": "0.0000",
               "chain_icc": "0.0000", "epsilon_squared_active": "0.0000",
               "epsilon_squared_null": "0.000000",
               "p_adjusted_active": "0.00E+00", "p_adjusted_null": "0.00E+00"}
    width = {"feature": 32, "family": 20, "block": 10, "description": 52, "formula": 40,
             "units": 12, "interpretation": 46, "requires": 34, "scope": 16,
             "na_rule": 44, "sign_convention": 40, "validated_by": 34, "transform": 12,
             "exclusion_reason": 18, "representative": 30}

    for index, column in enumerate(columns, start=1):
        letter = get_column_letter(index)
        sheet.column_dimensions[letter].width = width.get(column, 14)
        style = numeric.get(column)
        for cell in sheet[letter][1:]:
            cell.font = body
            cell.border = Border(bottom=hairline)
            cell.alignment = Alignment(vertical="top", wrap_text=column in width)
            if style:
                cell.number_format = style

    for cell in sheet[1]:
        cell.font = head
        cell.fill = fill
        cell.alignment = Alignment(vertical="center", wrap_text=True)
    sheet.row_dimensions[1].height = 42
    sheet.freeze_panes = "C2"
    sheet.auto_filter.ref = sheet.dimensions

    notes = workbook.create_sheet("Notes")
    notes.column_dimensions["A"].width = 38
    notes.column_dimensions["B"].width = 104
    title = (f"Supplementary Table S1. The declared registry of metabolic descriptors "
             f"({variant} reconstruction).")
    notes.append([title])
    notes["A1"].font = Font(name="Arial", size=11, bold=True)
    notes.append([])
    for heading, text in S1_NOTES:
        notes.append([heading, text])
    counts = [
        ("Descriptors declared", int(len(table))),
        ("Retained in the core phenotype", int(table["retained"].sum())),
        ("Tested on the imposed axis", int(table["epsilon_squared_active"].notna().sum())),
        ("Above the material-effect floor", int((table["epsilon_squared_active"] >= 0.01).sum())),
        ("Significant on the null axis", int((table["p_adjusted_null"] < 0.05).sum())),
    ]
    notes.append([])
    for label, value in counts:
        notes.append([label, value])
    for row in notes.iter_rows(min_row=2):
        for cell in row:
            cell.font = cell.font if cell.row == 1 else Font(name="Arial", size=9)
            cell.alignment = Alignment(vertical="top", wrap_text=True)

    workbook.save(path)
    return True


def run_23_supplementary_tables() -> None:
    """Write Supplementary Table S1 for every variant that has a registry."""
    out = Path(TABLE_ROOT).parent / "supplementary"
    out.mkdir(parents=True, exist_ok=True)

    written = []
    for variant in MODEL_VARIANTS:
        try:
            table = build_supplementary_table_s1(variant)
        except FileNotFoundError as absent:
            print(f"{variant}: {absent}")
            continue

        machine = out / f"table_s1_feature_registry_{variant}.csv"
        table.to_csv(machine, index=False)
        published = table.rename(columns=dict(S1_COLUMNS))
        published.to_csv(out / f"TableS1_{variant}.csv", index=False)
        workbook = out / f"TableS1_feature_registry_{variant}.xlsx"
        write_table_s1_workbook(table, workbook, variant)
        written.append((variant, table))
        print(f"{variant}: {len(table)} descriptors -> {machine.name}, {workbook.name}")

        core = int(table["retained"].sum()) if "retained" in table else 0
        tested = int(table["epsilon_squared_active"].notna().sum())
        material = int((table["epsilon_squared_active"] >= 0.01).sum())
        false_positives = int((table["p_adjusted_null"] < 0.05).sum())
        print(f"  {core} retained in the core phenotype")
        print(f"  {tested} tested on the imposed axis, {material} above the material floor")
        print(f"  {false_positives} significant on the axis that cannot respond")
        print(table.groupby(["block", "family"]).size().to_string())
        print()

    if not written:
        print("no registry written yet; run the feature stage first")


#: One sheet per supplementary table, in the order the paper cites them.
SUPPLEMENTARY_SHEETS = [
    ("S1 registry", "table_s1_feature_registry_{variant}.csv",
     "Every declared descriptor: definition, requirement, scope, what quality control "
     "decided and why, and the effect size on the imposed axis beside the effect size on "
     "the axis that cannot respond."),
    ("S2 importance", "table_s2_feature_importance_{variant}.csv",
     "The internal core ranked seven ways that are allowed to disagree, with the partial "
     "correlation separating descriptors that carry their own information from those that "
     "borrow it, and the exact share of the squared health distance each accounts for."),
    ("S2b boundary", "table_s2b_boundary_importance_{variant}.csv",
     "Model-based importances for the boundary block, kept separate because the predictor "
     "they come from does not predict: read them as a description of the fitted model, not "
     "as evidence about metabolism."),
    ("S4 factorial design", "../tables/{variant}/design/factorial_effects.csv",
     "Per descriptor, measured over the thirty chain medians: the rank correlation with "
     "the imposed capacity and whether it is monotone across the four ordered levels, the "
     "share of variance the control axis explains at fixed capacity, the share left to "
     "the interaction, and the size of the capacity effect in units of the spread between "
     "the chains of one condition."),
    ("S5 parsimony", "../tables/{variant}/benchmark/reduced_panel.csv",
     "Balanced accuracy and rank correlation for each representation and each panel size, "
     "with the selection fitted inside every training fold and whole chains held out, and "
     "the overlap between the sets chosen in different folds beside each one."),
    ("S6 structural support", "../tables/transfer/A_structural.csv",
     "Level A. For every declared descriptor: whether each reconstruction can compute it, "
     "the reactions its formula reads (declared support) in each, the fraction retained, and "
     "the part of that support that carries flux in the reference model's states (active "
     "support) — all of which the reduced model keeps, which is what makes any difference "
     "between the two a property of the descriptor's definition."),
    ("S7 response transfer", "../tables/transfer/B_response.csv",
     "Level B. For every descriptor computable in both: the chain-level gradient in each "
     "model, the standardised response at each capacity level in units of the model's own "
     "healthy robust spread, the shift of its healthy baseline between the models (Cliff's "
     "delta), and the class of what happened to the response."),
    ("S8 panel transfer", "../tables/transfer/C_panel.csv",
     "Level C. The three-descriptor panel selected in the original model, scored in the "
     "reduced model without being selected again, beside a panel selected inside the "
     "reduced model and the full internal core of each."),
    ("S9 prediction across", "../tables/transfer/D_predictive.csv",
     "Level D. Balanced accuracy within each model (chains held out) and across the two "
     "(trained on one, tested on the other), under strict and reference-relative "
     "preprocessing, for four representations."),
    ("S10 chain integrity", "../tables/{variant}/sampling/constraint_integrity.csv",
     "Every perturbed chain against the bounds it is named after: the imposed bound, the "
     "largest flux the chain reaches, whether the bound binds in the healthy distribution, "
     "and the verdict. Chains that fail are excluded from every analysis."),
    ("S11a equal terms O", "../tables/original/evidence/distance_same_terms.csv",
     "Original. The raw internal flux vector, the internal core and its biochemical and "
     "generic halves, standardised on the same healthy states and measured against the same "
     "healthy distribution, out of sample: rank correlation of chain medians with the "
     "capacity, effect-to-noise, null-axis share and state-level AUC, for two distances."),
    ("S11b equal terms T", "../tables/thermo/evidence/distance_same_terms.csv",
     "Thermo. As S11a."),
    ("S11c energy O", "../tables/original/evidence/energy_over_noise.csv",
     "Original. Energy distance of each capacity level to the healthy states, and its ratio "
     "to the energy distance between healthy chains."),
    ("S11d energy T", "../tables/thermo/evidence/energy_over_noise.csv", "Thermo. As S11c."),
    ("S11e families O", "../tables/original/evidence/family_groups_classification.csv",
     "Original. Balanced accuracy, chains held out, of the biochemical and the generic "
     "families of the internal core."),
    ("S11f families T", "../tables/thermo/evidence/family_groups_classification.csv",
     "Thermo. As S11e."),
    ("S11g importance O", "../tables/original/evidence/permutation_importance_folds.csv",
     "Original. Permutation importance of a random forest on the held-out chain, per fold."),
    ("S11h importance T", "../tables/thermo/evidence/permutation_importance_folds.csv",
     "Thermo. As S11g."),
    ("S12a blocked by FVA", "../tables/review/blocked_by_fva.csv",
     "Minimum and maximum feasible flux of every reaction removed in Thermo, in Original, healthy and with G6PDH2 at 0.10."),
    ("S12b peroxide exchange", "../tables/review/peroxide_exchange_by_condition.csv",
     "EX_h2o2_e flux in every condition of both models beside the imposed upper bound (secretion positive)."),
    ("S12c chain prefixes", "../tables/review/chain_prefix_stability.csv",
     "Chain medians of key descriptors from the first quarter, half and three quarters of each chain against the full chain."),
    ("S12d second selector", "../tables/review/panel_second_selector.csv",
     "The three-descriptor panel chosen by an L1-penalised logistic regression inside each fold."),
    ("S12e re-centring chain", "../tables/review/recentring_reference_chain.csv",
     "Cross-model prediction with each healthy chain as the reference of the re-centring."),
    ("S12f coupling and scale", "../tables/review/coupling_scale_control.csv",
     "First canonical correlation between the internal and boundary cores, its variates against total absolute flux, and the correlation after removing that scale."),
    ("S12g interventions by chain", "../tables/review/intervention_recovery_by_chain.csv",
     "Recovery of the leading intervention candidates computed within each chain."),
    ("S3 transferability", "table_s3_descriptor_transferability.csv",
     "What each descriptor requires from a reconstruction, whether each model has it, and "
     "how much of that requirement the smaller model keeps. Computable in both is not the "
     "same as comparable in both, which is why the support columns are there."),
]


def run_26_supplementary_workbook(variant: str = None) -> None:
    """Collect the supplementary tables into one workbook for submission."""
    try:
        from openpyxl import Workbook
        from openpyxl.styles import Alignment, Font, PatternFill
        from openpyxl.utils import get_column_letter
    except ImportError:
        print("openpyxl is not installed; the combined workbook was not written")
        return

    variant = variant or ACTIVE_VARIANT
    out = Path(TABLE_ROOT).parent / "supplementary"
    body = Font(name="Arial", size=9)
    head = Font(name="Arial", size=9, bold=True, color="FFFFFF")
    fill = PatternFill("solid", fgColor="1F5B7A")

    workbook = Workbook()
    contents = workbook.active
    contents.title = "Contents"
    contents.column_dimensions["A"].width = 24
    contents.column_dimensions["B"].width = 108
    contents.append(["Supplementary tables"])
    contents["A1"].font = Font(name="Arial", size=12, bold=True)
    contents.append([])

    written = 0
    for title, pattern, caption in SUPPLEMENTARY_SHEETS:
        path = out / pattern.format(variant=variant)
        if not path.exists():
            contents.append([title, f"not yet written ({path.name})"])
            continue
        table = pd.read_csv(path)
        sheet = workbook.create_sheet(title[:31])
        sheet.append(list(table.columns))
        for row in table.itertuples(index=False):
            sheet.append([None if (isinstance(v, float) and np.isnan(v)) else v for v in row])
        for index, column in enumerate(table.columns, start=1):
            letter = get_column_letter(index)
            longest = max([len(str(column))] + [len(str(v))
                                                for v in table[column].head(80)])
            sheet.column_dimensions[letter].width = min(max(longest + 2, 11), 52)
            for cell in sheet[letter][1:]:
                cell.font = body
                cell.alignment = Alignment(vertical="top", wrap_text=longest > 30)
        for cell in sheet[1]:
            cell.font = head
            cell.fill = fill
            cell.alignment = Alignment(vertical="center", wrap_text=True)
        sheet.row_dimensions[1].height = 34
        sheet.freeze_panes = "B2"
        sheet.auto_filter.ref = sheet.dimensions
        contents.append([title, f"{caption}  [{len(table)} rows]"])
        written += 1

    for row in contents.iter_rows(min_row=3):
        for cell in row:
            cell.font = Font(name="Arial", size=9, bold=cell.column == 1)
            cell.alignment = Alignment(vertical="top", wrap_text=True)

    target = out / f"Supplementary_tables_{variant}.xlsx"
    workbook.save(target)
    print(f"{written} supplementary tables collected into {target.name}")

## 50. Supplementary Table S2 — which descriptors matter, measured several ways (`run_25_importance_summary`)

Seven model-free views of the internal block, plus the partial correlation
and the exact share of the squared health distance, plus the model-based
importances of the boundary block kept separate. Where the measures
disagree is reported, not smoothed away.

In [ ]:
"""Stage — which descriptors matter, measured several ways that can disagree.

A single importance ranking is a claim about one estimator, not about the
representation. This stage puts the measures the pipeline already computes side
by side and reports where they agree and where they do not, because the
disagreement is informative: a descriptor with no univariate association that
the representation nevertheless cannot lose is a different object from one that
correlates strongly and is redundant with six others.

Seven views are collected, in two groups.

*Model-free, on the internal block:* rank correlation with the health distance,
covariance with it, mutual information, loading on the first principal
component, the closed-form sensitivity of the distance, the loss when the
descriptor is ablated, and the effect size over the severity axis. To these are
added the partial correlation — which separates a descriptor that carries its
own information from one that borrows it — and the share of the squared health
distance the descriptor accounts for, which is exact rather than attributed
because the distance is a quadratic form.

*Model-based, on the boundary block:* permutation importance and Shapley value
from the fitted predictors. These are reported separately and are not merged
into a single number, because a model-based importance inherits whatever the
model got wrong.
"""
import numpy as np
import pandas as pd


__all__ = ["IMPORTANCE_VIEWS", "build_importance_table", "run_25_importance_summary"]


#: The model-free views, and whether a larger absolute value means more
#: important. All are read from files earlier stages wrote.
IMPORTANCE_VIEWS = [
    ("spearman_with_health_distance", "rank correlation with the health distance"),
    ("covariance_with_health_distance", "covariance with the health distance"),
    ("mutual_information", "mutual information with the health distance"),
    ("pca1_loading", "loading on the first principal component"),
    ("sensitivity", "closed-form sensitivity of the distance"),
    ("ablation_delta_rho", "loss of rank correlation when ablated"),
    ("severity_epsilon_squared", "effect size over the severity axis"),
]


def _rank(frame: pd.DataFrame, column: str) -> pd.Series:
    """Rank descriptors by the magnitude of one view; 1 is the most important."""
    return frame[column].abs().rank(ascending=False, method="min")


def build_importance_table(variant: str) -> pd.DataFrame:
    """Join every importance view available for one reconstruction."""
    tables = TABLE_ROOT / variant
    contribution_path = tables / "features" / "feature_contribution.csv"
    if not contribution_path.exists():
        raise FileNotFoundError(
            f"no contribution table for {variant}: run the contribution stage first")

    table = pd.read_csv(contribution_path)
    available = [column for column, _ in IMPORTANCE_VIEWS if column in table.columns]
    for column in available:
        table[f"rank_{column}"] = _rank(table, column)
    rank_columns = [f"rank_{c}" for c in available]
    table["median_rank"] = table[rank_columns].median(axis=1)
    table["rank_spread"] = table[rank_columns].max(axis=1) - table[rank_columns].min(axis=1)

    partial_path = tables / "features" / "partial_correlations.csv"
    if partial_path.exists():
        partial = pd.read_csv(partial_path)
        keep = ["feature", "marginal_correlation_with_distance",
                "partial_correlation_with_distance", "partial_over_marginal", "role"]
        table = table.merge(partial[[c for c in keep if c in partial.columns]],
                            on="feature", how="left")

    mahalanobis_path = tables / "features" / "mahalanobis_decomposition.csv"
    if mahalanobis_path.exists():
        decomposition = pd.read_csv(mahalanobis_path)
        keep = ["feature", "share_of_squared_distance_disease",
                "share_of_squared_distance_healthy", "excess_share"]
        table = table.merge(decomposition[[c for c in keep if c in decomposition.columns]],
                            on="feature", how="left")

    minimal_path = tables / "robustness" / "minimal_phenotype.csv"
    if minimal_path.exists():
        minimal = pd.read_csv(minimal_path)[["feature", "step"]]
        table = table.merge(minimal.rename(columns={"step": "greedy_selection_step"}),
                            on="feature", how="left")

    table = table.sort_values("median_rank").reset_index(drop=True)
    table.insert(0, "n", np.arange(1, len(table) + 1))
    return table


def run_25_importance_summary() -> None:
    """Write Supplementary Table S2, and report where the measures disagree."""
    out = Path(TABLE_ROOT).parent / "supplementary"
    out.mkdir(parents=True, exist_ok=True)

    for variant in MODEL_VARIANTS:
        try:
            table = build_importance_table(variant)
        except FileNotFoundError as absent:
            print(f"{variant}: {absent}")
            continue

        table.to_csv(out / f"table_s2_feature_importance_{variant}.csv", index=False)
        print(f"{variant}: {len(table)} internal core descriptors ranked seven ways")

        columns = [f"rank_{c}" for c, _ in IMPORTANCE_VIEWS if f"rank_{c}" in table.columns]
        print("\n  the ten descriptors the measures agree on:")
        top = table.head(10)
        for _, row in top.iterrows():
            role = row.get("role", "")
            share = row.get("share_of_squared_distance_disease", np.nan)
            print(f"    {row['feature']:<34} {row['family']:<14} "
                  f"median rank {row['median_rank']:>5.1f}  spread {row['rank_spread']:>4.0f}"
                  f"  {role}" + (f"  {share:.1%} of D2" if np.isfinite(share) else ""))

        disagreement = table.nlargest(5, "rank_spread")
        print("\n  the five the measures disagree about most:")
        for _, row in disagreement.iterrows():
            print(f"    {row['feature']:<34} {row['family']:<14} "
                  f"ranks {row[columns].min():.0f}-{row[columns].max():.0f}"
                  f"  {row.get('role', '')}")

        if "role" in table.columns:
            print("\n  partial against marginal correlation:")
            print("   ", table["role"].value_counts().to_dict())

        agreement_path = TABLE_ROOT / variant / "features" / "contribution_agreement.csv"
        if agreement_path.exists():
            agreement = pd.read_csv(agreement_path, index_col=0)
            values = agreement.where(~np.eye(len(agreement), dtype=bool)).stack()
            print(f"\n  agreement between the seven measures: median rho "
                  f"{values.median():.2f}, range {values.min():.2f} to {values.max():.2f}")

        external = TABLE_ROOT / variant / "ml"
        permutation = external / "permutation_importance.csv"
        shapley = external / "shapley_importance.csv"
        if permutation.exists() and shapley.exists():
            left = pd.read_csv(permutation)
            left = left[left["experiment"] == "pooled_disease"] if "experiment" in left else left
            right = pd.read_csv(shapley)
            merged = left.merge(right, on="feature", how="outer")
            merged.to_csv(out / f"table_s2b_boundary_importance_{variant}.csv", index=False)
            print(f"\n  boundary block: {len(merged)} descriptors, model-based importances "
                  f"kept separate because the predictor they come from does not predict")
        print()

## 51. Supplementary Table S3 — what each descriptor requires, and what transfers (`run_24_descriptor_transferability`)

Whether a descriptor can be computed in the other reconstruction is a
property of the registry and the two models, answerable before any state
is sampled. Whether it *behaves* the same there is a different question,
answered by the cross-model stage once both variants have been run.

In [ ]:
"""Stage — what each descriptor requires, and whether the other model has it.

Two different questions are often run together and should not be. The first is
*whether a descriptor can be computed at all in another reconstruction*, which
is a property of the registry and of the two models' contents and can be
answered before any state is sampled. The second is *whether the descriptor
behaves the same way there*, which needs the phenotype computed on both and is
answered by the cross-model stage.

This stage answers only the first, and answers it exactly: every descriptor is
resolved to the model object it needs — a metabolite, a named reaction, a
pathway group, an exchange, or nothing at all — and that object is looked up in
each reconstruction. A descriptor whose requirement is missing is reported as
*not computable there*, which is the honest statement; it is never reported as
a difference in the phenotype, because no phenotype was computed.
"""
import numpy as np
import pandas as pd


__all__ = ["resolve_requirement", "descriptor_transferability",
           "pending_descriptor_specs", "run_24_descriptor_transferability"]


#: Descriptors of the biochemical family that are not a named reaction or a
#: metabolite turnover but an aggregate over the oxidative branch.
_OXIDATIVE_BRANCH = ("nadph_from_oxidative_ppp", "oxidative_ppp_flux",
                     "redox_turnover_budget")


def resolve_requirement(feature: str, family: str) -> tuple[str, str]:
    """Return ``(kind, identifier)``: what this descriptor needs from a model.

    ``kind`` is one of ``metabolite``, ``reaction``, ``pathway``, ``exchange``,
    ``exchange_role``, ``any_exchange`` or ``none``. ``none`` means the
    descriptor is a statistic of the flux vector itself and needs nothing
    named, which is what makes a descriptor transferable without conditions.
    """
    if feature.startswith("turnover_"):
        return "metabolite", feature[len("turnover_"):]
    if feature.startswith("rxn_"):
        label = feature[len("rxn_"):]
        return "reaction", KEY_REACTIONS.get(label, label)
    if feature.startswith("eff_"):
        label = feature[len("eff_"):]
        for name, numerator, denominator in EFFICIENCY_RATIOS:
            if name == label:
                return "composite", f"{numerator}+{denominator}"
        return "composite", label
    if feature.startswith("path_"):
        rest = feature[len("path_"):]
        for suffix in ("_activity", "_share", "_engagement", "_directionality"):
            if rest.endswith(suffix):
                return "pathway", rest[: -len(suffix)]
        return "pathway", rest
    if feature.startswith("exrole_"):
        return "exchange_role", feature[len("exrole_"):].rsplit("_", 1)[0]
    if feature in _OXIDATIVE_BRANCH:
        return "pathway", "pentose_phosphate"
    if feature.startswith("ex_"):
        label = feature[len("ex_"):]
        for met, name in TRACKED_EXCHANGES.items():
            for suffix in ("_net", "_uptake"):
                if label == f"{name}{suffix}":
                    return "exchange", met
        return "any_exchange", "at least one exchange reaction"
    if feature.startswith("cof_"):
        return "metabolite", "cofactor pool"
    if feature.startswith("cpl_"):
        return "composite", "both blocks non-empty"
    if feature.startswith("cap_"):
        return "bounds", "finite bounds"
    return "none", ""


def _available(kind: str, identifier: str, content: dict) -> bool | float:
    """Is the object a descriptor needs present in this reconstruction?"""
    if kind == "none":
        return True
    if kind == "metabolite":
        if identifier == "cofactor pool":
            needed = {m for pair in COFACTOR_PAIRS.values() for m in pair}
            return bool(needed <= content["mets"])
        return identifier in content["mets"]
    if kind == "reaction":
        return identifier in content["rxns"]
    if kind == "pathway":
        return content["pathway_sizes"].get(identifier, 0) > 0
    if kind == "exchange":
        return f"EX_{identifier}" in content["rxns"] or identifier in content["exchange_mets"]
    if kind == "exchange_role":
        return identifier in content["roles"]
    if kind == "any_exchange":
        return bool(content["exchange_mets"])
    if kind == "bounds":
        return bool(content.get("has_finite_bounds", True))
    if kind == "composite":
        parts = [p for p in identifier.split("+") if p.startswith(("turnover_", "rxn_", "path_"))]
        if not parts:
            return bool(content["exchange_mets"])
        return all(_available(*resolve_requirement(p, ""), content) for p in parts)
    return np.nan


def model_content(variant: str) -> dict | None:
    """What one reconstruction contains, in the terms the registry asks about."""
    spec = MODEL_VARIANTS[variant]
    path = Path(spec["model_file"])
    if not path.exists():
        return None
    gem = load_gem(path, spec["model_name"])
    rxns = list(gem.rxns)
    mets = set(gem.mets)

    # The annotation table the pipeline writes is the authority on which
    # reaction belongs to which pathway group and what role an exchange plays,
    # because it is what the feature engine itself used. Restricting it to the
    # reactions a variant actually has is exact whenever one model is a subset
    # of the other, which the structural comparison establishes first.
    annotation = None
    for variant_with_tables in MODEL_VARIANTS:
        candidate = TABLE_ROOT / variant_with_tables / "reaction_metadata.csv"
        if candidate.exists():
            annotation = pd.read_csv(candidate)
            break

    if annotation is not None and set(rxns) <= set(annotation["rxn_id"]):
        present = annotation[annotation["rxn_id"].isin(rxns)]
        pathway_sizes = present["pathway"].value_counts().to_dict()
        exchange = present[present.get("is_external", False).astype(bool)]
        exchange_mets = set(exchange["boundary_metabolite"].dropna())
        roles = set(exchange["boundary_role"].dropna())
    else:
        subsystems = list(getattr(gem, "subsystems", [""] * len(rxns)))
        pathway_sizes = {}
        for pathway, members in PATHWAY_GROUPS.items():
            wanted = set(members)
            pathway_sizes[pathway] = sum(1 for s in subsystems if s in wanted)
        shunt = sum(1 for r in rxns if r in RAPOPORT_LUEBERING_RXNS)
        if shunt:
            pathway_sizes["rapoport_luebering"] = shunt
        exchange_mets = {r[len("EX_"):] for r in rxns if r.startswith("EX_")}
        roles = set()
        for met in exchange_mets:
            record = EXCHANGE_ADMISSIBILITY.get(met)
            roles.add(record.role if record is not None else "metabolite")

    lb = np.asarray(gem.lb, float)
    ub = np.asarray(gem.ub, float)
    return {"rxns": set(rxns), "mets": mets, "pathway_sizes": pathway_sizes,
            "exchange_mets": exchange_mets, "roles": roles,
            "met_degree": metabolite_degree(gem),
            "has_finite_bounds": bool((np.isfinite(lb) & np.isfinite(ub)).any())}


def metabolite_degree(gem) -> dict:
    """How many reactions touch each metabolite.

    A turnover descriptor survives as long as its metabolite exists, but what
    it measures changes if half the reactions producing and consuming that
    metabolite are gone. Computability and comparability are different
    questions and this is what separates them.
    """
    precomputed = getattr(gem, "met_degree", None)
    if precomputed:
        return dict(precomputed)
    S = getattr(gem, "S", None)
    if S is None:
        return {}
    counts = np.asarray((abs(S) > 0).sum(axis=1)).ravel()
    return {met: int(n) for met, n in zip(gem.mets, counts)}


def requirement_support(kind: str, identifier: str, content: dict) -> float:
    """How much of what a descriptor needs this model still has.

    For a pathway it is the number of reactions in the group, for a metabolite
    turnover the number of reactions touching it, and for anything named
    exactly it is 1 or 0. Comparing this number between two reconstructions is
    what says whether a descriptor that is computable in both means the same
    thing in both.
    """
    if kind == "pathway":
        return float(content["pathway_sizes"].get(identifier, 0))
    if kind == "metabolite" and identifier != "cofactor pool":
        return float(content.get("met_degree", {}).get(identifier, np.nan))
    if kind in ("reaction", "exchange"):
        return float(bool(_available(kind, identifier, content)))
    if kind == "any_exchange":
        return float(len(content["exchange_mets"]))
    return np.nan


def pending_descriptor_specs() -> pd.DataFrame:
    """The descriptors the three new families declare, read from the classes.

    ``specs()`` depends on nothing but the family definition, so the declared
    registry can be listed before the pipeline has been run on any samples.
    This is what makes the difference between *declared* and *computed*
    auditable rather than a claim.
    """
    rows = []
    for family in (CofactorFamily, CapacityFamily, CouplingFamily):
        instance = family.__new__(family)
        for spec in instance.specs():
            rows.append({"feature": spec.name, "family": spec.family, "block": spec.block,
                         "description": spec.description, "formula": spec.formula,
                         "units": spec.units})
    return pd.DataFrame(rows)


def descriptor_transferability(registry: pd.DataFrame, contents: dict) -> pd.DataFrame:
    """One row per descriptor: what it needs, and which models have it."""
    rows = []
    for _, entry in registry.iterrows():
        kind, identifier = resolve_requirement(entry["feature"], entry.get("family", ""))
        row = {"feature": entry["feature"], "family": entry.get("family", ""),
               "block": entry.get("block", ""), "scope": entry.get("scope", ""),
               "requirement_kind": kind, "requirement": identifier or "none"}
        for variant, content in contents.items():
            row[f"computable_in_{variant}"] = _available(kind, identifier, content)
            row[f"support_in_{variant}"] = requirement_support(kind, identifier, content)
        rows.append(row)
    table = pd.DataFrame(rows)
    variants = list(contents)
    if len(variants) == 2:
        a, b = (f"computable_in_{v}" for v in variants)
        table["shared"] = table[a] & table[b]
        table["lost_in_" + variants[1]] = table[a] & ~table[b]
        first, second = (f"support_in_{v}" for v in variants)
        with np.errstate(invalid="ignore", divide="ignore"):
            table["support_retained"] = np.where(
                table[first] > 0, table[second] / table[first], np.nan)
    return table


def run_24_descriptor_transferability() -> None:
    """Write Supplementary Table S3: what transfers between the two models, and why not."""
    out = Path(TABLE_ROOT).parent / "supplementary"
    out.mkdir(parents=True, exist_ok=True)

    contents = {}
    for variant, spec in MODEL_VARIANTS.items():
        content = model_content(variant)
        if content is None:
            print(f"{variant}: model file not available")
            continue
        contents[variant] = content
    if len(contents) < 2:
        print("both reconstructions are needed for this comparison")
        return

    registry_path = METADATA_DIR / "feature_registry.csv"
    if not registry_path.exists():
        print("no registry yet; run the registry stage first")
        return
    registry = pd.read_csv(registry_path)
    registry["status"] = "computed"

    pending = pending_descriptor_specs()
    if len(pending):
        known = set(registry["feature"])
        pending = pending[~pending["feature"].isin(known)].copy()
        pending["status"] = "declared, not yet computed"
        registry = pd.concat([registry, pending], ignore_index=True)

    table = descriptor_transferability(registry, contents)
    table["status"] = registry["status"].to_numpy()
    table.to_csv(out / "table_s3_descriptor_transferability.csv", index=False)

    variants = list(contents)
    print(f"{len(table)} declared descriptors "
          f"({int((table['status'] == 'computed').sum())} already computed, "
          f"{int((table['status'] != 'computed').sum())} declared and pending)")
    print()
    for variant in variants:
        column = f"computable_in_{variant}"
        print(f"  computable in {variant:<10} {int(table[column].sum()):>4}")
    print(f"  computable in both     {int(table['shared'].sum()):>4} "
          f"({table['shared'].mean():.1%})")
    lost = table[table[f"lost_in_{variants[1]}"]]
    print(f"  lost in {variants[1]:<14} {len(lost):>4}")
    if len(lost):
        print()
        print(lost.groupby(["family", "requirement_kind"]).size().to_string())
        print()
        print("  the objects that are missing:")
        print("   ", ", ".join(sorted(set(lost["requirement"]))[:40]))
    print()
    print(table.groupby(["requirement_kind", "shared"]).size().unstack(fill_value=0).to_string())

    # Computable is not the same as comparable.
    comparable = table[table["shared"] & table["support_retained"].notna()]
    if len(comparable):
        print()
        print(f"  of the {len(comparable)} descriptors whose requirement can be counted, "
              f"the share of that requirement the subset keeps:")
        bands = pd.cut(comparable["support_retained"],
                       [-0.001, 0.499, 0.799, 0.949, 0.999, 1.001],
                       labels=["< 50%", "50-80%", "80-95%", "95-100%", "unchanged"])
        print(bands.value_counts().sort_index().to_string())
        weakened = comparable.nsmallest(12, "support_retained")
        print()
        print("  the descriptors whose support shrinks most:")
        for _, row in weakened.iterrows():
            print(f"    {row['feature']:<40} {row['requirement']:<26} "
                  f"{row[f'support_in_{variants[0]}']:>6.0f} -> "
                  f"{row[f'support_in_{variants[1]}']:>5.0f}  "
                  f"({row['support_retained']:.0%})")

## 52. The factorial design, at the condition level (`run_27_design_analysis`)

Thirty chain medians, not thirty thousand states: the imposed capacity as
an ordered gradient, the peroxide axis as the negative control, and the
interaction between them.

In [ ]:
"""Stage — the factorial design, analysed at the level it was designed at.

The experiment has two axes and three chains per condition. It does not have
30,000 independent replicates: the states within a chain are draws from one
Markov chain on one polytope, and treating them as replicates would turn any
difference, however small, into an arbitrarily significant one. So the design
is analysed twice, and the two answers are reported side by side.

* **At the level of the state**, which is what a classifier sees, and which is
  the right level for asking how much of a single state's identity the
  representation preserves.
* **At the level of the condition**, with the median of each chain as one
  observation — thirty observations, ten conditions, three chains each. This
  is the level at which the factorial question is asked: does the imposed
  enzyme capacity produce an ordered response, does the peroxide axis produce
  any response at all, and do the two interact?

The primary quantity is the *gradient*: the imposed capacity is an ordered
variable with four levels, and a descriptor that tracks it monotonically is
saying something a ten-class accuracy cannot. The peroxide axis is the
negative control, and the interaction term is what says whether the two axes
can be read separately.
"""
import itertools

import numpy as np
import pandas as pd
from scipy import stats


__all__ = ["chain_summaries", "factorial_effects", "run_27_design_analysis"]

#: The imposed capacity, as an ordered variable. The healthy reference is the
#: unrestricted level and belongs on the same axis.
G6PD_ORDER = (0.10, 0.28, 0.50, 1.00)


def chain_summaries(features: pd.DataFrame, metadata: pd.DataFrame,
                    columns: list[str]) -> pd.DataFrame:
    """One row per chain: the median of each descriptor over its states.

    The median rather than the mean, because the sampled distributions are
    skewed and a chain that wandered into a corner should not drag the summary
    with it.
    """
    frame = metadata[["condition", "chain", "g6pd_level", "h2o2_level", "state"]].copy()
    frame["unit"] = frame["condition"] + "|chain" + frame["chain"].astype(str)
    summary = (pd.concat([frame[["unit", "condition", "chain", "g6pd_level",
                                 "h2o2_level", "state"]], features[columns]], axis=1)
               .groupby(["unit", "condition", "chain", "g6pd_level", "h2o2_level", "state"],
                        as_index=False)[columns].median())
    return summary


def factorial_effects(summary: pd.DataFrame, columns: list[str]) -> pd.DataFrame:
    """Per descriptor: the gradient, the null axis, and the interaction.

    Three quantities, each answering a different question and none of them a
    ten-class accuracy:

    ``gradient_rho``      Spearman correlation with the imposed capacity over
                          the thirty chain medians. Sign and monotonicity are
                          the content; a descriptor that responds to severity
                          but not monotonically is a different finding from one
                          that does not respond.
    ``h2o2_eta_squared``  the share of variance the peroxide axis explains
                          *within* a fixed capacity, which is where a null axis
                          must show nothing.
    ``interaction_eta_squared``  the share left to the interaction after both
                          main effects, from a two-way decomposition on the
                          same thirty observations.
    """
    rows = []
    g6pd = summary["g6pd_level"].to_numpy(dtype=float)
    h2o2 = summary["h2o2_level"].to_numpy(dtype=float)
    restricted = summary["state"].to_numpy() == "disease"

    for name in columns:
        y = summary[name].to_numpy(dtype=float)
        if not np.isfinite(y).all() or np.ptp(y) == 0:
            rows.append({"feature": name, "gradient_rho": np.nan, "gradient_p": np.nan,
                         "monotone": False, "h2o2_eta_squared": np.nan,
                         "h2o2_p": np.nan, "interaction_eta_squared": np.nan,
                         "between_chain_sd": np.nan, "effect_to_noise": np.nan})
            continue

        gradient = stats.spearmanr(g6pd, y)
        # Monotone in the ordered capacity: the condition medians, taken in
        # order of imposed capacity, never reverse direction.
        by_level = [np.median(y[g6pd == level]) for level in G6PD_ORDER
                    if (g6pd == level).any()]
        steps = np.diff(by_level)
        monotone = bool(np.all(steps >= 0) or np.all(steps <= 0))

        # The peroxide axis is tested only among the restricted conditions,
        # where it varies with the capacity held fixed; the healthy reference
        # has no peroxide level to vary.
        grand = y[restricted].mean()
        total = float(((y[restricted] - grand) ** 2).sum())
        spread = float(np.ptp(y[restricted]))
        if spread > 1e-9 * max(1.0, abs(grand)) and total > 0:
            between = 0.0
            for level in np.unique(h2o2[restricted]):
                block = y[restricted & (h2o2 == level)]
                between += len(block) * (block.mean() - grand) ** 2
            h2o2_eta = float(between / total)
            groups = [y[restricted & (h2o2 == level)] for level in np.unique(h2o2[restricted])]
            h2o2_p = float(stats.kruskal(*groups).pvalue) if all(
                len(g) > 1 and np.ptp(g) > 0 for g in groups) else np.nan

            # Two-way decomposition on the restricted cells: what is left to
            # the interaction once both main effects are removed.
            cell_means, main = {}, 0.0
            for a, b in itertools.product(np.unique(g6pd[restricted]),
                                          np.unique(h2o2[restricted])):
                block = y[restricted & (g6pd == a) & (h2o2 == b)]
                if len(block):
                    cell_means[(a, b)] = block.mean()
            row_mean = {a: np.mean([m for (x, _), m in cell_means.items() if x == a])
                        for a in np.unique(g6pd[restricted])}
            col_mean = {b: np.mean([m for (_, x), m in cell_means.items() if x == b])
                        for b in np.unique(h2o2[restricted])}
            for (a, b), mean in cell_means.items():
                expected = row_mean[a] + col_mean[b] - grand
                main += 3 * (mean - expected) ** 2      # three chains per cell
            interaction_eta = float(main / total)
        else:
            h2o2_eta = h2o2_p = interaction_eta = np.nan

        # How large the gradient is against how much the chains of one
        # condition disagree: an effect smaller than the sampler's own spread
        # is not an effect.
        within = summary.groupby("condition")[name].std(ddof=1).mean()
        spread = float(np.ptp([np.median(y[g6pd == level]) for level in G6PD_ORDER
                               if (g6pd == level).any()]))
        rows.append({
            "feature": name,
            "gradient_rho": float(gradient.statistic),
            "gradient_p": float(gradient.pvalue),
            "monotone": monotone,
            "h2o2_eta_squared": h2o2_eta,
            "h2o2_p": h2o2_p,
            "interaction_eta_squared": interaction_eta,
            "between_chain_sd": float(within),
            "effect_to_noise": float(spread / within) if within > 0 else np.nan,
        })

    table = pd.DataFrame(rows)
    finite = table["gradient_p"].notna()
    table.loc[finite, "gradient_p_adjusted"] = benjamini_hochberg(
        table.loc[finite, "gradient_p"].to_numpy())
    finite = table["h2o2_p"].notna()
    table.loc[finite, "h2o2_p_adjusted"] = benjamini_hochberg(
        table.loc[finite, "h2o2_p"].to_numpy())
    return table


def run_27_design_analysis() -> None:
    """Write the condition-level factorial analysis of the core phenotype."""
    OUT = config.RESULTS_DIR / "design"
    Path(OUT).mkdir(parents=True, exist_ok=True)

    features = pd.read_csv(config.PROCESSED_DIR / "phenotypes_transformed.csv.gz")
    metadata = pd.read_csv(config.PROCESSED_DIR / "sample_metadata.csv.gz")
    core = pd.read_csv(config.RESULTS_DIR / "features" / "core_features.csv")
    columns = [f for f in core["feature"] if f in features.columns]
    print(f"{len(columns)} core descriptors, "
          f"{metadata['condition'].nunique()} conditions, "
          f"{metadata.groupby('condition')['chain'].nunique().max()} chains each")

    summary = chain_summaries(features, metadata, columns)
    summary.to_csv(OUT / "chain_medians.csv", index=False)
    print(f"{len(summary)} chain-level observations — the unit of the factorial analysis")

    effects = factorial_effects(summary, columns)
    effects = effects.merge(core[["feature", "family", "block"]], on="feature", how="left")
    effects = effects.sort_values("gradient_rho", key=lambda s: -s.abs())
    effects.to_csv(OUT / "factorial_effects.csv", index=False)

    strong = effects[(effects["gradient_p_adjusted"] < 0.05)
                     & (effects["gradient_rho"].abs() >= 0.7)]
    monotone = strong[strong["monotone"]]
    print()
    print(f"descriptors tracking the imposed capacity at |rho| >= 0.7 after FDR: "
          f"{len(strong)} of {len(columns)}, of which {len(monotone)} are monotone "
          f"in the {summary['g6pd_level'].nunique()} ordered levels")
    print()
    print(effects.head(12)[["feature", "family", "gradient_rho", "monotone",
                            "h2o2_eta_squared", "interaction_eta_squared",
                            "effect_to_noise"]].to_string(index=False))

    print()
    responding_to_null = effects[(effects["h2o2_p_adjusted"] < 0.05)]
    print(f"descriptors responding to the peroxide axis after FDR, at fixed capacity: "
          f"{len(responding_to_null)} of {len(columns)}")
    print(f"largest share of variance the peroxide axis explains: "
          f"{effects['h2o2_eta_squared'].max():.3f}; "
          f"median {effects['h2o2_eta_squared'].median():.3f}")
    print(f"largest share left to the interaction: "
          f"{effects['interaction_eta_squared'].max():.3f}; "
          f"median {effects['interaction_eta_squared'].median():.3f}")

    verdict = pd.DataFrame([
        {"question": "does the imposed capacity produce an ordered response?",
         "answer_from": "Spearman of chain medians against the ordered capacity levels",
         "n_descriptors": int(len(strong)),
         "n_monotone": int(len(monotone)),
         "largest_effect": float(effects["gradient_rho"].abs().max())},
        {"question": "does the peroxide axis produce any response at fixed capacity?",
         "answer_from": "variance share of the peroxide axis within the restricted cells",
         "n_descriptors": int(len(responding_to_null)),
         "n_monotone": np.nan,
         "largest_effect": float(effects["h2o2_eta_squared"].max())},
        {"question": "do the two axes interact?",
         "answer_from": "variance left to the interaction after both main effects",
         "n_descriptors": int((effects["interaction_eta_squared"] > 0.1).sum()),
         "n_monotone": np.nan,
         "largest_effect": float(effects["interaction_eta_squared"].max())},
    ])
    verdict.to_csv(OUT / "design_verdict.csv", index=False)
    print()
    print(verdict.to_string(index=False))

## 53. How few descriptors are enough, and is the choice stable? (`run_28_reduced_panel`)

Selection fitted inside each training fold, whole chains held out, and the
overlap between the sets chosen in different folds reported beside the
accuracy they achieve.

In [ ]:
"""Stage — how few descriptors are enough, and is the choice of them stable?

Three questions are run together because they are easy to confuse:

* **predictive value** — does a representation recover the imposed capacity?
* **parsimony** — how small can it be before that recovery degrades?
* **stability** — does the same small set get chosen when the data change?

A descriptor can be the most predictive in one fold and absent from the next;
that is a property worth reporting, not an inconvenience to average away.

Everything that looks at the data — standardisation, selection, the model —
is fitted inside the training folds only, and whole chains are held out, so a
descriptor cannot be selected using the states it is later scored on. The
target is the imposed capacity as an ordered variable with four levels rather
than the ten-cell condition label, because the second axis of the design was
shown not to produce a material effect and asking a classifier to separate
cells that differ only along it measures nothing.
"""
import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import balanced_accuracy_score
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from scipy.stats import spearmanr


__all__ = ["PANEL_SIZES", "MAX_FEATURES_FOR_SELECTION",
           "evaluate_representation", "run_28_reduced_panel"]

#: Panel sizes to test. The question is where the curve flattens, not which
#: single size wins.
PANEL_SIZES = (3, 5, 10, 20, 40, 80)

#: Selection is run on the engineered representations only. Cutting 2,166
#: fluxes down to a panel is a different question from cutting a declared
#: registry down, and it is not the one this paper asks.
MAX_FEATURES_FOR_SELECTION = 500


#: Which descriptors each fold chose, for every panel size (written by stage 28).
MEMBERS: list[dict] = []


def _panel_folds(groups: np.ndarray, n_splits: int = 3):
    return GroupKFold(n_splits=n_splits).split(np.zeros(len(groups)), groups=groups)


def _fold_local(X, y, train, test, names=None):
    """Training and test arrays, transformed with what the training fold alone says.

    With ``names`` the columns are descriptors: the variance-stabilising
    transform is chosen on the healthy states of the training fold (the highest
    capacity class), applied to both, and missing values are filled with the
    training-fold median. Without ``names`` (raw fluxes) nothing is done here.
    """
    if names is None:
        return X[train], X[test]
    train_frame = pd.DataFrame(X[train], columns=names)
    test_frame = pd.DataFrame(X[test], columns=names)
    healthy = y[train] == y.max()
    transform = PhenotypeTransform.fit(train_frame.loc[healthy])
    a = transform.apply(train_frame).to_numpy(dtype=float)
    b = transform.apply(test_frame).to_numpy(dtype=float)
    a = np.where(np.isfinite(a), a, np.nan)
    b = np.where(np.isfinite(b), b, np.nan)
    fill = np.nanmedian(a, axis=0)
    fill = np.where(np.isfinite(fill), fill, 0.0)
    return np.where(np.isnan(a), fill, a), np.where(np.isnan(b), fill, b)


def _panel_score(estimator, X, y, groups, *, seed=0, names=None):
    """Chain-held-out score for one representation, nothing selected."""
    accuracies, correlations = [], []
    for train, test in _panel_folds(groups):
        X_train, X_test = _fold_local(X, y, train, test, names)
        model = make_pipeline(StandardScaler(), clone(estimator)).fit(X_train, y[train])
        prediction = model.predict(X_test)
        accuracies.append(balanced_accuracy_score(y[test], prediction))
        if len(np.unique(y[test])) > 1:
            correlations.append(spearmanr(prediction, y[test]).statistic)
    return (float(np.mean(accuracies)), float(np.std(accuracies)),
            float(np.nanmean(correlations)) if correlations else np.nan)


def _fold_rankings(X, y, groups, *, seed=0, names=None):
    """Rank the descriptors inside each training fold, once per fold.

    An L1-penalised linear model is fitted on the training chains only and the
    descriptors are ranked by the size of their largest coefficient across the
    classes. Ranking once per fold rather than once per fold and panel size is
    the same experiment and a sixth of the work: the ranking does not depend on
    how many of its top entries are later kept.
    """
    rankings = []
    for train, test in _panel_folds(groups):
        X_train, X_test = _fold_local(X, y, train, test, names)
        scaler = StandardScaler().fit(X_train)
        picker = LinearSVC(penalty="l1", dual=False, C=0.05, max_iter=50000,
                           random_state=seed).fit(scaler.transform(X_train), y[train])
        strength = np.abs(np.atleast_2d(picker.coef_)).max(axis=0)
        rankings.append((train, test, np.argsort(-strength), X_train, X_test))
    return rankings


def evaluate_representation(name, X, y, groups, feature_names, *, seed=0,
                            transform=False) -> list[dict]:
    """The full representation, then the same representation cut down."""
    rows = []
    forest = RandomForestClassifier(n_estimators=300, random_state=seed, n_jobs=-1)
    logistic = LogisticRegression(max_iter=2000, random_state=seed)

    for model_name, model in (("forest", forest), ("logistic", logistic)):
        accuracy, sd, rho = _panel_score(model, X, y, groups, seed=seed,
                                         names=list(feature_names) if transform else None)
        rows.append({"representation": name, "selection": "none",
                     "n_features": X.shape[1], "model": model_name,
                     "balanced_accuracy_mean": accuracy, "balanced_accuracy_sd": sd,
                     "rank_correlation": rho, "selection_jaccard": np.nan,
                     "always_selected": np.nan})
        print(f"  {name:<28} {model_name:<9} all {X.shape[1]:>4}  "
              f"acc {accuracy:.3f} +- {sd:.3f}  rho {rho:+.3f}", flush=True)

    if X.shape[1] <= min(PANEL_SIZES) or X.shape[1] > MAX_FEATURES_FOR_SELECTION:
        if X.shape[1] > MAX_FEATURES_FOR_SELECTION:
            print(f"  {name:<28} selection not run: cutting {X.shape[1]} fluxes down to "
                  f"a panel is a different question from cutting a declared registry "
                  f"down, and it is not the one this paper asks", flush=True)
        return rows

    rankings = _fold_rankings(X, y, groups, seed=seed,
                              names=list(feature_names) if transform else None)
    for size in PANEL_SIZES:
        if size >= X.shape[1]:
            continue
        accuracies, correlations, chosen = [], [], []
        for train, test, order, X_train, X_test in rankings:
            keep = order[:size]
            chosen.append(frozenset(keep.tolist()))
            model = make_pipeline(StandardScaler(),
                                  clone(logistic)).fit(X_train[:, keep], y[train])
            prediction = model.predict(X_test[:, keep])
            accuracies.append(balanced_accuracy_score(y[test], prediction))
            if len(np.unique(y[test])) > 1:
                correlations.append(spearmanr(prediction, y[test]).statistic)
        overlaps = [len(a & b) / len(a | b) for i, a in enumerate(chosen)
                    for b in chosen[i + 1:]]
        for fold, (_, _, order, *_) in enumerate(rankings):
            for rank, index in enumerate(order[:size]):
                MEMBERS.append({"representation": name, "n_features": size,
                                "fold": fold, "rank": rank,
                                "feature": feature_names[index]})
        counts = pd.Series([feature_names[i] for s in chosen for i in s]).value_counts()
        always = counts[counts == len(chosen)]
        rows.append({"representation": name, "selection": "L1 inside each fold",
                     "n_features": size, "model": "logistic",
                     "balanced_accuracy_mean": float(np.mean(accuracies)),
                     "balanced_accuracy_sd": float(np.std(accuracies)),
                     "rank_correlation": float(np.nanmean(correlations)) if correlations else np.nan,
                     "selection_jaccard": float(np.mean(overlaps)) if overlaps else np.nan,
                     "always_selected": int(len(always))})
        print(f"  {name:<28} logistic  top {size:>4}  acc {np.mean(accuracies):.3f} "
              f"+- {np.std(accuracies):.3f}  rho {np.nanmean(correlations):+.3f}  "
              f"overlap {np.mean(overlaps):.2f}  {len(always)} in every fold", flush=True)
        if size == 10 and len(always):
            print(f"      chosen in every fold: {', '.join(always.index[:10])}", flush=True)
    return rows


def run_28_reduced_panel() -> None:
    """Compare full and reduced representations on the imposed-capacity gradient."""
    OUT = config.RESULTS_DIR / "benchmark"
    config.ensure_dirs()

    # untransformed: the transform is chosen inside each training fold
    features = pd.read_csv(config.PROCESSED_DIR / "phenotypes.csv.gz")
    metadata = pd.read_csv(config.PROCESSED_DIR / "sample_metadata.csv.gz")
    core = pd.read_csv(config.RESULTS_DIR / "features" / "core_features.csv")

    # The imposed capacity is ordered, so the classes are its four levels and
    # the rank correlation between the predicted and the true level is reported
    # beside the accuracy: getting "severe" wrong by one level is not the same
    # mistake as getting it wrong by three.
    levels = np.sort(metadata["g6pd_level"].unique())
    y = np.searchsorted(levels, metadata["g6pd_level"].to_numpy(dtype=float))
    groups = metadata["chain"].to_numpy()
    print(f"target: the imposed capacity, {len(levels)} ordered levels "
          f"({', '.join(f'{v:g}' for v in levels)}); "
          f"folds hold out whole chains; majority baseline "
          f"{1 / len(np.unique(y)):.3f}")

    internal = [f for f in core.loc[core["block"] == "internal", "feature"]
                if f in features.columns]
    everything = [f for f in core["feature"] if f in features.columns]

    gem = load_gem(config.MODEL_FILE, config.MODEL_NAME)
    reactions = annotate_reactions(gem)
    store = SampleStore(config.RAW_DIR)
    V, _ = store.load_all()
    raw = V[:, reactions["is_internal"].to_numpy().astype(bool)]
    varying = raw.std(axis=0) > 0
    raw = raw[:, varying]
    raw_names = [f"v_{i}" for i in range(raw.shape[1])]

    MEMBERS.clear()
    rows = []
    print()
    rows += evaluate_representation("raw internal flux", raw, y, groups, raw_names)
    print()
    rows += evaluate_representation("phenotype (internal core)",
                                    features[internal].to_numpy(dtype=float), y, groups,
                                    internal, transform=True)
    print()
    rows += evaluate_representation("phenotype (whole core)",
                                    features[everything].to_numpy(dtype=float), y, groups,
                                    everything, transform=True)

    table = pd.DataFrame(rows)
    table.to_csv(OUT / "reduced_panel.csv", index=False)
    members = pd.DataFrame(MEMBERS)
    if len(members):
        n_folds = members["fold"].nunique()
        members["in_every_fold"] = (members.groupby(["representation", "n_features", "feature"])
                                    ["fold"].transform("nunique") == n_folds)
        members.to_csv(OUT / "reduced_panel_members.csv", index=False)
        three = members[(members["representation"] == "phenotype (internal core)")
                        & (members["n_features"] == 3)]
        print(f"3-descriptor panel, per fold: "
              + " | ".join(", ".join(g.sort_values("rank")["feature"])
                           for _, g in three.groupby("fold")))

    print()
    # The comparison that means something is against the same model class: a
    # panel chosen for a linear model is not refuted by a forest on everything.
    full_linear = table[(table["selection"] == "none") & (table["model"] == "logistic")]
    best = full_linear["balanced_accuracy_mean"].max()
    reduced = table[table["selection"] != "none"]
    within = reduced[reduced["balanced_accuracy_mean"] >= best - 0.02]
    if len(within):
        smallest = within.nsmallest(1, "n_features").iloc[0]
        print(f"the smallest panel within 0.02 of the best full representation: "
              f"{int(smallest['n_features'])} descriptors, balanced accuracy "
              f"{smallest['balanced_accuracy_mean']:.3f}, selection overlap between "
              f"folds {smallest['selection_jaccard']:.2f}")
    else:
        print("no reduced panel came within 0.02 of the best full representation")
    print()
    print(table.to_string(index=False))

## 54. What survives when the network is reduced? (`run_29_transferability`)

The thermodynamically consistent subset is RBC-GEM with every reaction removed that cannot carry flux under stoichiometric and thermodynamic consistency. Comparing the two is a test of structural sensitivity — which parts of the phenotype are invariant to removing those reactions — not a validation in another tissue or cohort. Four levels: whether a descriptor can be computed and what it still reads (A), whether its response to the imposed capacity survives (B), whether the original panel works without being chosen again (C), and whether a classifier trained on one model works on the other (D).

In [ ]:
"""Stage 29 — what survives when the network is structurally reduced?

The thermodynamically consistent subset is not an independent reconstruction:
it is RBC-GEM with every reaction removed that cannot carry flux under
stoichiometric and thermodynamic consistency. That makes it a controlled test
of one specific question — *which parts of a phenotype are invariant to a
reduction of the network, and which depend on the reactions that were
removed* — and not a validation in another tissue, population or cohort.

The question is asked at four levels, each harder than the last.

**A. Structural.** Can the descriptor be computed in both models, and how much
of what it reads survives? Every descriptor is resolved to its *support*: the
set of reactions whose fluxes it reads. The retained support is the fraction
of those reactions that the reduced model still has. A descriptor can be
computable in both models and still read a different set of reactions in each;
it is then the same formula, not necessarily the same quantity.

**B. Biological response.** On the conditions both models share, does the
descriptor respond to the imposed G6PDH2 capacity in the same direction and by
a similar amount, in units of its own healthy spread (IQR/1.349)? The original model
defines what the response is (chain medians tracking the capacity at
|rho| >= 0.7 after FDR); the reduced model is then judged by the sign and size
of its own standardised response — conserved within a factor of two,
attenuated, amplified or reversed — and not by whether it also clears a
significance threshold, because that depends on how well its chains mix. The healthy baseline of the two models is compared too, with
Cliff's delta, because a descriptor whose resting value moves between
reconstructions can still respond in the same way to the same perturbation.

**C. The minimal panel.** The panel selected in the original model is scored
in the reduced one *without being selected again*. Beside it: a panel
selected afresh in the reduced model, and the reduced model's full core.

**D. Prediction across models.** A classifier trained on one reconstruction is
tested on the other. Two preprocessing regimes: *strict*, where every
parameter of the preprocessing is fitted on the training model alone; and
*reference-relative*, where, after the same training-fitted transform, each
model is re-centred on one healthy chain of its own unperturbed
reconstruction — the one thing always available for a new model, because the
unconstrained model can always be sampled. For a log-scaled descriptor that is
a fold change against the model's own healthy reference. In both regimes the
reference chain is left out of every evaluation, so the two are scored on the
same states. Within-model
accuracy, with whole chains held out, is the ceiling against which transfer is
read.

The comparison is restricted to the conditions present in both models after
the constraint-integrity check (stage 00b), so neither model is asked about a
condition the other lacks.
"""
import json

import numpy as np
import pandas as pd
from scipy import stats
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import balanced_accuracy_score
from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import StandardScaler


__all__ = ["descriptor_support", "structural_transferability", "response_transferability",
           "panel_transferability", "predictive_transferability", "run_29_transferability",
           "RESPONSIVE_RHO", "MAGNITUDE_FOLD"]

#: A descriptor responds to the gradient if its chain medians correlate with
#: the imposed capacity at least this strongly (and survive FDR at 5 %).
RESPONSIVE_RHO = 0.7
#: Two responses of the same sign are "the same size" within this fold change.
MAGNITUDE_FOLD = 2.0
#: Cliff's delta thresholds (Romano et al.): negligible, small, medium.
CLIFF_BANDS = (0.147, 0.33, 0.474)
PANEL_SIZE = 3


# ======================================================================
# A. structural transferability
# ======================================================================
def _tr_touching(gem, metabolites) -> set[str]:
    """Reactions with a non-zero coefficient on any of these metabolites."""
    S = gem.S.tocsr() if hasattr(gem.S, "tocsr") else gem.S
    rxns = np.asarray(gem.rxns)
    out = set()
    for met in metabolites:
        hits = np.flatnonzero(np.asarray(gem.mets) == met)
        if hits.size:
            row = S[hits[0]]
            cols = row.indices if hasattr(row, "indices") else np.flatnonzero(row)
            out |= set(rxns[cols])
    return out


_COFACTOR_METABOLITES = {
    "cof_nadph_nadp_share": ("nadph_c", "nadp_c"),
    "cof_nadh_nad_share": ("nadh_c", "nad_c"),
    "cof_glutathione_share": ("gthrd_c", "gthox_c"),
    "cof_nadph_over_nadh": ("nadph_c", "nadh_c"),
    "cof_energy_charge_proxy": ("atp_c", "adp_c", "amp_c"),
    "cof_redox_per_energy": ("nadph_c", "gthrd_c", "atp_c", "adp_c", "amp_c"),
}


def descriptor_support(feature: str, block: str, gem, reactions: pd.DataFrame) -> set[str]:
    """The reactions whose fluxes a descriptor reads, in the reference model.

    Resolved from the declaration, in the same order the engine resolves it:
    a named reaction, a metabolite's turnover, a ratio of two declared parts,
    a pathway group, a tracked exchange, an exchange role, and otherwise the
    whole block the descriptor is a statistic of.
    """
    internal = set(reactions.loc[reactions["is_internal"], "rxn_id"])
    external = set(reactions.loc[reactions["is_external"], "rxn_id"])
    pathway = reactions.groupby("pathway")["rxn_id"].apply(set).to_dict()
    ppp = pathway.get("pentose_phosphate", set())

    if feature.startswith("rxn_"):
        label = feature[len("rxn_"):]
        return {KEY_REACTIONS.get(label, label)}
    if feature == "oxidative_ppp_flux":
        return set(OXIDATIVE_PPP_RXNS)
    if feature == "nadph_from_oxidative_ppp":
        return _tr_touching(gem, ["nadph_c"]) | set(OXIDATIVE_PPP_RXNS)
    if feature == "redox_burden":
        return _tr_touching(gem, ["h2o2_c", "nadph_c", "gthrd_c"])
    if feature == "redox_turnover_budget":
        return _tr_touching(gem, ["nadph_c", "gthrd_c"])
    if feature.startswith("turnover_"):
        return _tr_touching(gem, [feature[len("turnover_"):]])
    if feature.startswith("eff_"):
        label = feature[len("eff_"):]
        for name, numerator, denominator in EFFICIENCY_RATIOS:
            if name == label:
                return (descriptor_support(numerator, block, gem, reactions)
                        | descriptor_support(denominator, block, gem, reactions))
    if feature.startswith("path_"):
        rest = feature[len("path_"):]
        for suffix in ("_activity", "_share", "_engagement", "_directionality"):
            if rest.endswith(suffix):
                rest = rest[: -len(suffix)]
                break
        if rest == "rapoport_luebering":
            return set(RAPOPORT_LUEBERING_RXNS)
        return set(pathway.get(rest, set()))
    if feature in _COFACTOR_METABOLITES:
        return _tr_touching(gem, _COFACTOR_METABOLITES[feature])
    if feature == "cpl_ppp_per_glucose_uptake":
        return ppp | {"EX_glc__D_e"}
    if feature == "cpl_ppp_per_oxygen_uptake":
        return ppp | {"EX_o2_e"}
    if feature.startswith("cpl_"):
        return internal | external
    if feature.startswith("exrole_"):
        role = feature[len("exrole_"):].rsplit("_", 1)[0]
        return set(reactions.loc[reactions["boundary_role"] == role, "rxn_id"])
    if feature.startswith("ex_"):
        label = feature[len("ex_"):]
        for met, name in TRACKED_EXCHANGES.items():
            if label in (f"{name}_net", f"{name}_uptake"):
                return {f"EX_{met}"}
        return external
    if feature.startswith("cap_"):
        return external if feature.endswith("_boundary") else internal
    if feature.startswith("ext_") or block in ("external", "boundary"):
        return external
    if block in ("combined", "coupling"):
        return internal | external
    return internal


def flux_space_equivalence(gem_ref, gem_red) -> tuple[dict, set[str]]:
    """Is the reduced model's flux space the reference's, minus inactive reactions?

    Three checks, in the reference model's own samples: whether any removed
    reaction ever carries flux; whether any shared reaction has a different
    bound; and whether the healthy distribution of each shared active reaction
    differs between the two models by more than it differs between two chains
    of the same model (two-sample Kolmogorov–Smirnov statistic). When the
    answers are no, no and no, the two models describe the same metabolic
    states, and any difference a descriptor shows between them is a property of
    the descriptor's definition, not of metabolism.
    """
    reference_rxns, reduced_rxns = list(np.asarray(gem_ref.rxns)), list(np.asarray(gem_red.rxns))
    V_ref, meta_ref = SampleStore(SAMPLE_DIR, "original").load_all()
    V_red, meta_red = SampleStore(SAMPLE_DIR, "thermo").load_all()
    index_ref = {r: i for i, r in enumerate(reference_rxns)}
    index_red = {r: i for i, r in enumerate(reduced_rxns)}
    active = {r for r in reference_rxns if np.ptp(V_ref[:, index_ref[r]]) > 1e-9
              or np.abs(V_ref[:, index_ref[r]]).max() > 1e-9}
    removed = [r for r in reference_rxns if r not in index_red]
    removed_active = [r for r in removed if r in active]

    shared = [r for r in reduced_rxns if r in index_ref]
    lb_ref, ub_ref = np.asarray(gem_ref.lb, float), np.asarray(gem_ref.ub, float)
    lb_red, ub_red = np.asarray(gem_red.lb, float), np.asarray(gem_red.ub, float)
    changed = [r for r in shared
               if lb_ref[index_ref[r]] != lb_red[index_red[r]]
               or ub_ref[index_ref[r]] != ub_red[index_red[r]]]

    healthy_ref = V_ref[(meta_ref["state"] == "healthy").to_numpy()]
    healthy_red = V_red[(meta_red["state"] == "healthy").to_numpy()]
    chain_ref = meta_ref.loc[meta_ref["state"] == "healthy", "chain"].to_numpy()
    compared = [r for r in shared if r in active]
    between = [stats.ks_2samp(healthy_ref[:, index_ref[r]], healthy_red[:, index_red[r]]).statistic
               for r in compared]
    within = [stats.ks_2samp(healthy_ref[chain_ref == 1, index_ref[r]],
                             healthy_ref[chain_ref == 2, index_ref[r]]).statistic
              for r in compared]
    summary = {
        "reactions_removed": len(removed),
        "removed_ever_active_in_reference": len(removed_active),
        "removed_max_abs_flux": float(np.abs(V_ref[:, [index_ref[r] for r in removed]]).max())
        if removed else 0.0,
        "shared_reactions": len(shared),
        "shared_with_changed_bounds": len(changed),
        "active_reactions_reference": len(active),
        "active_reactions_compared": len(compared),
        "ks_between_models_median": float(np.median(between)),
        "ks_between_models_p95": float(np.percentile(between, 95)),
        "ks_between_chains_median": float(np.median(within)),
        "ks_between_chains_p95": float(np.percentile(within, 95)),
    }
    return summary, active


def _tr_support_category(computable: bool, retained: float) -> str:
    if not computable:
        return "not computable"
    if not np.isfinite(retained):
        return "not resolved"
    if retained >= 1.0 - 1e-12:
        return "identical support"
    if retained >= 0.5:
        return "reduced support"
    return "largely lost"


def _tr_variant_tables(variant: str) -> dict:
    root = TABLE_ROOT / variant
    return {
        "raw": pd.read_csv(root / "phenotypes.csv.gz"),
        "meta": pd.read_csv(root / "sample_metadata.csv.gz"),
        "qc": pd.read_csv(root / "features" / "feature_qc_decision.csv"),
        "reactions": pd.read_csv(root / "reaction_metadata.csv"),
    }


def structural_transferability(ref: dict, red: dict, gem_ref, gem_red,
                               active: set[str] | None = None) -> pd.DataFrame:
    """Level A: per descriptor, computability in both and retained support.

    *Declared* support is every reaction the descriptor's formula reads.
    *Active* support is the part of it that carries flux in the reference
    model's sampled states. A descriptor can lose declared support and keep
    all of its active support — the removed reactions were at zero anyway — and
    still change value, if its formula counts reactions (a mean over the
    block, a fraction of reactions at zero, an entropy normalised by the number
    of terms). That is the case this distinction exists to expose.
    """
    red_rxns = set(np.asarray(gem_red.rxns))
    qc_ref = ref["qc"].set_index("feature")
    qc_red = red["qc"].set_index("feature")
    rows = []
    for feature, row in qc_ref.iterrows():
        support = descriptor_support(feature, str(row.get("block", "")), gem_ref,
                                     ref["reactions"])
        kept = support & red_rxns
        in_ref = feature in ref["raw"].columns and ref["raw"][feature].notna().any()
        in_red = feature in red["raw"].columns and red["raw"][feature].notna().any()
        retained = len(kept) / len(support) if support else np.nan
        live = support & active if active is not None else set()
        live_kept = live & red_rxns
        rows.append({
            "feature": feature, "family": row.get("family"), "block": row.get("block"),
            "computable_reference": bool(in_ref), "computable_reduced": bool(in_red),
            "support_size_reference": len(support), "support_size_reduced": len(kept),
            "retained_support": retained,
            "active_support_size_reference": len(live),
            "active_support_retained": (len(live_kept) / len(live)) if live else np.nan,
            "support_category": _tr_support_category(in_red, retained),
            "core_reference": bool(row.get("retained", False)),
            "core_reduced": bool(qc_red["retained"].get(feature, False))
            if feature in qc_red.index else False,
        })
    return pd.DataFrame(rows)


# ======================================================================
# B. response transferability
# ======================================================================
def _tr_shared_design(ref: dict, red: dict) -> list[str]:
    return sorted(set(ref["meta"]["condition"]) & set(red["meta"]["condition"]))


def _tr_chain_medians(tables: dict, columns: list[str], conditions: list[str]) -> pd.DataFrame:
    meta = tables["meta"]
    keep = meta["condition"].isin(conditions).to_numpy()
    frame = meta.loc[keep, ["condition", "chain", "g6pd_level", "state"]].reset_index(drop=True)
    values = tables["raw"].loc[keep, columns].reset_index(drop=True)
    return (pd.concat([frame, values], axis=1)
            .groupby(["condition", "chain", "g6pd_level", "state"], as_index=False)[columns]
            .median())


def _tr_cliff_delta(a: np.ndarray, b: np.ndarray) -> float:
    """P(a > b) - P(a < b), from the Mann-Whitney U statistic."""
    a, b = a[np.isfinite(a)], b[np.isfinite(b)]
    if not len(a) or not len(b):
        return np.nan
    u = stats.mannwhitneyu(a, b, alternative="two-sided").statistic
    return float(2 * u / (len(a) * len(b)) - 1)


def _tr_cliff_band(delta: float) -> str:
    if not np.isfinite(delta):
        return "undefined"
    d = abs(delta)
    return ("negligible" if d < CLIFF_BANDS[0] else "small" if d < CLIFF_BANDS[1]
            else "medium" if d < CLIFF_BANDS[2] else "large")


def _tr_response(tables: dict, medians: pd.DataFrame, feature: str, levels: list[float]) -> dict:
    """Gradient, monotonicity and standardised effect of one descriptor in one model."""
    y = medians[feature].to_numpy(dtype=float)
    g = medians["g6pd_level"].to_numpy(dtype=float)
    healthy = tables["raw"].loc[tables["meta"]["state"].to_numpy() == "healthy", feature]
    healthy = healthy.to_numpy(dtype=float)
    # A robust scale: the sampled distributions of ratios are heavy-tailed, and
    # a standard deviation would let a handful of extreme healthy states make
    # every response look small.
    finite_healthy = healthy[np.isfinite(healthy)]
    if finite_healthy.size > 1:
        q1, q3 = np.percentile(finite_healthy, [25, 75])
        sd = float((q3 - q1) / 1.349) if q3 > q1 else float(np.std(finite_healthy, ddof=1))
    else:
        sd = np.nan
    centre = float(np.nanmedian(healthy)) if np.isfinite(healthy).any() else np.nan
    by_level = [float(np.median(y[g == level])) for level in levels]
    out = {"healthy_median": centre, "healthy_sd": sd}
    if not np.isfinite(y).all() or np.ptp(y) == 0:
        out.update(rho=np.nan, p=np.nan, monotone=False,
                   **{f"delta_{level:g}": np.nan for level in levels if level < 1})
        return out
    test = stats.spearmanr(g, y)
    steps = np.diff(by_level)
    out.update(rho=float(test.statistic), p=float(test.pvalue),
               monotone=bool(np.all(steps >= 0) or np.all(steps <= 0)))
    for level, value in zip(levels, by_level):
        if level < 1:
            out[f"delta_{level:g}"] = ((value - by_level[-1]) / sd
                                       if sd and np.isfinite(sd) and sd > 0 else np.nan)
    return out


def _tr_response_class(row) -> str:
    """What happened to a descriptor's response, judged by effect, not by significance.

    The reference model defines what the response is: a descriptor responds
    if its chain medians track the imposed capacity there. Whether the reduced
    model reproduces it is then read from the size and sign of the reduced
    model's own standardised response, not from whether it also clears a
    significance threshold — that threshold depends on how well each model's
    chains mix, which is a property of the sampling, not of the network.
    """
    if not row["computable_reduced"]:
        return "not computable"
    if row["responsive_reference"]:
        a, b = row["effect_reference"], row["effect_reduced"]
        if not (np.isfinite(a) and np.isfinite(b)) or a == 0:
            return "not comparable"
        if np.sign(a) != np.sign(b):
            return "reversed"
        ratio = abs(b) / abs(a)
        if ratio < 1 / MAGNITUDE_FOLD:
            return "attenuated"
        if ratio > MAGNITUDE_FOLD:
            return "amplified"
        return "conserved"
    if row["responsive_reduced"]:
        return "gained in reduced model"
    return "unresponsive in both"


def response_transferability(ref: dict, red: dict, structural: pd.DataFrame) -> pd.DataFrame:
    """Level B: per descriptor, does the response to the gradient survive?"""
    conditions = _tr_shared_design(ref, red)
    levels = sorted(set(ref["meta"].loc[ref["meta"]["condition"].isin(conditions), "g6pd_level"]))
    usable = structural[structural["computable_reference"] & structural["computable_reduced"]]
    columns = [f for f in usable["feature"]
               if f in ref["raw"].columns and f in red["raw"].columns]
    med_ref = _tr_chain_medians(ref, columns, conditions)
    med_red = _tr_chain_medians(red, columns, conditions)

    rows = []
    healthy_ref = ref["meta"]["state"].to_numpy() == "healthy"
    healthy_red = red["meta"]["state"].to_numpy() == "healthy"
    for feature in columns:
        a = _tr_response(ref, med_ref, feature, levels)
        b = _tr_response(red, med_red, feature, levels)
        cliff = _tr_cliff_delta(red["raw"].loc[healthy_red, feature].to_numpy(float),
                             ref["raw"].loc[healthy_ref, feature].to_numpy(float))
        strongest = min(levels)
        da, db = a.get(f"delta_{strongest:g}", np.nan), b.get(f"delta_{strongest:g}", np.nan)
        ratio = abs(db) / abs(da) if np.isfinite(da) and np.isfinite(db) and da != 0 else np.nan
        rows.append({
            "feature": feature,
            "effect_reference": da, "effect_reduced": db,
            "rho_reference": a["rho"], "p_reference": a["p"], "monotone_reference": a["monotone"],
            "rho_reduced": b["rho"], "p_reduced": b["p"], "monotone_reduced": b["monotone"],
            **{f"delta_reference_{k[6:]}": v for k, v in a.items() if k.startswith("delta_")},
            **{f"delta_reduced_{k[6:]}": v for k, v in b.items() if k.startswith("delta_")},
            "magnitude_ratio": ratio,
            "healthy_median_reference": a["healthy_median"],
            "healthy_median_reduced": b["healthy_median"],
            "healthy_cliff_delta": cliff, "healthy_shift": _tr_cliff_band(cliff),
        })
    table = pd.DataFrame(rows)
    for side in ("reference", "reduced"):
        finite = table[f"p_{side}"].notna()
        table[f"q_{side}"] = np.nan
        table.loc[finite, f"q_{side}"] = benjamini_hochberg(
            table.loc[finite, f"p_{side}"].to_numpy())
        table[f"responsive_{side}"] = ((table[f"q_{side}"] < 0.05)
                                       & (table[f"rho_{side}"].abs() >= RESPONSIVE_RHO))
    table = table.merge(structural, on="feature", how="left")
    table["response_class"] = table.apply(_tr_response_class, axis=1)
    table.attrs["levels"] = levels
    table.attrs["conditions"] = conditions
    return table


# ======================================================================
# C and D. panels and prediction
# ======================================================================
def _tr_labels(meta: pd.DataFrame, levels: list[float]) -> np.ndarray:
    return np.searchsorted(np.asarray(levels), meta["g6pd_level"].to_numpy(dtype=float))


def _tr_subset(tables: dict, conditions: list[str]) -> tuple[pd.DataFrame, pd.DataFrame]:
    keep = tables["meta"]["condition"].isin(conditions).to_numpy()
    return (tables["raw"].loc[keep].reset_index(drop=True),
            tables["meta"].loc[keep].reset_index(drop=True))


def _tr_classifier(seed: int = 0) -> LogisticRegression:
    """The classifier of stage 28, so that the two stages' numbers can be compared."""
    return LogisticRegression(max_iter=3000, random_state=seed)


def _tr_clean(X: np.ndarray, fill: np.ndarray | None = None):
    """Replace non-finite entries by the training column medians."""
    X = np.asarray(X, dtype=float)
    if fill is None:
        fill = np.nanmedian(np.where(np.isfinite(X), X, np.nan), axis=0)
        fill = np.where(np.isfinite(fill), fill, 0.0)
    bad = ~np.isfinite(X)
    if bad.any():
        X = X.copy()
        X[bad] = np.take(fill, np.nonzero(bad)[1])
    return X, fill


def _tr_transform_from(X: np.ndarray, y: np.ndarray) -> "PhenotypeTransform":
    """The notebook's own variance-stabilising transform, fitted on training data.

    Stage 04 chooses identity, log or asinh per descriptor from the healthy
    states; the same rule is applied here, to the healthy states of whatever
    is being trained on (healthy is the highest class, the unrestricted
    capacity), and the fitted transform is then applied unchanged to whatever
    is scored. Within one model this reproduces stage 28's preprocessing; across
    models it is the strict version of it.
    """
    healthy = X[y == y.max()]
    return PhenotypeTransform.fit(pd.DataFrame(healthy if len(healthy) else X))


def _tr_stabilise(X: np.ndarray, transform) -> np.ndarray:
    return transform.apply(pd.DataFrame(X)).to_numpy(dtype=float)


def _tr_prepare(X, fitted, reference=None) -> np.ndarray:
    """Clean and stabilise with the training parameters; re-centre if asked.

    ``reference`` is the raw reference chain of the model ``X`` comes from.
    When given, every column is re-centred on that chain's median *after* the
    variance-stabilising transform, so that for a log-scaled descriptor the
    result is a fold change against that model's own healthy reference.
    """
    fill, transform = fitted
    Xc, _ = _tr_clean(X, fill)
    Z = _tr_stabilise(Xc, transform)
    if reference is not None:
        Rc, _ = _tr_clean(reference, fill)
        centre = np.nanmedian(_tr_stabilise(Rc, transform), axis=0)
        Z = Z - np.where(np.isfinite(centre), centre, 0.0)
    return Z


def _tr_score(model, scaler, X, y, fitted, reference=None) -> tuple[float, float]:
    prediction = model.predict(scaler.transform(_tr_prepare(X, fitted, reference)))
    rho = (stats.spearmanr(prediction, y).statistic
           if len(np.unique(y)) > 1 and len(np.unique(prediction)) > 1 else np.nan)
    return float(balanced_accuracy_score(y, prediction)), float(rho)


def _tr_fit(X, y, seed=0, reference=None):
    """Clean, stabilise, (re-centre,) standardise and fit — parameters from training data."""
    Xc, fill = _tr_clean(X)
    fitted = (fill, _tr_transform_from(Xc, y))
    Z = _tr_prepare(X, fitted, reference)
    scaler = StandardScaler().fit(Z)
    model = _tr_classifier(seed).fit(scaler.transform(Z), y)
    return model, scaler, fitted


def _tr_within(X, y, groups, seed=0, reference=None) -> tuple[float, float, float]:
    accuracies, correlations = [], []
    for train, test in GroupKFold(n_splits=len(np.unique(groups))).split(X, y, groups):
        model, scaler, fitted = _tr_fit(X[train], y[train], seed, reference)
        a, r = _tr_score(model, scaler, X[test], y[test], fitted, reference)
        accuracies.append(a)
        correlations.append(r)
    return float(np.mean(accuracies)), float(np.std(accuracies)), float(np.nanmean(correlations))


def _tr_top_panel(X, y, groups, names, size=PANEL_SIZE) -> tuple[list[str], list[list[str]]]:
    """The panel chosen inside each training fold, and the consensus panel."""
    per_fold, scores = [], np.zeros(X.shape[1])
    Xc, _ = _tr_clean(X)
    for _, _, order, *_ in _fold_rankings(Xc, y, groups, names=list(names)):
        per_fold.append([names[i] for i in order[:size]])
        rank = np.empty(len(order))
        rank[order] = np.arange(len(order))
        scores += rank
    counts = pd.Series([n for fold in per_fold for n in fold]).value_counts()
    consensus = list(counts[counts == len(per_fold)].index)
    for i in np.argsort(scores):                        # fill by mean rank across folds
        if len(consensus) >= size:
            break
        if names[i] not in consensus:
            consensus.append(names[i])
    return consensus[:size], per_fold


def _tr_stage28_panel(variant: str, size: int = PANEL_SIZE):
    """The size-``size`` panel stage 28 chose in ``variant``: consensus and per fold."""
    path = TABLE_ROOT / variant / "benchmark" / "reduced_panel_members.csv"
    if not path.exists():
        return None, None
    members = pd.read_csv(path)
    members = members[(members["representation"] == "phenotype (internal core)")
                      & (members["n_features"] == size)]
    if members.empty:
        return None, None
    folds = [list(g.sort_values("rank")["feature"]) for _, g in members.groupby("fold")]
    consensus = list(members.loc[members["in_every_fold"], "feature"].drop_duplicates())
    order = members.groupby("feature")["rank"].mean().sort_values()
    for name in order.index:
        if len(consensus) >= size:
            break
        if name not in consensus:
            consensus.append(name)
    return consensus[:size], folds


def panel_transferability(ref: dict, red: dict, structural: pd.DataFrame,
                          conditions: list[str]) -> tuple[pd.DataFrame, dict]:
    """Level C: the original panel, scored in the reduced model as it is."""
    core_ref = structural[structural["core_reference"] & (structural["block"] == "internal")]
    core_red = structural[structural["core_reduced"] & (structural["block"] == "internal")]
    names_ref = [f for f in core_ref["feature"] if f in ref["raw"].columns]
    names_red = [f for f in core_red["feature"] if f in red["raw"].columns]

    # The panel is the one stage 28 chose in the reference model — on its full
    # design, inside each training fold — so the paper reports one panel, not
    # two. Stage 28 writes its choices; if it has not run, the same procedure
    # is repeated here.
    panel, folds = _tr_stage28_panel("original")
    if panel is None:
        levels_full = sorted(ref["meta"]["g6pd_level"].unique())
        y_full = _tr_labels(ref["meta"], levels_full)
        panel, folds = _tr_top_panel(ref["raw"][names_ref].to_numpy(float), y_full,
                                  ref["meta"]["chain"].to_numpy(), names_ref)

    raw_ref, meta_ref = _tr_subset(ref, conditions)
    raw_red, meta_red = _tr_subset(red, conditions)
    levels = sorted(meta_ref["g6pd_level"].unique())
    y_ref, y_red = _tr_labels(meta_ref, levels), _tr_labels(meta_red, levels)
    g_ref, g_red = meta_ref["chain"].to_numpy(), meta_red["chain"].to_numpy()

    red_panel, red_folds = _tr_stage28_panel("thermo")
    if red_panel is None:
        red_panel, red_folds = _tr_top_panel(raw_red[names_red].to_numpy(float), y_red, g_red,
                                          names_red)
    rows = []

    def record(label, model_name, columns, raw, y, g, note=""):
        missing = [c for c in columns if c not in raw.columns]
        if missing:
            rows.append({"representation": label, "evaluated_in": model_name,
                         "n_features": len(columns), "balanced_accuracy_mean": np.nan,
                         "balanced_accuracy_sd": np.nan, "rank_correlation": np.nan,
                         "note": f"not computable: {', '.join(missing)}"})
            return
        accuracy, sd, rho = _tr_within(raw[columns].to_numpy(float), y, g)
        rows.append({"representation": label, "evaluated_in": model_name,
                     "n_features": len(columns), "balanced_accuracy_mean": accuracy,
                     "balanced_accuracy_sd": sd, "rank_correlation": rho, "note": note})

    record("panel selected in original", "original", panel, raw_ref, y_ref, g_ref)
    record("panel selected in original", "thermo", panel, raw_red, y_red, g_red,
           "not reselected")
    record("panel selected in thermo", "thermo", red_panel, raw_red, y_red, g_red,
           "selected inside each training fold")
    record("full internal core (original)", "original", names_ref, raw_ref, y_ref, g_ref)
    record("full internal core (thermo)", "thermo", names_red, raw_red, y_red, g_red)
    table = pd.DataFrame(rows)
    table["baseline"] = 1 / len(levels)

    jaccard = [len(set(panel) & set(f)) / len(set(panel) | set(f)) for f in red_folds]
    info = {"panel_original": panel, "panel_original_folds": folds,
            "panel_thermo": red_panel, "panel_thermo_folds": red_folds,
            "panel_overlap_jaccard": float(np.mean(jaccard)),
            "levels_shared": [float(x) for x in levels]}
    return table, info


def _tr_split_reference(raw: pd.DataFrame, meta: pd.DataFrame, columns: list[str], chain: int = 1):
    """Separate one healthy chain of a model's own unperturbed reconstruction.

    That chain is the model's reference: it anchors the re-centring and is
    never scored. The unperturbed model can always be sampled, so this is the
    one piece of a new model that is available without any label.
    """
    reference = ((meta["state"] == "healthy") & (meta["chain"] == chain)).to_numpy()
    X = raw[columns].to_numpy(float)
    return X[~reference], meta.loc[~reference].reset_index(drop=True), X[reference]


def predictive_transferability(ref: dict, red: dict, structural: pd.DataFrame,
                               conditions: list[str], panel: list[str],
                               gem_ref, gem_red) -> pd.DataFrame:
    """Level D: within-model accuracy against accuracy across models."""
    raw_ref, meta_ref = _tr_subset(ref, conditions)
    raw_red, meta_red = _tr_subset(red, conditions)
    levels = sorted(meta_ref["g6pd_level"].unique())

    # The perturbation is internal, and the boundary block carries no
    # information about it (stage 07), so the representations compared here
    # are internal: otherwise the descriptors with identical support — most of
    # them exchanges, which the reduction leaves alone — would look
    # untransferable for a reason that has nothing to do with transfer.
    comparable = structural[structural["core_reference"] & structural["computable_reduced"]
                            & (structural["block"] == "internal")]
    identical = comparable[comparable["support_category"] == "identical support"]
    representations = {
        "internal descriptors with identical support": list(identical["feature"]),
        "all comparable internal core descriptors": list(comparable["feature"]),
        "panel selected in original": list(panel),
    }

    # Raw fluxes, matched by reaction identity: the internal reactions both
    # models have and that vary in both. Read from the exported chains.
    reactions_ref = ref["reactions"]
    shared = sorted(set(reactions_ref.loc[reactions_ref["is_internal"], "rxn_id"])
                    & set(np.asarray(gem_red.rxns)))
    flux = {}
    for variant, gem, meta in (("original", gem_ref, meta_ref), ("thermo", gem_red, meta_red)):
        store = SampleStore(SAMPLE_DIR, variant)
        index = {r: i for i, r in enumerate(np.asarray(gem.rxns))}
        cols = [index[r] for r in shared]
        blocks = []
        for stem in store.stems:
            if parse_condition(stem)["condition"] in conditions:
                thin = store.load_chain(stem).thin
                blocks.append(pd.DataFrame(thin[:, cols], columns=shared)
                              .assign(_stem=stem, _k=np.arange(thin.shape[0])))
        frame = pd.concat(blocks, ignore_index=True)
        order = meta[["stem", "sample_index"]].merge(
            frame.rename(columns={"_stem": "stem", "_k": "sample_index"}),
            on=["stem", "sample_index"], how="left")
        flux[variant] = order[shared]
    varying = [r for r in shared
               if flux["original"][r].std() > 0 and flux["thermo"][r].std() > 0]
    data = {"original": (raw_ref.join(flux["original"][varying]), meta_ref),
            "thermo": (raw_red.join(flux["thermo"][varying]), meta_red)}
    representations["raw internal fluxes (shared reactions)"] = varying

    rows = []
    for regime in ("strict", "reference-relative"):
        for label, columns in representations.items():
            prepared = {}
            for variant, (raw, meta) in data.items():
                X, m, R = _tr_split_reference(raw, meta, columns)
                if regime == "strict":
                    R = None
                prepared[variant] = (X, _tr_labels(m, levels), m["chain"].to_numpy(), R)
            for train, test in (("original", "original"), ("thermo", "thermo"),
                                ("original", "thermo"), ("thermo", "original")):
                X_tr, y_tr, g_tr, R_tr = prepared[train]
                X_te, y_te, _, R_te = prepared[test]
                if train == test:
                    accuracy, sd, rho = _tr_within(X_tr, y_tr, g_tr, reference=R_tr)
                else:
                    model, scaler, fitted = _tr_fit(X_tr, y_tr, reference=R_tr)
                    accuracy, rho = _tr_score(model, scaler, X_te, y_te, fitted, reference=R_te)
                    sd = np.nan
                rows.append({"preprocessing": regime, "representation": label,
                             "n_features": len(columns), "trained_on": train,
                             "tested_on": test,
                             "kind": "within model" if train == test else "across models",
                             "balanced_accuracy": accuracy, "balanced_accuracy_sd": sd,
                             "rank_correlation": rho})
            print(f"  {regime:<19} {label:<44} "
                  + "  ".join(f"{r['trained_on'][0]}->{r['tested_on'][0]} "
                              f"{r['balanced_accuracy']:.3f}"
                              for r in rows[-4:]), flush=True)
    table = pd.DataFrame(rows)
    table["baseline"] = 1 / len(levels)
    return table


# ======================================================================
# the stage
# ======================================================================
def run_29_transferability() -> None:
    """All four levels, Original against its thermodynamically consistent subset."""
    OUT = Path(TABLE_ROOT) / "transfer"
    OUT.mkdir(parents=True, exist_ok=True)
    ref, red = _tr_variant_tables("original"), _tr_variant_tables("thermo")
    gem_ref = load_gem(MODEL_VARIANTS["original"]["model_file"],
                       MODEL_VARIANTS["original"]["model_name"])
    gem_red = load_gem(MODEL_VARIANTS["thermo"]["model_file"],
                       MODEL_VARIANTS["thermo"]["model_name"])

    conditions = _tr_shared_design(ref, red)
    levels = sorted(set(ref["meta"].loc[ref["meta"]["condition"].isin(conditions),
                                        "g6pd_level"]))
    print(f"shared design: {len(conditions)} conditions, capacity levels "
          f"{', '.join(f'{x:g}' for x in levels)}")
    missing = sorted(set(ref["meta"]["condition"]) - set(conditions))
    if missing:
        print(f"   conditions only the original model has (left out here): {', '.join(missing)}")

    # ---------------------------------------------------------------- A0
    print("\nA0. are the two flux spaces the same?")
    equivalence, active = flux_space_equivalence(gem_ref, gem_red)
    (OUT / "A0_flux_space.json").write_text(json.dumps(equivalence, indent=1))
    print(f"   {equivalence['reactions_removed']} reactions removed; "
          f"{equivalence['removed_ever_active_in_reference']} of them ever carry flux in the "
          f"original model's states (largest |v| {equivalence['removed_max_abs_flux']:.3g})")
    print(f"   {equivalence['shared_with_changed_bounds']} of {equivalence['shared_reactions']} "
          f"shared reactions have a different bound")
    print(f"   healthy distributions of {equivalence['active_reactions_compared']} shared active "
          f"reactions: KS between models median {equivalence['ks_between_models_median']:.3f} "
          f"(95th {equivalence['ks_between_models_p95']:.3f}); between two chains of the "
          f"original median {equivalence['ks_between_chains_median']:.3f} "
          f"(95th {equivalence['ks_between_chains_p95']:.3f})")

    # ---------------------------------------------------------------- A
    print("\nA. structural transferability")
    structural = structural_transferability(ref, red, gem_ref, gem_red, active)
    structural.to_csv(OUT / "A_structural.csv", index=False)
    counts = structural["support_category"].value_counts()
    for category, n in counts.items():
        print(f"   {category:<20} {n:>4}")
    core = structural[structural["core_reference"]]
    computable = structural[structural["computable_reduced"]]
    print(f"   active support fully retained: "
          f"{int((computable['active_support_retained'].fillna(1) >= 1 - 1e-12).sum())} of "
          f"{len(computable)} computable descriptors")
    print(f"   of the {len(core)} core descriptors: "
          + ", ".join(f"{n} {c}" for c, n in core["support_category"].value_counts().items()))

    # ---------------------------------------------------------------- B
    print("\nB. response transferability")
    response = response_transferability(ref, red, structural)
    response.to_csv(OUT / "B_response.csv", index=False)
    for subset, frame in (("all comparable", response),
                          ("core (original)", response[response["core_reference"]])):
        print(f"   {subset}: {len(frame)} descriptors")
        for cls, n in frame["response_class"].value_counts().items():
            print(f"      {cls:<40} {n:>4}")
    responsive = response[response["responsive_reference"]]
    finite = responsive[["effect_reference", "effect_reduced"]].dropna()
    same_sign = float((np.sign(finite["effect_reference"]) == np.sign(finite["effect_reduced"])).mean())
    every = response[["effect_reference", "effect_reduced"]].replace([np.inf, -np.inf], np.nan).dropna()
    effect_rho = float(stats.spearmanr(every["effect_reference"], every["effect_reduced"]).statistic)
    responsive_rho = float(stats.spearmanr(finite["effect_reference"],
                                           finite["effect_reduced"]).statistic)
    print(f"   {len(responsive)} descriptors respond in the original model; "
          f"same direction in the reduced one: {same_sign:.3f}")
    print(f"   rank correlation of the standardised response at the strongest restriction: "
          f"{effect_rho:.3f} over all {len(every)} comparable descriptors, "
          f"{responsive_rho:.3f} over the responsive ones")
    print("   healthy baseline shift (Cliff's delta, thermo vs original): "
          + ", ".join(f"{n} {b}" for b, n in response["healthy_shift"].value_counts().items()))
    by_support = (responsive.groupby("support_category")["response_class"]
                  .apply(lambda s: (s == "conserved").mean()).rename("fraction_conserved"))
    sign_by_support = (responsive.assign(same=np.sign(responsive["effect_reference"])
                                         == np.sign(responsive["effect_reduced"]))
                       .groupby("support_category")["same"].mean())
    n_by_support = responsive.groupby("support_category").size()
    print("   of the descriptors that respond in the original model, by retained support:")
    for category in by_support.index:
        print(f"      {category:<20} n={n_by_support[category]:>3}  same direction "
              f"{sign_by_support[category]:.3f}  conserved within {MAGNITUDE_FOLD:g}x "
              f"{by_support[category]:.3f}")
    both = response[response["responsive_reference"] & response["responsive_reduced"]]
    agree = float((np.sign(both["rho_reference"]) == np.sign(both["rho_reduced"])).mean()) \
        if len(both) else np.nan
    print(f"   significant in both models: {len(both)} (a count that depends on each "
          f"model's sampling precision as well as on its structure)")

    # ---------------------------------------------------------------- C
    print("\nC. the minimal panel")
    panel_table, panel_info = panel_transferability(ref, red, structural, conditions)
    panel_table.to_csv(OUT / "C_panel.csv", index=False)
    print(f"   panel selected in original: {', '.join(panel_info['panel_original'])}")
    print(f"   panel selected in thermo:   {', '.join(panel_info['panel_thermo'])}")
    print(f"   overlap (Jaccard, per thermo fold): {panel_info['panel_overlap_jaccard']:.2f}")
    print(panel_table.to_string(index=False))

    # ---------------------------------------------------------------- D
    print("\nD. prediction across models")
    predictive = predictive_transferability(ref, red, structural, conditions,
                                            panel_info["panel_original"], gem_ref, gem_red)
    predictive.to_csv(OUT / "D_predictive.csv", index=False)

    summary = {
        "flux_space": equivalence,
        "shared_conditions": conditions, "shared_levels": [float(x) for x in levels],
        "excluded_conditions_original_only": missing,
        "structural": counts.to_dict(),
        "structural_core": core["support_category"].value_counts().to_dict(),
        "response_classes_all": response["response_class"].value_counts().to_dict(),
        "response_classes_core": response.loc[response["core_reference"], "response_class"]
        .value_counts().to_dict(),
        "n_responsive_reference": int(len(responsive)),
        "same_direction_responsive": same_sign,
        "effect_rank_correlation_all": effect_rho,
        "effect_rank_correlation_all_n": int(len(every)),
        "effect_rank_correlation_responsive": responsive_rho,
        "n_significant_in_both": int(len(both)),
        "direction_agreement_significant_in_both": agree,
        "same_direction_by_support": sign_by_support.to_dict(),
        "n_responsive_by_support": n_by_support.to_dict(),
        "healthy_shift": response["healthy_shift"].value_counts().to_dict(),
        "fraction_conserved_by_support": by_support.to_dict(),
        **{k: v for k, v in panel_info.items()},
    }
    (OUT / "transfer_summary.json").write_text(json.dumps(summary, indent=1, default=str))
    print(f"\nwritten to {OUT.relative_to(ROOT)}")

## 55. The representation against the raw flux vector, on equal terms (`run_29b_representation_evidence`)

The same chains, the same healthy reference, the same two distances, always out of sample; then the registry split into its biochemical and its generic families; then which descriptors a classifier actually uses. Asked of both reconstructions, with the chain as the unit.

In [ ]:
"""Stage 29b — the representation against the raw flux vector, on equal terms.

Three questions the earlier stages answer only in part, asked of both
reconstructions with the chain as the unit:

1. **Same samples, same reference, same metric.** The raw internal flux vector
   and the internal core of Φ(v) are put through one identical procedure:
   standardised on the healthy states of the same model (median and IQR/1.349,
   the healthy reference of stage 05), then measured against the healthy
   distribution, always out of sample (the reference never contains the chain
   it measures), with the same two distances — Euclidean, and Mahalanobis with
   Ledoit–Wolf shrinkage. Only the representation differs. Each distance is
   summarised per chain (the median over its states) and judged on the chain
   medians: rank correlation with the imposed capacity, effect-to-noise, and
   the share of the variance at fixed capacity that the inert axis carries.
   The energy distance of each capacity level to the healthy states, in units
   of the energy distance between healthy chains of the same model, says how
   far a condition sits from health compared with how far the sampler's own
   chains sit from each other.
2. **Biochemistry or generic statistics?** The internal core is split into its
   biochemical families (pathway, turnover, biochemical, efficiency, cofactor,
   capacity) and its generic ones (magnitude, distribution, sparsity, entropy,
   architecture, network). Each half goes through the distance analysis and
   through the chain-held-out classification of stage 28.
3. **Which descriptors does a classifier use?** A random forest on the
   internal core, fitted on two chains of every condition and scored on the
   third, with permutation importance (drop in balanced accuracy) measured on
   the held-out chain. The top ten are compared between folds and between
   models.

The partition of families is declared here, before the result is read, and is
the only grouping used.
"""
import json
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import stats
from scipy.spatial.distance import cdist
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

__all__ = ["EVIDENCE_GROUPS", "run_29b_representation_evidence"]

EVIDENCE_GROUPS = {
    "biochemical": ("pathway", "turnover", "biochemical", "efficiency", "cofactor", "capacity"),
    "generic": ("magnitude", "distribution", "sparsity", "entropy", "architecture", "network"),
}
EVIDENCE_TOP = 10
EVIDENCE_ENERGY_N = 800
EVIDENCE_PERM_N = 3000
EVIDENCE_PERM_REPEATS = 5


def _ev_energy(a: np.ndarray, b: np.ndarray, seed: int) -> float:
    """Energy distance between two samples, on at most EVIDENCE_ENERGY_N states each."""
    rng = np.random.default_rng(seed)
    A = a[rng.choice(len(a), size=min(EVIDENCE_ENERGY_N, len(a)), replace=False)]
    B = b[rng.choice(len(b), size=min(EVIDENCE_ENERGY_N, len(b)), replace=False)]
    return float(2 * cdist(A, B).mean() - cdist(A, A).mean() - cdist(B, B).mean())


def _ev_chain_summary(distance: np.ndarray, meta: pd.DataFrame) -> pd.DataFrame:
    frame = meta[["condition", "chain", "g6pd_level", "h2o2_level"]].copy()
    frame["d"] = distance
    return frame.groupby(["condition", "chain", "g6pd_level", "h2o2_level"],
                         as_index=False)["d"].median()


def _ev_judge(chains: pd.DataFrame) -> dict:
    """Rank correlation, effect-to-noise and null-axis share, on chain medians."""
    rho = float(stats.spearmanr(chains["d"], chains["g6pd_level"]).statistic)
    within = chains.groupby("condition")["d"].std(ddof=1)
    pooled = float(np.sqrt((within ** 2).mean()))
    level = chains.groupby("g6pd_level")["d"].mean()
    etn = float((level.max() - level.min()) / pooled) if pooled > 0 else np.nan
    perturbed = chains[chains["g6pd_level"] < 1.0]
    ss_null = ss_total = 0.0
    for _, block in perturbed.groupby("g6pd_level"):
        grand = block["d"].mean()
        ss_total += float(((block["d"] - grand) ** 2).sum())
        ss_null += float(sum(len(g) * (g["d"].mean() - grand) ** 2
                             for _, g in block.groupby("h2o2_level")))
    healthy = chains[chains["g6pd_level"] == 1.0]["d"]
    severe = chains[chains["g6pd_level"] == chains["g6pd_level"].min()]["d"]
    return {"chain_rho": rho, "effect_to_noise": etn,
            "null_share": ss_null / ss_total if ss_total > 0 else np.nan,
            "levels_ordered": bool(np.all(np.diff(level.sort_index(ascending=False).to_numpy()) > 0)),
            "healthy_vs_severe_overlap": bool(severe.min() <= healthy.max())}


def _ev_distances(name: str, prepare, meta: pd.DataFrame, healthy: np.ndarray,
                  seed: int) -> tuple[list[dict], list[dict], pd.DataFrame]:
    """Distances to the healthy distribution, every one of them out of sample.

    A reference fitted on the healthy states and then used to measure those
    same states flatters them, and the flattery grows with the dimension: with
    2,166 coordinates and a few thousand healthy states an in-sample
    Mahalanobis distance is far smaller than the distance of any new state,
    healthy or not. So the reference is fitted on the healthy states of two
    chains and every condition is measured on the third, in turn: the healthy
    chain being measured was never seen by the reference that measures it.
    """
    chain = meta["chain"].to_numpy()
    disease = (meta["state"] != "healthy").to_numpy()
    rows, chain_rows = [], []
    per_metric = {"euclidean": np.full(len(meta), np.nan),
                  "mahalanobis": np.full(len(meta), np.nan)}
    aucs = {"euclidean": [], "mahalanobis": []}
    for held in np.unique(chain):
        # everything the representation learns (transform, projection) and the
        # reference itself come from the healthy states of the other two chains
        fitted_on = healthy & (chain != held)
        X = prepare(fitted_on)
        reference = HealthyReference(X.loc[fitted_on], seed=seed)
        rows_held = chain == held
        for metric in per_metric:
            d = getattr(reference, metric)(X.loc[rows_held])
            per_metric[metric][rows_held] = d
            dis = disease[rows_held]
            aucs[metric].append(float(stats.mannwhitneyu(d[dis], d[~dis]).statistic
                                      / (dis.sum() * (~dis).sum())))
    for metric, d in per_metric.items():
        chains = _ev_chain_summary(d, meta)
        rows.append({"representation": name, "metric": metric, "n_dimensions": X.shape[1],
                     "state_auc_healthy_vs_restricted": float(np.mean(aucs[metric])),
                     **_ev_judge(chains)})
        chain_rows.extend(chains.assign(representation=name, metric=metric).to_dict("records"))
    # energy distance of each capacity level to health, in units of chain-to-chain noise;
    # the standardisation is a median and an IQR, which a held-out chain does not change
    X = prepare(healthy)
    reference = HealthyReference(X.loc[healthy], seed=seed)
    Z = reference.standardise(X)
    Z = np.where(np.isfinite(Z), Z, 0.0)
    h_idx = np.flatnonzero(healthy)
    noise = float(np.mean([_ev_energy(Z[h_idx[chain[h_idx] == c]], Z[h_idx[chain[h_idx] != c]],
                                      seed + int(c)) for c in np.unique(chain[h_idx])]))
    energy = []
    for level in sorted(meta["g6pd_level"].unique(), reverse=True):
        if level == 1.0:
            continue
        idx = np.flatnonzero((meta["g6pd_level"] == level).to_numpy()
                             & (meta["h2o2_level"] == 1.0).to_numpy())
        e = _ev_energy(Z[idx], Z[h_idx], seed)
        energy.append({"representation": name, "g6pd_level": level, "energy": e,
                       "healthy_chain_noise": noise, "energy_over_noise": e / noise})
    return rows, chain_rows, pd.DataFrame(energy)


def _ev_classify(X: np.ndarray, y: np.ndarray, groups: np.ndarray, seed: int,
                 names=None, components=None) -> dict:
    """Chain-held-out accuracy; transform (descriptors) or projection (raw) inside the fold."""
    out = {}
    for label, model in (("forest", RandomForestClassifier(n_estimators=300, random_state=seed,
                                                           n_jobs=-1)),
                         ("logistic", LogisticRegression(max_iter=2000, random_state=seed))):
        if components:
            model = make_pipeline(PCA(n_components=components, random_state=seed), model)
        accuracy, sd, rho = _panel_score(model, X, y, groups, seed=seed, names=names)
        out[f"{label}_balanced_accuracy"] = accuracy
        out[f"{label}_sd"] = sd
        out[f"{label}_rho"] = rho
    return out


def _ev_importance(X: pd.DataFrame, y: np.ndarray, groups: np.ndarray, seed: int) -> pd.DataFrame:
    rows = []
    values = X.to_numpy(dtype=float)
    for fold, (train, test) in enumerate(GroupKFold(n_splits=3).split(X, y, groups)):
        X_train, X_test = _fold_local(values, y, train, test, list(X.columns))
        scaler = StandardScaler().fit(X_train)
        forest = RandomForestClassifier(n_estimators=300, random_state=seed, n_jobs=-1)
        forest.fit(scaler.transform(X_train), y[train])
        rng = np.random.default_rng(seed + fold)
        pick = rng.choice(len(test), size=min(EVIDENCE_PERM_N, len(test)), replace=False)
        result = permutation_importance(forest, scaler.transform(X_test[pick]), y[test][pick],
                                        scoring="balanced_accuracy",
                                        n_repeats=EVIDENCE_PERM_REPEATS,
                                        random_state=seed, n_jobs=-1)
        for name, mean, sd in zip(X.columns, result.importances_mean, result.importances_std):
            rows.append({"fold": fold, "feature": name, "importance": float(mean),
                         "importance_sd": float(sd)})
    return pd.DataFrame(rows)


def _ev_variant(variant: str, seed: int = 0) -> dict:
    use_variant(variant)
    OUT = Path(config.RESULTS_DIR) / "evidence"
    OUT.mkdir(parents=True, exist_ok=True)
    # untransformed: the transform is chosen on whichever healthy states are training data
    features = pd.read_csv(config.PROCESSED_DIR / "phenotypes.csv.gz")
    meta = pd.read_csv(config.PROCESSED_DIR / "sample_metadata.csv.gz")
    core = pd.read_csv(config.RESULTS_DIR / "features" / "core_features.csv")
    internal = core[(core["block"] == "internal") & core["feature"].isin(features.columns)]
    family = dict(zip(internal["feature"], internal["family"]))
    healthy = (meta["state"] == "healthy").to_numpy()

    gem = load_gem(config.MODEL_FILE, config.MODEL_NAME)
    reactions = annotate_reactions(gem)
    V, check = SampleStore(config.RAW_DIR).load_all()
    assert (check["sample_id"].to_numpy() == meta["sample_id"].to_numpy()).all()
    raw = V[:, reactions["is_internal"].to_numpy().astype(bool)]
    names = np.asarray(gem.rxns)[reactions["is_internal"].to_numpy().astype(bool)]
    varying = raw.std(axis=0) > 0
    raw = pd.DataFrame(raw[:, varying], columns=names[varying])
    del V

    sets = {"raw internal flux": raw,
            "internal core": features[internal["feature"].tolist()]}
    for group, fams in EVIDENCE_GROUPS.items():
        sets[f"{group} families"] = features[[f for f in internal["feature"] if family[f] in fams]]
    n_core = sets["internal core"].shape[1]
    pca_name = f"raw flux, {n_core} principal components"

    def descriptors(X):
        def prepare(mask):
            return PhenotypeTransform.fit(X.loc[mask]).apply(X)
        return prepare

    def projected(mask):
        # a matched baseline: the raw vector compressed to as many dimensions as
        # the internal core, by a projection learned from the reference states
        reference = HealthyReference(raw.loc[mask], seed=seed)
        Z = np.nan_to_num(reference.standardise(raw), nan=0.0, posinf=0.0, neginf=0.0)
        pca = PCA(n_components=n_core, random_state=seed).fit(Z[np.asarray(mask)])
        return pd.DataFrame(pca.transform(Z), columns=[f"pc{i + 1}" for i in range(n_core)])

    preparers = {"raw internal flux": lambda mask: raw, pca_name: projected}
    for name in sets:
        if name != "raw internal flux":
            preparers[name] = descriptors(sets[name])

    rows, chain_rows, energy = [], [], []
    for name, prepare in preparers.items():
        print(f"  {variant}: {name}", flush=True)
        r, c, e = _ev_distances(name, prepare, meta, healthy, seed)
        rows.extend(r)
        chain_rows.extend(c)
        energy.append(e)
    distances = pd.DataFrame(rows)
    distances.to_csv(OUT / "distance_same_terms.csv", index=False)
    pd.DataFrame(chain_rows).to_csv(OUT / "distance_chain_medians.csv", index=False)
    energy = pd.concat(energy, ignore_index=True)
    energy.to_csv(OUT / "energy_over_noise.csv", index=False)
    print(distances[["representation", "metric", "n_dimensions", "chain_rho",
                     "effect_to_noise", "null_share", "state_auc_healthy_vs_restricted"]]
          .round(3).to_string(index=False))

    levels = np.sort(meta["g6pd_level"].unique())
    y = np.searchsorted(levels, meta["g6pd_level"].to_numpy(dtype=float))
    groups = meta["chain"].to_numpy()
    classes = []
    for name in ("biochemical families", "generic families"):
        X = sets[name].to_numpy(dtype=float)
        scored = _ev_classify(X, y, groups, seed, names=list(sets[name].columns))
        classes.append({"representation": name, "n_features": X.shape[1], **scored})
        print(f"  {name:<22} forest {scored['forest_balanced_accuracy']:.3f}  "
              f"logistic {scored['logistic_balanced_accuracy']:.3f}", flush=True)
    for name, X, kwargs in ((pca_name, raw.to_numpy(dtype=float), {"components": n_core}),):
        scored = _ev_classify(X, y, groups, seed, **kwargs)
        classes.append({"representation": name, "n_features": n_core, **scored})
        print(f"  {name:<22} forest {scored['forest_balanced_accuracy']:.3f}  "
              f"logistic {scored['logistic_balanced_accuracy']:.3f}", flush=True)
    pd.DataFrame(classes).to_csv(OUT / "family_groups_classification.csv", index=False)

    perm = _ev_importance(sets["internal core"], y, groups, seed)
    perm["family"] = perm["feature"].map(family)
    perm.to_csv(OUT / "permutation_importance_folds.csv", index=False)
    summary = (perm.groupby(["feature", "family"], as_index=False)
               .agg(importance=("importance", "mean"), importance_sd=("importance", "std"))
               .sort_values("importance", ascending=False, ignore_index=True))
    summary["rank"] = np.arange(1, len(summary) + 1)
    tops = [set(g.nlargest(EVIDENCE_TOP, "importance")["feature"]) for _, g in perm.groupby("fold")]
    overlap = float(np.mean([len(a & b) / len(a | b) for i, a in enumerate(tops)
                             for b in tops[i + 1:]]))
    in_every = sorted(set.intersection(*tops))
    summary["top_in_every_fold"] = summary["feature"].isin(in_every)
    summary.to_csv(OUT / "permutation_importance.csv", index=False)
    print(f"  top {EVIDENCE_TOP} by permutation importance, overlap between folds {overlap:.2f}; "
          f"in every fold: {', '.join(in_every)}", flush=True)
    top_family = summary.head(EVIDENCE_TOP)["family"].map(
        lambda f: "biochemical" if f in EVIDENCE_GROUPS["biochemical"] else "generic")
    meta_out = {"top_overlap": overlap, "top_in_every_fold": in_every,
                "top_biochemical": int((top_family == "biochemical").sum()),
                "top": summary.head(EVIDENCE_TOP)["feature"].tolist()}
    (OUT / "permutation_importance_summary.json").write_text(json.dumps(meta_out, indent=1))
    return {"summary": summary, "distances": distances, "chains": pd.DataFrame(chain_rows)}


def run_29b_representation_evidence() -> None:
    """Raw against Φ(v) on equal terms, biochemical against generic families, and importance."""
    results = {}
    try:
        for variant in ("original", "thermo"):
            print(f"\n=== {variant}")
            results[variant] = _ev_variant(variant)
    finally:
        use_variant("original")

    # ---- the same questions across the two reconstructions -------------------
    o, t = results["original"], results["thermo"]
    shared = o["summary"].merge(t["summary"], on="feature", suffixes=("_o", "_t"))
    top_o = set(o["summary"].head(EVIDENCE_TOP)["feature"])
    top_t = set(t["summary"].head(EVIDENCE_TOP)["feature"])
    cross = {
        "importance_rho_shared": float(stats.spearmanr(shared["importance_o"],
                                                       shared["importance_t"]).statistic),
        "importance_shared_n": int(len(shared)),
        "top_jaccard": len(top_o & top_t) / len(top_o | top_t),
        "top_common": sorted(top_o & top_t),
    }
    for name in ("raw internal flux", "internal core"):
        key = "raw" if name.startswith("raw") else "phi"
        a = (o["chains"].query("representation == @name and metric == 'mahalanobis'")
             .groupby("condition")["d"].median())
        b = (t["chains"].query("representation == @name and metric == 'mahalanobis'")
             .groupby("condition")["d"].median())
        common = a.index.intersection(b.index)
        cross[f"condition_order_rho_{key}"] = float(stats.spearmanr(a[common], b[common]).statistic)
    OUT = Path(TABLE_ROOT) / "transfer"
    OUT.mkdir(parents=True, exist_ok=True)
    (OUT / "E_cross_model_evidence.json").write_text(json.dumps(cross, indent=1))
    print("\nacross models:", json.dumps(cross, indent=1))

## 56. ## 10hc. Checks of the premises (`run_29c_review_checks`)

Each check tests one premise the paper rests on, by a route independent of the one that established it: the removed reactions by optimisation rather than by sampling, the inert axis by the flux it constrains, the chain medians by shorter chains, the panel by a second selector, the re-centring by each healthy chain in turn, the internal–boundary coupling against a plain measure of scale, and the intervention screen chain by chain.

In [ ]:
"""Stage 29c — checks asked for in review, each a direct test of a stated premise.

1. **Blocked, by optimisation.** That the reactions Thermo removes carry no flux
   was shown on the sampled states; here each is maximised and minimised by
   linear programming in Original, healthy and at the strongest restriction.
2. **The peroxide axis.** The exchange flux of every condition beside the bound
   imposed on it, with the sign convention of the model (secretion positive).
3. **Chain-median stability.** The chain medians of the descriptors the paper
   leans on, recomputed from the first quarter, half and three quarters of
   each chain, against the full chain.
4. **A second selector.** The three-descriptor panel chosen again with an
   L1-penalised logistic regression instead of the L1 support-vector machine.
5. **Which healthy chain anchors the re-centring.** Cross-model prediction of
   stage 29 with each of the three healthy chains as the reference.
6. **Is the coupling scale?** The first canonical variates of the internal and
   boundary blocks against the total absolute flux of each block, and the
   canonical correlation after that scale is regressed out of both blocks.
7. **The intervention screen by chain.** Recovery of the leading candidates
   computed chain by chain, so that its spread can be read.

Tables go to ``results/tables/review``.
"""
import json
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import stats
from scipy.optimize import linprog
from sklearn.cross_decomposition import CCA
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

__all__ = ["run_29c_review_checks"]

RV_PANEL_FEATURES = ("eff_ppp_flux_split", "rxn_hexokinase", "turnover_6pgc_c",
                     "path_pentose_phosphate_activity")
RV_BLOCKED_EPS = 1e-6
RV_CCA_STATES = 1500
RV_TOP_INTERVENTIONS = 10


def _rv_fva(gem, indices) -> pd.DataFrame:
    """Minimum and maximum of each reaction in ``indices`` over the model's polytope."""
    n = gem.S.shape[1]
    bounds = list(zip(gem.lb, gem.ub))
    b = np.zeros(gem.S.shape[0])
    rows = []
    for k, j in enumerate(indices):
        out = {"rxn_id": str(gem.rxns[j])}
        for sense, sign in (("min", 1.0), ("max", -1.0)):
            c = np.zeros(n)
            c[j] = sign
            res = linprog(c, A_eq=gem.S, b_eq=b, bounds=bounds, method="highs")
            out[sense] = float(sign * res.fun) if res.status == 0 else np.nan
            out[f"{sense}_status"] = int(res.status)
        rows.append(out)
        if (k + 1) % 100 == 0:
            print(f"    {k + 1} of {len(indices)}", flush=True)
    return pd.DataFrame(rows)


def _rv_blocked(OUT: Path) -> dict:
    gem_o = load_gem(MODEL_VARIANTS["original"]["model_file"], MODEL_VARIANTS["original"]["model_name"])
    gem_t = load_gem(MODEL_VARIANTS["thermo"]["model_file"], MODEL_VARIANTS["thermo"]["model_name"])
    removed = sorted(set(map(str, gem_o.rxns)) - set(map(str, gem_t.rxns)))
    index = {r: i for i, r in enumerate(map(str, gem_o.rxns))}
    g6 = index["G6PDH2"]
    conditions = {"healthy": gem_o,
                  "G6PDH2 at 0.10": gem_o.with_bounds({"G6PDH2": (None, 0.10 * gem_o.ub[g6])})}
    tables, summary = [], {"removed": len(removed), "epsilon": RV_BLOCKED_EPS}
    for label, gem in conditions.items():
        print(f"  flux variability of {len(removed)} removed reactions, {label}", flush=True)
        t = _rv_fva(gem, [index[r] for r in removed]).assign(condition=label)
        t["max_abs"] = t[["min", "max"]].abs().max(axis=1)
        tables.append(t)
        summary[label] = {"solved": int(((t["min_status"] == 0) & (t["max_status"] == 0)).sum()),
                          "max_abs": float(t["max_abs"].max()),
                          "above_epsilon": int((t["max_abs"] > RV_BLOCKED_EPS).sum())}
        print(f"    largest |v| {summary[label]['max_abs']:.3g}; above {RV_BLOCKED_EPS:g}: "
              f"{summary[label]['above_epsilon']}", flush=True)
    pd.concat(tables, ignore_index=True).to_csv(OUT / "blocked_by_fva.csv", index=False)
    return summary


def _rv_peroxide(OUT: Path) -> pd.DataFrame:
    rows = []
    for variant in ("original", "thermo"):
        use_variant(variant)
        gem = load_gem(config.MODEL_FILE, config.MODEL_NAME)
        j = int(np.flatnonzero(np.asarray(gem.rxns) == "EX_h2o2_e")[0])
        V, meta = SampleStore(config.RAW_DIR).load_all()
        flux = V[:, j]
        for condition, g in meta.groupby("condition", sort=False):
            x = flux[g.index.to_numpy()]
            level = float(g["h2o2_level"].iloc[0])
            rows.append({"model": variant, "condition": condition,
                         "g6pd_level": float(g["g6pd_level"].iloc[0]), "h2o2_level": level,
                         "imposed_ub": level * float(gem.ub[j]), "lb": float(gem.lb[j]),
                         "flux_min": float(x.min()), "flux_median": float(np.median(x)),
                         "flux_max": float(x.max()), "fraction_uptake": float((x < 0).mean())})
        del V
    table = pd.DataFrame(rows)
    table.to_csv(OUT / "peroxide_exchange_by_condition.csv", index=False)
    return table


def _rv_prefix(OUT: Path) -> pd.DataFrame:
    rows = []
    for variant in ("original", "thermo"):
        root = Path(TABLE_ROOT) / variant
        raw = pd.read_csv(root / "phenotypes.csv.gz", usecols=list(RV_PANEL_FEATURES))
        meta = pd.read_csv(root / "sample_metadata.csv.gz")
        frame = pd.concat([meta[["condition", "chain", "stem", "sample_index"]], raw], axis=1)
        healthy = (meta["state"] == "healthy").to_numpy()
        for feature in RV_PANEL_FEATURES:
            q = raw.loc[healthy, feature].quantile([0.25, 0.75])
            scale = float(q.iloc[1] - q.iloc[0]) / 1.349 or float(raw.loc[healthy, feature].std())
            full = frame.groupby("stem")[feature].median()
            levels = frame.groupby(meta["g6pd_level"])[feature].median()
            effect = float(levels.max() - levels.min())
            noise = frame.groupby(["condition", "chain"])[feature].median() \
                .groupby(level=0).std().mean()
            for fraction in (0.25, 0.5, 0.75):
                part = frame[frame.groupby("stem")["sample_index"].rank(pct=True) <= fraction]
                shift = (part.groupby("stem")[feature].median() - full).abs()
                rows.append({"model": variant, "feature": feature, "prefix": fraction,
                             "max_shift_over_between_chain_sd": float(shift.max() / noise),
                             "median_shift_over_between_chain_sd": float(shift.median() / noise),
                             "max_shift_over_healthy_spread": float(shift.max() / scale),
                             "max_shift_over_capacity_effect": float(shift.max() / effect)})
    table = pd.DataFrame(rows)
    table.to_csv(OUT / "chain_prefix_stability.csv", index=False)
    return table


def _rv_l1_logistic(OUT: Path) -> pd.DataFrame:
    rows, scores = [], []
    for variant in ("original", "thermo"):
        root = Path(TABLE_ROOT) / variant
        meta = pd.read_csv(root / "sample_metadata.csv.gz")
        core = pd.read_csv(root / "features" / "core_features.csv")
        raw = pd.read_csv(root / "phenotypes.csv.gz")
        internal = [f for f in core.loc[core["block"] == "internal", "feature"] if f in raw]
        X = raw[internal].to_numpy(dtype=float)
        levels = np.sort(meta["g6pd_level"].unique())
        y = np.searchsorted(levels, meta["g6pd_level"].to_numpy(dtype=float))
        groups = meta["chain"].to_numpy()
        for fold, (train, test) in enumerate(_panel_folds(groups)):
            X_train, _ = _fold_local(X, y, train, test, internal)
            Z = StandardScaler().fit_transform(X_train)
            pick = np.random.default_rng(fold).choice(len(Z), size=min(12000, len(Z)), replace=False)
            model = LogisticRegression(l1_ratio=1.0, solver="saga", C=0.01, max_iter=3000,
                                       random_state=0).fit(Z[pick], y[train][pick])
            strength = np.abs(np.atleast_2d(model.coef_)).max(axis=0)
            top = [internal[i] for i in np.argsort(-strength)[:3]]
            rows.append({"model": variant, "fold": fold, "selector": "L1 logistic regression",
                         "panel": ", ".join(sorted(top))})
        # both selectors' panels scored with the same logistic model and folds
        members = pd.read_csv(root / "benchmark" / "reduced_panel_members.csv")
        members = members[(members["representation"] == "phenotype (internal core)")
                          & (members["n_features"] == 3)]
        for selector, panel in (("L1 support-vector machine", sorted(set(members["feature"]))),
                                ("L1 logistic regression",
                                 sorted(set(", ".join(r["panel"] for r in rows
                                                      if r["model"] == variant).split(", "))))):
            idx = [internal.index(f) for f in panel]
            accuracy, sd, _ = _panel_score(LogisticRegression(max_iter=2000, random_state=0),
                                           X[:, idx], y, groups, names=panel)
            scores.append({"model": variant, "selector": selector, "panel": ", ".join(panel),
                           "n_features": len(panel), "balanced_accuracy": accuracy,
                           "balanced_accuracy_sd": sd})
    table = pd.DataFrame(rows)
    table.to_csv(OUT / "panel_second_selector.csv", index=False)
    pd.DataFrame(scores).to_csv(OUT / "panel_second_selector_scores.csv", index=False)
    return table


def _rv_recentre(OUT: Path) -> pd.DataFrame:
    ref, red = _tr_variant_tables("original"), _tr_variant_tables("thermo")
    conditions = _tr_shared_design(ref, red)
    structural = pd.read_csv(Path(TABLE_ROOT) / "transfer" / "A_structural.csv")
    comparable = structural[structural["core_reference"] & structural["computable_reduced"]
                            & (structural["block"] == "internal")]["feature"].tolist()
    raw_o, meta_o = _tr_subset(ref, conditions)
    raw_t, meta_t = _tr_subset(red, conditions)
    levels = sorted(meta_o["g6pd_level"].unique())
    rows = []
    for chain in (1, 2, 3):
        prepared = {}
        for variant, raw, meta in (("original", raw_o, meta_o), ("thermo", raw_t, meta_t)):
            X, m, R = _tr_split_reference(raw, meta, comparable, chain=chain)
            prepared[variant] = (X, _tr_labels(m, levels), R)
        for train, test in (("original", "thermo"), ("thermo", "original")):
            X_tr, y_tr, R_tr = prepared[train]
            X_te, y_te, R_te = prepared[test]
            model, scaler, fitted = _tr_fit(X_tr, y_tr, reference=R_tr)
            accuracy, _ = _tr_score(model, scaler, X_te, y_te, fitted, reference=R_te)
            rows.append({"reference_chain": chain, "trained_on": train, "tested_on": test,
                         "n_features": len(comparable), "balanced_accuracy": accuracy})
            print(f"    reference chain {chain}: {train} -> {test} {accuracy:.3f}", flush=True)
    table = pd.DataFrame(rows)
    table.to_csv(OUT / "recentring_reference_chain.csv", index=False)
    return table


def _rv_scale_cca(OUT: Path) -> pd.DataFrame:
    rows = []
    for variant in ("original", "thermo"):
        use_variant(variant)
        root = Path(TABLE_ROOT) / variant
        gem = load_gem(config.MODEL_FILE, config.MODEL_NAME)
        reactions = annotate_reactions(gem)
        internal_rxn = reactions["is_internal"].to_numpy().astype(bool)
        V, meta = SampleStore(config.RAW_DIR).load_all()
        scale_int = np.log(np.abs(V[:, internal_rxn]).sum(axis=1))
        scale_ex = np.log(np.abs(V[:, ~internal_rxn]).sum(axis=1))
        del V
        raw = pd.read_csv(root / "phenotypes.csv.gz")
        core = pd.read_csv(root / "features" / "core_features.csv")
        healthy = (meta["state"] == "healthy").to_numpy()
        T = PhenotypeTransform.fit(raw.loc[healthy]).apply(raw)
        blocks = {b: [f for f in core.loc[core["block"] == b, "feature"] if f in T]
                  for b in ("internal", "external")}
        for condition, g in meta.groupby("condition", sort=False):
            idx = g.index.to_numpy()
            idx = np.random.default_rng(0).choice(idx, size=min(RV_CCA_STATES, len(idx)), replace=False)
            A = T.loc[idx, blocks["internal"]].to_numpy(float)
            B = T.loc[idx, blocks["external"]].to_numpy(float)
            ok = np.isfinite(A).all(axis=1) & np.isfinite(B).all(axis=1)
            A, B = A[ok], B[ok]
            s = np.c_[scale_int[idx][ok], scale_ex[idx][ok]]
            A = A[:, A.std(axis=0) > 0]
            B = B[:, B.std(axis=0) > 0]
            A = StandardScaler().fit_transform(A)
            B = StandardScaler().fit_transform(B)

            def first(a, b):
                u, w = CCA(n_components=1, max_iter=1000).fit(a, b).transform(a, b)
                return u[:, 0], w[:, 0], abs(float(np.corrcoef(u[:, 0], w[:, 0])[0, 1]))

            u, w, rho = first(A, B)
            design = np.c_[np.ones(len(s)), s]
            beta_a, *_ = np.linalg.lstsq(design, A, rcond=None)
            beta_b, *_ = np.linalg.lstsq(design, B, rcond=None)
            A_r, B_r = A - design @ beta_a, B - design @ beta_b
            *_, rho_r = first(A_r, B_r)
            rows.append({"model": variant, "condition": condition, "canonical_correlation": rho,
                         "canonical_correlation_scale_removed": rho_r,
                         "corr_internal_variate_internal_scale": abs(float(np.corrcoef(u, s[:, 0])[0, 1])),
                         "corr_boundary_variate_exchange_scale": abs(float(np.corrcoef(w, s[:, 1])[0, 1])),
                         "corr_internal_variate_exchange_scale": abs(float(np.corrcoef(u, s[:, 1])[0, 1]))})
        print(f"    {variant}: canonical correlation median "
              f"{np.median([r['canonical_correlation'] for r in rows if r['model'] == variant]):.2f}, "
              f"after removing scale {np.median([r['canonical_correlation_scale_removed'] for r in rows if r['model'] == variant]):.2f}",
              flush=True)
    table = pd.DataFrame(rows)
    table.to_csv(OUT / "coupling_scale_control.csv", index=False)
    return table


def _rv_interventions(OUT: Path) -> pd.DataFrame:
    rows = []
    for variant in ("original", "thermo"):
        use_variant(variant)
        root = Path(TABLE_ROOT) / variant
        gem = load_gem(config.MODEL_FILE, config.MODEL_NAME)
        reactions = annotate_reactions(gem)
        index = {r: i for i, r in enumerate(gem.rxns)}
        V, meta = SampleStore(config.RAW_DIR).load_all()
        raw = pd.read_csv(root / "phenotypes.csv.gz")
        core = pd.read_csv(root / "features" / "core_features.csv")
        internal = [f for f in core.loc[core["block"] == "internal", "feature"] if f in raw]
        healthy = (meta["state"] == "healthy").to_numpy()
        transform = PhenotypeTransform.fit(raw.loc[healthy])
        reference = HealthyReference(transform.apply(raw.loc[healthy])[internal],
                                     seed=config.RANDOM_SEED)
        target = (meta["condition"] == TARGET_CONDITION).to_numpy()
        d_target = reference.mahalanobis(transform.apply(raw.loc[target])[internal])
        d_healthy = float(np.median(reference.mahalanobis(transform.apply(raw.loc[healthy])[internal])))
        chains = meta.loc[target, "chain"].to_numpy()
        catalogue = {c.intervention_id: c for c in build_catalogue(reactions, V[healthy])}
        V_target = V[target]
        del V
        ranking = pd.read_csv(root / "interventions" / "ranking.csv")
        estimable = ranking[ranking["estimable"]]
        leading = pd.concat([estimable.nlargest(RV_TOP_INTERVENTIONS, "recovery"),
                             estimable[estimable["admissible"]].nlargest(RV_TOP_INTERVENTIONS, "utility")]
                            ).drop_duplicates("intervention_id")
        for _, row in leading.iterrows():
            lb, ub = catalogue[row["intervention_id"]].apply(gem.lb, gem.ub, index)
            keep = np.all((V_target >= lb - 1e-6) & (V_target <= ub + 1e-6), axis=1)
            per_chain = []
            for c in np.unique(chains):
                base = float(np.median(d_target[chains == c]))
                kept = keep & (chains == c)
                per_chain.append((base - float(np.median(d_target[kept]))) / max(base - d_healthy, 1e-9)
                                 if kept.sum() >= 50 else np.nan)
            rows.append({"model": variant, "intervention_id": row["intervention_id"],
                         "admissible": bool(row["admissible"]), "recovery_pooled": float(row["recovery"]),
                         "n_retained": int(keep.sum()),
                         **{f"recovery_chain{c}": v for c, v in zip(np.unique(chains), per_chain)},
                         "recovery_chain_min": float(np.nanmin(per_chain)),
                         "recovery_chain_max": float(np.nanmax(per_chain))})
    table = pd.DataFrame(rows)
    table.to_csv(OUT / "intervention_recovery_by_chain.csv", index=False)
    return table


def run_29c_review_checks() -> None:
    """Direct tests of the premises reviewers question (see the stage description)."""
    OUT = Path(TABLE_ROOT) / "review"
    OUT.mkdir(parents=True, exist_ok=True)
    summary = {}
    print("1. removed reactions, by flux variability analysis")
    summary["blocked"] = _rv_blocked(OUT)
    print("2. the peroxide exchange in every condition")
    print(_rv_peroxide(OUT).round(1).to_string(index=False))
    print("3. chain medians from chain prefixes")
    prefix = _rv_prefix(OUT)
    print(prefix.groupby(["model", "prefix"])["max_shift_over_between_chain_sd"].max().round(2).to_string())
    print("4. the panel with a second selector")
    print(_rv_l1_logistic(OUT).to_string(index=False))
    print("5. re-centring on each healthy chain")
    _rv_recentre(OUT)
    print("6. coupling and scale")
    _rv_scale_cca(OUT)
    print("7. the intervention screen by chain")
    print(_rv_interventions(OUT).round(3).to_string(index=False))
    (OUT / "review_summary.json").write_text(json.dumps(summary, indent=1))
    use_variant("original")
    print(f"\nwritten to {OUT.relative_to(ROOT)}")

## 57. Flux2feature — the tool for someone else's model (`run_30_flux2feature_check`)

The same declarations as the engine above, in one NumPy-only module that takes a model and a flux vector and returns named descriptors — or, asked first, says which descriptors the model can support and why the others cannot be computed. Stage 30 checks it against the engine on a real state and writes the values the MATLAB twin is tested against.

In [ ]:
"""The standalone tool, kept inside this notebook.

``flux2feature`` is the one-file version of the descriptor engine: NumPy only,
no COBRA, no SciPy. It is what someone else uses on their own model. Its
source lives in the string below so that the notebook stays the only source of
truth. Nothing is written to disk unless asked: ``write_flux2feature()``
writes it out as ``flux2feature.py`` when it is to be handed to someone, and
``flux2feature_tool`` is the same module, loaded here.

The MATLAB twin, ``code/flux2feature.m``, implements the same declarations.
Stage 30 checks that this tool and the notebook's engine agree descriptor by
descriptor on a real sampled state, and writes the values the MATLAB test
compares against (``code/flux2feature_reference.csv``).
"""
import sys
import types

FLUX2FEATURE_SOURCE = r'''"""flux2feature — interpretable descriptors from a metabolic flux vector.

One file, one dependency (NumPy), no toolbox. Give it a model and a flux
vector and it returns named biochemical descriptors instead of a list of
reaction fluxes. Ask it first and it tells you which descriptors your model
can support and why the others cannot be computed.

    >>> model = Model(S, rxns, mets, lb, ub, subsystems)
    >>> audit(model)                      # what is computable here, and what is not
    >>> features = flux2feature(model, v)  # {descriptor: value}
    >>> flux2feature(model, V)             # a matrix of states -> one row each

Three rules are enforced, and they are the point of the thing:

* every descriptor is **declared** before it is computed — name, units, block,
  what it needs from the model, what it means;
* a descriptor whose requirement is missing returns **NaN, never 0**, because
  zero is a measurement and absence is not;
* nothing is renamed, reordered or dropped silently: :func:`audit` is the
  contract and :func:`flux2feature` honours it.

The defaults describe human erythrocyte metabolism (RBC-GEM identifiers). On
another reconstruction, pass your own ``pathways``, ``turnover_metabolites``,
``named_reactions``, ``tracked_exchanges`` and ``cofactor_pairs``; everything
that does not depend on names keeps working untouched.
"""
from __future__ import annotations

import math
from dataclasses import dataclass, field

import numpy as np

__all__ = ["Model", "Descriptor", "audit", "flux2feature", "registry",
           "PATHWAYS", "TURNOVER_METABOLITES", "NAMED_REACTIONS",
           "TRACKED_EXCHANGES", "COFACTOR_PAIRS"]

__version__ = "1.0"

#: Below this, a flux counts as zero. Raise it if your model is in different units.
FLUX_TOLERANCE = 1e-6

#: Metabolites excluded from the flux-weighted network statistics: they take
#: part in so many reactions that they would dominate every centrality.
CURRENCY = ("h_c", "h_e", "h2o_c", "h2o_e", "pi_c", "pi_e", "ppi_c",
            "co2_c", "co2_e", "na1_c", "na1_e", "k_c", "k_e", "cl_c", "cl_e")

#: Pathway groups, as {label: (subsystem strings in the model,)}.
PATHWAYS: dict[str, tuple[str, ...]] = {
    "glycolysis": ("Glycolysis / Gluconeogenesis",),
    "pentose_phosphate": ("Pentose phosphate pathway",),
    "glutathione": ("Glutathione metabolism",),
    "redox_detox": ("Reactive species formation and detoxification",),
    "purine": ("Purine metabolism",),
    "pyrimidine": ("Pyrimidine metabolism",),
    "transport": ("Transport, extracellular",),
}

#: Pathway groups defined by reaction identity rather than by subsystem
#: annotation, as {label: (reaction identifiers,)}. A reaction claimed here is
#: removed from any subsystem-defined group, because a reaction belongs to one
#: group: the Rapoport-Luebering shunt is annotated as glycolysis in RBC-GEM
#: and is not glycolysis.
PATHWAY_REACTIONS: dict[str, tuple[str, ...]] = {
    "rapoport_luebering": ("DPGM", "DPGase"),
}

#: Metabolites whose gross turnover is reported individually.
TURNOVER_METABOLITES = ("atp_c", "adp_c", "amp_c", "nadh_c", "nad_c", "nadph_c",
                        "nadp_c", "gthrd_c", "gthox_c", "glc__D_c", "g6p_c",
                        "f6p_c", "pyr_c", "lac__L_c", "h2o2_c", "o2_c",
                        "13dpg_c", "r5p_c", "ru5p__D_c", "6pgc_c")

#: Named reactions, as {label: identifier in the model}.
NAMED_REACTIONS = {"g6pd": "G6PDH2", "pgl": "PGL", "gnd": "GND",
                   "catalase": "CAT", "gpx": "GTHP", "sod": "SPODM",
                   "hexokinase": "HEX1", "pfk": "PFK", "pyruvate_kinase": "PYK",
                   "ldh": "LDH_L", "adenylate_kinase": "ADK1"}

#: Boundary metabolites reported individually, as {metabolite: label}.
TRACKED_EXCHANGES = {"glc__D_e": "glucose", "lac__L_e": "lactate", "o2_e": "oxygen",
                     "co2_e": "carbon_dioxide", "h2o2_e": "hydrogen_peroxide",
                     "gthrd_e": "reduced_glutathione", "pyr_e": "pyruvate",
                     "cys__L_e": "cysteine", "pi_e": "phosphate"}

#: Redox and adenylate pools whose allocation the cofactor family reports.
COFACTOR_PAIRS = {"nadph_nadp": ("nadph_c", "nadp_c"),
                  "nadh_nad": ("nadh_c", "nad_c"),
                  "glutathione": ("gthrd_c", "gthox_c")}

#: Ratios, as (label, numerator descriptor, denominator descriptor).
EFFICIENCY_RATIOS = (
    ("nadph_per_glucose", "turnover_nadph_c", "turnover_glc__D_c"),
    ("nadph_per_atp", "turnover_nadph_c", "turnover_atp_c"),
    ("atp_per_glucose", "turnover_atp_c", "turnover_glc__D_c"),
    ("nadph_per_h2o2", "turnover_nadph_c", "turnover_h2o2_c"),
    ("redox_buffer_ratio", "turnover_gthrd_c", "turnover_gthox_c"),
    ("ppp_per_glycolysis", "path_pentose_phosphate_activity",
     "path_glycolysis_activity"),
    ("ppp_flux_split", "rxn_g6pd", "rxn_hexokinase"),
)


# --------------------------------------------------------------------------
# the model, in the least that is needed
# --------------------------------------------------------------------------
@dataclass
class Model:
    """The parts of a reconstruction these descriptors use.

    ``S`` may be a dense array or anything with a ``toarray``/``todense``
    method, so a SciPy sparse matrix or a COBRApy model's matrix both work.
    Build one from a COBRApy model with :meth:`from_cobra`.
    """

    S: np.ndarray
    rxns: list[str]
    mets: list[str]
    lb: np.ndarray | None = None
    ub: np.ndarray | None = None
    subsystems: list[str] | None = None
    exchange: np.ndarray | None = None          # bool per reaction

    def __post_init__(self) -> None:
        self.S = np.asarray(getattr(self.S, "toarray", lambda: self.S)(), dtype=float)
        self.rxns = [str(r) for r in self.rxns]
        self.mets = [str(m) for m in self.mets]
        if self.S.shape != (len(self.mets), len(self.rxns)):
            raise ValueError(f"S is {self.S.shape}, expected "
                             f"({len(self.mets)}, {len(self.rxns)})")
        n = len(self.rxns)
        self.lb = np.full(n, -np.inf) if self.lb is None else np.asarray(self.lb, float)
        self.ub = np.full(n, np.inf) if self.ub is None else np.asarray(self.ub, float)
        self.subsystems = ([""] * n if self.subsystems is None
                           else [str(s) for s in self.subsystems])
        if self.exchange is None:
            # A boundary reaction is one that exchanges mass with the outside.
            # Where identifiers follow the BiGG convention that is exactly the
            # EX_ prefix; otherwise fall back to "touches one metabolite",
            # which also catches sinks and demand reactions and is therefore
            # the looser rule. Pass ``exchange=`` explicitly if your model
            # distinguishes them and the distinction matters to you — it
            # changes every internal/boundary split downstream.
            prefixed = np.array([r.startswith("EX_") for r in self.rxns])
            single = (np.abs(self.S) > 0).sum(axis=0) == 1
            self.exchange = prefixed if prefixed.any() else single
        self.exchange = np.asarray(self.exchange, dtype=bool)
        self._rxn_index = {r: i for i, r in enumerate(self.rxns)}
        self._met_index = {m: i for i, m in enumerate(self.mets)}

    @classmethod
    def from_cobra(cls, model) -> "Model":
        """Build from a COBRApy model, without importing COBRApy."""
        rxns = [r.id for r in model.reactions]
        mets = [m.id for m in model.metabolites]
        index = {m: i for i, m in enumerate(mets)}
        S = np.zeros((len(mets), len(rxns)))
        for j, reaction in enumerate(model.reactions):
            for metabolite, coefficient in reaction.metabolites.items():
                S[index[metabolite.id], j] = coefficient
        return cls(S, rxns, mets,
                   [r.lower_bound for r in model.reactions],
                   [r.upper_bound for r in model.reactions],
                   [getattr(r, "subsystem", "") for r in model.reactions])

    def reaction(self, identifier: str) -> int | None:
        return self._rxn_index.get(identifier)

    def metabolite(self, identifier: str) -> int | None:
        return self._met_index.get(identifier)


@dataclass
class Descriptor:
    """One declared descriptor: what it is, what it needs, what it means."""

    name: str
    family: str
    block: str                 # internal | external | combined
    units: str
    description: str
    requires: str = "nothing named"
    computable: bool = True
    reason: str = ""
    scope: str = "common"      # common | conditional | model_specific
    compute: callable = field(default=None, repr=False)


def _safe(numerator, denominator):
    """Divide, returning NaN where the denominator vanishes."""
    denominator = np.asarray(denominator, dtype=float)
    out = np.full(np.shape(denominator), np.nan, dtype=float)
    usable = np.abs(denominator) > 0
    out[usable] = np.asarray(numerator, dtype=float)[usable] / denominator[usable]
    return out


# --------------------------------------------------------------------------
# the registry
# --------------------------------------------------------------------------
def registry(model: Model, *, pathways=None, pathway_reactions=None,
             turnover_metabolites=None,
             named_reactions=None, tracked_exchanges=None,
             cofactor_pairs=None, currency=CURRENCY,
             tolerance: float = FLUX_TOLERANCE) -> list[Descriptor]:
    """Declare every descriptor and decide, per descriptor, if this model has it.

    Nothing is computed here. The result is the contract: a descriptor marked
    ``computable=False`` carries the reason, and :func:`flux2feature` will
    return NaN for it rather than quietly leaving it out.
    """
    pathways = PATHWAYS if pathways is None else pathways
    pathway_reactions = (PATHWAY_REACTIONS if pathway_reactions is None
                         else pathway_reactions)
    turnover_metabolites = (TURNOVER_METABOLITES if turnover_metabolites is None
                            else turnover_metabolites)
    named_reactions = NAMED_REACTIONS if named_reactions is None else named_reactions
    tracked_exchanges = (TRACKED_EXCHANGES if tracked_exchanges is None
                         else tracked_exchanges)
    cofactor_pairs = COFACTOR_PAIRS if cofactor_pairs is None else cofactor_pairs

    internal = ~model.exchange
    external = model.exchange
    absS = np.abs(model.S)
    keep_met = np.array([m not in set(currency) for m in model.mets])

    out: list[Descriptor] = []

    def declare(name, family, block, units, description, compute,
                requires="nothing named", computable=True, reason="",
                scope="common"):
        out.append(Descriptor(name, family, block, units, description, requires,
                              computable, reason, scope, compute))

    # -- shape of the state, needing nothing named -------------------------
    for block, mask, tag in (("internal", internal, "int"), ("external", external, "ext")):
        if not mask.any():
            continue
        declare(f"{tag}_L1", "magnitude", block, "flux",
                f"total {block} flux magnitude",
                lambda V, mask=mask: np.abs(V[:, mask]).sum(axis=1))
        declare(f"{tag}_L2", "magnitude", block, "flux",
                f"Euclidean norm of the {block} fluxes",
                lambda V, mask=mask: np.sqrt((V[:, mask] ** 2).sum(axis=1)))
        declare(f"{tag}_Linf", "magnitude", block, "flux",
                f"largest single {block} flux",
                lambda V, mask=mask: np.abs(V[:, mask]).max(axis=1))
        declare(f"{tag}_active_fraction", "sparsity", block, "ratio",
                f"fraction of {block} reactions carrying flux",
                lambda V, mask=mask: (np.abs(V[:, mask]) > tolerance).mean(axis=1))
        declare(f"{tag}_gini", "sparsity", block, "dimensionless",
                f"Gini concentration of the {block} flux magnitudes",
                lambda V, mask=mask: _gini(np.abs(V[:, mask])))
        declare(f"{tag}_shannon_entropy", "entropy", block, "nats",
                f"entropy of the {block} flux shares",
                lambda V, mask=mask: _entropy(np.abs(V[:, mask])))
        declare(f"{tag}_directionality", "architecture", block, "dimensionless",
                f"net directional bias of the {block} fluxes",
                lambda V, mask=mask: _safe(
                    np.maximum(V[:, mask], 0).sum(axis=1)
                    - np.maximum(-V[:, mask], 0).sum(axis=1),
                    np.abs(V[:, mask]).sum(axis=1)))
        declare(f"{tag}_cv_abs", "distribution", block, "dimensionless",
                f"coefficient of variation of the {block} flux magnitudes",
                lambda V, mask=mask: _cv(np.abs(V[:, mask])))
        declare(f"{tag}_top10pct_share", "distribution", block, "ratio",
                f"share of {block} flux carried by its busiest tenth",
                lambda V, mask=mask: _top_share(np.abs(V[:, mask]), 0.10))

    # -- network, needing only the stoichiometry ---------------------------
    if internal.any():
        declare("net_active_metabolites", "network", "internal", "count",
                "metabolites touched by a carrying reaction, currency excluded",
                lambda V: ((np.abs(V[:, internal]) > tolerance)
                           @ (absS[np.ix_(keep_met, internal)].T > 0) > 0).sum(axis=1),
                requires="stoichiometric matrix")
        declare("net_top_decile_turnover_share", "network", "internal", "ratio",
                "share of gross metabolite turnover carried by the busiest tenth of "
                "metabolites, currency excluded",
                lambda V: _top_share(
                    np.abs(V[:, internal]) @ absS[np.ix_(keep_met, internal)].T, 0.10),
                requires="stoichiometric matrix")

    # -- pathways ----------------------------------------------------------
    subsystems = np.array(model.subsystems)
    by_identity = {label: np.array([model.reaction(r) for r in members
                                    if model.reaction(r) is not None], dtype=int)
                   for label, members in pathway_reactions.items()}
    claimed = np.zeros(len(model.rxns), dtype=bool)
    for columns in by_identity.values():
        claimed[columns] = True

    for label, members in list(pathways.items()) + [(k, None) for k in by_identity]:
        if members is None:
            columns = by_identity[label]
        else:
            columns = np.flatnonzero(np.isin(subsystems, list(members))
                                     & internal & ~claimed)
        present = columns.size > 0
        reason = "" if present else f"no reaction with subsystem in {members}"
        declare(f"path_{label}_activity", "pathway", "internal", "flux",
                f"total flux magnitude carried by {label} ({columns.size} reactions)",
                (lambda V, c=columns: np.abs(V[:, c]).sum(axis=1)) if present else None,
                requires=f"a reaction in {label}", computable=present, reason=reason,
                scope="conditional")
        declare(f"path_{label}_share", "pathway", "internal", "ratio",
                f"share of internal flux carried by {label}",
                (lambda V, c=columns: _safe(np.abs(V[:, c]).sum(axis=1),
                                            np.abs(V[:, internal]).sum(axis=1)))
                if present else None,
                requires=f"a reaction in {label}", computable=present, reason=reason,
                scope="conditional")
        declare(f"path_{label}_engagement", "pathway", "internal", "ratio",
                f"fraction of {label} reactions carrying flux",
                (lambda V, c=columns: (np.abs(V[:, c]) > tolerance).mean(axis=1))
                if present else None,
                requires=f"a reaction in {label}", computable=present, reason=reason,
                scope="conditional")
        two = present and columns.size >= 2
        declare(f"path_{label}_directionality", "pathway", "internal", "dimensionless",
                f"net directional bias of {label}",
                (lambda V, c=columns: _safe(
                    np.maximum(V[:, c], 0).sum(axis=1) - np.maximum(-V[:, c], 0).sum(axis=1),
                    np.abs(V[:, c]).sum(axis=1))) if two else None,
                requires=f"at least two reactions in {label}", computable=two,
                reason=reason or ("only one reaction in the group" if not two else ""),
                scope="conditional")

    # -- metabolite turnover ----------------------------------------------
    for metabolite in turnover_metabolites:
        row = model.metabolite(metabolite)
        present = row is not None
        declare(f"turnover_{metabolite}", "turnover", "internal", "flux",
                f"gross stoichiometric throughput of {metabolite}; not a measurable "
                f"turnover",
                (lambda V, r=row: np.abs(V[:, internal]) @ absS[r, internal])
                if present else None,
                requires=f"the metabolite {metabolite}", computable=present,
                reason="" if present else f"{metabolite} is not in this model",
                scope="conditional")

    # -- named reactions ---------------------------------------------------
    for label, identifier in named_reactions.items():
        column = model.reaction(identifier)
        present = column is not None
        declare(f"rxn_{label}", "biochemical", "internal", "flux",
                f"flux through {identifier}, in the model's own sign convention",
                (lambda V, c=column: V[:, c]) if present else None,
                requires=f"the reaction {identifier}", computable=present,
                reason="" if present else f"{identifier} is not in this model",
                scope="conditional")

    # -- efficiency ratios -------------------------------------------------
    declared = {d.name: d for d in out}
    for label, numerator, denominator in EFFICIENCY_RATIOS:
        parts = [declared.get(numerator), declared.get(denominator)]
        present = all(p is not None and p.computable for p in parts)
        missing = [n for n, p in zip((numerator, denominator), parts)
                   if p is None or not p.computable]
        declare(f"eff_{label}", "efficiency", "internal", "ratio",
                f"{numerator} relative to {denominator}",
                (lambda V, a=parts[0], b=parts[1]:
                 _safe(np.abs(a.compute(V)), np.abs(b.compute(V)))) if present else None,
                requires=f"{numerator} and {denominator}", computable=present,
                reason="" if present else f"needs {', '.join(missing)}",
                scope="conditional")

    # -- cofactor allocation ----------------------------------------------
    for label, (reduced, oxidised) in cofactor_pairs.items():
        rows = [model.metabolite(reduced), model.metabolite(oxidised)]
        present = all(r is not None for r in rows)
        declare(f"cof_{label}_share", "cofactor", "internal", "ratio",
                f"share of the {reduced}/{oxidised} pool turnover carried by {reduced}",
                (lambda V, a=rows[0], b=rows[1]: _safe(
                    np.abs(V[:, internal]) @ absS[a, internal],
                    np.abs(V[:, internal]) @ absS[a, internal]
                    + np.abs(V[:, internal]) @ absS[b, internal])) if present else None,
                requires=f"both {reduced} and {oxidised}", computable=present,
                reason="" if present else "one member of the pair is absent",
                scope="conditional")

    # -- capacity: the only family that depends on the constraints ---------
    finite = np.isfinite(model.lb) & np.isfinite(model.ub) & (model.ub > model.lb)
    for block, mask, tag in (("internal", internal, "int"), ("external", external, "ext")):
        usable = finite & mask
        present = bool(usable.any())
        declare(f"cap_{tag}_mean_saturation", "capacity", block, "ratio",
                f"mean position within the admissible interval, over bounded {block} "
                f"reactions",
                (lambda V, u=usable: ((V[:, u] - model.lb[u])
                                      / (model.ub[u] - model.lb[u])).mean(axis=1))
                if present else None,
                requires="finite bounds", computable=present,
                reason="" if present else f"no {block} reaction has finite bounds",
                scope="model_specific")
        declare(f"cap_{tag}_at_bound", "capacity", block, "ratio",
                f"fraction of bounded {block} reactions within tolerance of a bound",
                (lambda V, u=usable: (
                    (V[:, u] <= model.lb[u] + tolerance)
                    | (V[:, u] >= model.ub[u] - tolerance)).mean(axis=1))
                if present else None,
                requires="finite bounds", computable=present,
                reason="" if present else f"no {block} reaction has finite bounds",
                scope="model_specific")

    # -- the boundary ------------------------------------------------------
    if external.any():
        declare("ex_total_uptake", "exchange_burden", "external", "flux",
                "total magnitude taken up across the boundary",
                lambda V: np.maximum(-V[:, external], 0).sum(axis=1),
                requires="an exchange reaction")
        declare("ex_total_secretion", "exchange_burden", "external", "flux",
                "total magnitude secreted across the boundary",
                lambda V: np.maximum(V[:, external], 0).sum(axis=1),
                requires="an exchange reaction")
        declare("ex_n_uptaken", "exchange_burden", "external", "count",
                "number of metabolites taken up",
                lambda V: (V[:, external] < -tolerance).sum(axis=1),
                requires="an exchange reaction")
        declare("ex_n_secreted", "exchange_burden", "external", "count",
                "number of metabolites secreted",
                lambda V: (V[:, external] > tolerance).sum(axis=1),
                requires="an exchange reaction")

    boundary_metabolite = {}
    for column in np.flatnonzero(external):
        rows = np.flatnonzero(np.abs(model.S[:, column]) > 0)
        if rows.size == 1:
            boundary_metabolite[model.mets[rows[0]]] = int(column)
    for metabolite, label in tracked_exchanges.items():
        column = boundary_metabolite.get(metabolite)
        present = column is not None
        declare(f"ex_{label}_net", "boundary_metabolite", "external", "flux",
                f"net exchange of {label}; negative is uptake",
                (lambda V, c=column: V[:, c]) if present else None,
                requires=f"an exchange for {metabolite}", computable=present,
                reason="" if present else f"no exchange reaction for {metabolite}",
                scope="conditional")

    # -- coupling: only meaningful because the blocks are kept apart -------
    both = bool(internal.any() and external.any())
    declare("cpl_boundary_share", "coupling", "combined", "ratio",
            "share of the state's total flux magnitude that crosses the boundary",
            (lambda V: _safe(np.abs(V[:, external]).sum(axis=1),
                             np.abs(V).sum(axis=1))) if both else None,
            requires="both blocks non-empty", computable=both,
            reason="" if both else "one of the two blocks is empty", scope="conditional")
    declare("cpl_internal_per_boundary", "coupling", "combined", "ratio",
            "internal activity supported per unit of boundary exchange",
            (lambda V: _safe(np.abs(V[:, internal]).sum(axis=1),
                             np.abs(V[:, external]).sum(axis=1))) if both else None,
            requires="both blocks non-empty", computable=both,
            reason="" if both else "one of the two blocks is empty", scope="conditional")
    declare("cpl_secretion_over_uptake", "coupling", "combined", "ratio",
            "secreted magnitude relative to taken-up magnitude",
            (lambda V: _safe(np.maximum(V[:, external], 0).sum(axis=1),
                             np.maximum(-V[:, external], 0).sum(axis=1)))
            if external.any() else None,
            requires="an exchange reaction", computable=bool(external.any()),
            reason="" if external.any() else "no exchange reactions",
            scope="conditional")
    return out


# --------------------------------------------------------------------------
# the small numerical helpers, kept explicit rather than imported
# --------------------------------------------------------------------------
def _cv(A: np.ndarray) -> np.ndarray:
    """Coefficient of variation, with the sample standard deviation."""
    if A.shape[1] < 2:
        return np.full(A.shape[0], np.nan)
    return _safe(A.std(axis=1, ddof=1), A.mean(axis=1))


def _gini(A: np.ndarray) -> np.ndarray:
    ordered = np.sort(A, axis=1)
    n = ordered.shape[1]
    index = np.arange(1, n + 1)
    total = ordered.sum(axis=1)
    return _safe((2 * (ordered * index).sum(axis=1) - (n + 1) * total), n * total)


def _entropy(A: np.ndarray) -> np.ndarray:
    total = A.sum(axis=1, keepdims=True)
    p = np.divide(A, total, out=np.zeros_like(A), where=total > 0)
    return -np.where(p > 0, p * np.log(np.where(p > 0, p, 1.0)), 0.0).sum(axis=1)


def _top_share(A: np.ndarray, fraction: float) -> np.ndarray:
    k = max(1, int(math.ceil(A.shape[1] * fraction)))
    top = np.sort(A, axis=1)[:, -k:].sum(axis=1)
    return _safe(top, A.sum(axis=1))


# --------------------------------------------------------------------------
# the two functions you call
# --------------------------------------------------------------------------
def audit(model: Model, **declaration) -> list[dict]:
    """What this model can and cannot support, descriptor by descriptor.

    Call it before computing anything. Each entry says what the descriptor is,
    what it needs, whether this model has it and, when it does not, why.

        >>> rows = audit(model)
        >>> sum(r["computable"] for r in rows), len(rows)
    """
    return [{"name": d.name, "family": d.family, "block": d.block, "units": d.units,
             "description": d.description, "requires": d.requires, "scope": d.scope,
             "computable": d.computable, "reason": d.reason}
            for d in registry(model, **declaration)]


def flux2feature(model: Model, fluxes, which="computable", *, as_frame=False,
                 **declaration):
    """Descriptors of one flux vector, or of a matrix of them.

    ``fluxes``   a vector of length ``len(model.rxns)``, or a matrix with one
                 state per row.
    ``which``    ``"computable"`` returns only the descriptors this model
                 supports; ``"all"`` returns every declared descriptor, with
                 NaN where the model cannot support it — use this when you
                 intend to compare two reconstructions, so that the columns
                 line up and an absence stays visibly an absence;
                 a list of names returns exactly those.
    ``as_frame`` return a pandas DataFrame instead of a dict of arrays.

    A descriptor that cannot be computed returns NaN. It never returns 0.
    """
    V = np.atleast_2d(np.asarray(fluxes, dtype=float))
    if V.shape[1] != len(model.rxns):
        raise ValueError(f"{V.shape[1]} fluxes for {len(model.rxns)} reactions")
    single = np.ndim(fluxes) == 1

    declared = registry(model, **declaration)
    if isinstance(which, (list, tuple, set)):
        wanted = set(which)
        unknown = wanted - {d.name for d in declared}
        if unknown:
            raise KeyError(f"not declared: {sorted(unknown)}")
        declared = [d for d in declared if d.name in wanted]
    elif which == "computable":
        declared = [d for d in declared if d.computable]
    elif which != "all":
        raise ValueError('which must be "computable", "all", or a list of names')

    out = {}
    for descriptor in declared:
        if descriptor.computable and descriptor.compute is not None:
            values = np.asarray(descriptor.compute(V), dtype=float)
        else:
            values = np.full(V.shape[0], np.nan)
        out[descriptor.name] = float(values[0]) if single else values

    if as_frame:
        import pandas as pd
        return pd.DataFrame(out, index=[0] if single else None)
    return out


# --------------------------------------------------------------------------
# a self-test, so the file can be trusted without a test framework
# --------------------------------------------------------------------------
def _self_test() -> None:
    """A three-reaction model whose every descriptor can be checked by hand."""
    #            EX_a   a->b   EX_b
    S = np.array([[1.0, -1.0, 0.0],
                  [0.0, 1.0, -1.0]])
    model = Model(S, ["EX_a", "R1", "EX_b"], ["a_c", "b_c"],
                  lb=[-10, 0, 0], ub=[0, 10, 10],
                  subsystems=["", "Glycolysis / Gluconeogenesis", ""])
    v = np.array([-4.0, 4.0, 4.0])
    f = flux2feature(model, v)

    assert model.exchange.tolist() == [True, False, True], model.exchange
    assert math.isclose(f["int_L1"], 4.0)
    assert math.isclose(f["ext_L1"], 8.0)
    assert math.isclose(f["ex_total_uptake"], 4.0)
    assert math.isclose(f["ex_total_secretion"], 4.0)
    assert math.isclose(f["turnover_a_c"] if "turnover_a_c" in f else 4.0, 4.0)
    assert math.isclose(f["cpl_boundary_share"], 8.0 / 12.0)
    assert math.isclose(f["path_glycolysis_activity"], 4.0)
    assert math.isclose(f["path_glycolysis_share"], 1.0)
    # R1 sits at neither bound, so nothing is saturated; it is at 40% of its span
    assert math.isclose(f["cap_int_mean_saturation"], 0.4)
    assert math.isclose(f["cap_int_at_bound"], 0.0)

    # the absence rule
    rows = {r["name"]: r for r in audit(model)}
    assert rows["rxn_g6pd"]["computable"] is False
    assert "G6PDH2" in rows["rxn_g6pd"]["reason"]
    every = flux2feature(model, v, which="all")
    assert math.isnan(every["rxn_g6pd"]), "an absent reaction must give NaN, not 0"
    assert math.isnan(every["eff_ppp_flux_split"]), "a ratio of absent parts must be NaN"

    # a matrix of states returns one row each, and matches the single-state call
    V = np.vstack([v, v * 2])
    many = flux2feature(model, V)
    assert many["int_L1"].shape == (2,)
    assert math.isclose(many["int_L1"][0], f["int_L1"])
    assert math.isclose(many["int_L1"][1], 8.0)

    computable = sum(r["computable"] for r in rows.values())
    print(f"self-test passed: {len(rows)} descriptors declared, "
          f"{computable} computable in this toy model")


if __name__ == "__main__":
    _self_test()
'''

flux2feature_tool = types.ModuleType("flux2feature")
sys.modules["flux2feature"] = flux2feature_tool          # dataclasses look the module up
exec(compile(FLUX2FEATURE_SOURCE, "flux2feature.py", "exec"), flux2feature_tool.__dict__)


def write_flux2feature(path: str | Path = None) -> Path:
    """Write the standalone tool next to this notebook (or to ``path``)."""
    path = Path(path) if path else Path(ROOT) / "code" / "flux2feature.py"
    path.write_text(FLUX2FEATURE_SOURCE)
    return path


def tool_model(gem) -> "flux2feature_tool.Model":
    """The notebook's model, as the standalone tool expects it."""
    return flux2feature_tool.Model(
        gem.S, list(gem.rxns), list(gem.mets), gem.lb, gem.ub, list(gem.subsystems))


#: The state the three implementations are checked on: the first retained
#: state of the first healthy chain, which is column 1 of modelOr_samples1.mat.
REFERENCE_STEM = "original_healthy_chain01"


def run_30_flux2feature_check() -> None:
    """The standalone tool against the notebook's engine, on one real state."""
    use_variant("original")
    gem = load_gem(config.MODEL_FILE, config.MODEL_NAME)
    model = tool_model(gem)
    declared = flux2feature_tool.audit(model)
    print(f"{len(declared)} descriptors declared by the tool, "
          f"{sum(r['computable'] for r in declared)} computable on {config.MODEL_NAME}")

    store = SampleStore(SAMPLE_DIR, "original")
    state = store.load_chain(REFERENCE_STEM).thin[0]
    mine = flux2feature_tool.flux2feature(model, state)

    metadata = pd.read_csv(config.PROCESSED_DIR / "sample_metadata.csv.gz")
    row = int(np.flatnonzero((metadata["stem"] == REFERENCE_STEM).to_numpy()
                             & (metadata["sample_index"] == 0).to_numpy())[0])
    theirs = pd.read_csv(config.PROCESSED_DIR / "phenotypes.csv.gz",
                         skiprows=range(1, row + 1), nrows=1)
    shared = sorted(k for k in mine if k in theirs.columns)
    disagree = [(k, mine[k], float(theirs[k].iloc[0])) for k in shared
                if not ((np.isnan(mine[k]) and np.isnan(float(theirs[k].iloc[0])))
                        or abs(mine[k] - float(theirs[k].iloc[0]))
                        <= 1e-5 * max(1.0, abs(float(theirs[k].iloc[0]))))]
    print(f"{len(shared) - len(disagree)} of {len(shared)} shared descriptors agree "
          f"to 1e-5 relative ({REFERENCE_STEM}, state 0)")
    for k, a, b in disagree:
        print(f"   {k}: tool {a:.6g} vs engine {b:.6g}")

    reference = Path(ROOT) / "code" / "flux2feature_reference.csv"
    pd.DataFrame({"descriptor": list(mine), "value": [mine[k] for k in mine]}).to_csv(
        reference, index=False, float_format="%.10g")
    print(f"MATLAB reference values written to {reference.relative_to(ROOT)} "
          f"({len(mine)} descriptors)")
    if disagree:
        raise AssertionError(f"{len(disagree)} descriptors disagree")

## 58. The numbers the manuscript quotes (`run_31_manuscript_numbers`)

The manuscript contains no typed-in results: every value it prints is a token filled from `reports/paper/numbers.json`, which this stage writes from the tables. Re-running the notebook and rebuilding the manuscript updates the text.

In [ ]:
"""Stage 31 — every number the manuscript quotes, taken from the tables.

The manuscript does not contain typed-in results. Its source,
``reports/paper/manuscript_annotated.md``, writes ``{{n:key}}`` wherever a
result appears, and the build replaces each token with the value this stage
writes to ``reports/paper/numbers.json``. Re-running the notebook on new
samples and rebuilding the manuscript therefore updates every figure in the
text; a token with no value stops the build instead of printing a stale one.

Numbers are stored already formatted, the way the text prints them, so that
a change of rounding is made here once and not in fifty places.
"""
import json

import numpy as np
import pandas as pd


__all__ = ["manuscript_numbers", "run_31_manuscript_numbers"]


def _fmt(value, digits=3) -> str:
    if value is None or (isinstance(value, float) and not np.isfinite(value)):
        return "n/a"
    if isinstance(value, (bool, np.bool_)):
        return "yes" if value else "no"
    if isinstance(value, (int, np.integer)):
        return f"{int(value):,}"
    if float(value).is_integer() and abs(value) >= 1 and digits == 0:
        return f"{int(value):,}"
    return f"{float(value):.{digits}f}"


def _pct(value, digits=0) -> str:
    return f"{100 * float(value):.{digits}f}%"


def _t(variant, *parts):
    path = Path(TABLE_ROOT) / variant
    for part in parts:
        path = path / part
    return pd.read_csv(path) if path.exists() else None


def _gate(table: pd.DataFrame, fragment: str) -> tuple[str, bool]:
    row = table[table["criterion"].str.contains(fragment, regex=False)].iloc[0]
    return str(row["observed"]).split(" (")[0], bool(row["passed"])


def _bench(table, representation, model, selection="none", n=None):
    rows = table[(table["representation"] == representation) & (table["model"] == model)
                 & (table["selection"].eq("none") if selection == "none"
                    else table["selection"].ne("none"))]
    if n is not None:
        rows = rows[rows["n_features"] == n]
    return rows.iloc[0] if len(rows) else None


def manuscript_numbers() -> dict:
    N: dict[str, str] = {}

    # ------------------------------------------------------------ registry
    qc = _t("original", "features", "feature_qc_decision.csv")
    N["declared"] = _fmt(len(qc))
    N["families"] = _fmt(qc["family"].nunique())
    for block, key in (("internal", "internal"), ("external", "boundary"), ("combined", "coupling")):
        N[f"declared_{key}"] = _fmt(int((qc["block"] == block).sum()))
    registry = pd.read_csv(Path(METADATA_DIR) / "feature_registry.csv")
    if "scope" in registry.columns:
        scope = registry.drop_duplicates("feature")["scope"].value_counts()
        for name in ("common", "conditional", "model_specific"):
            N[f"scope_{name}"] = _fmt(int(scope.get(name, 0)))
    for variant in ("original", "thermo"):
        q = _t(variant, "features", "feature_qc_decision.csv")
        retained = q[q["retained"]]
        N[f"{variant}_retained"] = _fmt(len(retained))
        N[f"{variant}_excluded"] = _fmt(int((~q["retained"]).sum()))
        for block, key in (("internal", "internal"), ("external", "boundary"),
                           ("combined", "coupling")):
            N[f"{variant}_core_{key}"] = _fmt(int((retained["block"] == block).sum()))
        for criterion in ("missing", "constant", "outliers", "chain_dependence",
                          "no_healthy_unit", "redundant"):
            N[f"{variant}_fail_{criterion}"] = _fmt(int(q[f"fail_{criterion}"].sum()))
        fails = q[[c for c in q.columns if c.startswith("fail_")]].sum(axis=1)
        N[f"{variant}_multiple_fail"] = _fmt(int((fails > 1).sum()))
        N[f"{variant}_n_states"] = _fmt(len(pd.read_csv(Path(TABLE_ROOT) / variant
                                                        / "sample_metadata.csv.gz",
                                                        usecols=["stem"])))
    reasons = qc.loc[~qc["retained"], "exclusion_reason"].value_counts()
    for reason, n in reasons.items():
        N["original_reason_" + reason.replace(";", "_and_")] = _fmt(int(n))

    # -------------------------------------------------------------- models
    summary = pd.read_csv(Path(METADATA_DIR) / "model_comparison_summary.csv").set_index("quantity")
    for quantity, key in (("reactions", "reactions"), ("metabolites", "metabolites"),
                          ("exchange reactions", "exchanges"),
                          ("reversible reactions", "reversible"), ("genes", "genes")):
        if quantity in summary.index:
            o, t = float(summary.loc[quantity, "original"]), float(summary.loc[quantity, "thermo"])
            N[f"model_{key}_original"] = _fmt(int(o))
            N[f"model_{key}_thermo"] = _fmt(int(t))
            N[f"model_{key}_removed"] = _fmt(int(o - t))
            N[f"model_{key}_kept_pct"] = _pct(t / o)
            N[f"model_{key}_removed_pct"] = _pct(1 - t / o)
    diff = pd.read_csv(Path(METADATA_DIR) / "model_reaction_diff.csv")
    groups = diff.assign(removed=diff["status"] != "shared").groupby("pathway")["removed"]
    for group in ("pentose_phosphate", "glycolysis", "glutathione", "redox_detox", "lipid"):
        N[f"removed_{group}"] = f"{int(groups.sum().get(group, 0))} of {int(groups.size().get(group, 0))}"

    # ------------------------------------------------- sampling and integrity
    integrity = pd.concat([_t(v, "sampling", "constraint_integrity.csv") for v in ("original", "thermo")])
    g6 = integrity[integrity["reaction"] == G6PD_REACTION]
    for variant in ("original", "thermo"):
        sub = g6[g6["model_variant"] == variant]
        N[f"{variant}_chains_checked"] = _fmt(len(sub))
        N[f"{variant}_chains_pass"] = _fmt(int((sub["verdict"] == "pass").sum()))
        N[f"{variant}_chains_fail"] = _fmt(int((sub["verdict"] == "fail").sum()))
    failed = g6[g6["verdict"] == "fail"]
    N["thermo_fail_capped"] = _fmt(int((failed["chain_max"] < failed["imposed_ub"]).sum()))
    N["thermo_fail_exceed"] = _fmt(int((failed["chain_max"] > failed["imposed_ub"]).sum()))
    N["thermo_fail_capped_at"] = _fmt(float(failed.loc[failed["chain_max"] < failed["imposed_ub"],
                                                       "chain_max"].median()), 0)
    h2 = integrity[integrity["reaction"] == H2O2_EXCHANGE]
    N["h2o2_not_binding"] = _fmt(int((h2["verdict"] == "not_binding").sum()))
    N["h2o2_healthy_max"] = _fmt(float(h2["healthy_max"].max()), 0)
    N["h2o2_lowest_bound"] = _fmt(float(h2["imposed_ub"].min()), 0)
    N["h2o2_feasible_max"] = _fmt(float(h2["feasible_max"].max()), 0)
    N["thermo_failed_level"] = ", ".join(f"{x:g}" for x in sorted(failed["level"].unique()))
    if len(failed):
        reached = float(failed["chain_max"].median())
        N["thermo_fail_reached"] = _fmt(reached, 0)
        bounds = (g6[g6["model_variant"] == "thermo"].drop_duplicates("level")
                  .set_index("level")["imposed_ub"])
        N["thermo_fail_reached_level"] = f"{(bounds - reached).abs().idxmin():.2f}"
    chains = {v: _t(v, "sampling", "chain_inventory.csv") for v in ("original", "thermo")}
    for variant, inventory in chains.items():
        perturbed = inventory[inventory["state"] != "healthy"]
        healthy = inventory[inventory["state"] == "healthy"]
        N[f"{variant}_points_perturbed"] = _fmt(int(perturbed["n_raw_samples"].median()))
        N[f"{variant}_points_healthy"] = _fmt(int(healthy["n_raw_samples"].median()))
        N[f"{variant}_kept_per_chain"] = _fmt(int(inventory["n_retained_samples"].median()))
        N[f"{variant}_n_chains"] = _fmt(len(inventory))
        N[f"{variant}_n_conditions"] = _fmt(inventory["condition"].nunique())
        gates = _t(variant, "sampling", "qc_decision.csv")
        for fragment, key in (("median total ESS", "ess"), ("fraction of reactions with total ESS", "ess_frac"),
                              ("split R-hat across chains", "rhat"),
                              ("fraction of reactions with R-hat", "rhat_frac"),
                              ("running-mean drift", "drift")):
            value, passed = _gate(gates, fragment)
            N[f"{variant}_gate_{key}"] = value
            N[f"{variant}_gate_{key}_passed"] = "met" if passed else "not met"
        N[f"{variant}_gates_passed"] = _fmt(int(gates["passed"].sum()))
        cond = _t(variant, "sampling", "condition_diagnostics.csv")
        N[f"{variant}_tau_median"] = _fmt(float(cond["tau_median"].median()), 0)
        if variant == "thermo":
            h = cond[cond["condition"] == "healthy"].iloc[0]
            p = cond[cond["condition"] != "healthy"]
            N["thermo_healthy_ess"] = _fmt(float(h["ess_total_median"]), 0)
            N["thermo_healthy_rhat_frac"] = _fmt(float(h["frac_rhat_above_target"]), 3)
            N["thermo_perturbed_ess"] = _fmt(float(p["ess_total_median"].median()), 0)
            N["thermo_perturbed_rhat_frac"] = _fmt(float(p["frac_rhat_above_target"].median()), 2)

    # -------------------------------------------------- the factorial design
    for variant in ("original", "thermo"):
        effects = _t(variant, "design", "factorial_effects.csv")
        strong = effects[(effects["gradient_p_adjusted"] < 0.05)
                         & (effects["gradient_rho"].abs() >= 0.7)]
        N[f"{variant}_design_core"] = _fmt(len(effects))
        N[f"{variant}_track"] = _fmt(len(strong))
        N[f"{variant}_monotone"] = _fmt(int(strong["monotone"].sum()))
        N[f"{variant}_h2o2_responders"] = _fmt(int((effects["h2o2_p_adjusted"] < 0.05).sum()))
        N[f"{variant}_h2o2_eta_max"] = _fmt(float(effects["h2o2_eta_squared"].max()), 2)
        N[f"{variant}_h2o2_eta_median"] = _fmt(float(effects["h2o2_eta_squared"].median()), 3)
        N[f"{variant}_interaction_max"] = _fmt(float(effects["interaction_eta_squared"].max()), 2)
        N[f"{variant}_interaction_median"] = _fmt(float(effects["interaction_eta_squared"].median()), 3)
        N[f"{variant}_rho_max"] = _fmt(float(effects["gradient_rho"].abs().max()), 3)
        top = effects.sort_values("effect_to_noise", ascending=False)
        for k in range(3):
            N[f"{variant}_etn_{k + 1}_feature"] = f"`{top.iloc[k]['feature']}`"
            N[f"{variant}_etn_{k + 1}"] = _fmt(float(top.iloc[k]["effect_to_noise"]), 0)
        calibration = _t(variant, "features", "null_axis_calibration.csv").set_index("axis")
        for axis, key in (("g6pd_including_healthy", "g6pd"), ("h2o2_null_axis", "h2o2")):
            row = calibration.loc[axis]
            N[f"{variant}_state_{key}_tested"] = _fmt(int(row["n_features_tested"]))
            N[f"{variant}_state_{key}_sig"] = _fmt(int(row["n_fdr_significant"]))
            N[f"{variant}_state_{key}_material"] = _fmt(int(row["n_significant_and_material"]))
            N[f"{variant}_state_{key}_eps_max"] = _fmt(float(row["max_epsilon_squared"]), 4)

    # ------------------------------------------- representation and parsimony
    for variant in ("original", "thermo"):
        table = _t(variant, "benchmark", "reduced_panel.csv")
        levels = int(pd.read_csv(Path(TABLE_ROOT) / variant / "sample_metadata.csv.gz",
                                 usecols=["g6pd_level"])["g6pd_level"].nunique())
        N[f"{variant}_levels"] = _fmt(levels)
        N[f"{variant}_baseline"] = _fmt(1 / levels, 3)
        for representation, key in (("raw internal flux", "raw"),
                                    ("phenotype (internal core)", "int"),
                                    ("phenotype (whole core)", "all")):
            for model in ("forest", "logistic"):
                row = _bench(table, representation, model)
                if row is not None:
                    N[f"{variant}_{key}_{model}"] = _fmt(float(row["balanced_accuracy_mean"]))
                    N[f"{variant}_{key}_{model}_sd"] = _fmt(float(row["balanced_accuracy_sd"]))
                    N[f"{variant}_{key}_n"] = _fmt(int(row["n_features"]))
                    N[f"{variant}_{key}_{model}_rho"] = _fmt(float(row["rank_correlation"]))
        reduced = table[(table["selection"] != "none")
                        & (table["representation"] == "phenotype (internal core)")]
        for _, row in reduced.iterrows():
            n = int(row["n_features"])
            N[f"{variant}_panel{n}"] = _fmt(float(row["balanced_accuracy_mean"]))
            N[f"{variant}_panel{n}_sd"] = _fmt(float(row["balanced_accuracy_sd"]))
            N[f"{variant}_panel{n}_overlap"] = _fmt(float(row["selection_jaccard"]), 2)
            N[f"{variant}_panel{n}_always"] = _fmt(int(row["always_selected"]))
        members = _t(variant, "benchmark", "reduced_panel_members.csv")
        three = members[(members["representation"] == "phenotype (internal core)")
                        & (members["n_features"] == 3)]
        consensus = list(three.loc[three["in_every_fold"], "feature"].drop_duplicates())
        N[f"{variant}_panel3_members"] = ", ".join(f"`{f}`" for f in consensus) or "none"
        N[f"{variant}_panel3_n_consensus"] = _fmt(len(consensus))
        ten = members[(members["representation"] == "phenotype (internal core)")
                      & (members["n_features"] == 10) & members["in_every_fold"]]
        N[f"{variant}_panel10_members"] = ", ".join(f"`{f}`" for f in ten["feature"].drop_duplicates())

    supervised = _t("original", "benchmark", "supervised_condition.csv")
    for representation, key in (("raw flux", "raw"), ("phenotype (internal)", "int"),
                                ("phenotype (all)", "all"), ("learned embedding (32d)", "vae"),
                                ("phenotype (boundary)", "bnd")):
        sub = supervised[supervised["representation"] == representation]
        best = sub[sub["model"] != "majority"]["balanced_accuracy_mean"].max()
        N[f"ten_class_{key}"] = _fmt(float(best))

    # ----------------------------------------------- what distinguishes states
    decomposition = _t("original", "features", "mahalanobis_decomposition.csv").set_index("feature")
    for feature, key in (("eff_ppp_flux_split", "split"),
                         ("path_pentose_phosphate_activity", "ppp"),
                         ("turnover_6pgc_c", "6pgc")):
        if feature in decomposition.index:
            row = decomposition.loc[feature]
            N[f"d2_{key}_disease"] = _pct(row["share_of_squared_distance_disease"], 1)
            N[f"d2_{key}_healthy"] = _pct(row["share_of_squared_distance_healthy"], 1)
    partial = _t("original", "features", "partial_correlations.csv").set_index("feature")
    roles = partial["role"].value_counts()
    for role in ("redundant", "complementary", "independent"):
        N[f"partial_{role}"] = _fmt(int(roles.get(role, 0)))
    if "rxn_hexokinase" in partial.index:
        N["hk_marginal"] = _fmt(float(partial.loc["rxn_hexokinase", "marginal_correlation_with_distance"]), 2)
        N["hk_partial"] = _fmt(float(partial.loc["rxn_hexokinase", "partial_correlation_with_distance"]), 2)
    power = _t("original", "ml", "negative_result_power.csv")
    N["boundary_best_r2"] = _fmt(float(power["best_cv_r2"].max()), 3)
    N["boundary_detectable_rho"] = _fmt(float(power["detectable_spearman_at_80pct_power"].iloc[0]), 3)
    N["boundary_detectable_r2"] = _fmt(float(power["implied_detectable_r2"].iloc[0]), 3)

    # ------------------------------------------------------------ transfer
    transfer = Path(TABLE_ROOT) / "transfer"
    summary = json.loads((transfer / "transfer_summary.json").read_text())
    structural = pd.read_csv(transfer / "A_structural.csv")
    response = pd.read_csv(transfer / "B_response.csv")
    panel = pd.read_csv(transfer / "C_panel.csv")
    predictive = pd.read_csv(transfer / "D_predictive.csv")

    space = summary["flux_space"]
    N["fs_removed"] = _fmt(space["reactions_removed"])
    N["fs_removed_active"] = _fmt(space["removed_ever_active_in_reference"])
    N["fs_shared"] = _fmt(space["shared_reactions"])
    N["fs_changed_bounds"] = _fmt(space["shared_with_changed_bounds"])
    N["fs_active"] = _fmt(space["active_reactions_reference"])
    N["fs_compared"] = _fmt(space["active_reactions_compared"])
    N["fs_ks_models"] = _fmt(space["ks_between_models_median"], 3)
    N["fs_ks_models_p95"] = _fmt(space["ks_between_models_p95"], 3)
    N["fs_ks_chains"] = _fmt(space["ks_between_chains_median"], 3)
    N["fs_ks_chains_p95"] = _fmt(space["ks_between_chains_p95"], 3)
    computable = structural[structural["computable_reduced"]]
    N["A_active_full"] = _fmt(int((computable["active_support_retained"].fillna(1) >= 1 - 1e-12).sum()))
    N["shared_conditions"] = _fmt(len(summary["shared_conditions"]))
    N["shared_levels"] = ", ".join(f"{x:g}" for x in summary["shared_levels"])
    for frame, prefix in ((structural, "A"), (structural[structural["core_reference"]], "A_core")):
        counts = frame["support_category"].value_counts()
        for category in ("identical support", "reduced support", "largely lost", "not computable"):
            N[f"{prefix}_{category.replace(' ', '_')}"] = _fmt(int(counts.get(category, 0)))
        N[f"{prefix}_computable_both"] = _fmt(int(frame["computable_reduced"].sum()))
    internal = structural[structural["block"] == "internal"]
    N["A_internal_identical"] = _fmt(int((internal["support_category"] == "identical support").sum()))
    external = structural[structural["block"] == "external"]
    N["A_boundary_identical"] = _fmt(int((external["support_category"] == "identical support").sum()))
    not_computable = structural.loc[structural["support_category"] == "not computable", "feature"]
    N["A_not_computable_list"] = ", ".join(f"`{f}`" for f in not_computable)
    for feature in ("turnover_nadph_c", "turnover_nadp_c", "int_L1"):
        row = structural[structural["feature"] == feature]
        if len(row):
            N[f"support_{feature}"] = (f"{int(row['support_size_reduced'].iloc[0])} of "
                                       f"{int(row['support_size_reference'].iloc[0])}")

    classes = response["response_class"].value_counts()
    responsive = response[response["responsive_reference"]]
    N["B_comparable"] = _fmt(len(response))
    N["B_responsive"] = _fmt(len(responsive))
    for cls in ("conserved", "attenuated", "amplified", "reversed", "gained in reduced model",
                "unresponsive in both"):
        N[f"B_{cls.replace(' ', '_')}"] = _fmt(int(classes.get(cls, 0)))
    N["B_same_direction"] = _pct(summary["same_direction_responsive"])
    N["B_conserved_pct"] = _pct(classes.get("conserved", 0) / len(responsive))
    N["B_effect_rho_all"] = _fmt(summary["effect_rank_correlation_all"], 2)
    N["B_effect_rho_all_n"] = _fmt(int(summary["effect_rank_correlation_all_n"]))
    N["B_effect_rho_responsive"] = _fmt(summary["effect_rank_correlation_responsive"], 2)
    N["B_significant_both"] = _fmt(int(summary["n_significant_in_both"]))
    shift = response["healthy_shift"].value_counts()
    for band in ("negligible", "small", "medium", "large"):
        N[f"B_healthy_{band}"] = _fmt(int(shift.get(band, 0)))
    large = response[response["healthy_shift"] == "large"]
    N["B_large_identical"] = _fmt(int((large["support_category"] == "identical support").sum()))
    N["B_large_internal"] = _fmt(int((large["block"] == "internal").sum()))
    N["B_large_boundary"] = _fmt(int((large["block"] == "external").sum()))
    families = large["family"].value_counts()
    N["B_large_families"] = ", ".join(f"{name} ({n})" for name, n in families.head(4).items())
    N["B_healthy_negligible_pct"] = _pct(int(shift.get("negligible", 0)) / len(response))
    identical = response[response["support_category"] == "identical support"]
    N["B_identical_negligible"] = _fmt(int((identical["healthy_shift"] == "negligible").sum()))
    N["B_identical_total"] = _fmt(len(identical))
    for category, key in (("identical support", "identical"), ("reduced support", "reduced"),
                          ("largely lost", "lost")):
        sub = responsive[responsive["support_category"] == category]
        N[f"B_{key}_n"] = _fmt(len(sub))
        if len(sub):
            same = (np.sign(sub["effect_reference"]) == np.sign(sub["effect_reduced"])).mean()
            N[f"B_{key}_same"] = _pct(same)
            N[f"B_{key}_conserved"] = _pct((sub["response_class"] == "conserved").mean())
            N[f"B_{key}_reversed"] = _fmt(int((sub["response_class"] == "reversed").sum()))
    reversed_ = responsive[responsive["response_class"] == "reversed"]
    N["B_reversed_examples"] = ", ".join(f"`{f}`" for f in reversed_["feature"].head(4))
    for feature in ("eff_ppp_flux_split", "turnover_6pgc_c", "path_pentose_phosphate_activity"):
        row = response[response["feature"] == feature]
        if len(row):
            N[f"B_{feature}_ref"] = _fmt(float(row["effect_reference"].iloc[0]), 1)
            N[f"B_{feature}_red"] = _fmt(float(row["effect_reduced"].iloc[0]), 1)

    def acc(representation, evaluated):
        row = panel[(panel["representation"] == representation) & (panel["evaluated_in"] == evaluated)]
        return float(row["balanced_accuracy_mean"].iloc[0]), float(row["balanced_accuracy_sd"].iloc[0])

    for (representation, evaluated), key in (
            (("panel selected in original", "original"), "C_orig_in_orig"),
            (("panel selected in original", "thermo"), "C_orig_in_thermo"),
            (("panel selected in thermo", "thermo"), "C_thermo_in_thermo"),
            (("full internal core (original)", "original"), "C_core_orig"),
            (("full internal core (thermo)", "thermo"), "C_core_thermo")):
        a, s = acc(representation, evaluated)
        N[key], N[key + "_sd"] = _fmt(a), _fmt(s)
    N["C_baseline"] = _fmt(float(panel["baseline"].iloc[0]), 3)
    N["C_panel_thermo"] = ", ".join(f"`{f}`" for f in summary["panel_thermo"])
    N["C_jaccard"] = _fmt(summary["panel_overlap_jaccard"], 2)
    N["C_core_thermo_n"] = _fmt(int(panel.loc[panel["representation"] == "full internal core (thermo)",
                                              "n_features"].iloc[0]))

    short = {"internal descriptors with identical support": "ident",
             "all comparable internal core descriptors": "comp",
             "panel selected in original": "panel",
             "raw internal fluxes (shared reactions)": "raw"}
    for _, row in predictive.iterrows():
        key = (f"D_{'strict' if row['preprocessing'] == 'strict' else 'ref'}_"
               f"{short[row['representation']]}_{row['trained_on'][0]}{row['tested_on'][0]}")
        N[key] = _fmt(float(row["balanced_accuracy"]))
        N[f"D_{short[row['representation']]}_n"] = _fmt(int(row["n_features"]))
    N["D_baseline"] = _fmt(float(predictive["baseline"].iloc[0]), 3)

    # ------------------------------------- raw against phi, on equal terms (29b)
    short_rep = {"raw internal flux": "raw", "internal core": "core",
                 "biochemical families": "bio", "generic families": "gen"}

    class _Short(dict):
        def __missing__(self, name):                 # "raw flux, 103 principal components"
            return "pca" if str(name).startswith("raw flux, ") else str(name)

    short_rep = _Short(short_rep)
    for variant in ("original", "thermo"):
        ev = _t(variant, "evidence", "distance_same_terms.csv")
        for _, row in ev.iterrows():
            key = f"{variant}_ev_{short_rep[row['representation']]}_{row['metric'][:3]}"
            N[f"{key}_rho"] = _fmt(float(row["chain_rho"]), 2)
            N[f"{key}_etn"] = _fmt(float(row["effect_to_noise"]), 0)
            N[f"{key}_null"] = _pct(float(row["null_share"]))
            N[f"{key}_auc"] = _fmt(float(row["state_auc_healthy_vs_restricted"]), 2)
            N[f"{variant}_ev_{short_rep[row['representation']]}_n"] = _fmt(int(row["n_dimensions"]))
        energy = _t(variant, "evidence", "energy_over_noise.csv")
        for _, row in energy.iterrows():
            level = f"{row['g6pd_level']:.2f}".replace("0.", "").replace(".", "")
            N[f"{variant}_en_{short_rep[row['representation']]}_{level}"] = \
                _fmt(float(row["energy_over_noise"]), 1)
        groups = _t(variant, "evidence", "family_groups_classification.csv").set_index("representation")
        for name, key in (("biochemical families", "bio"), ("generic families", "gen")):
            for model in ("forest", "logistic"):
                N[f"{variant}_cls_{key}_{model}"] = _fmt(float(groups.loc[name, f"{model}_balanced_accuracy"]))
        for name in groups.index:
            if str(name).startswith("raw flux, "):
                for model in ("forest", "logistic"):
                    N[f"{variant}_cls_pca_{model}"] = _fmt(float(groups.loc[name, f"{model}_balanced_accuracy"]))
        imp = _t(variant, "evidence", "permutation_importance.csv")
        meta_imp = json.loads((Path(TABLE_ROOT) / variant / "evidence"
                               / "permutation_importance_summary.json").read_text())
        N[f"{variant}_imp_top1"] = f"`{imp.iloc[0]['feature']}`"
        N[f"{variant}_imp_top1_value"] = _fmt(float(imp.iloc[0]["importance"]), 2)
        N[f"{variant}_imp_second_value"] = _fmt(float(imp.iloc[1]["importance"]), 3)
        N[f"{variant}_imp_overlap"] = _fmt(float(meta_imp["top_overlap"]), 2)
        N[f"{variant}_imp_every"] = ", ".join(f"`{f}`" for f in meta_imp["top_in_every_fold"])
        N[f"{variant}_imp_every_n"] = _fmt(len(meta_imp["top_in_every_fold"]))
        N[f"{variant}_imp_top_bio"] = _fmt(int(meta_imp["top_biochemical"]))
    cross_ev = json.loads((Path(TABLE_ROOT) / "transfer" / "E_cross_model_evidence.json").read_text())
    N["E_top_jaccard"] = _fmt(float(cross_ev["top_jaccard"]), 2)
    N["E_top_common_n"] = _fmt(len(cross_ev["top_common"]))
    N["E_top_common"] = ", ".join(f"`{f}`" for f in cross_ev["top_common"])
    N["E_order_rho_phi"] = _fmt(float(cross_ev["condition_order_rho_phi"]), 2)
    N["E_order_rho_raw"] = _fmt(float(cross_ev["condition_order_rho_raw"]), 2)

    # ------------------------------------------------ secondary validations
    for variant in ("original", "thermo"):
        clusters = _t(variant, "landscape", "representation_clusters.csv")
        for name, key in (("raw flux", "raw"), ("phenotype", "phi")):
            sub = clusters[clusters["representation"] == name]
            best = sub.loc[sub["ari_with_severity"].idxmax()]
            N[f"{variant}_clu_{key}_ari"] = _fmt(float(best["ari_with_severity"]), 2)
            N[f"{variant}_clu_{key}_k"] = _fmt(int(best["k"]))
            N[f"{variant}_clu_{key}_null"] = _fmt(float(max(sub["ari_with_h2o2_null"].max(),
                                                            sub["ari_with_chain_null"].max())), 3)
        coupling = _t(variant, "robustness", "internal_external_coupling.csv")
        N[f"{variant}_cca_min"] = _fmt(float(coupling["first_canonical_correlation"].min()), 2)
        N[f"{variant}_cca_max"] = _fmt(float(coupling["first_canonical_correlation"].max()), 2)
        heldout = _t(variant, "robustness", "cca_validation.csv")
        N[f"{variant}_cca_heldout_min"] = _fmt(float(heldout["held_out_mean"].min()), 2)
        N[f"{variant}_cca_heldout_max"] = _fmt(float(heldout["held_out_mean"].max()), 2)
        N[f"{variant}_cca_null"] = _fmt(float(heldout["permutation_null_mean"].mean()), 2)
        pred = _t(variant, "robustness", "internal_feature_predictability.csv").set_index("feature")
        N[f"{variant}_ext_r2_median"] = _fmt(float(pred["predictability_r2_from_external"].median()), 3)
        for feature, key in (("turnover_6pgc_c", "6pgc"), ("path_pentose_phosphate_activity", "ppp"),
                             ("path_pentose_phosphate_directionality", "pppdir")):
            if feature in pred.index:
                N[f"{variant}_ext_r2_{key}"] = _fmt(float(pred.loc[feature, "predictability_r2_from_external"]), 3)
                N[f"{variant}_eps_{key}"] = _fmt(float(pred.loc[feature, "severity_epsilon_squared"]), 2)
        screening = _t(variant, "interventions", "screening.csv")
        ranking = _t(variant, "interventions", "ranking.csv")
        estimable = screening[screening["estimable"]]
        N[f"{variant}_iv_candidates"] = _fmt(len(screening))
        N[f"{variant}_iv_estimable"] = _fmt(len(estimable))
        N[f"{variant}_iv_max_recovery"] = _pct(float(estimable["recovery"].max()), 1)
        N[f"{variant}_iv_min_recovery"] = _pct(float(estimable["recovery"].min()), 1)
        N[f"{variant}_iv_over5"] = _fmt(int((estimable["recovery"] > 0.05).sum()))
        N[f"{variant}_iv_zero"] = _fmt(int((screening["n_retained"] == 0).sum()))
        N[f"{variant}_iv_few"] = _fmt(int(((screening["n_retained"] > 0) & ~screening["estimable"]).sum()))
        admissible = ranking[ranking["clinically_modifiable"] & (ranking["max_toxicity"] <= 0.5)]
        top = admissible.iloc[0]
        kind, met, quantile = str(top["intervention_id"]).split("::")
        meta_ = _t(variant, "reaction_metadata.csv").set_index("rxn_id")
        name = str(meta_.loc[f"EX_{met}", "boundary_metabolite_name"]) if f"EX_{met}" in meta_.index else met
        action = {"restrict": "capping the uptake of", "supplement": "forcing the uptake of",
                  "blockexport": "capping the secretion of"}.get(kind, kind)
        N[f"{variant}_iv_best"] = (f"{action} {name.lower()} at its healthy "
                                   f"{quantile.lstrip('q')}th percentile")
        N[f"{variant}_iv_best_recovery"] = _pct(float(top["recovery"]), 1)
        control = _t(variant, "interventions", "sampler_control.csv")
        N[f"{variant}_ctrl_ks_min"] = _fmt(float(control["ks_statistic"].min()), 2)
        N[f"{variant}_ctrl_ks_max"] = _fmt(float(control["ks_statistic"].max()), 2)
        N[f"{variant}_ctrl_pass"] = _fmt(int(control["reproduces_reference"].sum()))
        N[f"{variant}_ctrl_budgets"] = _fmt(len(control))

    # ------------------------------------------- the biology of the response
    for feature, key in (("path_pentose_phosphate_activity", "ppp"), ("turnover_6pgc_c", "6pgc"),
                         ("turnover_nadph_c", "nadph"), ("eff_nadph_per_h2o2", "nadph_h2o2"),
                         ("turnover_g6p_c", "g6p"), ("path_glycolysis_activity", "glyc"),
                         ("path_rapoport_luebering_activity", "rl"), ("turnover_r5p_c", "r5p"),
                         ("turnover_gthrd_c", "gsh"), ("path_glutathione_activity", "gshpath"),
                         ("turnover_atp_c", "atp"), ("turnover_h2o2_c", "h2o2"),
                         ("eff_oxidative_ppp_per_glycolysis", "oxppp")):
        row = response[response["feature"] == feature]
        if len(row):
            row = row.iloc[0]
            N[f"bio_{key}_eff"] = _fmt(float(row["effect_reference"]), 1)
            N[f"bio_{key}_rho"] = _fmt(abs(float(row["rho_reference"])), 2)

    # ------------------------------------------------------------ the tool
    tool_model_ = tool_model(load_gem(MODEL_VARIANTS["original"]["model_file"],
                                      MODEL_VARIANTS["original"]["model_name"]))
    N["tool_declared"] = _fmt(len(flux2feature_tool.audit(tool_model_)))
    gem_o = load_gem(MODEL_VARIANTS["original"]["model_file"], MODEL_VARIANTS["original"]["model_name"])
    exchange = annotate_reactions(gem_o)["reaction_class"].eq("exchange").to_numpy()
    lb, ub = gem_o.lb[exchange], gem_o.ub[exchange]
    N["exchanges_open"] = _fmt(int((np.isclose(lb, -1000) & np.isclose(ub, 1000)).sum()))
    import hashlib
    N["model_md5"] = f"`{hashlib.md5(Path(MODEL_VARIANTS['original']['model_file']).read_bytes()).hexdigest()}`"

    # ------------------------------------------- checks of the premises (29c)
    review = Path(TABLE_ROOT) / "review"
    if (review / "blocked_by_fva.csv").exists():
        fva = pd.read_csv(review / "blocked_by_fva.csv")
        largest = float(fva["max_abs"].max())
        N["rv_fva_max"] = "0" if largest == 0 else f"{largest:.1e}"
        N["rv_fva_above"] = _fmt(int((fva["max_abs"] > 1e-6).sum()))
    if (review / "peroxide_exchange_by_condition.csv").exists():
        h2o2 = pd.read_csv(review / "peroxide_exchange_by_condition.csv")
        N["rv_h2o2_max"] = _fmt(float(h2o2["flux_max"].max()), 0)
        N["rv_h2o2_uptake_min"] = _pct(float(h2o2["fraction_uptake"].min()), 1)
    if (review / "chain_prefix_stability.csv").exists():
        prefix = pd.read_csv(review / "chain_prefix_stability.csv")
        half = prefix[prefix["prefix"] == 0.5]
        N["rv_prefix_half"] = _fmt(float(half["max_shift_over_healthy_spread"].max()), 2)
        N["rv_prefix_half_chains"] = _fmt(float(half["max_shift_over_between_chain_sd"].max()), 0)
    if (review / "panel_second_selector.csv").exists():
        second = pd.read_csv(review / "panel_second_selector.csv")
        agree = 0
        for _, row in second.iterrows():
            members = _t(row["model"], "benchmark", "reduced_panel_members.csv")
            members = members[(members["representation"] == "phenotype (internal core)")
                              & (members["n_features"] == 3) & (members["fold"] == row["fold"])]
            agree += set(row["panel"].split(", ")) == set(members["feature"])
        N["rv_l1_agree"] = f"{agree} of {len(second)}"
        modal = second.groupby("model")["panel"].agg(lambda x: x.mode().iloc[0])
        same = int(sum((second["panel"] == second["model"].map(modal)).tolist()))
        N["rv_l1_consistent"] = f"{same} of {len(second)} folds"
        N["rv_l1_panel"] = ", ".join(f"`{f}`" for f in modal["original"].split(", "))
        scores = pd.read_csv(review / "panel_second_selector_scores.csv")
        pick = scores[(scores["model"] == "original") & (scores["selector"] == "L1 logistic regression")]
        N["rv_l1_acc_original"] = _fmt(float(pick["balanced_accuracy"].iloc[0]), 3)
    if (review / "recentring_reference_chain.csv").exists():
        recentre = pd.read_csv(review / "recentring_reference_chain.csv")
        N["rv_recentre_min"] = _fmt(float(recentre["balanced_accuracy"].min()), 3)
        N["rv_recentre_max"] = _fmt(float(recentre["balanced_accuracy"].max()), 3)
    if (review / "coupling_scale_control.csv").exists():
        cca = pd.read_csv(review / "coupling_scale_control.csv")
        cca = cca[cca["model"] == "original"]
        N["rv_cca_before"] = _fmt(float(cca["canonical_correlation"].median()), 2)
        N["rv_cca_after"] = _fmt(float(cca["canonical_correlation_scale_removed"].median()), 2)
        N["rv_cca_scale"] = _fmt(float(cca["corr_internal_variate_internal_scale"].median()), 2)
    if (review / "intervention_recovery_by_chain.csv").exists():
        chains = pd.read_csv(review / "intervention_recovery_by_chain.csv")
        N["rv_iv_chain_max"] = _pct(float(chains.loc[chains["model"] == "original",
                                                     "recovery_chain_max"].max()), 1)
        N["rv_iv_chain_min"] = _pct(float(chains.loc[chains["model"] == "original",
                                                     "recovery_chain_min"].min()), 1)
    return N


def run_31_manuscript_numbers() -> None:
    """Write reports/paper/numbers.json for the manuscript build."""
    numbers = manuscript_numbers()
    path = Path(PAPER_DIR) / "numbers.json"
    path.write_text(json.dumps(numbers, indent=1, ensure_ascii=False, sort_keys=True))
    print(f"{len(numbers)} numbers written to {path.relative_to(ROOT)}")

## 59. Figure 1 , the method (`run_31b_methods_figure`)

The method in one figure, placed at the head of the Methods: the two reconstructions, the perturbation design with its null axis, sampling and the per-chain check against the imposed bound, what happens to one sampled state on its way to Φ(v), and what is evaluated. It describes the method and carries no result; the counts it prints are read from `numbers.json`, so it is drawn after stage 31.

In [ ]:
"""Stage 31b — Figure 7, the method in one figure.

(a) the two reconstructions, (b) the perturbation design with its null axis,
(c) sampling and the per-chain check against the imposed bound, (d) what
happens to one sampled state: the split into internal and exchange reactions,
the declared descriptor families, the phenotype Φ(v) and its dimension against
that of the flux vector in both models, and (e) what is evaluated. Panels a–e
are a description of the method, so the figure carries no result; the counts
it prints (reactions, descriptors, chains, states) are read from
``reports/paper/numbers.json``, which is why it is drawn after stage 31. It is
written as ``Figure7_methodology`` to ``results/figures/paper`` and copied to
``reports/paper/figures``.
"""
import json
import shutil
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.patches import Circle, FancyBboxPatch, Polygon, Rectangle

__all__ = ["run_31b_methods_figure"]

MF_WIDTH, MF_HEIGHT = 7.2, 4.95                      # inches; one canvas unit = 1 inch
CHAIN_TONES = ("#1B2A38", "#2E7D8F", "#8FB3C4")
CPL = "#0F3346"                                      # coupling block
STATE = ACCENT["signal"]                             # the one state followed through panel d


# -- small primitives ---------------------------------------------------------
def _mf_label(ax, x, y, letter, title):
    ax.text(x, y, letter, fontsize=9.0, fontweight="bold", color=INK, va="baseline")
    ax.text(x + 0.17, y, title, fontsize=7.0, color=INK, va="baseline")


def _mf_arrow(ax, x0, y0, x1, y1, colour=MUTED, lw=0.9, style="-|>", ms=7, rad=0.0, z=4):
    ax.annotate("", xy=(x1, y1), xytext=(x0, y0),
                arrowprops=dict(arrowstyle=style, lw=lw, color=colour, mutation_scale=ms,
                                shrinkA=0, shrinkB=0, connectionstyle=f"arc3,rad={rad}"),
                zorder=z)


def _mf_chip(ax, x, y, w, h, text, face, edge, colour, size=5.3, bold=False):
    ax.add_patch(FancyBboxPatch((x, y), w, h, boxstyle=f"round,pad=0,rounding_size={h / 2}",
                                facecolor=face, edgecolor=edge, linewidth=0.7, zorder=4))
    ax.text(x + w / 2, y + h / 2 - 0.004, text, fontsize=size, color=colour, ha="center",
            va="center", zorder=5, fontweight="bold" if bold else "normal")


# -- a: the nested reconstructions ------------------------------------------
def _mf_network(ax, cx, cy, scale, colour, removed, *, full):
    rng = np.random.default_rng(7)
    angles = np.linspace(0, 2 * np.pi, 15, endpoint=False) + rng.normal(0, 0.18, 15)
    radii = rng.uniform(0.45, 1.0, 15)
    pts = np.c_[np.cos(angles) * radii, np.sin(angles) * radii * 0.92]
    pts = np.vstack([pts, [[0.02, 0.0], [-0.25, 0.22], [0.28, -0.18]]])
    d = np.linalg.norm(pts[:, None] - pts[None], axis=-1)
    edges = [(i, j) for i in range(len(pts)) for j in range(i + 1, len(pts)) if d[i, j] < 0.62]
    xy = np.c_[cx + pts[:, 0] * scale, cy + pts[:, 1] * scale]
    for k, (i, j) in enumerate(edges):
        if k in removed and not full:
            ax.plot(*xy[[i, j]].T, color="#C9D1DA", lw=0.6, ls=(0, (1.2, 1.4)), zorder=3)
        else:
            ax.plot(*xy[[i, j]].T, color=colour, lw=0.9, alpha=0.85, zorder=3,
                    solid_capstyle="round")
    live = {i for k, e in enumerate(edges) if k not in removed for i in e}
    for i, (px, py) in enumerate(xy):
        on = full or i in live
        ax.add_patch(Circle((px, py), 0.026 if on else 0.02, facecolor=colour if on else "white",
                            edgecolor=colour if on else "#C9D1DA", linewidth=0.7, zorder=4))


# -- c: hit-and-run inside a polytope cut by the imposed bound ----------------
_POLY = np.array([[0.05, 0.30], [0.30, 0.04], [0.80, 0.08], [0.98, 0.50],
                  [0.70, 0.96], [0.20, 0.86]])
_CUT = 0.64


def _inside(p):
    if p[0] > _CUT:
        return False
    for a, b in zip(_POLY, np.roll(_POLY, -1, axis=0)):
        if (b[0] - a[0]) * (p[1] - a[1]) - (b[1] - a[1]) * (p[0] - a[0]) < 0:
            return False
    return True


def _chrr_path(start, steps, seed):
    rng = np.random.default_rng(seed)
    grid = np.linspace(0, 1, 401)
    p = np.array(start, float)
    path = [p.copy()]
    for step in range(steps):
        axis = step % 2
        ok = np.array([_inside(np.where(np.arange(2) == axis, g, p)) for g in grid])
        lo, hi = grid[ok].min(), grid[ok].max()
        p = p.copy()
        p[axis] = rng.uniform(lo, hi)
        path.append(p.copy())
    return np.array(path)


def _mf_polytope(ax, x0, y0, w, h):
    P = np.c_[x0 + _POLY[:, 0] * w, y0 + _POLY[:, 1] * h]
    ax.add_patch(Polygon(P, closed=True, facecolor="white", edgecolor="#AEB9C5", lw=0.8,
                         hatch="//////", zorder=3))
    feasible = Polygon(P, closed=True, facecolor="#E9F0F5", edgecolor="#AEB9C5", lw=0.8, zorder=3.2)
    ax.add_patch(feasible)
    feasible.set_clip_path(Rectangle((x0 - 0.05, y0 - 0.05), _CUT * w + 0.05, h + 0.1,
                                     transform=ax.transData))
    for k, (start, seed) in enumerate((((0.20, 0.45), 1), ((0.42, 0.25), 2), ((0.40, 0.70), 5))):
        path = _chrr_path(start, 26, seed)
        X, Y = x0 + path[:, 0] * w, y0 + path[:, 1] * h
        ax.plot(X, Y, color=CHAIN_TONES[k], lw=0.5, alpha=0.75, zorder=5)
        ax.scatter(X, Y, s=2.6, color=CHAIN_TONES[k], zorder=6, linewidths=0)
    xc = x0 + _CUT * w
    ax.plot([xc, xc], [y0 + 0.03 * h, y0 + 0.97 * h], color=STATE, lw=1.0, ls=(0, (3, 1.6)),
            zorder=6)
    ax.text(xc + 0.03, y0 + 0.97 * h, "$u$", fontsize=6.4, color=STATE, va="top", zorder=7)
    state = (x0 + 0.46 * w, y0 + 0.55 * h)       # the state followed through panel d
    ax.add_patch(Circle(state, 0.033, facecolor=STATE, edgecolor="white", lw=0.8, zorder=8))
    return state


def _mf_check(ax, x0, y0, w, h):
    """Each chain's distribution of the bounded flux, reaching the bound, never above it."""
    ax.plot([x0, x0], [y0, y0 + h], color="#B9C3CE", lw=0.6, zorder=3)
    ub = y0 + 0.80 * h
    ax.plot([x0 - 0.03, x0 + w], [ub, ub], color=STATE, lw=1.0, ls=(0, (3, 1.6)), zorder=4)
    ax.text(x0 + w + 0.03, ub, "$u$", fontsize=6.4, color=STATE, va="center")
    grid = np.linspace(0, 1, 60)
    for k in range(3):
        dens = np.clip(grid ** (1.6 + 0.4 * k) * (1.05 - grid) * 6, 0, None)
        dens = dens / dens.max() * (w / 3.4)
        base = x0 + 0.05 + k * (w / 3.1)
        yy = y0 + 0.05 * h + grid * (ub - y0 - 0.05 * h)
        ax.fill_betweenx(yy, base, base + dens, color=CHAIN_TONES[k], alpha=0.55, lw=0, zorder=5)
    ax.add_patch(Circle((x0 + w * 0.82, y0 + h * 0.98), 0.075, facecolor=ACCENT["ok"],
                        edgecolor="white", lw=0.8, zorder=6))
    ax.text(x0 + w * 0.82, y0 + h * 0.98 - 0.005, "✓", fontsize=6.6, color="white",
            ha="center", va="center", fontweight="bold", zorder=7)


# -- d: one state, split, described, compressed -------------------------------
def _mf_column(ax, x, y, w, h, n, colours, *, gap=0.14):
    ch = h / n
    for k, colour in enumerate(colours):
        ax.add_patch(Rectangle((x, y + h - (k + 1) * ch), w, ch * (1 - gap), facecolor=colour,
                               edgecolor="none", zorder=4))


def _shade(base, rng, n, lo=0.0, hi=0.55):
    from matplotlib.colors import to_rgb
    b = np.array(to_rgb(base))
    return [tuple(b + (1 - b) * t) for t in rng.uniform(lo, hi, n)]


# -- e: what is evaluated -----------------------------------------------------
def _mf_response(ax, x0, y0, w, h):
    xs = x0 + np.linspace(0.08, 0.92, 4) * w
    ax.plot([x0, x0 + w], [y0, y0], color="#B9C3CE", lw=0.6)
    ax.plot([x0, x0], [y0, y0 + h], color="#B9C3CE", lw=0.6)
    steep = y0 + np.array([0.86, 0.62, 0.38, 0.14]) * h
    flat = y0 + np.array([0.50, 0.52, 0.49, 0.51]) * h
    ax.plot(xs, steep, color=SEVERITY["severe"], lw=1.3, marker="o", ms=2.6, zorder=5)
    ax.plot(xs, flat, color=BLOCK["neutral"], lw=1.3, marker="o", ms=2.6, ls=(0, (3, 1.5)),
            zorder=5)
    ax.text(x0 + w + 0.04, steep[-1], "G6PDH2", fontsize=4.9, color=SEVERITY["severe"],
            va="center")
    ax.text(x0 + w + 0.04, flat[-1], "H$_2$O$_2$", fontsize=4.9, color=BLOCK["neutral"],
            va="center")


def _mf_heldout(ax, x0, y0, w, h):
    cw = w / 9.6
    for k in range(9):
        held = k % 3 == 0
        ax.add_patch(FancyBboxPatch((x0 + k * cw * 1.07, y0 + h * 0.62), cw * 0.9, h * 0.30,
                                    boxstyle="round,pad=0,rounding_size=0.012",
                                    facecolor="white" if held else "#C9D6E0",
                                    edgecolor=STATE if held else "none", lw=0.9, zorder=4))
    for k, (label, colour, frac) in enumerate((("$v$", BLOCK["neutral"], 0.62),
                                              ("$\\Phi$", BLOCK["internal"], 0.86))):
        bx = x0 + 0.18 * w + k * 0.42 * w
        ax.add_patch(Rectangle((bx, y0), 0.17 * w, frac * h * 0.40, facecolor=colour,
                               edgecolor="none", zorder=4))
        ax.text(bx + 0.17 * w + 0.04, y0 + frac * h * 0.20, label, fontsize=6.0, ha="left",
                va="center", color=colour, fontweight="bold")


def _mf_panel(ax, x0, y0, w, h):
    rng = np.random.default_rng(4)
    chosen = {3, 9, 14}
    for f in range(3):
        fx = x0 + f * (w / 3)
        ax.add_patch(FancyBboxPatch((fx + 0.02, y0), w / 3 - 0.06, h,
                                    boxstyle="round,pad=0,rounding_size=0.03",
                                    facecolor="#F6F8FA", edgecolor="#E3E8ED", lw=0.6, zorder=3))
        for k in range(20):
            px = fx + 0.07 + (k % 4) * (w / 3 - 0.14) / 3
            py = y0 + 0.08 + (k // 4) * (h - 0.16) / 4
            on = k in chosen
            ax.add_patch(Circle((px, py), 0.028 if on else 0.018,
                                facecolor=BLOCK["internal"] if on else "#C9D1DA",
                                edgecolor="none", zorder=4))
    rng.random()


def _mf_transfer(ax, x0, y0, w, h):
    cy = y0 + h * 0.70
    for k, (letter, key) in enumerate((("O", "original"), ("T", "thermo"))):
        cx = x0 + 0.30 * w + k * 0.40 * w
        ax.add_patch(Circle((cx, cy), 0.13, facecolor=VARIANT_COLOUR[key], edgecolor="white",
                            lw=1.0, zorder=5))
        ax.text(cx, cy - 0.005, letter, fontsize=6.6, color="white", fontweight="bold",
                ha="center", va="center", zorder=6)
    _mf_arrow(ax, x0 + 0.30 * w + 0.15, cy, x0 + 0.70 * w - 0.15, cy, colour=INK, lw=1.0)
    for k, word in enumerate(("support", "response", "panel", "prediction")):
        col, row = k % 2, k // 2
        _mf_chip(ax, x0 + 0.02 * w + col * 0.50 * w, y0 + 0.20 * h - row * 0.21 * h,
                 0.46 * w, 0.17 * h, word, "white", "#C9D1DA", INK, size=4.9)


# -- the figure ---------------------------------------------------------------
def _mf_numbers() -> dict:
    path = Path(PAPER_DIR) / "numbers.json"
    return json.loads(path.read_text()) if path.exists() else {}


def paper_methods_figure(out: Path) -> None:
    N = _mf_numbers()

    def n(key, default="–"):
        return str(N.get(key, default)).replace("`", "")

    def num(key, default=0.0):
        try:
            return float(str(N.get(key, default)).replace(",", ""))
        except ValueError:
            return float(default)

    W, H = MF_WIDTH, MF_HEIGHT
    fig = plt.figure(figsize=(W, H))
    ax = fig.add_axes([0, 0, 1, 1])
    ax.set_axis_off()
    ax.set_xlim(0, W)
    ax.set_ylim(0, H)
    ax.set_aspect("equal")

    # ======================= row 1: a, b, c ====================================
    r1 = H - 0.20                                     # baseline of the row's labels
    _mf_label(ax, 0.10, r1, "a", "Two reconstructions")
    _mf_label(ax, 2.50, r1, "b", "Perturbation design")
    _mf_label(ax, 4.80, r1, "c", "Sampling and integrity check")

    # a
    removed = {1, 4, 9, 13, 17, 22, 26, 31}
    ny = H - 0.80
    _mf_network(ax, 0.62, ny, 0.34, VARIANT_COLOUR["original"], removed, full=True)
    _mf_network(ax, 1.78, ny, 0.34, VARIANT_COLOUR["thermo"], removed, full=False)
    ax.text(1.20, ny + 0.02, "$\\supset$", fontsize=12, ha="center", va="center", color=MUTED)
    ax.text(1.20, ny - 0.20, f"−{n('model_reactions_removed')}\nblocked", fontsize=4.9,
            ha="center", va="top", color=MUTED, linespacing=1.1)
    for cx, label, key, count in ((0.62, "Original", "original", "model_reactions_original"),
                                  (1.78, "Thermo", "thermo", "model_reactions_thermo")):
        ax.text(cx, ny - 0.47, label, fontsize=6.4, ha="center", va="center", fontweight="bold",
                color=VARIANT_COLOUR[key])
        ax.text(cx, ny - 0.62, f"{n(count)} reactions", fontsize=5.0, ha="center",
                va="center", color=MUTED)

    # b
    cell, gap = 0.17, 0.035
    gx, gy = 3.42, H - 0.52                           # top-left of the grid
    rows = (("1", SEVERITY["healthy"], (0,)), (".50", SEVERITY["mild"], (0, 1, 2)),
            (".28", SEVERITY["moderate"], (0, 1, 2)), (".10", SEVERITY["severe"], (0, 1, 2)))
    for r, (tick, colour, cols) in enumerate(rows):
        y = gy - (r + 1) * cell - r * gap
        for c in cols:
            ax.add_patch(FancyBboxPatch((gx + c * (cell + gap), y), cell, cell,
                                        boxstyle="round,pad=0,rounding_size=0.018",
                                        facecolor=colour, edgecolor="white", lw=0.5, zorder=4))
            for k in range(3):                     # three chains per condition
                ax.add_patch(Circle((gx + c * (cell + gap) + cell * (0.27 + 0.23 * k),
                                     y + cell * 0.5), 0.012, facecolor="white", alpha=0.85,
                                    edgecolor="none", zorder=5))
        ax.text(gx - 0.05, y + cell / 2, tick, fontsize=5.1, color=MUTED, ha="right", va="center")
    for c, tick in enumerate(("1", ".75", ".50")):
        ax.text(gx + c * (cell + gap) + cell / 2, gy + 0.05, tick, fontsize=5.1, color=MUTED,
                ha="center", va="bottom")
    grid_bottom = gy - 4 * cell - 3 * gap
    ax.text(gx - 0.30, (gy + grid_bottom) / 2, "G6PDH2 capacity", fontsize=5.6,
            color=SEVERITY["severe"], rotation=90, ha="center", va="center", fontweight="bold")
    ax.text(gx + 1.5 * cell + gap, gy + 0.20, "H$_2$O$_2$ bound (null axis)", fontsize=5.6,
            color=BLOCK["neutral"], ha="center", va="bottom", fontweight="bold")
    ax.text(gx + 3 * (cell + gap) + 0.06, gy - 0.5 * cell, "reference", fontsize=5.0, color=MUTED,
            va="center")

    # c
    state = _mf_polytope(ax, 4.92, H - 1.52, 1.25, 1.15)
    _mf_check(ax, 6.42, H - 1.42, 0.55, 0.95)
    ax.text(5.55, H - 1.62, f"3 CHRR chains · {n('original_kept_per_chain')} states each",
            fontsize=5.0, color=MUTED, ha="center", va="top")
    ax.text(6.70, H - 1.62, "checked per chain", fontsize=5.0, color=MUTED, ha="center", va="top")

    # row-1 flow
    for x in (2.36, 4.68):
        _mf_arrow(ax, x - 0.10, H - 0.80, x + 0.04, H - 0.80, colour="#B9C3CE", lw=1.4, ms=8)

    # ======================= row 2: d ==========================================
    top_d = H - 1.92
    ax.add_patch(FancyBboxPatch((0.06, top_d - 1.70), W - 0.12, 1.86,
                                boxstyle="round,pad=0,rounding_size=0.06",
                                facecolor="#F6F8FA", edgecolor="none", zorder=0))
    _mf_label(ax, 0.16, top_d - 0.02, "d", "One sampled state, from flux to phenotype")
    y_lo, y_hi = top_d - 1.52, top_d - 0.22          # vertical extent of the columns
    rng = np.random.default_rng(12)

    # the state v, with exchange reactions interleaved
    n_cells = 64
    is_ex = rng.random(n_cells) < 0.15
    greys = _shade("#7D8794", rng, n_cells, 0.15, 0.85)
    colours = [(_shade(BLOCK["external"], rng, 1, 0.25, 0.6)[0] if e else g)
               for e, g in zip(is_ex, greys)]
    vx = 0.40
    _mf_column(ax, vx, y_lo, 0.11, y_hi - y_lo, n_cells, colours, gap=0.18)
    ax.text(vx + 0.055, y_lo - 0.07, "$v$", fontsize=8.0, ha="center", va="top", color=INK)
    ax.add_patch(Circle((vx - 0.10, y_hi - 0.02), 0.033, facecolor=STATE, edgecolor="white",
                        lw=0.8, zorder=6))
    ax.text(vx + 0.055, y_hi + 0.06, f"n = {n('model_reactions_original')}", fontsize=4.8,
            ha="center", va="bottom", color=MUTED)

    # the split
    sx = 0.86
    int_lo, int_hi = y_lo + 0.50, y_hi
    bnd_lo, bnd_hi = y_lo, y_lo + 0.30
    _mf_column(ax, sx, int_lo, 0.11, int_hi - int_lo, 34,
               _shade(BLOCK["internal"], rng, 34, 0.25, 0.85), gap=0.18)
    _mf_column(ax, sx, bnd_lo, 0.11, bnd_hi - bnd_lo, 8,
               _shade(BLOCK["external"], rng, 8, 0.2, 0.7), gap=0.18)
    _mf_arrow(ax, vx + 0.14, (y_lo + y_hi) / 2 + 0.1, sx - 0.03, (int_lo + int_hi) / 2,
              colour=BLOCK["internal"], lw=0.9, rad=-0.2)
    _mf_arrow(ax, vx + 0.14, (y_lo + y_hi) / 2 - 0.1, sx - 0.03, (bnd_lo + bnd_hi) / 2,
              colour=BLOCK["external"], lw=0.9, rad=0.2)
    ax.text(sx + 0.055, int_hi + 0.06, "internal", fontsize=5.0, color=BLOCK["internal"],
            ha="center", va="bottom", fontweight="bold")
    ax.text(sx + 0.055, bnd_lo - 0.07, "exchange", fontsize=5.0, color=BLOCK["external"],
            ha="center", va="top", fontweight="bold")

    # the declared descriptor families
    fx0 = 1.38
    chip_w, chip_h, cgx, cgy = 0.66, 0.155, 0.07, 0.065
    internal = ("pathway", "turnover", "cofactor · redox", "efficiency", "network",
                "flux shape")
    for k, word in enumerate(internal):
        col, row = k % 3, k // 3
        _mf_chip(ax, fx0 + col * (chip_w + cgx), int_hi - chip_h - row * (chip_h + cgy) - 0.04,
                 chip_w, chip_h, word, BLOCK["internal_soft"], BLOCK["internal"],
                 BLOCK["internal"])
    cpl_y = int_hi - 2 * chip_h - cgy - 0.04 - 0.07 - chip_h - 0.02
    _mf_chip(ax, fx0 + 0.5 * (chip_w + cgx), cpl_y, 2 * chip_w + cgx, chip_h,
             "coupling", CPL, CPL, "white", bold=True)
    boundary = ("uptake/secretion", "exchange role", "burden")
    for k, word in enumerate(boundary):
        _mf_chip(ax, fx0 + k * (chip_w + cgx), bnd_lo + 0.05, chip_w, chip_h, word,
                 BLOCK["external_soft"], BLOCK["external"], BLOCK["external"])
    groups_r = fx0 + 3 * chip_w + 2 * cgx
    ax.text((fx0 + groups_r) / 2, y_hi + 0.06,
            f"{n('declared')} declared descriptors · {n('families')} families",
            fontsize=5.0, color=MUTED, ha="center", va="bottom")
    _mf_arrow(ax, sx + 0.14, (int_lo + int_hi) / 2 + 0.05, fx0 - 0.05,
              int_hi - chip_h - 0.04 - 0.02, colour=BLOCK["internal"], lw=0.8)
    _mf_arrow(ax, sx + 0.14, (bnd_lo + bnd_hi) / 2, fx0 - 0.05, bnd_lo + 0.05 + chip_h / 2,
              colour=BLOCK["external"], lw=0.8)

    # Φ(v)
    px = groups_r + 0.42
    blocks = [BLOCK["internal"]] * 8 + [CPL] + [BLOCK["external"]] * 5
    _mf_column(ax, px, y_lo, 0.15, y_hi - y_lo, len(blocks), blocks, gap=0.10)
    ch = (y_hi - y_lo) / len(blocks)
    nan_k = 11
    ax.add_patch(Rectangle((px, y_hi - (nan_k + 1) * ch), 0.15, ch * 0.9, facecolor="white",
                           edgecolor=BLOCK["external"], lw=0.6, hatch="//////", zorder=5))
    ax.text(px + 0.20, y_hi - (nan_k + 0.55) * ch, "NaN, not 0", fontsize=4.7,
            color=STATE, va="center")
    ax.text(px + 0.075, y_lo - 0.07, "$\\Phi(v)$", fontsize=8.0, ha="center", va="top",
            color=BLOCK["internal"])
    ax.text(px + 0.075, y_hi + 0.06, f"p = {n('declared')}", fontsize=4.8, ha="center",
            va="bottom", color=MUTED)
    int_chip_mid = int_hi - chip_h - 0.04 - (chip_h + cgy) / 2 + chip_h / 2
    cpl_right = fx0 + 0.5 * (chip_w + cgx) + 2 * chip_w + cgx
    for (x0_, y0_), y1_ in (((groups_r + 0.05, int_chip_mid), y_hi - 4 * ch),
                             ((cpl_right + 0.05, cpl_y + chip_h / 2), y_hi - 8.5 * ch),
                             ((groups_r + 0.05, bnd_lo + 0.05 + chip_h / 2), y_hi - 11 * ch)):
        _mf_arrow(ax, x0_, y0_, px - 0.05, y1_, colour="#9AA5B1", lw=0.8)
    # coupling reads both sides of the split
    _mf_arrow(ax, sx + 0.14, int_lo + 0.05, fx0 + 0.5 * (chip_w + cgx) - 0.05, cpl_y + chip_h * 0.7,
              colour=CPL, lw=0.7)
    _mf_arrow(ax, sx + 0.14, bnd_hi - 0.03, fx0 + 0.5 * (chip_w + cgx) - 0.05, cpl_y + chip_h * 0.3,
              colour=CPL, lw=0.7)
    ax.add_patch(Circle((px - 0.10, y_hi - 0.02), 0.033, facecolor=STATE, edgecolor="white",
                        lw=0.8, zorder=6))

    # dimension: the flux vector depends on the model, the phenotype does not
    dx0, dw = px + 0.88, W - 0.30 - (px + 0.88)
    raw_o, raw_t = num("model_reactions_original", 3275), num("model_reactions_thermo", 2580)
    p = num("declared", 282)
    scale = (dw - 0.42) / max(raw_o, 1)
    rows = (("$v$", "original", raw_o, False), ("", "thermo", raw_t, False),
            ("$\\Phi(v)$", "original", p, True), ("", "thermo", p, True))
    bar_h, row_gap = 0.13, 0.065
    ytop = y_hi - 0.12
    ax.text(dx0, y_hi + 0.06, "dimension", fontsize=5.0, color=MUTED, va="bottom")
    for k, (group, key, value, phi) in enumerate(rows):
        yy = ytop - k * (bar_h + row_gap) - (0.12 if k >= 2 else 0)
        ax.add_patch(Rectangle((dx0 + 0.38, yy - bar_h / 2), value * scale, bar_h,
                               facecolor=VARIANT_COLOUR[key] if phi else "#D5DBE2",
                               edgecolor=VARIANT_COLOUR[key], lw=0.8, zorder=4))
        ax.text(dx0 + 0.38 + value * scale + 0.04, yy, f"{value:,.0f}", fontsize=5.0,
                color=VARIANT_COLOUR[key], va="center", fontweight="bold")
        if not phi:
            ax.text(dx0 + 0.43, yy, VARIANT_LABEL[key], fontsize=5.0, color=VARIANT_COLOUR[key],
                    va="center", fontweight="bold", zorder=5)
        if group:
            ax.text(dx0 + 0.30, yy - (bar_h + row_gap) / 2, group, fontsize=6.6, ha="right",
                    va="center", color=BLOCK["internal"] if phi else INK)

    # ======================= row 3: e ==========================================
    top_e = top_d - 1.90
    _mf_label(ax, 0.10, top_e - 0.06, "e", "What is evaluated")
    slot = (W - 0.20) / 4
    ey, eh = 0.42, 0.62
    pictos = (
        (_mf_response, "response to the perturbation\nagainst the null axis"),
        (_mf_heldout, "$v$ against $\\Phi(v)$, same\nreference and metric"),
        (_mf_panel, "smallest panel, selected\ninside each fold"),
        (_mf_transfer, "Original → Thermo,\nat four levels"),
    )
    for k, (draw, caption) in enumerate(pictos):
        sx0 = 0.10 + k * slot
        draw(ax, sx0 + 0.28, ey, slot - 0.75 if k == 0 else slot - 0.56, eh)
        ax.text(sx0 + slot / 2, ey - 0.10, caption, fontsize=5.3, color=INK, ha="center",
                va="top", linespacing=1.2)
        if k:
            ax.plot([sx0, sx0], [ey - 0.30, ey + eh], color=RULE, lw=0.6)
    save(fig, out / "Figure7_methodology")
    return state


def run_31b_methods_figure() -> None:
    """Draw Figure 7, the method, from numbers.json."""
    apply_theme()
    out = Path(FIGURE_ROOT) / "paper"
    out.mkdir(parents=True, exist_ok=True)
    Path(PAPER_FIGURE_DIR).mkdir(parents=True, exist_ok=True)
    paper_methods_figure(out)
    for path in sorted(out.glob("Figure7_methodology*")):
        shutil.copy2(path, Path(PAPER_FIGURE_DIR) / path.name)
    print(f"Figure 7 (methodology) written to {out.relative_to(ROOT)}")

## 60. The manuscript (`run_32_build_manuscript`)

The text, the figures and the numbers are assembled into the Word document here. Every result in the text is a token filled from `numbers.json`; the build stops on a missing value and lists any sentence whose wording no longer matches the numbers.

In [ ]:
"""Stage 32 — build the manuscript from its source, its figures and the numbers.

The manuscript lives in ``reports/paper``: ``manuscript_annotated.md`` (the
text, with a ``{{n:key}}`` token wherever a result appears), ``references.json``,
``figure_legends.json``, the figures stage 22 copies there, and the
``numbers.json`` stage 31 writes. This stage fills every token, numbers the
citations, figures and equations in order of first mention, checks that every
qualitative statement in the prose still holds for the numbers, and writes the
Word document with pandoc. It needs ``pandoc`` on the path.

The build logic is kept as a module, so that its helpers (``fill``, ``expand``,
``main``) do not collide with the notebook's own names.
"""
import shutil
import sys
import types

MANUSCRIPT_BUILD_SOURCE = r'''"""Build the manuscript: numbered superscript citations, linked figures, linked equations.

The rules this enforces, because they are easy to break by hand:

* **Citations** are numbered in order of first mention, rendered as a
  superscript number, and hyperlinked to their entry in the reference list.
* **Figures** are numbered in order of first mention, and a figure must be
  mentioned in the text *before* it is placed. A forward reference — naming
  Figure 3 in the Introduction, before Figures 1 and 2 exist — is an error, not
  a warning, and the build stops.
* **Equations** are numbered in order of appearance. A reference to an equation
  is hyperlinked; an equation that is never mentioned is still numbered but
  nothing links to it.

Markup in the annotated source:

    [@key]              a citation
    {{FIGURE:n}}        where figure n is placed, with its legend
    [[F3|Figure 3]]     a mention of figure n
    @@EQ:n@@ latex      a numbered display equation
    [[EQ3|Equation 3]]  a mention of it
    $...$               inline mathematics
    {{n:key}}           a result, filled from numbers.json

**Results** are never typed into the source. Every number the text quotes is a
``{{n:key}}`` token, filled from ``numbers.json``, which the notebook's stage 31
writes from the result tables. A token with no value stops the build: a stale
or missing number is an error, not a blank.

Everything becomes pandoc markdown with bracketed spans as anchors, which
pandoc turns into Word bookmarks and internal hyperlinks.
"""
import json
import re
import subprocess
import sys
from pathlib import Path

HERE = Path(PAPER_DIRECTORY)
SOURCE = HERE / "manuscript_annotated.md"
TARGET = HERE / "Metabolic_flux_phenotyping_manuscript.docx"
REFERENCES = json.loads((HERE / "references.json").read_text())
LEGENDS = json.loads((HERE / "figure_legends.json").read_text()) if (HERE / "figure_legends.json").exists() else {}
FIGURE_DIR = HERE / "figures"
NUMBERS = json.loads((HERE / "numbers.json").read_text()) if (HERE / "numbers.json").exists() else {}
TOKEN = re.compile(r"\{\{n:([A-Za-z0-9_.]+)\}\}")


def fill(text: str, missing: set[str]) -> str:
    """Replace every {{n:key}} with its value; record the keys that have none."""
    def value(match: re.Match) -> str:
        key = match.group(1)
        if key not in NUMBERS:
            missing.add(key)
            return f"[{key}?]"
        return NUMBERS[key]
    return TOKEN.sub(value, text)


# --------------------------------------------------------------- mathematics
def expand(latex: str) -> str:
    """Expand the \\br{} \\sq{} \\cu{} shorthands into sized delimiters."""
    out, stack, index = [], [], 0
    openers = {"\\br{": (r"\left(", r"\right)"), "\\sq{": (r"\left[", r"\right]"),
               "\\cu{": (r"\left\{", r"\right\}")}
    while index < len(latex):
        for pattern, (opening, closing) in openers.items():
            if latex.startswith(pattern, index):
                out.append(opening)
                stack.append(closing)
                index += len(pattern)
                break
        else:
            character = latex[index]
            if character == "{":
                stack.append("}")
                out.append("{")
            elif character == "}":
                out.append(stack.pop() if stack else "}")
            else:
                out.append(character)
            index += 1
    return "".join(out)


#: Qualitative statements the prose makes about the numbers. The numbers are
#: filled automatically; the words around them are not, so after a re-run each
#: statement is checked and any that no longer holds is listed with the
#: sentence to rewrite. A failed claim does not stop the build.
def _f(key):
    value = NUMBERS.get(key, "nan").replace(",", "").replace("%", "")
    try:
        return float(value)
    except ValueError:
        return float("nan")


CLAIMS = [
    ("the internal core matches raw flux under a forest",
     lambda: _f("original_int_forest") >= _f("original_raw_forest") - 0.01,
     "Against the raw flux vector: 'matches the raw flux vector under a flexible model'"),
    ("the internal core beats raw flux under logistic regression",
     lambda: _f("original_int_logistic") > _f("original_raw_logistic"),
     "Abstract and Discussion: 'better under a linear model'"),
    ("the three-descriptor panel is chosen identically in every fold",
     lambda: _f("original_panel3_overlap") == 1.0,
     "Abstract and Main message 2: 'chosen identically in every fold'"),
    ("three descriptors reach the full internal core",
     lambda: _f("original_panel3") >= _f("original_int_logistic") - 0.01,
     "Main message 2: 'reach the accuracy of the full internal representation'"),
    ("selection overlap drops beyond ten descriptors",
     lambda: _f("original_panel10_overlap") < 0.6,
     "Main message 2 and Results: 'beyond ten the choice stops being reproducible'"),
    ("no descriptor responds to the control axis in Original (chain level)",
     lambda: _f("original_h2o2_responders") == 0,
     "Abstract: 'reports no effect on the control axis'"),
    ("no material state-level effect on the control axis in Original",
     lambda: _f("original_state_h2o2_material") == 0,
     "Results, control axis: 'none clears the material-effect floor'"),
    ("no removed reaction ever carries flux",
     lambda: _f("fs_removed_active") == 0,
     "Abstract and Results: 'removes only reactions that never carry flux'"),
    ("no shared bound changes",
     lambda: _f("fs_changed_bounds") == 0,
     "Results: 'none of the shared reactions has a different bound'"),
    ("flux distributions differ between models no more than between chains",
     lambda: _f("fs_ks_models") <= _f("fs_ks_chains") * 1.25,
     "Results: 'differ ... no more than two chains of the same model'"),
    ("every computable descriptor keeps its active support",
     lambda: _f("A_active_full") == _f("A_computable_both"),
     "Results, structure: 'every one ... keeps all of its active support'"),
    ("identical-support responses are all conserved",
     lambda: _f("B_identical_conserved") == 100,
     "Results: 'Every responsive descriptor with identical support keeps its direction and size'"),
    ("no identical-support descriptor shifts its healthy baseline",
     lambda: _f("B_large_identical") == 0 and _f("B_identical_negligible") == _f("B_identical_total"),
     "Results: 'including all ... with identical support ... none of them with identical support'"),
    ("strict transfer of all comparable descriptors collapses to the baseline",
     lambda: max(_f("D_strict_comp_ot"), _f("D_strict_comp_to")) <= _f("D_baseline") + 0.05,
     "Results and Discussion: 'makes transfer collapse to the baseline'"),
    ("re-centring restores transfer",
     lambda: min(_f("D_ref_comp_ot"), _f("D_ref_comp_to")) >= 0.8,
     "Results: 'Re-centring ... restores it'"),
    ("identical-support descriptors transfer with little loss",
     lambda: _f("D_strict_ident_ot") >= _f("D_strict_ident_oo") - 0.05,
     "Results: 'transfer with almost no loss'"),
    ("the Original panel carries over to Thermo",
     lambda: _f("C_orig_in_thermo") >= _f("C_thermo_in_thermo"),
     "Results, the panel: 'Selected where the chains are precise, the panel carries over'"),
    ("the largest effect-to-noise is 6-phosphogluconate turnover",
     lambda: NUMBERS.get("original_etn_1_feature") == "`turnover_6pgc_c`"
     and NUMBERS.get("original_etn_2_feature") == "`path_pentose_phosphate_activity`",
     "Results, gradient: the three named descriptors with the largest effect-to-noise"),
    ("every chain of both models has the same length",
     lambda: _f("thermo_points_perturbed") == _f("thermo_points_healthy")
     == _f("original_points_perturbed") and _f("thermo_kept_per_chain") == _f("original_kept_per_chain"),
     "Results, mixing; Methods: 'Every chain returned ... points'"),
    ("the same panel is selected in Thermo",
     lambda: NUMBERS.get("thermo_panel3_members") == NUMBERS.get("original_panel3_members")
     and _f("C_jaccard") == 1.0 and _f("thermo_panel3_overlap") == 1.0,
     "Abstract, Main message 2, Results (panel), Discussion H2/H3: 'chosen again, independently'"),
    ("no control-axis effect in Thermo at either level",
     lambda: _f("thermo_h2o2_responders") == 0 and _f("thermo_state_h2o2_material") == 0,
     "Results, control axis: 'Thermo reproduces both halves'"),
    ("Thermo's gradient is led by the same three descriptors",
     lambda: NUMBERS.get("thermo_etn_1_feature") == NUMBERS.get("original_etn_1_feature"),
     "Results, control axis: 'led by the same three'"),
    ("responses survive the reduction almost everywhere",
     lambda: _f("B_same_direction") >= 90 and _f("B_reduced_reversed") <= 5,
     "Results, response; Discussion H3: 'survived almost without exception'"),
    ("Thermo has four capacity levels",
     lambda: _f("thermo_levels") == 4,
     "Figure 4 legend: 'four capacity levels each'"),
    ("every chain of both models carries its bound, so the full design is shared",
     lambda: _f("thermo_chains_fail") == 0 and _f("original_chains_fail") == 0
     and _f("thermo_chains_checked") == 27 and _f("original_chains_checked") == 27
     and _f("shared_conditions") == 10,
     "Results, integrity: 'the comparison between models uses the full design'; S1 legend"),
    ("the boundary block does not predict the internal state",
     lambda: _f("boundary_best_r2") < 0.05,
     "Results, boundary: 'does not predict the internal state'"),
    ("on equal terms both representations order the conditions",
     lambda: max(_f("original_ev_raw_euc_rho"), _f("original_ev_core_euc_rho"),
                 _f("thermo_ev_raw_euc_rho"), _f("thermo_ev_core_euc_rho")) <= -0.85,
     "Results, equal terms; Abstract; Discussion H1: 'both order the conditions'"),
    ("phi separates the perturbation from chain noise more clearly than raw fluxes",
     lambda: _f("original_ev_core_euc_etn") > _f("original_ev_raw_euc_etn")
     and _f("thermo_ev_core_euc_etn") > _f("thermo_ev_raw_euc_etn")
     and _f("original_ev_core_euc_auc") > _f("original_ev_raw_euc_auc")
     and _f("thermo_ev_core_euc_auc") > _f("thermo_ev_raw_euc_auc"),
     "Abstract, Main message 1, Results: 'a wider margin over the sampler's own noise'"),
    ("phi is less sensitive to the inert axis than raw fluxes",
     lambda: _f("original_ev_core_euc_null") < _f("original_ev_raw_euc_null")
     and _f("thermo_ev_core_euc_null") < _f("thermo_ev_raw_euc_null"),
     "Results, equal terms: null-axis share"),
    ("the mildest restriction is at chain noise in raw fluxes and well above it in phi",
     lambda: _f("original_en_raw_50") <= 1.5 and _f("original_en_core_50") >= 3
     and _f("thermo_en_raw_50") <= 1.5 and _f("thermo_en_core_50") >= 3,
     "Results, equal terms: 'no further from health than one healthy chain from another ... four times further'"),
    ("biochemical families match the core, generic families stay near chance",
     lambda: _f("original_cls_bio_forest") >= _f("original_int_forest") - 0.01
     and _f("thermo_cls_bio_forest") >= _f("thermo_int_forest") - 0.01
     and max(_f("original_cls_gen_forest"), _f("original_cls_gen_logistic"),
             _f("thermo_cls_gen_forest"), _f("thermo_cls_gen_logistic")) < 0.40,
     "Main message 1, Abstract, Results: 'biochemistry carries the signal; generic statistics do not'"),
    ("the forest leans on 6-phosphogluconate turnover in both models, all top ten biochemical in Original",
     lambda: NUMBERS.get("original_imp_top1") == "`turnover_6pgc_c`"
     and NUMBERS.get("thermo_imp_top1") == "`turnover_6pgc_c`"
     and _f("original_imp_top_bio") == 10,
     "Results, what a classifier uses"),
    ("unsupervised structure follows capacity better in phi, nulls near zero",
     lambda: _f("original_clu_phi_ari") > _f("original_clu_raw_ari")
     and _f("thermo_clu_phi_ari") > _f("thermo_clu_raw_ari")
     and max(_f("original_clu_phi_null"), _f("thermo_clu_phi_null")) < 0.05,
     "Results, secondary validations: clustering"),
    ("the boundary is coupled overall but does not predict the lesion",
     lambda: _f("original_cca_min") > 0.9
     and max(_f("original_ext_r2_6pgc"), _f("original_ext_r2_ppp")) < _f("original_ext_r2_median"),
     "Results, secondary: 'coupled to the interior, but not where it matters'"),
    ("no boundary intervention recovers a tenth of the distance",
     lambda: max(_f("original_iv_max_recovery"), _f("thermo_iv_max_recovery")) < 10,
     "Results, intervention screening; Discussion"),
    ("the confirmatory sampler fails its control at every budget",
     lambda: _f("original_ctrl_pass") == 0 and _f("thermo_ctrl_pass") == 0,
     "Results, intervention screening; Discussion, audit"),
    ("every exchange is open",
     lambda: _f("exchanges_open") == _f("model_exchanges_original"),
     "Methods, Reconstructions; Discussion: 'all exchanges open'"),
    ("no removed reaction can carry flux, by optimisation",
     lambda: _f("rv_fva_above") == 0,
     "Results, two reconstructions: 'none of them can carry more than'"),
    ("the coupling is not overall scale",
     lambda: _f("rv_cca_after") >= _f("rv_cca_before") - 0.05 and _f("rv_cca_scale") < 0.3,
     "Results, boundary: 'The coupling is not the overall scale of metabolism'"),
    ("compression explains part, not all, of the margin",
     lambda: _f("original_ev_pca_euc_etn") < _f("original_ev_core_euc_etn")
     and _f("original_cls_pca_forest") < _f("original_int_forest")
     and _f("original_en_pca_50") > _f("original_en_raw_50"),
     "Results, equal terms: 'Compression accounts for part of the difference, not all of it'"),
    ("chain medians stable against the effect",
     lambda: _f("rv_prefix_half") < 1.0,
     "Results, mixing: 'move by at most ... of a healthy robust spread'"),
    ("re-centring does not depend on the reference chain",
     lambda: _f("rv_recentre_min") > 0.8,
     "Results, prediction across models: 'the choice of that chain does not matter'"),
]


def check_claims() -> list[str]:
    failed = []
    for name, test, where in CLAIMS:
        try:
            ok = bool(test())
        except Exception:                                   # noqa: BLE001
            ok = False
        if not ok:
            failed.append(f"{name}  ->  {where}")
    return failed


def place_figures(text: str) -> str:
    """Put every figure right after the paragraph that first mentions it.

    Placement markers in the source are ignored: a figure belongs where the
    reader first meets it, so the build decides, for every figure, the end of
    the paragraph holding its first mention.
    """
    text = re.sub(r"\n*^\{\{FIGURE:\d+\}\}[ \t]*$\n*", "\n\n", text, flags=re.M)
    first = {}
    for m in re.finditer(r"\[\[F(\d+)\|", text):
        first.setdefault(int(m.group(1)), m.start())
    for number, position in sorted(first.items(), key=lambda kv: -kv[1]):
        end = text.find("\n\n", position)
        end = len(text) if end < 0 else end
        text = text[:end] + "\n\n{{FIGURE:%d}}" % number + text[end:]
    return text


def main() -> None:
    missing: set[str] = set()
    text = fill(SOURCE.read_text(), missing)
    text = place_figures(text)
    for number, legend in LEGENDS.items():
        for field in ("title", "body"):
            if field in legend:
                legend[field] = fill(legend[field], missing)
    if missing:
        print("results quoted in the text with no value in numbers.json:")
        for key in sorted(missing):
            print(f"  - {key}")
        raise SystemExit(1)
    problems: list[str] = []

    # ------------------------------------------------- 1. order of first mention
    figure_mentions = [int(m.group(1)) for m in re.finditer(r"\[\[F(\d+)\|", text)]
    figure_placements = [int(m.group(1)) for m in re.finditer(r"\{\{FIGURE:(\d+)\}\}", text)]
    first_mention = {}
    for number in figure_mentions:
        first_mention.setdefault(number, len(first_mention) + 1)
    for number in sorted(set(figure_placements)):
        if number not in first_mention:
            problems.append(f"Figure {number} is placed but never mentioned in the text")
    for number, rank in first_mention.items():
        if rank != number:
            problems.append(
                f"Figure {number} is the figure mentioned in position {rank}: figures must be "
                f"numbered in order of first mention")
    for number in sorted(set(figure_placements)):
        mention = text.find(f"[[F{number}|")
        placement = text.find("{{FIGURE:%d}}" % number)
        if mention == -1 or mention > placement:
            problems.append(
                f"Figure {number} is placed before it is mentioned; a figure must be "
                f"introduced in the text first")

    equation_numbers = [int(m.group(1)) for m in re.finditer(r"^@@EQ:(\d+)@@", text, re.M)]
    if equation_numbers != sorted(equation_numbers):
        problems.append("equations are not in ascending order in the source")
    for number in {int(m.group(1)) for m in re.finditer(r"\[\[EQ(\d+)\|", text)}:
        if number not in equation_numbers:
            problems.append(f"Equation {number} is referenced but never written")

    if problems:
        print("the manuscript does not satisfy its own ordering rules:")
        for problem in problems:
            print(f"  - {problem}")
        raise SystemExit(1)

    # ------------------------------------------------------------ 2. citations
    order: list[str] = []

    def cite(match: re.Match) -> str:
        keys = [k.strip() for k in match.group(1).split(",")]
        numbers = []
        for key in keys:
            if key not in REFERENCES:
                raise SystemExit(f"unknown citation key: {key}")
            if key not in order:
                order.append(key)
            numbers.append(order.index(key) + 1)
        return "^" + ",".join(f"[{n}](#ref{n})" for n in sorted(numbers)) + "^"

    text = re.sub(r"\[@([^\]]+)\]", cite, text)

    # ------------------------------------------- 3. figures, equations, mentions
    text = re.sub(r"\[\[F(\d+)\|([^\]]+)\]\]", lambda m: f"[{m.group(2)}](#fig{m.group(1)})", text)
    text = re.sub(r"\[\[FS(\d+)\|([^\]]+)\]\]", lambda m: f"[{m.group(2)}](#figs{m.group(1)})", text)
    text = re.sub(r"\[\[EQ(\d+)\|([^\]]+)\]\]", lambda m: f"[{m.group(2)}](#eq{m.group(1)})", text)
    text = re.sub(r"\[\[T(\d+)\|([^\]]+)\]\]", lambda m: f"[{m.group(2)}](#tab{m.group(1)})", text)
    text = re.sub(r"\[\[TS(\d+)\|([^\]]+)\]\]", lambda m: f"[{m.group(2)}](#tabs{m.group(1)})", text)

    def equation(match: re.Match) -> str:
        number, latex = match.group(1), match.group(2).strip()
        return (f"[]{{#eq{number}}}\n\n"
                f"$${expand(latex)} \\qquad ({number})$$")

    text = re.sub(r"^@@EQ:(\d+)@@\s*(.+)$", equation, text, flags=re.M)
    text = re.sub(r"\$([^$\n]+)\$", lambda m: "$" + expand(m.group(1)) + "$", text)

    def figure(match: re.Match) -> str:
        number = match.group(1)
        legend = LEGENDS.get(number, {})
        stem = f"Figure{number}_{legend.get('slug', '')}"
        image = next((candidate for suffix in (".png", ".jpg", ".jpeg")
                      if (candidate := FIGURE_DIR / f"{stem}{suffix}").exists()), None)
        block = [f"[]{{#fig{number}}}"]
        if image is not None:
            block.append(f"![]({image}){{width=100%}}")
        else:
            block.append(f"*[Figure {number} is generated by run_22_figures() into "
                         f"results/python/figures/]*")
        title = legend.get("title", "")
        body = legend.get("body", "")
        block.append(f"**Figure {number}. {title}** {body}".strip())
        return "\n\n".join(block)

    text = re.sub(r"\{\{FIGURE:(\d+)\}\}", figure, text)

    text = re.sub(r"^\*\*Table (\d+)\.", lambda m: f"[]{{#tab{m.group(1)}}}\n\n**Table {m.group(1)}.",
                  text, flags=re.M)

    # ------------------------------------------------------- 4. reference list
    if order:
        lines = ["", "---", "", "## References", ""]
        for index, key in enumerate(order, start=1):
            entry = REFERENCES[key]
            doi = f" doi:{entry['doi']}" if entry.get("doi") else ""
            flag = "" if entry.get("verified") else "  [citation to verify]"
            lines.append(f"[]{{#ref{index}}}{index}. {entry['text']}{doi}{flag}")
            lines.append("")
        text = text.rstrip() + "\n" + "\n".join(lines) + "\n"

    import tempfile
    staged = Path(tempfile.mkdtemp()) / "manuscript.prepared.md"
    staged.write_text(text)

    subprocess.run(
        ["pandoc", str(staged), "-o", str(TARGET),
         "--from", "markdown+tex_math_dollars+bracketed_spans+superscript",
         "--to", "docx"],
        check=True)

    failed_claims = check_claims()
    unverified = [k for k in order if not REFERENCES[k].get("verified")]
    audit = HERE / "references_audit.csv"
    audit.write_text("number,key,verified,entry\n" + "\n".join(
        f"{i},{k},{REFERENCES[k].get('verified', False)},\"{REFERENCES[k]['text']}\""
        for i, k in enumerate(order, start=1)) + "\n")

    print(f"wrote {TARGET}")
    print(f"{len(order)} references, numbered in order of mention")
    print(f"{len(set(figure_placements))} figures, {len(equation_numbers)} equations, "
          f"{len(TOKEN.findall(SOURCE.read_text()))} results filled from numbers.json")
    if unverified:
        print(f"{len(unverified)} citations still to verify — see {audit.name}")
    if failed_claims:
        print(f"{len(failed_claims)} statements in the prose no longer match the numbers; rewrite:")
        for line in failed_claims:
            print(f"  - {line}")
    else:
        print(f"all {len(CLAIMS)} qualitative statements in the prose hold for these numbers")

'''


def run_32_build_manuscript() -> None:
    """Fill the numbers into the manuscript and write the .docx."""
    if shutil.which("pandoc") is None:
        print("pandoc is not installed; install it (https://pandoc.org) to build the .docx")
        return
    module = types.ModuleType("build_manuscript")
    module.__dict__["PAPER_DIRECTORY"] = str(PAPER_DIR)
    sys.modules["build_manuscript"] = module
    exec(compile(MANUSCRIPT_BUILD_SOURCE, "build_manuscript", "exec"), module.__dict__)
    module.main()

## 61. Checks

The assertions that guard the numerical core. They are cheap and they have caught
real defects: the steady-state check below is what exposed the sampler drift
described in section 19. Run them after any change.

In [ ]:
class _Approx:
    """pytest.approx, in the three forms the checks below use."""

    def __init__(self, value, abs=1e-6):
        self.value, self.tol = value, abs

    def __eq__(self, other):
        return abs(other - self.value) <= max(self.tol, 1e-6 * abs(self.value))


class expect:
    """Context manager asserting that a given exception is raised."""

    def __init__(self, exception):
        self.exception = exception

    def __enter__(self):
        return self

    def __exit__(self, kind, value, tb):
        assert kind is not None and issubclass(kind, self.exception), (
            f"expected {self.exception.__name__}")
        return True


class pytest:
    approx = staticmethod(_Approx)

from pathlib import Path

import numpy as np
import pandas as pd


# --------------------------------------------------------------------- helpers
def test_safe_divide_returns_fill_on_zero_denominator():
    out = safe_divide(np.array([1.0, 2.0]), np.array([2.0, 0.0]), fill=-1.0)
    assert out[0] == pytest.approx(0.5)
    assert out[1] == -1.0


def test_positive_and_negative_parts_reconstruct_the_vector():
    V = np.array([[-2.0, 0.0, 3.0]])
    assert np.allclose(positive_part(V) - negative_part(V), V)
    assert np.all(negative_part(V) >= 0)


# ------------------------------------------------------------------- sparsity
def test_gini_is_zero_for_a_uniform_vector_and_maximal_for_a_spike():
    uniform = np.ones((1, 100))
    spike = np.zeros((1, 100))
    spike[0, 0] = 1.0
    assert gini(uniform)[0] == pytest.approx(0.0, abs=1e-9)
    assert gini(spike)[0] == pytest.approx(0.99, abs=0.01)


def test_hoyer_sparsity_bounds():
    uniform = np.ones((1, 50))
    spike = np.zeros((1, 50))
    spike[0, 3] = 7.0
    assert hoyer_sparsity(uniform)[0] == pytest.approx(0.0, abs=1e-9)
    assert hoyer_sparsity(spike)[0] == pytest.approx(1.0, abs=1e-9)


# -------------------------------------------------------------------- entropy
def test_shannon_entropy_of_a_uniform_vector_is_log_n():
    V = np.ones((1, 16))
    assert shannon_entropy(V)[0] == pytest.approx(np.log(16))


def test_entropy_ignores_sign():
    V = np.array([[1.0, -1.0, 1.0, -1.0]])
    assert shannon_entropy(V)[0] == pytest.approx(np.log(4))


def test_renyi_entropy_order_two_is_negative_log_of_sum_of_squares():
    V = np.array([[3.0, 1.0]])
    p = np.array([0.75, 0.25])
    assert renyi_entropy(V, alpha=2.0)[0] == pytest.approx(-np.log((p ** 2).sum()))


def test_top_fraction_is_one_when_all_reactions_are_included():
    rng = np.random.default_rng(0)
    V = rng.normal(size=(5, 40))
    assert np.allclose(top_fraction(V, 1.0), 1.0)


# ----------------------------------------------------------------- structural
def test_structural_block_norms_are_correct():
    V = np.array([[3.0, -4.0, 0.0]])
    block = structural_block(V, "int")
    assert block["int_L0"].iloc[0] == 2
    assert block["int_L1"].iloc[0] == pytest.approx(7.0)
    assert block["int_L2"].iloc[0] == pytest.approx(5.0)
    assert block["int_Linf"].iloc[0] == pytest.approx(4.0)


def test_structural_block_directionality_is_signed():
    forward = structural_block(np.array([[1.0, 2.0, 3.0]]), "x")
    reverse = structural_block(np.array([[-1.0, -2.0, -3.0]]), "x")
    assert forward["x_directionality"].iloc[0] == pytest.approx(1.0)
    assert reverse["x_directionality"].iloc[0] == pytest.approx(-1.0)


def test_reduced_block_is_a_strict_subset():
    rng = np.random.default_rng(1)
    V = rng.normal(size=(10, 30))
    full = structural_block(V, "e", reduced=False)
    reduced = structural_block(V, "e", reduced=True)
    assert set(reduced.columns).issubset(full.columns)
    assert len(reduced.columns) < len(full.columns)


# ----------------------------------------------------------------- transforms
def test_transform_is_chosen_from_healthy_data_and_is_monotone():
    rng = np.random.default_rng(2)
    healthy = pd.DataFrame({
        "ratio": np.exp(rng.normal(0, 3, size=2000)),      # spans orders of magnitude
        "plain": rng.normal(0, 1, size=2000),
    })
    transform = PhenotypeTransform.fit(healthy)
    kinds = transform.summary().set_index("feature")["transform"].to_dict()
    assert kinds["ratio"] == "log"
    assert kinds["plain"] == "identity"

    probe = pd.DataFrame({"ratio": [1.0, 10.0, 100.0], "plain": [0.0, 1.0, 2.0]})
    out = transform.apply(probe)
    assert np.all(np.diff(out["ratio"]) > 0)
    assert np.allclose(out["plain"], probe["plain"])


def test_log_transform_survives_zeros_absent_from_the_healthy_set():
    healthy = pd.DataFrame({"ratio": np.exp(np.linspace(-8, 8, 500))})
    transform = PhenotypeTransform.fit(healthy)
    out = transform.apply(pd.DataFrame({"ratio": [0.0, 1.0]}))
    assert np.isfinite(out["ratio"]).all()


# ------------------------------------------------------------ MCMC diagnostics
def test_autocorrelation_of_white_noise_decays_immediately():
    rng = np.random.default_rng(3)
    x = rng.normal(size=20000)
    acf = autocorrelation(x, max_lag=10)
    assert acf[0] == pytest.approx(1.0)
    assert abs(acf[1]) < 0.05


def test_autocorrelation_time_detects_a_correlated_chain():
    rng = np.random.default_rng(4)
    n = 40000
    x = np.zeros(n)
    for i in range(1, n):
        x[i] = 0.8 * x[i - 1] + rng.normal()
    tau = integrated_autocorr_time(x)
    # for an AR(1) process the analytic value is (1 + phi) / (1 - phi) = 9
    assert 6.0 < tau < 12.0
    assert effective_sample_size(x) < n / 5


def test_integrated_autocorr_time_is_nan_for_a_constant_trace():
    assert np.isnan(integrated_autocorr_time(np.ones(1000)))


def test_split_rhat_is_near_one_for_agreeing_chains_and_large_otherwise():
    rng = np.random.default_rng(5)
    agreeing = rng.normal(size=(3, 4000))
    assert split_rhat(agreeing) == pytest.approx(1.0, abs=0.02)

    disagreeing = rng.normal(size=(3, 4000)) + np.array([[0.0], [5.0], [10.0]])
    assert split_rhat(disagreeing) > 1.5


# ------------------------------------------------------------------- multiple testing
def test_benjamini_hochberg_is_monotone_and_bounded():
    p = np.array([0.001, 0.01, 0.02, 0.5, 0.9])
    q = benjamini_hochberg(p)
    assert np.all(q >= p - 1e-12)
    assert np.all(np.diff(q) >= -1e-12)
    assert q.max() <= 1.0


# ------------------------------------------------------------------- sampling
def test_rejection_sample_selects_exactly_the_feasible_points():
    V = np.array([[0.0, 0.0], [1.0, 1.0], [2.0, 2.0]])
    lb = np.array([-1.0, -1.0])
    ub = np.array([1.5, 1.5])
    kept, report = rejection_sample(V, lb, ub, min_retained=1)
    assert report.n_retained == 2
    assert report.retention == pytest.approx(2 / 3)
    assert kept.shape == (2, 2)


def test_achr_preserves_the_null_space_and_the_bounds():
    rng = np.random.default_rng(6)
    # points on the plane x + y + z = 0, which stands in for Sv = 0
    basis = np.array([[1.0, -1.0, 0.0], [0.0, 1.0, -1.0]])
    warmup = rng.normal(size=(60, 2)) @ basis
    lb = np.full(3, -50.0)
    ub = np.full(3, 50.0)
    sampler = ACHRSampler(warmup, lb, ub, seed=0)
    out = sampler.sample(200, thinning=5, burn_in=50)
    # the constraint is preserved up to accumulated floating-point error, so
    # the residual is judged relative to the magnitude of the fluxes themselves
    assert np.abs(out.sum(axis=1)).max() / np.abs(out).max() < 1e-6
    assert np.all(out >= lb - 1e-9) and np.all(out <= ub + 1e-9)


def test_achr_requires_at_least_two_warmup_points():
    with expect(ValueError):
        ACHRSampler(np.zeros((1, 3)), np.zeros(3), np.ones(3))

In [ ]:
checks = [v for k, v in sorted(globals().items()) if k.startswith("test_") and callable(v)]
failed = 0
for check in checks:
    try:
        check()
        print(f"  PASS {check.__name__}")
    except AssertionError as error:
        failed += 1
        print(f"  FAIL {check.__name__}: {error}")
print(f"\n{len(checks) - failed}/{len(checks)} passed")

## Running everything

Running this notebook from top to bottom defines every stage above and then,
in the cell below, runs all of them: the export of the MATLAB archives, every
per-model stage for the original reconstruction and for the thermodynamically
consistent subset, and the stages that compare the two. Nothing runs outside
this notebook.

Any single stage can also be run on its own once the definitions are loaded:
`use_variant("thermo")` followed by, for example, `run_12_raw_vs_phenotype()`.
Set `RUN_EVERYTHING = False` to load the definitions without running anything.

In [ ]:
import time
import traceback

RUN_EVERYTHING = True

#: Every per-model stage, in the order it depends on the previous ones.
VARIANT_STAGES = [
    "run_01_prepare", "run_02_sampling_qc", "run_03_features", "run_04_feature_qc",
    "run_05_health_distance", "run_06_landscape", "run_07_external_inference",
    "run_08_interventions", "run_08b_sampler_control", "run_09_robustness",
    "run_09b_coupling_detail", "run_11_threshold_sensitivity",
    "run_12_raw_vs_phenotype", "run_13_feature_contribution",
    "run_14_minimal_phenotype", "run_15_recovery_profiles", "run_16_geometry",
    "run_17_sensitivity", "run_18_chain_level", "run_19_supervised_separability",
    "run_27_design_analysis", "run_28_reduced_panel",
]

#: The stages that run once, after both models: they read both models' tables.
CROSS_STAGES = [
    "run_00_model_comparison", "run_20_cross_model", "run_21_publish_registry",
    "run_29_transferability", "run_29b_representation_evidence", "run_29c_review_checks",
    "run_23_supplementary_tables", "run_25_importance_summary",
    "run_24_descriptor_transferability", "run_26_supplementary_workbook",
    "run_30_flux2feature_check", "run_22_figures", "run_31_manuscript_numbers", "run_31b_methods_figure", "run_32_build_manuscript",
]


def _run(names: list[str]) -> list[str]:
    failed = []
    for name in names:
        stage = globals().get(name)
        if stage is None:
            print(f"!! no stage named {name}")
            failed.append(name)
            continue
        print(f"\n{'=' * 72}\n=== {name}  [{ACTIVE_VARIANT}]\n{'=' * 72}", flush=True)
        started = time.time()
        try:
            stage()
        except Exception:                            # noqa: BLE001 - report and go on
            traceback.print_exc()
            failed.append(name)
            print(f"!! {name} failed after {time.time() - started:.0f}s", flush=True)
            continue
        print(f"--- {name} finished in {time.time() - started:.0f}s", flush=True)
    return failed


def run_variant(variant: str) -> list[str]:
    """Every per-model stage, in order, for one reconstruction."""
    use_variant(variant)
    return _run(VARIANT_STAGES)


def run_all() -> None:
    """Export, both models, then everything that compares them."""
    started = time.time()
    run_00_export_samples()
    run_00b_constraint_integrity()
    failed = []
    for variant in MODEL_VARIANTS:
        failed += [f"{variant}:{n}" for n in run_variant(variant)]
    use_variant("original")
    failed += _run(CROSS_STAGES)
    print(f"\nall stages finished in {(time.time() - started) / 60:.1f} min")
    print("every stage succeeded" if not failed else f"failed: {', '.join(failed)}")


if RUN_EVERYTHING:
    run_all()